# Greenhouse ERW — cumulative-alkalinity figures (companion to Hammes et al.)

Self-contained: the per-pot leachate data are embedded inline, so this notebook runs in Google Colab with **no external files**. It renders one figure per soil (mean ±1 SE cumulative alkalinity vs. days, with endpoint Hedges' g and Mann-Whitney significance). Each pot's line starts at 0 on its recorded creation day. See the methods paragraph for the processing steps.

In [ ]:
# Self-contained — runs in Google Colab, no external files.
import io, os, math
from itertools import cycle, islice
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu
from matplotlib import patheffects as pe
plt.rcParams['font.family']='DejaVu Sans'; plt.rcParams['font.sans-serif']=['DejaVu Sans']


In [ ]:
PERPOT_CSV = '''
Column/Pot,Treatment,soil,rock,days,creation_day,HCO3- mmol/m2
0.0.A,0.0,LUFA 2.2 B,Control,406,365,38.97937461941152
0.0.A,0.0,LUFA 2.2 B,Control,442,365,212.94287986040075
0.0.A,0.0,LUFA 2.2 B,Control,469,365,221.74933115109212
0.0.A,0.0,LUFA 2.2 B,Control,497,365,182.62558849509597
0.0.A,0.0,LUFA 2.2 B,Control,525,365,103.75250823755944
0.0.A,0.0,LUFA 2.2 B,Control,561,365,346.62769793609726
0.0.A,0.0,LUFA 2.2 B,Control,595,365,204.87270562609064
0.0.A,0.0,LUFA 2.2 B,Control,623,365,100.74484045971478
0.0.A,0.0,LUFA 2.2 B,Control,651,365,173.74695315000903
0.0.A,0.0,LUFA 2.2 B,Control,679,365,408.12848907876526
0.0.A,0.0,LUFA 2.2 B,Control,714,365,370.6794158493291
0.0.A,0.0,LUFA 2.2 B,Control,742,365,99.3733439430313
0.0.A,0.0,LUFA 2.2 B,Control,770,365,110.6581135086347
0.0.A,0.0,LUFA 2.2 B,Control,798,365,176.48032165593597
0.0.A,0.0,LUFA 2.2 B,Control,833,365,171.4081906733257
0.0.A,0.0,LUFA 2.2 B,Control,868,365,367.31564017088874
0.0.A,0.0,LUFA 2.2 B,Control,892,365,265.8393349780372
0.0.A,0.0,LUFA 2.2 B,Control,925,365,152.45266516637582
0.0.A,0.0,LUFA 2.2 B,Control,955,365,122.61660066189302
0.0.A,0.0,LUFA 2.2 B,Control,987,365,94.32046204946148
0.0.A,0.0,LUFA 2.2 B,Control,1016,365,83.38698807389133
0.0.A,0.0,LUFA 2.2 B,Control,1043,365,
0.0.B,0.0,LUFA 2.2 B,Control,406,365,32.91591633672303
0.0.B,0.0,LUFA 2.2 B,Control,442,365,234.4537199350141
0.0.B,0.0,LUFA 2.2 B,Control,469,365,262.46112232986343
0.0.B,0.0,LUFA 2.2 B,Control,497,365,180.5563130392924
0.0.B,0.0,LUFA 2.2 B,Control,525,365,188.44362106023223
0.0.B,0.0,LUFA 2.2 B,Control,561,365,383.8842804019496
0.0.B,0.0,LUFA 2.2 B,Control,595,365,634.569782056682
0.0.B,0.0,LUFA 2.2 B,Control,623,365,180.46006768156929
0.0.B,0.0,LUFA 2.2 B,Control,651,365,282.14330055960045
0.0.B,0.0,LUFA 2.2 B,Control,679,365,247.73558096155003
0.0.B,0.0,LUFA 2.2 B,Control,714,365,169.53621825621275
0.0.B,0.0,LUFA 2.2 B,Control,742,365,61.46710518896169
0.0.B,0.0,LUFA 2.2 B,Control,770,365,40.44711650520488
0.0.B,0.0,LUFA 2.2 B,Control,798,365,37.99767184547807
0.0.B,0.0,LUFA 2.2 B,Control,833,365,46.154466911366505
0.0.B,0.0,LUFA 2.2 B,Control,868,365,100.33579762921956
0.0.B,0.0,LUFA 2.2 B,Control,892,365,92.71316437812143
0.0.B,0.0,LUFA 2.2 B,Control,925,365,119.531936554546
0.0.B,0.0,LUFA 2.2 B,Control,955,365,224.92542836512425
0.0.B,0.0,LUFA 2.2 B,Control,987,365,233.45758035982908
0.0.B,0.0,LUFA 2.2 B,Control,1016,365,153.30443670497624
0.0.B,0.0,LUFA 2.2 B,Control,1043,365,
0.0.C,0.0,LUFA 2.2 B,Control,406,365,63.52194382333474
0.0.C,0.0,LUFA 2.2 B,Control,442,365,238.6885161802756
0.0.C,0.0,LUFA 2.2 B,Control,469,365,320.0158534207834
0.0.C,0.0,LUFA 2.2 B,Control,497,365,316.8156948071484
0.0.C,0.0,LUFA 2.2 B,Control,525,365,171.3167575907094
0.0.C,0.0,LUFA 2.2 B,Control,561,365,528.8201824417835
0.0.C,0.0,LUFA 2.2 B,Control,595,365,516.7413885311993
0.0.C,0.0,LUFA 2.2 B,Control,623,365,475.3751287081052
0.0.C,0.0,LUFA 2.2 B,Control,651,365,214.1459469763524
0.0.C,0.0,LUFA 2.2 B,Control,679,365,179.54573666285577
0.0.C,0.0,LUFA 2.2 B,Control,714,365,118.67054051386968
0.0.C,0.0,LUFA 2.2 B,Control,742,365,50.28820552321035
0.0.C,0.0,LUFA 2.2 B,Control,770,365,61.139870918827846
0.0.C,0.0,LUFA 2.2 B,Control,798,365,63.05996603887117
0.0.C,0.0,LUFA 2.2 B,Control,833,365,37.79555657981828
0.0.C,0.0,LUFA 2.2 B,Control,868,365,168.20321988085925
0.0.C,0.0,LUFA 2.2 B,Control,892,365,128.0352149467477
0.0.C,0.0,LUFA 2.2 B,Control,925,365,29.287465912509777
0.0.C,0.0,LUFA 2.2 B,Control,955,365,47.91094489439798
0.0.C,0.0,LUFA 2.2 B,Control,987,365,59.96086515434141
0.0.C,0.0,LUFA 2.2 B,Control,1016,365,42.41533430410976
0.0.C,0.0,LUFA 2.2 B,Control,1043,365,
0.0.D,0.0,LUFA 2.2 B,Control,406,365,17.32416649617907
0.0.D,0.0,LUFA 2.2 B,Control,442,365,164.09835487093088
0.0.D,0.0,LUFA 2.2 B,Control,469,365,103.46377214032132
0.0.D,0.0,LUFA 2.2 B,Control,497,365,74.15705715145316
0.0.D,0.0,LUFA 2.2 B,Control,525,365,62.309252157169496
0.0.D,0.0,LUFA 2.2 B,Control,561,365,176.50919525843912
0.0.D,0.0,LUFA 2.2 B,Control,595,365,97.98259835128468
0.0.D,0.0,LUFA 2.2 B,Control,623,365,27.213378205668217
0.0.D,0.0,LUFA 2.2 B,Control,651,365,70.88471457969793
0.0.D,0.0,LUFA 2.2 B,Control,679,365,82.34753807088272
0.0.D,0.0,LUFA 2.2 B,Control,714,365,82.33791354473794
0.0.E,0.0,LUFA 2.2 B,Control,406,365,45.90904121788314
0.0.E,0.0,LUFA 2.2 B,Control,442,365,288.73610830976594
0.0.E,0.0,LUFA 2.2 B,Control,469,365,292.34530958541427
0.0.E,0.0,LUFA 2.2 B,Control,497,365,314.3854992478489
0.0.E,0.0,LUFA 2.2 B,Control,525,365,202.44732231782896
0.0.F,0.0,LUFA 2.2 B,Control,406,365,57.74722166195319
0.0.F,0.0,LUFA 2.2 B,Control,442,365,352.2580521090318
0.0.F,0.0,LUFA 2.2 B,Control,469,365,375.4531861122811
0.0.F,0.0,LUFA 2.2 B,Control,497,365,337.82124676575006
0.0.F,0.0,LUFA 2.2 B,Control,525,365,114.86884840243094
0.0.F,0.0,LUFA 2.2 B,Control,561,365,426.86746254287266
0.0.F,0.0,LUFA 2.2 B,Control,595,365,392.2480031289488
0.0.F,0.0,LUFA 2.2 B,Control,623,365,179.37730727480596
0.0.F,0.0,LUFA 2.2 B,Control,651,365,286.5032155966063
0.0.F,0.0,LUFA 2.2 B,Control,679,365,219.4394423009808
0.0.G,0.0,LUFA 2.2 B,Control,406,365,47.64145787351826
0.0.G,0.0,LUFA 2.2 B,Control,442,365,287.7736544918467
0.0.G,0.0,LUFA 2.2 B,Control,469,365,413.1813709609483
0.0.G,0.0,LUFA 2.2 B,Control,497,365,123.86779045670616
0.0.G,0.0,LUFA 2.2 B,Control,525,365,507.6365764486431
0.0.G,0.0,LUFA 2.2 B,Control,561,365,396.5357342800409
0.0.G,0.0,LUFA 2.2 B,Control,595,365,498.4740174499067
0.0.G,0.0,LUFA 2.2 B,Control,623,365,193.45319253866057
0.0.G,0.0,LUFA 2.2 B,Control,651,365,352.79702629520426
0.0.G,0.0,LUFA 2.2 B,Control,679,365,341.6903106083398
0.1.A,0.1,LUFA 2.2 B,Limestone 10,406,365,65.8318326975149
0.1.A,0.1,LUFA 2.2 B,Limestone 10,442,365,496.8667197785667
0.1.A,0.1,LUFA 2.2 B,Limestone 10,469,365,565.4415454600156
0.1.A,0.1,LUFA 2.2 B,Limestone 10,497,365,409.28343341958
0.1.A,0.1,LUFA 2.2 B,Limestone 10,525,365,98.459012912931
0.1.A,0.1,LUFA 2.2 B,Limestone 10,561,365,452.70453192129486
0.1.A,0.1,LUFA 2.2 B,Limestone 10,595,365,871.0302179433179
0.1.A,0.1,LUFA 2.2 B,Limestone 10,623,365,716.86438510139
0.1.A,0.1,LUFA 2.2 B,Limestone 10,651,365,703.9771300318912
0.1.A,0.1,LUFA 2.2 B,Limestone 10,679,365,571.3317619592033
0.1.A,0.1,LUFA 2.2 B,Limestone 10,714,365,499.2247312112642
0.1.A,0.1,LUFA 2.2 B,Limestone 10,742,365,172.16852909372983
0.1.A,0.1,LUFA 2.2 B,Limestone 10,770,365,45.37969167819965
0.1.A,0.1,LUFA 2.2 B,Limestone 10,798,365,67.48244077260966
0.1.A,0.1,LUFA 2.2 B,Limestone 10,833,365,59.71062718575125
0.1.A,0.1,LUFA 2.2 B,Limestone 10,868,365,144.9359018232144
0.1.A,0.1,LUFA 2.2 B,Limestone 10,892,365,121.56752613273964
0.1.A,0.1,LUFA 2.2 B,Limestone 10,925,365,72.04928354293278
0.1.A,0.1,LUFA 2.2 B,Limestone 10,955,365,90.78344470786448
0.1.A,0.1,LUFA 2.2 B,Limestone 10,987,365,154.4545688669595
0.1.A,0.1,LUFA 2.2 B,Limestone 10,1016,365,110.87466557554606
0.1.A,0.1,LUFA 2.2 B,Limestone 10,1043,365,
0.1.B,0.1,LUFA 2.2 B,Limestone 10,406,365,68.62294840844815
0.1.B,0.1,LUFA 2.2 B,Limestone 10,442,365,317.0322469462663
0.1.B,0.1,LUFA 2.2 B,Limestone 10,469,365,429.1581021722125
0.1.B,0.1,LUFA 2.2 B,Limestone 10,497,365,253.60654840844813
0.1.B,0.1,LUFA 2.2 B,Limestone 10,525,365,140.05144932908118
0.1.B,0.1,LUFA 2.2 B,Limestone 10,561,365,697.7211810578253
0.1.B,0.1,LUFA 2.2 B,Limestone 10,595,365,1216.3730402551296
0.1.B,0.1,LUFA 2.2 B,Limestone 10,623,365,443.9317665322823
0.1.B,0.1,LUFA 2.2 B,Limestone 10,651,365,451.36672122269687
0.1.B,0.1,LUFA 2.2 B,Limestone 10,679,365,424.4589221387101
0.1.B,0.1,LUFA 2.2 B,Limestone 10,714,365,323.42293952704733
0.1.B,0.1,LUFA 2.2 B,Limestone 10,742,365,81.56795060998053
0.1.B,0.1,LUFA 2.2 B,Limestone 10,770,365,10.259756380046934
0.1.B,0.1,LUFA 2.2 B,Limestone 10,798,365,49.58561432095794
0.1.B,0.1,LUFA 2.2 B,Limestone 10,833,365,34.60983483964137
0.1.B,0.1,LUFA 2.2 B,Limestone 10,868,365,205.0315104879957
0.1.B,0.1,LUFA 2.2 B,Limestone 10,892,365,158.45837621998916
0.1.B,0.1,LUFA 2.2 B,Limestone 10,925,365,131.37492927372284
0.1.B,0.1,LUFA 2.2 B,Limestone 10,955,365,105.05663301040975
0.1.B,0.1,LUFA 2.2 B,Limestone 10,987,365,128.89179873638605
0.1.B,0.1,LUFA 2.2 B,Limestone 10,1016,365,87.58328619050484
0.1.B,0.1,LUFA 2.2 B,Limestone 10,1043,365,
0.1.C,0.1,LUFA 2.2 B,Limestone 10,406,365,87.92014496660448
0.1.C,0.1,LUFA 2.2 B,Limestone 10,442,365,361.01638076899934
0.1.C,0.1,LUFA 2.2 B,Limestone 10,469,365,363.3262696913171
0.1.C,0.1,LUFA 2.2 B,Limestone 10,497,365,492.53567795896265
0.1.C,0.1,LUFA 2.2 B,Limestone 10,525,365,195.15192331668572
0.1.C,0.1,LUFA 2.2 B,Limestone 10,561,365,523.9838524580299
0.1.C,0.1,LUFA 2.2 B,Limestone 10,595,365,362.07507984836633
0.1.C,0.1,LUFA 2.2 B,Limestone 10,623,365,278.91426824718695
0.1.C,0.1,LUFA 2.2 B,Limestone 10,651,365,241.98973223418977
0.1.C,0.1,LUFA 2.2 B,Limestone 10,679,365,
0.1.D,0.1,LUFA 2.2 B,Limestone 10,406,365,
0.1.D,0.1,LUFA 2.2 B,Limestone 10,442,365,440.9481600577651
0.1.D,0.1,LUFA 2.2 B,Limestone 10,469,365,379.97671845478067
0.1.D,0.1,LUFA 2.2 B,Limestone 10,497,365,368.7881942355136
0.1.D,0.1,LUFA 2.2 B,Limestone 10,525,365,135.0900005295144
0.1.D,0.1,LUFA 2.2 B,Limestone 10,561,365,435.7605345688669
0.1.D,0.1,LUFA 2.2 B,Limestone 10,595,365,1295.000694626632
0.1.D,0.1,LUFA 2.2 B,Limestone 10,623,365,581.4856484746374
0.1.D,0.1,LUFA 2.2 B,Limestone 10,651,365,506.6356247668332
0.1.D,0.1,LUFA 2.2 B,Limestone 10,679,365,476.9848325230295
0.1.D,0.1,LUFA 2.2 B,Limestone 10,714,365,407.63763764366087
0.1.D,0.1,LUFA 2.2 B,Limestone 10,742,365,148.0710947224236
0.1.D,0.1,LUFA 2.2 B,Limestone 10,770,365,121.87551130633612
0.1.D,0.1,LUFA 2.2 B,Limestone 10,798,365,85.7546241530778
0.1.D,0.1,LUFA 2.2 B,Limestone 10,833,365,47.98794119983152
0.1.D,0.1,LUFA 2.2 B,Limestone 10,868,365,90.04235537637643
0.1.D,0.1,LUFA 2.2 B,Limestone 10,892,365,74.06081176966123
0.1.D,0.1,LUFA 2.2 B,Limestone 10,925,365,41.905233840784646
0.1.D,0.1,LUFA 2.2 B,Limestone 10,955,365,26.462664323966543
0.1.D,0.1,LUFA 2.2 B,Limestone 10,987,365,104.5080343943679
0.1.D,0.1,LUFA 2.2 B,Limestone 10,1016,365,56.361288332631325
0.1.D,0.1,LUFA 2.2 B,Limestone 10,1043,365,
0.1.E,0.1,LUFA 2.2 B,Limestone 10,406,365,64.6768882604248
0.1.E,0.1,LUFA 2.2 B,Limestone 10,442,365,525.2591036765148
0.1.E,0.1,LUFA 2.2 B,Limestone 10,469,365,460.4378472832301
0.1.E,0.1,LUFA 2.2 B,Limestone 10,497,365,276.2482714964799
0.1.E,0.1,LUFA 2.2 B,Limestone 10,525,365,100.04706153198146
0.1.E,0.1,LUFA 2.2 B,Limestone 10,561,365,404.5385368554065
0.1.E,0.1,LUFA 2.2 B,Limestone 10,595,365,664.1074858896444
0.1.E,0.1,LUFA 2.2 B,Limestone 10,623,365,259.4293933449666
0.1.E,0.1,LUFA 2.2 B,Limestone 10,651,365,425.4045328840483
0.1.E,0.1,LUFA 2.2 B,Limestone 10,679,365,
0.1.F,0.1,LUFA 2.2 B,Limestone 10,406,365,49.75885598411457
0.1.F,0.1,LUFA 2.2 B,Limestone 10,442,365,358.0327742944822
0.1.F,0.1,LUFA 2.2 B,Limestone 10,469,365,429.6393292015164
0.1.F,0.1,LUFA 2.2 B,Limestone 10,497,365,331.1081321379144
0.1.F,0.1,LUFA 2.2 B,Limestone 10,525,365,333.4420824357663
0.1.G,0.1,LUFA 2.2 B,Limestone 10,406,365,62.36699938624466
0.1.G,0.1,LUFA 2.2 B,Limestone 10,442,365,425.01955135688064
0.1.G,0.1,LUFA 2.2 B,Limestone 10,469,365,358.4177558216499
0.1.G,0.1,LUFA 2.2 B,Limestone 10,497,365,287.9661452554305
0.1.G,0.1,LUFA 2.2 B,Limestone 10,525,365,181.03753990011433
0.1.G,0.1,LUFA 2.2 B,Limestone 10,561,365,517.4391674589325
0.1.G,0.1,LUFA 2.2 B,Limestone 10,595,365,1112.8178350081232
0.1.G,0.1,LUFA 2.2 B,Limestone 10,623,365,500.7261589746675
0.1.G,0.1,LUFA 2.2 B,Limestone 10,651,365,347.95588398820627
0.1.G,0.1,LUFA 2.2 B,Limestone 10,679,365,
0.2.A,0.2,LUFA 2.2 B,Basanite,406,365,70.16287432456826
0.2.A,0.2,LUFA 2.2 B,Basanite,442,365,292.5859231000662
0.2.A,0.2,LUFA 2.2 B,Basanite,469,365,282.2395458210482
0.2.A,0.2,LUFA 2.2 B,Basanite,497,365,205.8447838738793
0.2.A,0.2,LUFA 2.2 B,Basanite,525,365,83.14637465551476
0.2.A,0.2,LUFA 2.2 B,Basanite,561,365,660.4357251338829
0.2.A,0.2,LUFA 2.2 B,Basanite,595,365,355.289781334617
0.2.A,0.2,LUFA 2.2 B,Basanite,623,365,267.0375897466755
0.2.A,0.2,LUFA 2.2 B,Basanite,651,365,363.5331971839461
0.2.A,0.2,LUFA 2.2 B,Basanite,679,365,459.09041217883146
0.2.A,0.2,LUFA 2.2 B,Basanite,714,365,190.31078125037607
0.2.A,0.2,LUFA 2.2 B,Basanite,742,365,80.92551275275157
0.2.A,0.2,LUFA 2.2 B,Basanite,770,365,35.22580521090318
0.2.A,0.2,LUFA 2.2 B,Basanite,798,365,36.452933678319994
0.2.A,0.2,LUFA 2.2 B,Basanite,833,365,40.14875585775317
0.2.A,0.2,LUFA 2.2 B,Basanite,868,365,116.09116459474096
0.2.A,0.2,LUFA 2.2 B,Basanite,892,365,79.77778672603647
0.2.A,0.2,LUFA 2.2 B,Basanite,925,365,32.4539385763283
0.2.A,0.2,LUFA 2.2 B,Basanite,955,365,19.865044250556593
0.2.A,0.2,LUFA 2.2 B,Basanite,987,365,105.0999434141645
0.2.A,0.2,LUFA 2.2 B,Basanite,1016,365,30.538655731391778
0.2.A,0.2,LUFA 2.2 B,Basanite,1043,365,
0.2.B,0.2,LUFA 2.2 B,Basanite,406,365,65.8318326975149
0.2.B,0.2,LUFA 2.2 B,Basanite,442,365,266.79216414946745
0.2.B,0.2,LUFA 2.2 B,Basanite,469,365,200.43098184006257
0.2.B,0.2,LUFA 2.2 B,Basanite,497,365,127.26043973764966
0.2.B,0.2,LUFA 2.2 B,Basanite,525,365,56.67889805644142
0.2.B,0.2,LUFA 2.2 B,Basanite,561,365,130.02749409711777
0.2.B,0.2,LUFA 2.2 B,Basanite,595,365,99.11348143690957
0.2.B,0.2,LUFA 2.2 B,Basanite,623,365,46.04378472832301
0.2.B,0.2,LUFA 2.2 B,Basanite,651,365,82.13579827907816
0.2.B,0.2,LUFA 2.2 B,Basanite,679,365,88.0885743709818
0.2.B,0.2,LUFA 2.2 B,Basanite,714,365,99.44071568686442
0.2.B,0.2,LUFA 2.2 B,Basanite,742,365,78.86104956883229
0.2.B,0.2,LUFA 2.2 B,Basanite,770,365,96.27905529815273
0.2.B,0.2,LUFA 2.2 B,Basanite,798,365,65.1918009747879
0.2.B,0.2,LUFA 2.2 B,Basanite,833,365,81.42358252602443
0.2.B,0.2,LUFA 2.2 B,Basanite,868,365,204.117179493351
0.2.B,0.2,LUFA 2.2 B,Basanite,892,365,125.50396173054936
0.2.B,0.2,LUFA 2.2 B,Basanite,925,365,75.62479903724653
0.2.B,0.2,LUFA 2.2 B,Basanite,955,365,61.54410147421626
0.2.B,0.2,LUFA 2.2 B,Basanite,987,365,80.49962698116613
0.2.B,0.2,LUFA 2.2 B,Basanite,1016,365,37.540506336121304
0.2.B,0.2,LUFA 2.2 B,Basanite,1043,365,
0.2.C,0.2,LUFA 2.2 B,Basanite,406,365,60.87519617305494
0.2.C,0.2,LUFA 2.2 B,Basanite,442,365,237.7741851615621
0.2.C,0.2,LUFA 2.2 B,Basanite,469,365,284.11633046513026
0.2.C,0.2,LUFA 2.2 B,Basanite,497,365,265.54097430651666
0.2.C,0.2,LUFA 2.2 B,Basanite,525,365,237.41326503399725
0.2.C,0.2,LUFA 2.2 B,Basanite,561,365,582.4769757506468
0.2.C,0.2,LUFA 2.2 B,Basanite,595,365,529.6286433600096
0.2.C,0.2,LUFA 2.2 B,Basanite,623,365,414.17751056020217
0.2.C,0.2,LUFA 2.2 B,Basanite,651,365,383.8842804019496
0.2.C,0.2,LUFA 2.2 B,Basanite,679,365,348.7932188458993
0.2.C,0.2,LUFA 2.2 B,Basanite,714,365,337.82124676575006
0.2.D,0.2,LUFA 2.2 B,Basanite,406,365,83.68534872134305
0.2.D,0.2,LUFA 2.2 B,Basanite,442,365,425.01955135688064
0.2.D,0.2,LUFA 2.2 B,Basanite,469,365,406.5404404597148
0.2.D,0.2,LUFA 2.2 B,Basanite,497,365,291.6234692821469
0.2.D,0.2,LUFA 2.2 B,Basanite,525,365,116.97180973584452
0.2.D,0.2,LUFA 2.2 B,Basanite,561,365,302.3596404115771
0.2.D,0.2,LUFA 2.2 B,Basanite,595,365,232.6635560503039
0.2.D,0.2,LUFA 2.2 B,Basanite,623,365,119.65224326373428
0.2.D,0.2,LUFA 2.2 B,Basanite,651,365,128.42982097599133
0.2.D,0.2,LUFA 2.2 B,Basanite,679,365,114.63785953426802
0.2.D,0.2,LUFA 2.2 B,Basanite,714,365,130.38360197364463
0.2.D,0.2,LUFA 2.2 B,Basanite,742,365,77.89378361516833
0.2.D,0.2,LUFA 2.2 B,Basanite,770,365,122.67916014200614
0.2.D,0.2,LUFA 2.2 B,Basanite,798,365,55.0764126602082
0.2.D,0.2,LUFA 2.2 B,Basanite,833,365,55.10528626271136
0.2.D,0.2,LUFA 2.2 B,Basanite,868,365,102.18852099404296
0.2.D,0.2,LUFA 2.2 B,Basanite,892,365,78.08868047415609
0.2.D,0.2,LUFA 2.2 B,Basanite,925,365,41.039025525001506
0.2.D,0.2,LUFA 2.2 B,Basanite,955,365,49.373874505084544
0.2.D,0.2,LUFA 2.2 B,Basanite,987,365,73.95494186172452
0.2.D,0.2,LUFA 2.2 B,Basanite,1016,365,50.90417589505987
0.2.D,0.2,LUFA 2.2 B,Basanite,1043,365,
0.2.E,0.2,LUFA 2.2 B,Basanite,406,365,41.57799959082977
0.2.E,0.2,LUFA 2.2 B,Basanite,442,365,240.6134235754257
0.2.E,0.2,LUFA 2.2 B,Basanite,469,365,248.6017892773332
0.2.E,0.2,LUFA 2.2 B,Basanite,497,365,200.8881473494193
0.2.E,0.2,LUFA 2.2 B,Basanite,525,365,140.47974121186593
0.2.E,0.2,LUFA 2.2 B,Basanite,561,365,214.2229432817859
0.2.E,0.2,LUFA 2.2 B,Basanite,595,365,162.3322523376858
0.2.E,0.2,LUFA 2.2 B,Basanite,623,365,98.19433816715808
0.2.E,0.2,LUFA 2.2 B,Basanite,651,365,193.68418143089232
0.2.E,0.2,LUFA 2.2 B,Basanite,679,365,256.74414754197005
0.2.E,0.2,LUFA 2.2 B,Basanite,714,365,211.88418080510257
0.2.F,0.2,LUFA 2.2 B,Basanite,406,365,57.554730922438175
0.2.F,0.2,LUFA 2.2 B,Basanite,442,365,331.5652977916842
0.2.F,0.2,LUFA 2.2 B,Basanite,469,365,358.6102465852337
0.2.F,0.2,LUFA 2.2 B,Basanite,497,365,242.10522678861545
0.2.F,0.2,LUFA 2.2 B,Basanite,525,365,91.6159671701065
0.2.F,0.2,LUFA 2.2 B,Basanite,561,365,814.2935726578012
0.2.F,0.2,LUFA 2.2 B,Basanite,595,365,490.89950682953247
0.2.F,0.2,LUFA 2.2 B,Basanite,623,365,285.69475467838015
0.2.F,0.2,LUFA 2.2 B,Basanite,651,365,413.494168361514
0.2.F,0.2,LUFA 2.2 B,Basanite,679,365,429.8943794452133
0.2.G,0.2,LUFA 2.2 B,Basanite,406,365,78.53622145736807
0.2.G,0.2,LUFA 2.2 B,Basanite,442,365,311.8349970515675
0.2.G,0.2,LUFA 2.2 B,Basanite,469,365,294.75144376917984
0.2.G,0.2,LUFA 2.2 B,Basanite,497,365,177.57270658884408
0.2.G,0.2,LUFA 2.2 B,Basanite,525,365,193.1067092123473
0.7.A,0.7,LUFA 2.2 B,Peridotite,406,365,46.294022696913174
0.7.A,0.7,LUFA 2.2 B,Peridotite,442,365,265.63721956796434
0.7.A,0.7,LUFA 2.2 B,Peridotite,469,365,263.32733064564655
0.7.A,0.7,LUFA 2.2 B,Peridotite,497,365,180.5563130392924
0.7.A,0.7,LUFA 2.2 B,Peridotite,525,365,119.96504071243756
0.7.A,0.7,LUFA 2.2 B,Peridotite,561,365,430.8905190444672
0.7.A,0.7,LUFA 2.2 B,Peridotite,595,365,224.86768113604907
0.7.A,0.7,LUFA 2.2 B,Peridotite,623,365,161.97614448522776
0.7.A,0.7,LUFA 2.2 B,Peridotite,651,365,268.00485564715086
0.7.A,0.7,LUFA 2.2 B,Peridotite,679,365,198.0729702596798
0.7.A,0.7,LUFA 2.2 B,Peridotite,714,365,205.1758785486491
0.7.A,0.7,LUFA 2.2 B,Peridotite,742,365,211.05165834983936
0.7.A,0.7,LUFA 2.2 B,Peridotite,770,365,204.94488965641736
0.7.A,0.7,LUFA 2.2 B,Peridotite,798,365,223.8426679583609
0.7.A,0.7,LUFA 2.2 B,Peridotite,833,365,295.5935909501173
0.7.A,0.7,LUFA 2.2 B,Peridotite,868,365,482.8630184728323
0.7.A,0.7,LUFA 2.2 B,Peridotite,892,365,246.78275179011973
0.7.A,0.7,LUFA 2.2 B,Peridotite,925,365,93.81036158613634
0.7.A,0.7,LUFA 2.2 B,Peridotite,955,365,64.84531764847463
0.7.A,0.7,LUFA 2.2 B,Peridotite,987,365,63.63743825741621
0.7.A,0.7,LUFA 2.2 B,Peridotite,1016,365,39.53759776159817
0.7.A,0.7,LUFA 2.2 B,Peridotite,1043,365,
0.7.B,0.7,LUFA 2.2 B,Peridotite,406,365,65.8318326975149
0.7.B,0.7,LUFA 2.2 B,Peridotite,442,365,317.6097189963295
0.7.B,0.7,LUFA 2.2 B,Peridotite,469,365,321.5076565376978
0.7.B,0.7,LUFA 2.2 B,Peridotite,497,365,264.2176003369637
0.7.B,0.7,LUFA 2.2 B,Peridotite,525,365,194.29533952704736
0.7.B,0.7,LUFA 2.2 B,Peridotite,561,365,531.9962794391961
0.7.B,0.7,LUFA 2.2 B,Peridotite,595,365,878.7202228774294
0.7.B,0.7,LUFA 2.2 B,Peridotite,623,365,512.4632817859077
0.7.B,0.7,LUFA 2.2 B,Peridotite,651,365,865.7078488477043
0.7.B,0.7,LUFA 2.2 B,Peridotite,679,365,
0.7.C,0.7,LUFA 2.2 B,Peridotite,406,365,74.06081176966123
0.7.C,0.7,LUFA 2.2 B,Peridotite,442,365,172.47170200373066
0.7.C,0.7,LUFA 2.2 B,Peridotite,469,365,103.0787906612913
0.7.C,0.7,LUFA 2.2 B,Peridotite,497,365,78.82495755460617
0.7.C,0.7,LUFA 2.2 B,Peridotite,525,365,54.35457238100969
0.7.C,0.7,LUFA 2.2 B,Peridotite,561,365,173.94425616463084
0.7.C,0.7,LUFA 2.2 B,Peridotite,595,365,111.327018809796
0.7.C,0.7,LUFA 2.2 B,Peridotite,623,365,64.46514844455142
0.7.C,0.7,LUFA 2.2 B,Peridotite,651,365,87.04431210060774
0.7.C,0.7,LUFA 2.2 B,Peridotite,679,365,93.80314317814695
0.7.C,0.7,LUFA 2.2 B,Peridotite,714,365,82.36197489620314
0.7.C,0.7,LUFA 2.2 B,Peridotite,742,365,64.26543930429136
0.7.C,0.7,LUFA 2.2 B,Peridotite,770,365,116.3606516396895
0.7.C,0.7,LUFA 2.2 B,Peridotite,798,365,85.68725239785788
0.7.C,0.7,LUFA 2.2 B,Peridotite,833,365,108.2760406041278
0.7.C,0.7,LUFA 2.2 B,Peridotite,868,365,152.29867257957758
0.7.C,0.7,LUFA 2.2 B,Peridotite,892,365,114.29618845899272
0.7.C,0.7,LUFA 2.2 B,Peridotite,925,365,52.4056036584632
0.7.C,0.7,LUFA 2.2 B,Peridotite,955,365,62.64611095733798
0.7.C,0.7,LUFA 2.2 B,Peridotite,987,365,121.6782082917143
0.7.C,0.7,LUFA 2.2 B,Peridotite,1016,365,255.70469751489256
0.7.C,0.7,LUFA 2.2 B,Peridotite,1043,365,
0.7.D,0.7,LUFA 2.2 B,Peridotite,406,365,47.49708981286479
0.7.D,0.7,LUFA 2.2 B,Peridotite,442,365,219.4394423009808
0.7.D,0.7,LUFA 2.2 B,Peridotite,469,365,310.39131644503277
0.7.D,0.7,LUFA 2.2 B,Peridotite,497,365,323.55287056982974
0.7.D,0.7,LUFA 2.2 B,Peridotite,525,365,214.38656039472892
0.7.D,0.7,LUFA 2.2 B,Peridotite,561,365,428.34001660749743
0.7.D,0.7,LUFA 2.2 B,Peridotite,595,365,266.6189224381732
0.7.D,0.7,LUFA 2.2 B,Peridotite,623,365,323.02352103014624
0.7.D,0.7,LUFA 2.2 B,Peridotite,651,365,643.4628540826765
0.7.D,0.7,LUFA 2.2 B,Peridotite,679,365,583.9399054094711
0.7.D,0.7,LUFA 2.2 B,Peridotite,714,365,491.4096073169264
0.7.D,0.7,LUFA 2.2 B,Peridotite,742,365,234.1168611675172
0.7.D,0.7,LUFA 2.2 B,Peridotite,770,365,190.989311101751
0.7.D,0.7,LUFA 2.2 B,Peridotite,798,365,121.26916548528791
0.7.D,0.7,LUFA 2.2 B,Peridotite,833,365,63.59894012876828
0.7.D,0.7,LUFA 2.2 B,Peridotite,868,365,179.55536121306938
0.7.D,0.7,LUFA 2.2 B,Peridotite,892,365,170.8355307298875
0.7.D,0.7,LUFA 2.2 B,Peridotite,925,365,97.27038262229976
0.7.D,0.7,LUFA 2.2 B,Peridotite,955,365,101.01913975570127
0.7.D,0.7,LUFA 2.2 B,Peridotite,987,365,79.86440755761477
0.7.D,0.7,LUFA 2.2 B,Peridotite,1016,365,67.67974378723147
0.7.D,0.7,LUFA 2.2 B,Peridotite,1043,365,
0.7.E,0.7,LUFA 2.2 B,Peridotite,406,365,50.91380042120464
0.7.E,0.7,LUFA 2.2 B,Peridotite,442,365,344.31780901377937
0.7.E,0.7,LUFA 2.2 B,Peridotite,469,365,385.4627046151994
0.7.E,0.7,LUFA 2.2 B,Peridotite,497,365,310.7522365966664
0.7.E,0.7,LUFA 2.2 B,Peridotite,525,365,107.63119663036284
0.7.E,0.7,LUFA 2.2 B,Peridotite,561,365,303.24028545640533
0.7.E,0.7,LUFA 2.2 B,Peridotite,595,365,245.3583202358746
0.7.E,0.7,LUFA 2.2 B,Peridotite,623,365,332.0272755280101
0.7.E,0.7,LUFA 2.2 B,Peridotite,651,365,706.4843220410373
0.7.E,0.7,LUFA 2.2 B,Peridotite,679,365,518.6855449786389
0.7.F,0.7,LUFA 2.2 B,Peridotite,406,365,66.313059534268
0.7.F,0.7,LUFA 2.2 B,Peridotite,442,365,301.729233046513
0.7.F,0.7,LUFA 2.2 B,Peridotite,469,365,226.17661815993745
0.7.F,0.7,LUFA 2.2 B,Peridotite,497,365,135.7059709007762
0.7.F,0.7,LUFA 2.2 B,Peridotite,525,365,72.11665532222155
0.7.F,0.7,LUFA 2.2 B,Peridotite,561,365,439.4178585955833
0.7.F,0.7,LUFA 2.2 B,Peridotite,595,365,223.9148519646188
0.7.F,0.7,LUFA 2.2 B,Peridotite,623,365,148.85790061977255
0.7.F,0.7,LUFA 2.2 B,Peridotite,651,365,168.2609671099344
0.7.F,0.7,LUFA 2.2 B,Peridotite,679,365,175.27725453998437
0.7.G,0.7,LUFA 2.2 B,Peridotite,406,365,48.70015692881641
0.7.G,0.7,LUFA 2.2 B,Peridotite,442,365,334.16392273903364
0.7.G,0.7,LUFA 2.2 B,Peridotite,469,365,398.5039521030146
0.7.G,0.7,LUFA 2.2 B,Peridotite,497,365,302.5954413622961
0.7.G,0.7,LUFA 2.2 B,Peridotite,525,365,169.56027958360912
0.8.A,0.8,LUFA 2.2 B,Steel Slag,406,365,100.57641104759612
0.8.A,0.8,LUFA 2.2 B,Steel Slag,442,365,758.0766522654792
0.8.A,0.8,LUFA 2.2 B,Steel Slag,469,365,933.5800835188638
0.8.A,0.8,LUFA 2.2 B,Steel Slag,497,365,683.2458776099645
0.8.A,0.8,LUFA 2.2 B,Steel Slag,525,365,404.1294939527048
0.8.A,0.8,LUFA 2.2 B,Steel Slag,561,365,1537.9047581683617
0.8.A,0.8,LUFA 2.2 B,Steel Slag,595,365,1421.5633554365486
0.8.A,0.8,LUFA 2.2 B,Steel Slag,623,365,640.3300672723991
0.8.A,0.8,LUFA 2.2 B,Steel Slag,651,365,769.4624795715747
0.8.A,0.8,LUFA 2.2 B,Steel Slag,679,365,627.7507975209097
0.8.B,0.8,LUFA 2.2 B,Steel Slag,406,365,84.16657555809616
0.8.B,0.8,LUFA 2.2 B,Steel Slag,442,365,592.8714757807329
0.8.B,0.8,LUFA 2.2 B,Steel Slag,469,365,1749.6445709128104
0.8.B,0.8,LUFA 2.2 B,Steel Slag,497,365,1528.1839756904749
0.8.B,0.8,LUFA 2.2 B,Steel Slag,525,365,243.57778085324028
0.8.B,0.8,LUFA 2.2 B,Steel Slag,561,365,994.7247665924544
0.8.B,0.8,LUFA 2.2 B,Steel Slag,595,365,624.9693064564655
0.8.B,0.8,LUFA 2.2 B,Steel Slag,623,365,175.33500174499068
0.8.B,0.8,LUFA 2.2 B,Steel Slag,651,365,1111.7110131776883
0.8.B,0.8,LUFA 2.2 B,Steel Slag,679,365,979.6142436969732
0.8.B,0.8,LUFA 2.2 B,Steel Slag,714,365,876.4103339551116
0.8.B,0.8,LUFA 2.2 B,Steel Slag,742,365,584.2142046567575
0.8.B,0.8,LUFA 2.2 B,Steel Slag,770,365,691.9079607677959
0.8.B,0.8,LUFA 2.2 B,Steel Slag,798,365,433.58057693001984
0.8.B,0.8,LUFA 2.2 B,Steel Slag,833,365,205.118131319574
0.8.B,0.8,LUFA 2.2 B,Steel Slag,868,365,843.9275217522113
0.8.B,0.8,LUFA 2.2 B,Steel Slag,892,365,749.8332364161502
0.8.B,0.8,LUFA 2.2 B,Steel Slag,925,365,289.4290749142548
0.8.B,0.8,LUFA 2.2 B,Steel Slag,955,365,45.88979214152476
0.8.B,0.8,LUFA 2.2 B,Steel Slag,987,365,167.21670485588783
0.8.B,0.8,LUFA 2.2 B,Steel Slag,1016,365,121.61564881160118
0.8.B,0.8,LUFA 2.2 B,Steel Slag,1043,365,
0.8.C,0.8,LUFA 2.2 B,Steel Slag,406,365,50.288205523798055
0.8.C,0.8,LUFA 2.2 B,Steel Slag,442,365,468.9074399181659
0.8.C,0.8,LUFA 2.2 B,Steel Slag,469,365,1082.760406041278
0.8.C,0.8,LUFA 2.2 B,Steel Slag,497,365,915.9671809374812
0.8.C,0.8,LUFA 2.2 B,Steel Slag,525,365,404.5770349599855
0.8.C,0.8,LUFA 2.2 B,Steel Slag,561,365,798.817317287442
0.8.C,0.8,LUFA 2.2 B,Steel Slag,595,365,477.4732778145496
0.8.C,0.8,LUFA 2.2 B,Steel Slag,623,365,400.1882459835128
0.8.C,0.8,LUFA 2.2 B,Steel Slag,651,365,685.7482570551778
0.8.C,0.8,LUFA 2.2 B,Steel Slag,679,365,
0.8.D,0.8,LUFA 2.2 B,Steel Slag,406,365,69.77789284553823
0.8.D,0.8,LUFA 2.2 B,Steel Slag,442,365,669.6271578313978
0.8.D,0.8,LUFA 2.2 B,Steel Slag,469,365,1440.0713400324928
0.8.D,0.8,LUFA 2.2 B,Steel Slag,497,365,1227.128460196161
0.8.D,0.8,LUFA 2.2 B,Steel Slag,525,365,426.7664049581804
0.8.D,0.8,LUFA 2.2 B,Steel Slag,561,365,1187.4272452012756
0.8.D,0.8,LUFA 2.2 B,Steel Slag,595,365,692.3073790240087
0.8.D,0.8,LUFA 2.2 B,Steel Slag,623,365,440.00495553282383
0.8.D,0.8,LUFA 2.2 B,Steel Slag,651,365,967.2370891148684
0.8.D,0.8,LUFA 2.2 B,Steel Slag,679,365,875.4478803778808
0.8.D,0.8,LUFA 2.2 B,Steel Slag,714,365,742.1817295866177
0.8.D,0.8,LUFA 2.2 B,Steel Slag,742,365,277.40321605040424
0.8.D,0.8,LUFA 2.2 B,Steel Slag,770,365,150.07059228593778
0.8.D,0.8,LUFA 2.2 B,Steel Slag,798,365,97.0153323786028
0.8.D,0.8,LUFA 2.2 B,Steel Slag,833,365,116.57720373066972
0.8.D,0.8,LUFA 2.2 B,Steel Slag,868,365,362.1617007040134
0.8.D,0.8,LUFA 2.2 B,Steel Slag,892,365,331.90215656778383
0.8.D,0.8,LUFA 2.2 B,Steel Slag,925,365,178.05393344966603
0.8.D,0.8,LUFA 2.2 B,Steel Slag,955,365,159.09359566760935
0.8.D,0.8,LUFA 2.2 B,Steel Slag,987,365,333.60569950057163
0.8.D,0.8,LUFA 2.2 B,Steel Slag,1016,365,181.4225213791443
0.8.D,0.8,LUFA 2.2 B,Steel Slag,1043,365,
0.8.E,0.8,LUFA 2.2 B,Steel Slag,406,365,61.30830033094651
0.8.E,0.8,LUFA 2.2 B,Steel Slag,442,365,614.0454568866959
0.8.E,0.8,LUFA 2.2 B,Steel Slag,469,365,1174.1935069498766
0.8.E,0.8,LUFA 2.2 B,Steel Slag,497,365,943.1564977435464
0.8.E,0.8,LUFA 2.2 B,Steel Slag,525,365,365.0490616763945
0.8.E,0.8,LUFA 2.2 B,Steel Slag,561,365,1473.285616944461
0.8.E,0.8,LUFA 2.2 B,Steel Slag,595,365,877.3968491485649
0.8.E,0.8,LUFA 2.2 B,Steel Slag,623,365,1438.338923400927
0.8.E,0.8,LUFA 2.2 B,Steel Slag,651,365,1032.4000166074975
0.8.E,0.8,LUFA 2.2 B,Steel Slag,679,365,755.4058431915278
0.8.F,0.8,LUFA 2.2 B,Steel Slag,406,365,62.36699938624466
0.8.F,0.8,LUFA 2.2 B,Steel Slag,442,365,404.23055153739693
0.8.F,0.8,LUFA 2.2 B,Steel Slag,469,365,2289.677338708707
0.8.F,0.8,LUFA 2.2 B,Steel Slag,497,365,1427.8722394849267
0.8.F,0.8,LUFA 2.2 B,Steel Slag,525,365,216.956311763644
0.8.F,0.8,LUFA 2.2 B,Steel Slag,561,365,879.1437025091763
0.8.F,0.8,LUFA 2.2 B,Steel Slag,595,365,714.1887639448823
0.8.F,0.8,LUFA 2.2 B,Steel Slag,623,365,446.19353270353207
0.8.F,0.8,LUFA 2.2 B,Steel Slag,651,365,828.0662848546845
0.8.F,0.8,LUFA 2.2 B,Steel Slag,679,365,1038.410539743667
0.8.F,0.8,LUFA 2.2 B,Steel Slag,714,365,786.2380472952644
0.8.F,0.8,LUFA 2.2 B,Steel Slag,742,365,523.365476000981
0.8.F,0.8,LUFA 2.2 B,Steel Slag,770,365,462.0355205487695
0.8.F,0.8,LUFA 2.2 B,Steel Slag,798,365,684.3190134183766
0.8.F,0.8,LUFA 2.2 B,Steel Slag,833,365,864.6202763102472
0.8.F,0.8,LUFA 2.2 B,Steel Slag,868,365,1551.4176080389914
0.8.F,0.8,LUFA 2.2 B,Steel Slag,892,365,1400.601114146459
0.8.F,0.8,LUFA 2.2 B,Steel Slag,925,365,602.4960125157951
0.8.F,0.8,LUFA 2.2 B,Steel Slag,955,365,895.6401588543234
0.8.F,0.8,LUFA 2.2 B,Steel Slag,987,365,919.0133468921116
0.8.F,0.8,LUFA 2.2 B,Steel Slag,1016,365,
0.8.F,0.8,LUFA 2.2 B,Steel Slag,1043,365,
0.8.G,0.8,LUFA 2.2 B,Steel Slag,406,365,56.30354110355617
0.8.G,0.8,LUFA 2.2 B,Steel Slag,442,365,685.7482570551778
0.8.G,0.8,LUFA 2.2 B,Steel Slag,469,365,1289.2067233888922
0.8.G,0.8,LUFA 2.2 B,Steel Slag,497,365,1110.4309498766472
0.8.G,0.8,LUFA 2.2 B,Steel Slag,525,365,462.555245441964
0.9.A,0.9,LUFA 2.2 B,Cement,406,365,100.48016568987305
0.9.A,0.9,LUFA 2.2 B,Cement,442,365,731.994157289849
0.9.A,0.9,LUFA 2.2 B,Cement,469,365,1006.7265642938804
0.9.A,0.9,LUFA 2.2 B,Cement,497,365,893.1570282207113
0.9.A,0.9,LUFA 2.2 B,Cement,525,365,328.8704273421987
0.9.A,0.9,LUFA 2.2 B,Cement,561,365,1194.3665363740297
0.9.A,0.9,LUFA 2.2 B,Cement,595,365,1506.2400315301763
0.9.A,0.9,LUFA 2.2 B,Cement,623,365,1205.4540030086046
0.9.A,0.9,LUFA 2.2 B,Cement,651,365,1113.525238341657
0.9.A,0.9,LUFA 2.2 B,Cement,679,365,774.8089098020338
0.9.B,0.9,LUFA 2.2 B,Cement,406,365,71.3178187375895
0.9.B,0.9,LUFA 2.2 B,Cement,442,365,714.6218679824298
0.9.B,0.9,LUFA 2.2 B,Cement,469,365,1221.1612472471268
0.9.B,0.9,LUFA 2.2 B,Cement,497,365,719.0491550634815
0.9.B,0.9,LUFA 2.2 B,Cement,525,365,218.04388444551415
0.9.B,0.9,LUFA 2.2 B,Cement,561,365,964.5614679583608
0.9.B,0.9,LUFA 2.2 B,Cement,595,365,891.2128717732714
0.9.B,0.9,LUFA 2.2 B,Cement,623,365,647.7457731512125
0.9.B,0.9,LUFA 2.2 B,Cement,651,365,671.6338737589506
0.9.B,0.9,LUFA 2.2 B,Cement,679,365,464.720766351766
0.9.C,0.9,LUFA 2.2 B,Cement,406,365,77.95874923882303
0.9.C,0.9,LUFA 2.2 B,Cement,442,365,636.663118839882
0.9.C,0.9,LUFA 2.2 B,Cement,469,365,1416.7318380167278
0.9.C,0.9,LUFA 2.2 B,Cement,497,365,1089.0644777664118
0.9.C,0.9,LUFA 2.2 B,Cement,525,365,638.3955354714484
0.9.C,0.9,LUFA 2.2 B,Cement,561,365,1274.5004310728684
0.9.C,0.9,LUFA 2.2 B,Cement,595,365,930.2307445694688
0.9.C,0.9,LUFA 2.2 B,Cement,623,365,528.0405947409591
0.9.C,0.9,LUFA 2.2 B,Cement,651,365,585.0178535411276
0.9.C,0.9,LUFA 2.2 B,Cement,679,365,574.8158442746254
0.9.D,0.9,LUFA 2.2 B,Cement,406,365,50.81755506348156
0.9.D,0.9,LUFA 2.2 B,Cement,442,365,480.023780010831
0.9.D,0.9,LUFA 2.2 B,Cement,469,365,1108.265428966845
0.9.D,0.9,LUFA 2.2 B,Cement,497,365,818.0856400505446
0.9.D,0.9,LUFA 2.2 B,Cement,525,365,446.9057483603104
0.9.D,0.9,LUFA 2.2 B,Cement,561,365,875.2505772910523
0.9.D,0.9,LUFA 2.2 B,Cement,595,365,539.2435559299597
0.9.D,0.9,LUFA 2.2 B,Cement,623,365,469.2442986942656
0.9.D,0.9,LUFA 2.2 B,Cement,651,365,604.7289052289548
0.9.D,0.9,LUFA 2.2 B,Cement,679,365,407.0601653529093
1.0.A,1.0,LUFA 6S B,Control,407,365,1580.8301929117276
1.0.A,1.0,LUFA 6S B,Control,442,365,1905.6583147000424
1.0.A,1.0,LUFA 6S B,Control,470,365,1672.263293820326
1.0.A,1.0,LUFA 6S B,Control,498,365,1461.0047078644925
1.0.A,1.0,LUFA 6S B,Control,525,365,842.5079025212107
1.0.A,1.0,LUFA 6S B,Control,560,365,897.4158859137133
1.0.A,1.0,LUFA 6S B,Control,596,365,922.7524794512304
1.0.A,1.0,LUFA 6S B,Control,624,365,474.4896713400325
1.0.A,1.0,LUFA 6S B,Control,651,365,293.98148095553285
1.0.A,1.0,LUFA 6S B,Control,679,365,222.18243532537485
1.0.A,1.0,LUFA 6S B,Control,714,365,644.2424415428123
1.0.A,1.0,LUFA 6S B,Control,742,365,
1.0.A,1.0,LUFA 6S B,Control,770,365,662.4328164149467
1.0.A,1.0,LUFA 6S B,Control,798,365,1142.336289788796
1.0.A,1.0,LUFA 6S B,Control,833,365,723.8614234310127
1.0.A,1.0,LUFA 6S B,Control,868,365,808.4611030747939
1.0.A,1.0,LUFA 6S B,Control,892,365,733.0047336181478
1.0.A,1.0,LUFA 6S B,Control,925,365,563.8775582164992
1.0.A,1.0,LUFA 6S B,Control,955,365,334.64514952764904
1.0.A,1.0,LUFA 6S B,Control,987,365,204.2086125759673
1.0.A,1.0,LUFA 6S B,Control,1016,365,
1.0.A,1.0,LUFA 6S B,Control,1043,365,
1.0.B,1.0,LUFA 6S B,Control,407,365,1770.9147975209098
1.0.B,1.0,LUFA 6S B,Control,442,365,2312.2950004212043
1.0.B,1.0,LUFA 6S B,Control,470,365,1476.8851938143089
1.0.B,1.0,LUFA 6S B,Control,498,365,1805.5631303929235
1.0.B,1.0,LUFA 6S B,Control,525,365,598.4296458270654
1.0.B,1.0,LUFA 6S B,Control,560,365,857.546241530778
1.0.B,1.0,LUFA 6S B,Control,596,365,825.1356134544798
1.0.B,1.0,LUFA 6S B,Control,624,365,615.4410147421626
1.0.B,1.0,LUFA 6S B,Control,651,365,392.68110716649613
1.0.B,1.0,LUFA 6S B,Control,679,365,372.9989291420128
1.0.B,1.0,LUFA 6S B,Control,714,365,361.64197557013057
1.0.B,1.0,LUFA 6S B,Control,742,365,
1.0.B,1.0,LUFA 6S B,Control,770,365,375.0441432095794
1.0.B,1.0,LUFA 6S B,Control,798,365,808.2686125518984
1.0.B,1.0,LUFA 6S B,Control,833,365,326.5124156688128
1.0.B,1.0,LUFA 6S B,Control,868,365,1694.4478514952764
1.0.B,1.0,LUFA 6S B,Control,892,365,1382.4684864311932
1.0.B,1.0,LUFA 6S B,Control,925,365,548.9595258439135
1.0.B,1.0,LUFA 6S B,Control,955,365,548.9595258439135
1.0.B,1.0,LUFA 6S B,Control,987,365,754.5636962512787
1.0.B,1.0,LUFA 6S B,Control,1016,365,
1.0.B,1.0,LUFA 6S B,Control,1043,365,
1.0.C,1.0,LUFA 6S B,Control,407,365,1905.6583147000424
1.0.C,1.0,LUFA 6S B,Control,442,365,2623.167543173476
1.0.C,1.0,LUFA 6S B,Control,470,365,1563.024799566761
1.0.C,1.0,LUFA 6S B,Control,498,365,2088.524516517239
1.0.C,1.0,LUFA 6S B,Control,525,365,1659.510782357543
1.0.C,1.0,LUFA 6S B,Control,560,365,1614.0348454178952
1.0.C,1.0,LUFA 6S B,Control,596,365,1354.148286419159
1.0.C,1.0,LUFA 6S B,Control,624,365,418.2823755941994
1.0.C,1.0,LUFA 6S B,Control,651,365,370.30405896865034
1.0.C,1.0,LUFA 6S B,Control,679,365,395.8090818318344
1.0.C,1.0,LUFA 6S B,Control,714,365,425.93388242373186
1.0.C,1.0,LUFA 6S B,Control,742,365,
1.0.C,1.0,LUFA 6S B,Control,770,365,422.9262145736807
1.0.C,1.0,LUFA 6S B,Control,798,365,451.3907825982309
1.0.C,1.0,LUFA 6S B,Control,833,365,386.7620169685299
1.0.C,1.0,LUFA 6S B,Control,868,365,633.6313896142968
1.0.C,1.0,LUFA 6S B,Control,892,365,617.5102901498285
1.0.C,1.0,LUFA 6S B,Control,925,365,437.91643083218
1.0.C,1.0,LUFA 6S B,Control,955,365,635.9894012876828
1.0.C,1.0,LUFA 6S B,Control,987,365,367.6813726457669
1.0.C,1.0,LUFA 6S B,Control,1016,365,
1.0.C,1.0,LUFA 6S B,Control,1043,365,
1.0.D,1.0,LUFA 6S B,Control,407,365,2242.5171076478728
1.0.D,1.0,LUFA 6S B,Control,442,365,2553.8708779108247
1.0.D,1.0,LUFA 6S B,Control,470,365,2334.431435585775
1.0.D,1.0,LUFA 6S B,Control,498,365,1799.7884082074731
1.0.D,1.0,LUFA 6S B,Control,525,365,1010.3117041939948
1.0.D,1.0,LUFA 6S B,Control,560,365,1180.2329040255129
1.0.D,1.0,LUFA 6S B,Control,596,365,1114.1604577892772
1.0.D,1.0,LUFA 6S B,Control,624,365,724.2945277092484
1.0.D,1.0,LUFA 6S B,Control,651,365,256.2532961068656
1.0.D,1.0,LUFA 6S B,Control,679,365,
1.0.E,1.0,LUFA 6S B,Control,407,365,1754.3605940188938
1.0.E,1.0,LUFA 6S B,Control,442,365,2869.3150755159754
1.0.E,1.0,LUFA 6S B,Control,470,365,1238.677904567062
1.0.E,1.0,LUFA 6S B,Control,498,365,1318.8021746194115
1.0.E,1.0,LUFA 6S B,Control,525,365,737.7207567242313
1.0.E,1.0,LUFA 6S B,Control,560,365,865.2458711113785
1.0.E,1.0,LUFA 6S B,Control,596,365,1043.2757431855105
1.0.E,1.0,LUFA 6S B,Control,624,365,546.4812075335459
1.0.E,1.0,LUFA 6S B,Control,651,365,467.2231457969794
1.0.E,1.0,LUFA 6S B,Control,679,365,
1.0.F,1.0,LUFA 6S B,Control,407,365,1984.0982908718936
1.0.F,1.0,LUFA 6S B,Control,442,365,2691.50175582165
1.0.F,1.0,LUFA 6S B,Control,470,365,1702.5805851134244
1.0.F,1.0,LUFA 6S B,Control,498,365,1110.912176665263
1.0.F,1.0,LUFA 6S B,Control,525,365,559.5705777724291
1.0.F,1.0,LUFA 6S B,Control,560,365,739.8862773933449
1.0.F,1.0,LUFA 6S B,Control,596,365,781.5123997833805
1.0.F,1.0,LUFA 6S B,Control,624,365,420.39977375293336
1.0.F,1.0,LUFA 6S B,Control,651,365,560.5330315903484
1.0.F,1.0,LUFA 6S B,Control,679,365,
1.0.G,1.0,LUFA 6S B,Control,407,365,1802.4351559058907
1.0.G,1.0,LUFA 6S B,Control,442,365,2935.4837667729707
1.0.G,1.0,LUFA 6S B,Control,470,365,1493.2469067934292
1.0.G,1.0,LUFA 6S B,Control,498,365,1758.8841263613936
1.0.G,1.0,LUFA 6S B,Control,525,365,1176.5033958721945
1.1.A,1.1,LUFA 6S B,Limestone 10,407,365,2651.752418316385
1.1.A,1.1,LUFA 6S B,Limestone 10,442,365,2564.6984824598353
1.1.A,1.1,LUFA 6S B,Limestone 10,470,365,2468.693726457669
1.1.A,1.1,LUFA 6S B,Limestone 10,498,365,2183.5668189421745
1.1.A,1.1,LUFA 6S B,Limestone 10,525,365,339.2168046212167
1.1.A,1.1,LUFA 6S B,Limestone 10,560,365,1686.4835470244898
1.1.A,1.1,LUFA 6S B,Limestone 10,596,365,1311.078483663277
1.1.A,1.1,LUFA 6S B,Limestone 10,624,365,548.5023604308321
1.1.A,1.1,LUFA 6S B,Limestone 10,651,365,384.5964962994163
1.1.A,1.1,LUFA 6S B,Limestone 10,679,365,286.498403471309
1.1.A,1.1,LUFA 6S B,Limestone 10,714,365,512.5065922137312
1.1.A,1.1,LUFA 6S B,Limestone 10,742,365,
1.1.A,1.1,LUFA 6S B,Limestone 10,770,365,248.38523713821527
1.1.A,1.1,LUFA 6S B,Limestone 10,798,365,222.9283369396474
1.1.A,1.1,LUFA 6S B,Limestone 10,833,365,
1.1.A,1.1,LUFA 6S B,Limestone 10,868,365,59.96086515434141
1.1.A,1.1,LUFA 6S B,Limestone 10,892,365,796.3341866538299
1.1.A,1.1,LUFA 6S B,Limestone 10,925,365,290.9016289788796
1.1.A,1.1,LUFA 6S B,Limestone 10,955,365,197.01427120765388
1.1.A,1.1,LUFA 6S B,Limestone 10,987,365,576.196965401047
1.1.A,1.1,LUFA 6S B,Limestone 10,1016,365,
1.1.A,1.1,LUFA 6S B,Limestone 10,1043,365,
1.1.B,1.1,LUFA 6S B,Limestone 10,407,365,2281.0152555508753
1.1.B,1.1,LUFA 6S B,Limestone 10,442,365,2699.6826114688006
1.1.B,1.1,LUFA 6S B,Limestone 10,470,365,2176.107802876226
1.1.B,1.1,LUFA 6S B,Limestone 10,498,365,1809.412945183224
1.1.B,1.1,LUFA 6S B,Limestone 10,525,365,1162.4515720560803
1.1.B,1.1,LUFA 6S B,Limestone 10,560,365,1108.0248154521933
1.1.B,1.1,LUFA 6S B,Limestone 10,596,365,1197.0277208014925
1.1.B,1.1,LUFA 6S B,Limestone 10,624,365,505.504741560864
1.1.B,1.1,LUFA 6S B,Limestone 10,651,365,331.42092953848004
1.1.B,1.1,LUFA 6S B,Limestone 10,679,365,357.0943819042048
1.1.B,1.1,LUFA 6S B,Limestone 10,714,365,695.5171622841326
1.1.B,1.1,LUFA 6S B,Limestone 10,742,365,
1.1.B,1.1,LUFA 6S B,Limestone 10,770,365,621.8413319694325
1.1.B,1.1,LUFA 6S B,Limestone 10,798,365,333.4902051868343
1.1.B,1.1,LUFA 6S B,Limestone 10,833,365,420.7005405486358
1.1.B,1.1,LUFA 6S B,Limestone 10,868,365,373.1914199410313
1.1.B,1.1,LUFA 6S B,Limestone 10,892,365,382.5753434021301
1.1.B,1.1,LUFA 6S B,Limestone 10,925,365,351.1031077682171
1.1.B,1.1,LUFA 6S B,Limestone 10,955,365,312.8455733798664
1.1.B,1.1,LUFA 6S B,Limestone 10,987,365,336.4738114206631
1.1.B,1.1,LUFA 6S B,Limestone 10,1016,365,
1.1.B,1.1,LUFA 6S B,Limestone 10,1043,365,
1.1.C,1.1,LUFA 6S B,Limestone 10,407,365,2940.296037066009
1.1.C,1.1,LUFA 6S B,Limestone 10,442,365,2839.238397015464
1.1.C,1.1,LUFA 6S B,Limestone 10,470,365,2155.896275106805
1.1.C,1.1,LUFA 6S B,Limestone 10,498,365,1597.9137459534268
1.1.C,1.1,LUFA 6S B,Limestone 10,525,365,751.7725805403453
1.1.C,1.1,LUFA 6S B,Limestone 10,560,365,1243.2736210361634
1.1.C,1.1,LUFA 6S B,Limestone 10,596,365,917.7958428304952
1.1.C,1.1,LUFA 6S B,Limestone 10,624,365,356.05974414826403
1.1.C,1.1,LUFA 6S B,Limestone 10,651,365,471.40981936337926
1.1.C,1.1,LUFA 6S B,Limestone 10,679,365,276.1039034872875
1.1.C,1.1,LUFA 6S B,Limestone 10,714,365,352.5467881340634
1.1.C,1.1,LUFA 6S B,Limestone 10,742,365,
1.1.C,1.1,LUFA 6S B,Limestone 10,770,365,682.6684053192129
1.1.C,1.1,LUFA 6S B,Limestone 10,798,365,1087.9576559359768
1.1.C,1.1,LUFA 6S B,Limestone 10,833,365,1107.591711414646
1.1.C,1.1,LUFA 6S B,Limestone 10,868,365,2706.8047704434684
1.1.C,1.1,LUFA 6S B,Limestone 10,892,365,2068.914522654793
1.1.C,1.1,LUFA 6S B,Limestone 10,925,365,1178.7651620434442
1.1.C,1.1,LUFA 6S B,Limestone 10,955,365,1183.6255531620434
1.1.C,1.1,LUFA 6S B,Limestone 10,987,365,771.0697772429148
1.1.C,1.1,LUFA 6S B,Limestone 10,1016,365,
1.1.C,1.1,LUFA 6S B,Limestone 10,1043,365,
1.1.D,1.1,LUFA 6S B,Limestone 10,407,365,2791.1157133401525
1.1.D,1.1,LUFA 6S B,Limestone 10,442,365,2897.4668463806483
1.1.D,1.1,LUFA 6S B,Limestone 10,470,365,2457.8661219086584
1.1.D,1.1,LUFA 6S B,Limestone 10,498,365,2669.8465491305133
1.1.D,1.1,LUFA 6S B,Limestone 10,525,365,632.0433409952464
1.1.D,1.1,LUFA 6S B,Limestone 10,560,365,1697.1186605692278
1.1.D,1.1,LUFA 6S B,Limestone 10,596,365,964.7395217522112
1.1.D,1.1,LUFA 6S B,Limestone 10,624,365,614.0213955111619
1.1.D,1.1,LUFA 6S B,Limestone 10,651,365,491.64540826764545
1.1.D,1.1,LUFA 6S B,Limestone 10,679,365,
1.1.E,1.1,LUFA 6S B,Limestone 10,407,365,2217.493311510921
1.1.E,1.1,LUFA 6S B,Limestone 10,442,365,2208.83122835309
1.1.E,1.1,LUFA 6S B,Limestone 10,470,365,698.7413820326133
1.1.E,1.1,LUFA 6S B,Limestone 10,498,365,1637.1337339190084
1.1.E,1.1,LUFA 6S B,Limestone 10,525,365,263.32733064564655
1.1.E,1.1,LUFA 6S B,Limestone 10,560,365,978.5747936698958
1.1.E,1.1,LUFA 6S B,Limestone 10,596,365,1379.4367574462965
1.1.E,1.1,LUFA 6S B,Limestone 10,624,365,557.0922597027499
1.1.E,1.1,LUFA 6S B,Limestone 10,651,365,406.8772992358144
1.1.E,1.1,LUFA 6S B,Limestone 10,679,365,
1.1.F,1.1,LUFA 6S B,Limestone 10,407,365,1902.0491134243937
1.1.F,1.1,LUFA 6S B,Limestone 10,442,365,3415.7481605391417
1.1.F,1.1,LUFA 6S B,Limestone 10,470,365,1865.2352594018896
1.1.F,1.1,LUFA 6S B,Limestone 10,498,365,1876.7847037727904
1.1.F,1.1,LUFA 6S B,Limestone 10,525,365,738.3704129008966
1.1.F,1.1,LUFA 6S B,Limestone 10,560,365,1179.4870023467115
1.1.F,1.1,LUFA 6S B,Limestone 10,596,365,787.094631205247
1.1.F,1.1,LUFA 6S B,Limestone 10,624,365,523.4785642938805
1.1.F,1.1,LUFA 6S B,Limestone 10,651,365,448.6237282628317
1.1.F,1.1,LUFA 6S B,Limestone 10,679,365,
1.1.G,1.1,LUFA 6S B,Limestone 10,407,365,1630.637171430291
1.1.G,1.1,LUFA 6S B,Limestone 10,442,365,2148.918485829472
1.1.G,1.1,LUFA 6S B,Limestone 10,470,365,1390.264361273241
1.1.G,1.1,LUFA 6S B,Limestone 10,498,365,1767.0649827306097
1.1.G,1.1,LUFA 6S B,Limestone 10,525,365,761.9745898068476
1.2.A,1.2,LUFA 6S B,Basanite,407,365,2203.0565061676393
1.2.A,1.2,LUFA 6S B,Basanite,442,365,2545.2087947529935
1.2.A,1.2,LUFA 6S B,Basanite,470,365,1765.3806888501113
1.2.A,1.2,LUFA 6S B,Basanite,498,365,1896.9962315422104
1.2.A,1.2,LUFA 6S B,Basanite,525,365,597.683744148264
1.2.A,1.2,LUFA 6S B,Basanite,560,365,707.6922014561646
1.2.A,1.2,LUFA 6S B,Basanite,596,365,849.605998676214
1.2.A,1.2,LUFA 6S B,Basanite,624,365,518.2572530236476
1.2.A,1.2,LUFA 6S B,Basanite,651,365,389.0237831397797
1.2.A,1.2,LUFA 6S B,Basanite,679,365,277.9566268692033
1.2.A,1.2,LUFA 6S B,Basanite,714,365,341.0695278897647
1.2.A,1.2,LUFA 6S B,Basanite,742,365,
1.2.A,1.2,LUFA 6S B,Basanite,770,365,79.79944191587941
1.2.A,1.2,LUFA 6S B,Basanite,798,365,127.33262374390756
1.2.A,1.2,LUFA 6S B,Basanite,833,365,186.11448313376255
1.2.A,1.2,LUFA 6S B,Basanite,868,365,207.88999795414887
1.2.A,1.2,LUFA 6S B,Basanite,892,365,159.67106788615442
1.2.A,1.2,LUFA 6S B,Basanite,925,365,94.70544352849149
1.2.A,1.2,LUFA 6S B,Basanite,955,365,72.42464049581804
1.2.A,1.2,LUFA 6S B,Basanite,987,365,61.59703642818461
1.2.A,1.2,LUFA 6S B,Basanite,1016,365,
1.2.A,1.2,LUFA 6S B,Basanite,1043,365,
1.2.B,1.2,LUFA 6S B,Basanite,407,365,3414.304480413984
1.2.B,1.2,LUFA 6S B,Basanite,442,365,2444.6323846200135
1.2.B,1.2,LUFA 6S B,Basanite,470,365,2175.1453490583067
1.2.B,1.2,LUFA 6S B,Basanite,498,365,1695.6027958360912
1.2.B,1.2,LUFA 6S B,Basanite,525,365,856.6800332149949
1.2.B,1.2,LUFA 6S B,Basanite,560,365,1507.2024853480957
1.2.B,1.2,LUFA 6S B,Basanite,596,365,1399.7926529875444
1.2.B,1.2,LUFA 6S B,Basanite,624,365,661.9756510018653
1.2.B,1.2,LUFA 6S B,Basanite,651,365,540.2011971839461
1.2.B,1.2,LUFA 6S B,Basanite,679,365,
1.2.C,1.2,LUFA 6S B,Basanite,407,365,2858.4874709669657
1.2.C,1.2,LUFA 6S B,Basanite,442,365,2810.1241735363137
1.2.C,1.2,LUFA 6S B,Basanite,470,365,2016.8217163487575
1.2.C,1.2,LUFA 6S B,Basanite,498,365,1966.292897526927
1.2.C,1.2,LUFA 6S B,Basanite,525,365,1274.625550033095
1.2.C,1.2,LUFA 6S B,Basanite,560,365,2079.621819844756
1.2.C,1.2,LUFA 6S B,Basanite,596,365,1238.677904567062
1.2.C,1.2,LUFA 6S B,Basanite,624,365,394.1247877730309
1.2.C,1.2,LUFA 6S B,Basanite,651,365,363.0375334255972
1.2.C,1.2,LUFA 6S B,Basanite,679,365,365.11883958517166
1.2.C,1.2,LUFA 6S B,Basanite,714,365,247.0137406582827
1.2.C,1.2,LUFA 6S B,Basanite,742,365,
1.2.C,1.2,LUFA 6S B,Basanite,770,365,550.4032064504482
1.2.C,1.2,LUFA 6S B,Basanite,798,365,782.8357735122449
1.2.C,1.2,LUFA 6S B,Basanite,833,365,1236.271770383296
1.2.C,1.2,LUFA 6S B,Basanite,868,365,1313.989906011192
1.2.C,1.2,LUFA 6S B,Basanite,892,365,1022.1017620795476
1.2.C,1.2,LUFA 6S B,Basanite,925,365,670.0121393585655
1.2.C,1.2,LUFA 6S B,Basanite,955,365,622.6353561586136
1.2.C,1.2,LUFA 6S B,Basanite,987,365,363.0134720500632
1.2.C,1.2,LUFA 6S B,Basanite,1016,365,
1.2.C,1.2,LUFA 6S B,Basanite,1043,365,
1.2.D,1.2,LUFA 6S B,Basanite,407,365,1539.6852973103075
1.2.D,1.2,LUFA 6S B,Basanite,442,365,2623.6487706841567
1.2.D,1.2,LUFA 6S B,Basanite,470,365,1945.359529454239
1.2.D,1.2,LUFA 6S B,Basanite,498,365,1889.7778287502256
1.2.D,1.2,LUFA 6S B,Basanite,525,365,811.9259365786147
1.2.D,1.2,LUFA 6S B,Basanite,560,365,1369.427238943378
1.2.D,1.2,LUFA 6S B,Basanite,596,365,1171.4986367410793
1.2.D,1.2,LUFA 6S B,Basanite,624,365,478.1469953667489
1.2.D,1.2,LUFA 6S B,Basanite,651,365,300.67053420783435
1.2.D,1.2,LUFA 6S B,Basanite,679,365,
1.2.E,1.2,LUFA 6S B,Basanite,407,365,1660.23262266081
1.2.E,1.2,LUFA 6S B,Basanite,442,365,2029.814841326193
1.2.E,1.2,LUFA 6S B,Basanite,470,365,1738.9132121066248
1.2.E,1.2,LUFA 6S B,Basanite,498,365,1998.775709489139
1.2.E,1.2,LUFA 6S B,Basanite,525,365,814.8132975509958
1.2.E,1.2,LUFA 6S B,Basanite,560,365,1407.9494479812265
1.2.E,1.2,LUFA 6S B,Basanite,596,365,1054.849248931945
1.2.E,1.2,LUFA 6S B,Basanite,624,365,525.4997171911667
1.2.E,1.2,LUFA 6S B,Basanite,651,365,371.45900330946506
1.2.E,1.2,LUFA 6S B,Basanite,679,365,
1.2.F,1.2,LUFA 6S B,Basanite,407,365,1842.136370900776
1.2.F,1.2,LUFA 6S B,Basanite,442,365,2822.3954582104816
1.2.F,1.2,LUFA 6S B,Basanite,470,365,1842.136370900776
1.2.F,1.2,LUFA 6S B,Basanite,498,365,1609.2225768096755
1.2.F,1.2,LUFA 6S B,Basanite,525,365,666.5473058547445
1.2.G,1.2,LUFA 6S B,Basanite,407,365,2269.4658111799745
1.2.G,1.2,LUFA 6S B,Basanite,442,365,2519.70377279018
1.2.G,1.2,LUFA 6S B,Basanite,470,365,2080.1030468740596
1.2.G,1.2,LUFA 6S B,Basanite,498,365,2130.3912521812385
1.2.G,1.2,LUFA 6S B,Basanite,525,365,1048.112073169264
1.2.G,1.2,LUFA 6S B,Basanite,560,365,1246.0887978819424
1.2.G,1.2,LUFA 6S B,Basanite,596,365,941.544387748962
1.2.G,1.2,LUFA 6S B,Basanite,624,365,651.8217644864311
1.2.G,1.2,LUFA 6S B,Basanite,651,365,435.34186726036467
1.2.G,1.2,LUFA 6S B,Basanite,679,365,427.3294403271065
1.2.G,1.2,LUFA 6S B,Basanite,714,365,391.0930587881341
1.2.G,1.2,LUFA 6S B,Basanite,742,365,
1.2.G,1.2,LUFA 6S B,Basanite,770,365,763.2257796497984
1.2.G,1.2,LUFA 6S B,Basanite,798,365,1125.637718033576
1.2.G,1.2,LUFA 6S B,Basanite,833,365,1196.185573861243
1.2.G,1.2,LUFA 6S B,Basanite,868,365,1228.5721408026957
1.2.G,1.2,LUFA 6S B,Basanite,892,365,1190.8920787050963
1.2.G,1.2,LUFA 6S B,Basanite,925,365,873.1379914555629
1.2.G,1.2,LUFA 6S B,Basanite,955,365,1019.5993823936456
1.2.G,1.2,LUFA 6S B,Basanite,987,365,1154.58351308743
1.2.G,1.2,LUFA 6S B,Basanite,1016,365,
1.2.G,1.2,LUFA 6S B,Basanite,1043,365,
1.7.A,1.7,LUFA 6S B,Peridotite,407,365,2397.231539081774
1.7.A,1.7,LUFA 6S B,Peridotite,442,365,2770.663573018834
1.7.A,1.7,LUFA 6S B,Peridotite,470,365,2087.5620629400087
1.7.A,1.7,LUFA 6S B,Peridotite,498,365,2052.9137298273063
1.7.A,1.7,LUFA 6S B,Peridotite,525,365,868.0129254467778
1.7.A,1.7,LUFA 6S B,Peridotite,560,365,1274.625550033095
1.7.A,1.7,LUFA 6S B,Peridotite,596,365,1000.83153523076
1.7.A,1.7,LUFA 6S B,Peridotite,624,365,480.74562031409835
1.7.A,1.7,LUFA 6S B,Peridotite,651,365,357.8402835308984
1.7.A,1.7,LUFA 6S B,Peridotite,679,365,368.8603782915292
1.7.A,1.7,LUFA 6S B,Peridotite,714,365,430.2168012515796
1.7.A,1.7,LUFA 6S B,Peridotite,742,365,
1.7.A,1.7,LUFA 6S B,Peridotite,770,365,532.212831578314
1.7.A,1.7,LUFA 6S B,Peridotite,798,365,1004.4407367470968
1.7.A,1.7,LUFA 6S B,Peridotite,833,365,763.0814113965943
1.7.A,1.7,LUFA 6S B,Peridotite,868,365,430.722089415729
1.7.A,1.7,LUFA 6S B,Peridotite,892,365,426.6797841025332
1.7.A,1.7,LUFA 6S B,Peridotite,925,365,387.7725935375173
1.7.A,1.7,LUFA 6S B,Peridotite,955,365,289.60231662554907
1.7.A,1.7,LUFA 6S B,Peridotite,987,365,177.86144271015104
1.7.A,1.7,LUFA 6S B,Peridotite,1016,365,
1.7.A,1.7,LUFA 6S B,Peridotite,1043,365,
1.7.B,1.7,LUFA 6S B,Peridotite,407,365,873.4267274805945
1.7.B,1.7,LUFA 6S B,Peridotite,442,365,1165.772037306697
1.7.B,1.7,LUFA 6S B,Peridotite,470,365,1003.3579762921958
1.7.B,1.7,LUFA 6S B,Peridotite,498,365,1096.234757807329
1.7.B,1.7,LUFA 6S B,Peridotite,525,365,620.5901421264817
1.7.B,1.7,LUFA 6S B,Peridotite,560,365,1055.6913958721943
1.7.B,1.7,LUFA 6S B,Peridotite,596,365,1024.7244484024309
1.7.B,1.7,LUFA 6S B,Peridotite,624,365,466.3088149708165
1.7.B,1.7,LUFA 6S B,Peridotite,651,365,511.1591571093327
1.7.B,1.7,LUFA 6S B,Peridotite,679,365,
1.7.C,1.7,LUFA 6S B,Peridotite,407,365,2225.6741681208255
1.7.C,1.7,LUFA 6S B,Peridotite,442,365,2318.310336121307
1.7.C,1.7,LUFA 6S B,Peridotite,470,365,1597.9137459534268
1.7.C,1.7,LUFA 6S B,Peridotite,498,365,1520.1956100848429
1.7.C,1.7,LUFA 6S B,Peridotite,525,365,1317.3584940128767
1.7.C,1.7,LUFA 6S B,Peridotite,560,365,1625.4399215355918
1.7.C,1.7,LUFA 6S B,Peridotite,596,365,1064.7625219327276
1.7.C,1.7,LUFA 6S B,Peridotite,624,365,322.37386485348094
1.7.C,1.7,LUFA 6S B,Peridotite,651,365,358.51400108309764
1.7.C,1.7,LUFA 6S B,Peridotite,679,365,280.61540519886876
1.7.C,1.7,LUFA 6S B,Peridotite,714,365,596.3844315542451
1.7.C,1.7,LUFA 6S B,Peridotite,742,365,
1.7.C,1.7,LUFA 6S B,Peridotite,770,365,512.9878190023467
1.7.C,1.7,LUFA 6S B,Peridotite,798,365,385.36645911306334
1.7.C,1.7,LUFA 6S B,Peridotite,833,365,646.2154716890307
1.7.C,1.7,LUFA 6S B,Peridotite,868,365,767.3643304651303
1.7.C,1.7,LUFA 6S B,Peridotite,892,365,802.9269944039954
1.7.C,1.7,LUFA 6S B,Peridotite,925,365,985.9616258499308
1.7.C,1.7,LUFA 6S B,Peridotite,955,365,634.449475179012
1.7.C,1.7,LUFA 6S B,Peridotite,987,365,471.5060648655153
1.7.C,1.7,LUFA 6S B,Peridotite,1016,365,
1.7.C,1.7,LUFA 6S B,Peridotite,1043,365,
1.7.D,1.7,LUFA 6S B,Peridotite,407,365,2151.5652335278896
1.7.D,1.7,LUFA 6S B,Peridotite,442,365,2829.613861243155
1.7.D,1.7,LUFA 6S B,Peridotite,470,365,2069.756669595042
1.7.D,1.7,LUFA 6S B,Peridotite,498,365,1940.78787460136
1.7.D,1.7,LUFA 6S B,Peridotite,525,365,976.8904997893976
1.7.D,1.7,LUFA 6S B,Peridotite,560,365,1666.969798423491
1.7.D,1.7,LUFA 6S B,Peridotite,596,365,1075.2532672242614
1.7.D,1.7,LUFA 6S B,Peridotite,624,365,181.6631347975209
1.7.D,1.7,LUFA 6S B,Peridotite,651,365,310.87254323364823
1.7.D,1.7,LUFA 6S B,Peridotite,679,365,
1.7.E,1.7,LUFA 6S B,Peridotite,407,365,1381.60227811541
1.7.E,1.7,LUFA 6S B,Peridotite,442,365,2465.806363800469
1.7.E,1.7,LUFA 6S B,Peridotite,470,365,1721.5890455502736
1.7.E,1.7,LUFA 6S B,Peridotite,498,365,1559.1749847764606
1.7.E,1.7,LUFA 6S B,Peridotite,525,365,704.0348774294482
1.7.E,1.7,LUFA 6S B,Peridotite,560,365,830.3569247247126
1.7.E,1.7,LUFA 6S B,Peridotite,596,365,1210.3095818039592
1.7.E,1.7,LUFA 6S B,Peridotite,624,365,478.0026273542331
1.7.E,1.7,LUFA 6S B,Peridotite,651,365,485.4616434201817
1.7.E,1.7,LUFA 6S B,Peridotite,679,365,
1.7.F,1.7,LUFA 6S B,Peridotite,407,365,2012.731288043805
1.7.F,1.7,LUFA 6S B,Peridotite,442,365,2456.4224417835007
1.7.F,1.7,LUFA 6S B,Peridotite,470,365,2012.490674769842
1.7.F,1.7,LUFA 6S B,Peridotite,498,365,1610.1850306275946
1.7.F,1.7,LUFA 6S B,Peridotite,525,365,702.2062152957458
1.7.G,1.7,LUFA 6S B,Peridotite,407,365,1812.540919670257
1.7.G,1.7,LUFA 6S B,Peridotite,442,365,2697.2764787291653
1.7.G,1.7,LUFA 6S B,Peridotite,470,365,1737.4695315000904
1.7.G,1.7,LUFA 6S B,Peridotite,498,365,2273.7968527588905
1.7.G,1.7,LUFA 6S B,Peridotite,525,365,905.1876993802276
1.7.G,1.7,LUFA 6S B,Peridotite,560,365,1582.5626095432938
1.7.G,1.7,LUFA 6S B,Peridotite,596,365,1244.163890486792
1.7.G,1.7,LUFA 6S B,Peridotite,624,365,537.0491613213791
1.7.G,1.7,LUFA 6S B,Peridotite,651,365,371.45900330946506
1.7.G,1.7,LUFA 6S B,Peridotite,679,365,212.0044874849812
1.7.G,1.7,LUFA 6S B,Peridotite,714,365,388.8312923761959
1.7.G,1.7,LUFA 6S B,Peridotite,742,365,
1.7.G,1.7,LUFA 6S B,Peridotite,770,365,639.1895596606294
1.7.G,1.7,LUFA 6S B,Peridotite,798,365,454.61500258739994
1.7.G,1.7,LUFA 6S B,Peridotite,833,365,217.49047355436548
1.7.G,1.7,LUFA 6S B,Peridotite,868,365,
1.7.G,1.7,LUFA 6S B,Peridotite,892,365,1563.3616583428604
1.7.G,1.7,LUFA 6S B,Peridotite,925,365,978.815407184548
1.7.G,1.7,LUFA 6S B,Peridotite,955,365,957.1601990492808
1.7.G,1.7,LUFA 6S B,Peridotite,987,365,1164.809583488778
1.7.G,1.7,LUFA 6S B,Peridotite,1016,365,
1.7.G,1.7,LUFA 6S B,Peridotite,1043,365,
1.8.A,1.8,LUFA 6S B,Steel Slag,407,365,2210.0342954449725
1.8.A,1.8,LUFA 6S B,Steel Slag,442,365,3392.6492713159637
1.8.A,1.8,LUFA 6S B,Steel Slag,470,365,2691.50175582165
1.8.A,1.8,LUFA 6S B,Steel Slag,498,365,2765.1294638666586
1.8.A,1.8,LUFA 6S B,Steel Slag,525,365,747.10468018533
1.8.A,1.8,LUFA 6S B,Steel Slag,560,365,1869.061013057344
1.8.A,1.8,LUFA 6S B,Steel Slag,596,365,1614.1070293038088
1.8.A,1.8,LUFA 6S B,Steel Slag,624,365,417.0793085023166
1.8.A,1.8,LUFA 6S B,Steel Slag,651,365,343.0906807870509
1.8.A,1.8,LUFA 6S B,Steel Slag,679,365,222.8320915889453
1.8.A,1.8,LUFA 6S B,Steel Slag,714,365,466.8862870208797
1.8.A,1.8,LUFA 6S B,Steel Slag,742,365,
1.8.A,1.8,LUFA 6S B,Steel Slag,770,365,846.7186374631447
1.8.A,1.8,LUFA 6S B,Steel Slag,798,365,977.1792358144291
1.8.A,1.8,LUFA 6S B,Steel Slag,833,365,605.3593123533304
1.8.A,1.8,LUFA 6S B,Steel Slag,868,365,1131.4846242759113
1.8.A,1.8,LUFA 6S B,Steel Slag,892,365,477.35297093687944
1.8.A,1.8,LUFA 6S B,Steel Slag,925,365,386.37703568205063
1.8.A,1.8,LUFA 6S B,Steel Slag,955,365,483.7292267886154
1.8.A,1.8,LUFA 6S B,Steel Slag,987,365,509.76359925386606
1.8.A,1.8,LUFA 6S B,Steel Slag,1016,365,
1.8.A,1.8,LUFA 6S B,Steel Slag,1043,365,
1.8.B,1.8,LUFA 6S B,Steel Slag,407,365,1697.7683167458931
1.8.B,1.8,LUFA 6S B,Steel Slag,442,365,3308.434574884169
1.8.B,1.8,LUFA 6S B,Steel Slag,470,365,2425.383310668512
1.8.B,1.8,LUFA 6S B,Steel Slag,498,365,2357.7709376015405
1.8.B,1.8,LUFA 6S B,Steel Slag,525,365,799.4862225163969
1.8.B,1.8,LUFA 6S B,Steel Slag,560,365,1600.7529844154278
1.8.B,1.8,LUFA 6S B,Steel Slag,596,365,1636.5562618689453
1.8.B,1.8,LUFA 6S B,Steel Slag,624,365,914.0663349178652
1.8.B,1.8,LUFA 6S B,Steel Slag,651,365,711.8307522714964
1.8.B,1.8,LUFA 6S B,Steel Slag,679,365,
1.8.C,1.8,LUFA 6S B,Steel Slag,407,365,2240.1109734641072
1.8.C,1.8,LUFA 6S B,Steel Slag,442,365,3250.4467392743245
1.8.C,1.8,LUFA 6S B,Steel Slag,470,365,2286.549364221674
1.8.C,1.8,LUFA 6S B,Steel Slag,498,365,2043.2891928515555
1.8.C,1.8,LUFA 6S B,Steel Slag,525,365,1900.6776167037729
1.8.C,1.8,LUFA 6S B,Steel Slag,560,365,2215.929324267405
1.8.C,1.8,LUFA 6S B,Steel Slag,596,365,1715.381219327276
1.8.C,1.8,LUFA 6S B,Steel Slag,624,365,649.6562435766292
1.8.C,1.8,LUFA 6S B,Steel Slag,651,365,233.8040636861423
1.8.C,1.8,LUFA 6S B,Steel Slag,679,365,242.15334950456128
1.8.C,1.8,LUFA 6S B,Steel Slag,714,365,448.14250147421626
1.8.C,1.8,LUFA 6S B,Steel Slag,742,365,
1.8.C,1.8,LUFA 6S B,Steel Slag,770,365,260.4399696732655
1.8.C,1.8,LUFA 6S B,Steel Slag,798,365,281.61395101991695
1.8.C,1.8,LUFA 6S B,Steel Slag,833,365,331.1081321379144
1.8.C,1.8,LUFA 6S B,Steel Slag,868,365,438.5179644984656
1.8.C,1.8,LUFA 6S B,Steel Slag,892,365,375.9344129008966
1.8.C,1.8,LUFA 6S B,Steel Slag,925,365,471.5060648655153
1.8.C,1.8,LUFA 6S B,Steel Slag,955,365,265.63721956796434
1.8.C,1.8,LUFA 6S B,Steel Slag,987,365,214.8196645526205
1.8.C,1.8,LUFA 6S B,Steel Slag,1016,365,
1.8.C,1.8,LUFA 6S B,Steel Slag,1043,365,
1.8.D,1.8,LUFA 6S B,Steel Slag,407,365,2339.724930741922
1.8.D,1.8,LUFA 6S B,Steel Slag,442,365,2767.5355990131775
1.8.D,1.8,LUFA 6S B,Steel Slag,470,365,2413.833864853481
1.8.D,1.8,LUFA 6S B,Steel Slag,498,365,2487.94280040917
1.8.D,1.8,LUFA 6S B,Steel Slag,525,365,1358.214653348577
1.8.D,1.8,LUFA 6S B,Steel Slag,560,365,1954.1419194897403
1.8.D,1.8,LUFA 6S B,Steel Slag,596,365,1389.783134484626
1.8.D,1.8,LUFA 6S B,Steel Slag,624,365,258.2022648775498
1.8.D,1.8,LUFA 6S B,Steel Slag,651,365,210.70517501654732
1.8.D,1.8,LUFA 6S B,Steel Slag,679,365,226.5375382819723
1.8.D,1.8,LUFA 6S B,Steel Slag,714,365,184.55049586617724
1.8.D,1.8,LUFA 6S B,Steel Slag,742,365,
1.8.D,1.8,LUFA 6S B,Steel Slag,770,365,1383.1903267344603
1.8.D,1.8,LUFA 6S B,Steel Slag,798,365,1093.539887357843
1.8.D,1.8,LUFA 6S B,Steel Slag,833,365,909.711231722727
1.8.D,1.8,LUFA 6S B,Steel Slag,868,365,2336.187913352187
1.8.D,1.8,LUFA 6S B,Steel Slag,892,365,1648.442564775257
1.8.D,1.8,LUFA 6S B,Steel Slag,925,365,583.5837975810819
1.8.D,1.8,LUFA 6S B,Steel Slag,955,365,788.0089622720982
1.8.D,1.8,LUFA 6S B,Steel Slag,987,365,1283.1432651784105
1.8.D,1.8,LUFA 6S B,Steel Slag,1016,365,
1.8.D,1.8,LUFA 6S B,Steel Slag,1043,365,
1.8.E,1.8,LUFA 6S B,Steel Slag,407,365,1973.030073289608
1.8.E,1.8,LUFA 6S B,Steel Slag,442,365,3462.9083915999754
1.8.E,1.8,LUFA 6S B,Steel Slag,470,365,2826.0046597268188
1.8.E,1.8,LUFA 6S B,Steel Slag,498,365,1970.6239391058427
1.8.E,1.8,LUFA 6S B,Steel Slag,525,365,900.1588788735784
1.8.F,1.8,LUFA 6S B,Steel Slag,407,365,1819.0374821589744
1.8.F,1.8,LUFA 6S B,Steel Slag,442,365,2607.287056982971
1.8.F,1.8,LUFA 6S B,Steel Slag,470,365,2265.3753828750228
1.8.F,1.8,LUFA 6S B,Steel Slag,498,365,2407.096689331488
1.8.F,1.8,LUFA 6S B,Steel Slag,525,365,1049.2670175100789
1.8.F,1.8,LUFA 6S B,Steel Slag,560,365,1829.672595463024
1.8.F,1.8,LUFA 6S B,Steel Slag,596,365,1594.0158084120585
1.8.F,1.8,LUFA 6S B,Steel Slag,624,365,642.4859635357121
1.8.F,1.8,LUFA 6S B,Steel Slag,651,365,339.2649271315963
1.8.F,1.8,LUFA 6S B,Steel Slag,679,365,
1.8.G,1.8,LUFA 6S B,Steel Slag,407,365,1798.344727841627
1.8.G,1.8,LUFA 6S B,Steel Slag,442,365,3350.0606967928275
1.8.G,1.8,LUFA 6S B,Steel Slag,470,365,2105.367456284975
1.8.G,1.8,LUFA 6S B,Steel Slag,498,365,2555.314558035983
1.8.G,1.8,LUFA 6S B,Steel Slag,525,365,785.7471961008484
1.8.G,1.8,LUFA 6S B,Steel Slag,560,365,2392.6598839882063
1.8.G,1.8,LUFA 6S B,Steel Slag,596,365,1710.7614414826403
1.8.G,1.8,LUFA 6S B,Steel Slag,624,365,485.1488460196161
1.8.G,1.8,LUFA 6S B,Steel Slag,651,365,718.0145173596486
1.8.G,1.8,LUFA 6S B,Steel Slag,679,365,
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,406,365,234.59808797159877
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,442,365,509.37861772669834
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,470,365,415.77999590829774
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,498,365,422.2765583970155
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,525,365,185.51294956375233
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,560,365,361.1607487815151
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,596,365,454.2781438113003
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,624,365,289.81886876466695
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,651,365,663.9486809073951
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,679,365,448.69591238943383
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,714,365,257.6969767134003
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,742,365,
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,770,365,60.76451399001143
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,798,365,58.54605822251639
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,833,365,58.51718462001323
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,868,365,141.7838659847163
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,892,365,106.11051979060112
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,925,365,21.145107662314217
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,955,365,44.17662456224803
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,987,365,52.453726337324746
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,1016,365,
1.9.A,0.A,LUFA 2.2 B,Glacial Dust,1043,365,
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,406,365,251.63351826223
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,442,365,990.9423236055115
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,470,365,754.5636962512787
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,498,365,488.4452498946988
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,525,365,161.21099380227452
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,560,365,404.2810804500873
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,596,365,230.74827320536735
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,624,365,238.59227082255248
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,651,365,323.5769319453637
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,679,365,338.35059630543356
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,714,365,259.38127059389853
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,742,365,
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,770,365,775.1698299536674
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,798,365,420.712571153499
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,833,365,205.72447714062216
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,868,365,193.80448814008065
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,892,365,154.2187676996209
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,925,365,56.770331163126535
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,955,365,67.51612665021962
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,987,365,101.85166219387447
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,1016,365,
1.9.B,0.A,LUFA 2.2 B,Glacial Dust,1043,365,
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,406,365,181.90374821589745
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,442,365,581.0814178951802
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,470,365,532.7181197424635
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,498,365,358.0327742944822
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,525,365,278.05287225464826
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,560,365,544.8690977796498
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,596,365,620.205160599314
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,624,365,553.434935435345
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,651,365,478.1469953667489
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,679,365,483.9217175521993
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,714,365,269.00580756964916
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,742,365,
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,770,365,217.88026730850228
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,798,365,228.49131928515553
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,833,365,627.7556098441543
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,868,365,472.67063385281904
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,892,365,184.88735466634577
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,925,365,239.7664643119321
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,955,365,197.89972862386423
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,987,365,236.47487268788737
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,1016,365,
1.9.C,0.A,LUFA 2.2 B,Glacial Dust,1043,365,
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,406,365,145.5229985679042
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,442,365,441.76624562248026
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,470,365,808.4611030747939
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,498,365,664.0930491605993
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,525,365,336.16582634334196
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,560,365,455.7699469282147
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,596,365,398.45582935194653
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,624,365,361.8585277092485
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,651,365,298.28846115891446
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,679,365,308.73108369938024
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,714,365,117.10895938383776
1.9.D,0.A,LUFA 2.2 B,Glacial Dust,742,365,
2.0.A,2.0,Fürth 2A,Control,92,65,94.32046204946148
2.0.A,2.0,Fürth 2A,Control,115,65,232.09570838197243
2.0.A,2.0,Fürth 2A,Control,150,65,0.0
2.0.A,2.0,Fürth 2A,Control,177,65,317.6097189963295
2.0.A,2.0,Fürth 2A,Control,198,65,340.1311354473795
2.0.A,2.0,Fürth 2A,Control,228,65,108.75726746494976
2.0.A,2.0,Fürth 2A,Control,259,65,218.28449786389072
2.0.A,2.0,Fürth 2A,Control,284,65,370.54467224261384
2.0.A,2.0,Fürth 2A,Control,315,65,293.7408674408809
2.0.A,2.0,Fürth 2A,Control,351,65,363.1337789277333
2.0.A,2.0,Fürth 2A,Control,385,65,271.55630976593056
2.0.A,2.0,Fürth 2A,Control,410,65,233.8281250135387
2.0.A,2.0,Fürth 2A,Control,448,65,312.79745062879834
2.0.A,2.0,Fürth 2A,Control,471,65,209.42992387026896
2.0.A,2.0,Fürth 2A,Control,499,65,319.96773066971537
2.0.A,2.0,Fürth 2A,Control,528,65,104.78233369035442
2.0.A,2.0,Fürth 2A,Control,560,65,428.0657173284854
2.0.A,2.0,Fürth 2A,Control,595,65,375.8742596228871
2.0.A,2.0,Fürth 2A,Control,626,65,215.56556618328415
2.0.A,2.0,Fürth 2A,Control,652,65,176.71612280670715
2.0.A,2.0,Fürth 2A,Control,680,65,196.77365778927737
2.0.A,2.0,Fürth 2A,Control,714,65,290.5840194957579
2.0.A,2.0,Fürth 2A,Control,742,65,
2.0.A,2.0,Fürth 2A,Control,770,65,243.9820114326975
2.0.A,2.0,Fürth 2A,Control,798,65,192.5148001925507
2.0.A,2.0,Fürth 2A,Control,833,65,229.06397922859375
2.0.A,2.0,Fürth 2A,Control,868,65,361.7622824478007
2.0.A,2.0,Fürth 2A,Control,892,65,297.07095733798667
2.0.A,2.0,Fürth 2A,Control,925,65,231.82622133702387
2.0.A,2.0,Fürth 2A,Control,955,65,277.74969925988324
2.0.A,2.0,Fürth 2A,Control,987,65,229.49227113544737
2.0.A,2.0,Fürth 2A,Control,1016,65,
2.0.A,2.0,Fürth 2A,Control,1043,65,
2.0.B,2.0,Fürth 2A,Control,92,65,96.0047560021662
2.0.B,2.0,Fürth 2A,Control,115,65,260.1512336482339
2.0.B,2.0,Fürth 2A,Control,150,65,0.0
2.0.B,2.0,Fürth 2A,Control,177,65,444.6536068355497
2.0.B,2.0,Fürth 2A,Control,198,65,354.18295950418195
2.0.B,2.0,Fürth 2A,Control,228,65,94.41670740718456
2.0.B,2.0,Fürth 2A,Control,259,65,210.199886828329
2.0.B,2.0,Fürth 2A,Control,284,65,282.14330055960045
2.0.B,2.0,Fürth 2A,Control,315,65,285.2712750466334
2.0.B,2.0,Fürth 2A,Control,351,65,309.81384415428124
2.0.B,2.0,Fürth 2A,Control,385,65,222.32680336963716
2.0.B,2.0,Fürth 2A,Control,410,65,268.9095620675131
2.0.B,2.0,Fürth 2A,Control,448,65,337.82124676575006
2.0.B,2.0,Fürth 2A,Control,471,65,224.34795614657924
2.0.B,2.0,Fürth 2A,Control,499,65,321.363288525182
2.0.B,2.0,Fürth 2A,Control,528,65,102.154835116433
2.0.B,2.0,Fürth 2A,Control,560,65,379.1105101811275
2.0.B,2.0,Fürth 2A,Control,595,65,367.4455714063235
2.0.B,2.0,Fürth 2A,Control,626,65,218.1016316505205
2.0.B,2.0,Fürth 2A,Control,652,65,177.32728089466423
2.0.B,2.0,Fürth 2A,Control,680,65,199.28566192911725
2.0.C,2.0,Fürth 2A,Control,92,65,90.85562873819124
2.0.C,2.0,Fürth 2A,Control,115,65,247.8318262229978
2.0.C,2.0,Fürth 2A,Control,150,65,0.0
2.0.C,2.0,Fürth 2A,Control,177,65,627.5679314038149
2.0.C,2.0,Fürth 2A,Control,198,65,391.0930587881341
2.0.C,2.0,Fürth 2A,Control,228,65,187.10099815873397
2.0.C,2.0,Fürth 2A,Control,259,65,285.84874709669657
2.0.C,2.0,Fürth 2A,Control,284,65,228.58275239184064
2.0.C,2.0,Fürth 2A,Control,315,65,350.33314471388167
2.0.C,2.0,Fürth 2A,Control,351,65,437.2427132799808
2.0.C,2.0,Fürth 2A,Control,385,65,280.55525194054997
2.0.C,2.0,Fürth 2A,Control,410,65,266.35905987123175
2.0.C,2.0,Fürth 2A,Control,448,65,362.8450426620133
2.0.C,2.0,Fürth 2A,Control,471,65,212.70226644202415
2.0.C,2.0,Fürth 2A,Control,499,65,355.289781334617
2.0.C,2.0,Fürth 2A,Control,528,65,266.4553053733678
2.0.C,2.0,Fürth 2A,Control,560,65,338.19660371192606
2.0.C,2.0,Fürth 2A,Control,595,65,309.12087760094596
2.0.C,2.0,Fürth 2A,Control,626,65,224.86768113604907
2.0.C,2.0,Fürth 2A,Control,652,65,176.87011540031722
2.0.C,2.0,Fürth 2A,Control,680,65,209.43473614537575
2.0.C,2.0,Fürth 2A,Control,714,65,263.1540891750406
2.0.C,2.0,Fürth 2A,Control,742,65,
2.0.C,2.0,Fürth 2A,Control,770,65,208.10655004512907
2.0.C,2.0,Fürth 2A,Control,798,65,124.40676452253444
2.0.C,2.0,Fürth 2A,Control,833,65,151.17741402009747
2.0.C,2.0,Fürth 2A,Control,868,65,483.3923680125158
2.0.C,2.0,Fürth 2A,Control,892,65,183.66022619892897
2.0.C,2.0,Fürth 2A,Control,925,65,228.6549364221674
2.0.C,2.0,Fürth 2A,Control,955,65,94.6573208255611
2.0.C,2.0,Fürth 2A,Control,987,65,75.8413511041579
2.0.C,2.0,Fürth 2A,Control,1016,65,
2.0.C,2.0,Fürth 2A,Control,1043,65,
2.0.D,2.0,Fürth 2A,Control,92,65,87.58328619050484
2.0.D,2.0,Fürth 2A,Control,115,65,304.3278582345508
2.0.D,2.0,Fürth 2A,Control,150,65,0.0
2.0.D,2.0,Fürth 2A,Control,177,65,597.2025173596486
2.0.D,2.0,Fürth 2A,Control,198,65,418.8598476442626
2.0.D,2.0,Fürth 2A,Control,228,65,146.10047078644925
2.0.D,2.0,Fürth 2A,Control,259,65,437.38708129249653
2.0.D,2.0,Fürth 2A,Control,284,65,122.424109922378
2.0.D,2.0,Fürth 2A,Control,315,65,316.2622838919309
2.0.D,2.0,Fürth 2A,Control,351,65,349.75567242313014
2.0.D,2.0,Fürth 2A,Control,385,65,284.7419255069499
2.0.D,2.0,Fürth 2A,Control,410,65,380.74668150911606
2.0.D,2.0,Fürth 2A,Control,448,65,291.0459972320837
2.0.D,2.0,Fürth 2A,Control,471,65,224.92542836512425
2.0.D,2.0,Fürth 2A,Control,499,65,333.4902051868343
2.0.D,2.0,Fürth 2A,Control,528,65,251.24853697575065
2.0.D,2.0,Fürth 2A,Control,560,65,409.4927672444849
2.0.D,2.0,Fürth 2A,Control,595,65,338.2543509153442
2.0.D,2.0,Fürth 2A,Control,626,65,220.238278861544
2.0.D,2.0,Fürth 2A,Control,652,65,209.14600004022532
2.0.D,2.0,Fürth 2A,Control,680,65,228.10152555508756
2.1.A,2.1,Fürth 2A,EW 1,92,65,107.79481376737468
2.1.A,2.1,Fürth 2A,EW 1,115,65,382.6715889042662
2.1.A,2.1,Fürth 2A,EW 1,150,65,0.0
2.1.A,2.1,Fürth 2A,EW 1,177,65,743.0142518803779
2.1.A,2.1,Fürth 2A,EW 1,198,65,418.8598476442626
2.1.A,2.1,Fürth 2A,EW 1,228,65,240.37281015704917
2.1.A,2.1,Fürth 2A,EW 1,259,65,316.2622838919309
2.1.A,2.1,Fürth 2A,EW 1,284,65,314.52986726036465
2.1.A,2.1,Fürth 2A,EW 1,315,65,342.87412864793305
2.1.A,2.1,Fürth 2A,EW 1,351,65,402.97936169444614
2.1.A,2.1,Fürth 2A,EW 1,385,65,296.43573789036645
2.1.A,2.1,Fürth 2A,EW 1,410,65,270.2569974126
2.1.A,2.1,Fürth 2A,EW 1,448,65,356.5890936879475
2.1.A,2.1,Fürth 2A,EW 1,471,65,237.3410810036705
2.1.A,2.1,Fürth 2A,EW 1,499,65,365.7324038750827
2.1.A,2.1,Fürth 2A,EW 1,528,65,131.54817093687947
2.1.A,2.1,Fürth 2A,EW 1,560,65,424.70675399084774
2.1.A,2.1,Fürth 2A,EW 1,595,65,383.56185855578184
2.1.A,2.1,Fürth 2A,EW 1,626,65,247.73558096155003
2.1.A,2.1,Fürth 2A,EW 1,652,65,195.9074494903142
2.1.A,2.1,Fürth 2A,EW 1,680,65,218.85715780732895
2.1.B,2.1,Fürth 2A,EW 1,92,65,102.02009158192428
2.1.B,2.1,Fürth 2A,EW 1,115,65,392.2480031289488
2.1.B,2.1,Fürth 2A,EW 1,150,65,0.0
2.1.B,2.1,Fürth 2A,EW 1,177,65,713.9000276791623
2.1.B,2.1,Fürth 2A,EW 1,198,65,449.65836596666463
2.1.B,2.1,Fürth 2A,EW 1,228,65,124.34901729345928
2.1.B,2.1,Fürth 2A,EW 1,259,65,304.3278582345508
2.1.B,2.1,Fürth 2A,EW 1,284,65,257.9375899873638
2.1.B,2.1,Fürth 2A,EW 1,315,65,394.6541373127144
2.1.B,2.1,Fürth 2A,EW 1,351,65,452.4013589265299
2.1.B,2.1,Fürth 2A,EW 1,385,65,320.9301842469463
2.1.B,2.1,Fürth 2A,EW 1,410,65,279.11157133401525
2.1.B,2.1,Fürth 2A,EW 1,448,65,404.23055153739693
2.1.B,2.1,Fürth 2A,EW 1,471,65,237.7260624827005
2.1.B,2.1,Fürth 2A,EW 1,499,65,382.91220217822973
2.1.B,2.1,Fürth 2A,EW 1,528,65,118.17247071424272
2.1.B,2.1,Fürth 2A,EW 1,560,65,442.9091594291701
2.1.B,2.1,Fürth 2A,EW 1,595,65,411.3069923780076
2.1.B,2.1,Fürth 2A,EW 1,626,65,298.841872074132
2.1.B,2.1,Fürth 2A,EW 1,652,65,261.111281123382
2.1.B,2.1,Fürth 2A,EW 1,680,65,270.6323542932788
2.1.C,2.1,Fürth 2A,EW 1,92,65,100.57641104759612
2.1.C,2.1,Fürth 2A,EW 1,115,65,341.4304480413984
2.1.C,2.1,Fürth 2A,EW 1,150,65,0.0
2.1.C,2.1,Fürth 2A,EW 1,177,65,767.0274716890307
2.1.C,2.1,Fürth 2A,EW 1,198,65,644.3627484204826
2.1.C,2.1,Fürth 2A,EW 1,228,65,314.52986726036465
2.1.C,2.1,Fürth 2A,EW 1,259,65,410.2940099885673
2.1.C,2.1,Fürth 2A,EW 1,284,65,372.469579637764
2.1.C,2.1,Fürth 2A,EW 1,315,65,382.38285263854624
2.1.C,2.1,Fürth 2A,EW 1,351,65,394.990996088814
2.1.C,2.1,Fürth 2A,EW 1,385,65,312.6049598652145
2.1.C,2.1,Fürth 2A,EW 1,410,65,239.2659883988206
2.1.C,2.1,Fürth 2A,EW 1,448,65,350.33314471388167
2.1.C,2.1,Fürth 2A,EW 1,471,65,237.48544906432397
2.1.C,2.1,Fürth 2A,EW 1,499,65,354.6641862927974
2.1.C,2.1,Fürth 2A,EW 1,528,65,191.82183353992417
2.1.C,2.1,Fürth 2A,EW 1,560,65,346.0983483881871
2.1.C,2.1,Fürth 2A,EW 1,595,65,353.9134723840494
2.1.C,2.1,Fürth 2A,EW 1,626,65,241.88386232625305
2.1.C,2.1,Fürth 2A,EW 1,652,65,219.1507061573149
2.1.C,2.1,Fürth 2A,EW 1,680,65,241.72986990793672
2.1.D,2.1,Fürth 2A,EW 1,92,65,80.17239273121126
2.1.D,2.1,Fürth 2A,EW 1,115,65,325.5499620915819
2.1.D,2.1,Fürth 2A,EW 1,150,65,0.0
2.1.D,2.1,Fürth 2A,EW 1,177,65,728.0962197484806
2.1.D,2.1,Fürth 2A,EW 1,198,65,412.8926349359168
2.1.D,2.1,Fürth 2A,EW 1,228,65,157.6017924303508
2.1.D,2.1,Fürth 2A,EW 1,259,65,295.66577483603106
2.1.D,2.1,Fürth 2A,EW 1,284,65,335.5113578434322
2.1.D,2.1,Fürth 2A,EW 1,315,65,343.59596895120046
2.1.D,2.1,Fürth 2A,EW 1,351,65,382.91220217822973
2.1.D,2.1,Fürth 2A,EW 1,385,65,249.94922438173177
2.1.D,2.1,Fürth 2A,EW 1,410,65,220.9793682171009
2.1.D,2.1,Fürth 2A,EW 1,448,65,336.85879294783075
2.1.D,2.1,Fürth 2A,EW 1,471,65,230.41141440519883
2.1.D,2.1,Fürth 2A,EW 1,499,65,362.07507984836633
2.1.D,2.1,Fürth 2A,EW 1,528,65,262.3456280161261
2.1.D,2.1,Fürth 2A,EW 1,560,65,418.5807361961842
2.1.D,2.1,Fürth 2A,EW 1,595,65,367.70062164602143
2.1.D,2.1,Fürth 2A,EW 1,626,65,229.06397922859375
2.1.D,2.1,Fürth 2A,EW 1,652,65,199.01617489465656
2.1.D,2.1,Fürth 2A,EW 1,680,65,222.74547072627712
2.2.A,2.2,Fürth 2A,EW 2,92,65,76.99629554124797
2.2.A,2.2,Fürth 2A,EW 2,115,65,250.2379604067633
2.2.A,2.2,Fürth 2A,EW 2,150,65,0.0
2.2.A,2.2,Fürth 2A,EW 2,177,65,283.73134917865093
2.2.A,2.2,Fürth 2A,EW 2,198,65,312.3162238401829
2.2.A,2.2,Fürth 2A,EW 2,228,65,142.49126943859437
2.2.A,2.2,Fürth 2A,EW 2,259,65,314.14488573319693
2.2.A,2.2,Fürth 2A,EW 2,284,65,255.2427197785667
2.2.A,2.2,Fürth 2A,EW 2,315,65,296.43573789036645
2.2.A,2.2,Fürth 2A,EW 2,351,65,359.66894566460076
2.2.A,2.2,Fürth 2A,EW 2,385,65,260.2474789096817
2.2.A,2.2,Fürth 2A,EW 2,410,65,227.6202986942656
2.2.A,2.2,Fürth 2A,EW 2,448,65,262.74985859558336
2.2.A,2.2,Fürth 2A,EW 2,471,65,222.27868069077564
2.2.A,2.2,Fürth 2A,EW 2,499,65,387.4838572717973
2.2.A,2.2,Fürth 2A,EW 2,528,65,105.50417396955292
2.2.A,2.2,Fürth 2A,EW 2,560,65,333.0282273038893
2.2.A,2.2,Fürth 2A,EW 2,595,65,308.2618876151507
2.2.A,2.2,Fürth 2A,EW 2,626,65,193.30882450207596
2.2.A,2.2,Fürth 2A,EW 2,652,65,145.43637774700915
2.2.A,2.2,Fürth 2A,EW 2,680,65,183.65541394789096
2.2.B,2.2,Fürth 2A,EW 2,92,65,76.22633258318791
2.2.B,2.2,Fürth 2A,EW 2,115,65,277.9085042421325
2.2.B,2.2,Fürth 2A,EW 2,150,65,0.0
2.2.B,2.2,Fürth 2A,EW 2,177,65,612.5055310187136
2.2.B,2.2,Fürth 2A,EW 2,198,65,412.3151626451652
2.2.B,2.2,Fürth 2A,EW 2,228,65,157.74616049100428
2.2.B,2.2,Fürth 2A,EW 2,259,65,372.3733343763163
2.2.B,2.2,Fürth 2A,EW 2,284,65,364.1924780071003
2.2.B,2.2,Fürth 2A,EW 2,315,65,429.8318199651002
2.2.B,2.2,Fürth 2A,EW 2,351,65,491.28448835670014
2.2.B,2.2,Fürth 2A,EW 2,385,65,309.23637186352966
2.2.B,2.2,Fürth 2A,EW 2,410,65,248.3130530116132
2.2.B,2.2,Fürth 2A,EW 2,448,65,323.3844411817799
2.2.B,2.2,Fürth 2A,EW 2,471,65,229.6895741500692
2.2.B,2.2,Fürth 2A,EW 2,499,65,355.3379038449967
2.2.B,2.2,Fürth 2A,EW 2,528,65,216.26334511101751
2.2.B,2.2,Fürth 2A,EW 2,560,65,307.30905845599455
2.2.B,2.2,Fürth 2A,EW 2,595,65,329.72219886202333
2.2.B,2.2,Fürth 2A,EW 2,626,65,232.91379401889404
2.2.B,2.2,Fürth 2A,EW 2,652,65,192.95752889466792
2.2.B,2.2,Fürth 2A,EW 2,680,65,215.48375760274385
2.2.C,2.2,Fürth 2A,EW 2,92,65,83.15599918165954
2.2.C,2.2,Fürth 2A,EW 2,115,65,312.7012053673506
2.2.C,2.2,Fürth 2A,EW 2,150,65,0.0
2.2.C,2.2,Fürth 2A,EW 2,177,65,732.3791385763283
2.2.C,2.2,Fürth 2A,EW 2,198,65,435.1253151212468
2.2.C,2.2,Fürth 2A,EW 2,228,65,213.5203520789458
2.2.C,2.2,Fürth 2A,EW 2,259,65,358.12901955593
2.2.C,2.2,Fürth 2A,EW 2,284,65,316.45477465551477
2.2.C,2.2,Fürth 2A,EW 2,315,65,378.82177411396594
2.2.C,2.2,Fürth 2A,EW 2,351,65,422.1321903844996
2.2.C,2.2,Fürth 2A,EW 2,385,65,331.18031626451653
2.2.C,2.2,Fürth 2A,EW 2,410,65,262.79798110596306
2.2.C,2.2,Fürth 2A,EW 2,448,65,345.3283855827667
2.2.C,2.2,Fürth 2A,EW 2,471,65,222.51929410915216
2.2.C,2.2,Fürth 2A,EW 2,499,65,400.4769822492328
2.2.C,2.2,Fürth 2A,EW 2,528,65,273.46196810879115
2.2.C,2.2,Fürth 2A,EW 2,560,65,470.42330431869686
2.2.C,2.2,Fürth 2A,EW 2,595,65,429.7476050730952
2.2.C,2.2,Fürth 2A,EW 2,626,65,302.9371124616403
2.2.C,2.2,Fürth 2A,EW 2,652,65,222.32680334653077
2.2.C,2.2,Fürth 2A,EW 2,680,65,218.23156290992236
2.2.D,2.2,Fürth 2A,EW 2,92,65,86.23585101389975
2.2.D,2.2,Fürth 2A,EW 2,115,65,331.90215656778383
2.2.D,2.2,Fürth 2A,EW 2,150,65,0.0
2.2.D,2.2,Fürth 2A,EW 2,177,65,709.4246180877309
2.2.D,2.2,Fürth 2A,EW 2,198,65,425.5970236476322
2.2.D,2.2,Fürth 2A,EW 2,228,65,169.34372751669775
2.2.D,2.2,Fürth 2A,EW 2,259,65,360.5832764907636
2.2.D,2.2,Fürth 2A,EW 2,284,65,309.23637186352966
2.2.D,2.2,Fürth 2A,EW 2,315,65,332.19089259281543
2.2.D,2.2,Fürth 2A,EW 2,351,65,381.1316630362838
2.2.D,2.2,Fürth 2A,EW 2,385,65,313.2786774174138
2.2.D,2.2,Fürth 2A,EW 2,410,65,304.90533028461397
2.2.D,2.2,Fürth 2A,EW 2,448,65,340.70860773813104
2.2.D,2.2,Fürth 2A,EW 2,471,65,232.24007641855707
2.2.D,2.2,Fürth 2A,EW 2,499,65,377.6668295324628
2.2.D,2.2,Fürth 2A,EW 2,528,65,276.7054371502497
2.2.D,2.2,Fürth 2A,EW 2,560,65,422.2067805698164
2.2.D,2.2,Fürth 2A,EW 2,595,65,397.0819268748467
2.2.D,2.2,Fürth 2A,EW 2,626,65,285.6755056260906
2.2.D,2.2,Fürth 2A,EW 2,652,65,237.60094351611724
2.2.D,2.2,Fürth 2A,EW 2,680,65,250.72881184186772
2.3.A,2.3,Fürth 2A,EW 3,92,65,106.83236006979962
2.3.A,2.3,Fürth 2A,EW 3,115,65,297.3981914675973
2.3.A,2.3,Fürth 2A,EW 3,150,65,0.0
2.3.A,2.3,Fürth 2A,EW 3,177,65,488.1083911185992
2.3.A,2.3,Fürth 2A,EW 3,198,65,403.46058872374994
2.3.A,2.3,Fürth 2A,EW 3,228,65,162.41406091822614
2.3.A,2.3,Fürth 2A,EW 3,259,65,326.0793116312654
2.3.A,2.3,Fürth 2A,EW 3,284,65,309.140126602082
2.3.A,2.3,Fürth 2A,EW 3,315,65,414.72129682893075
2.3.A,2.3,Fürth 2A,EW 3,351,65,390.3712184848667
2.3.A,2.3,Fürth 2A,EW 3,385,65,295.28079330886334
2.3.A,2.3,Fürth 2A,EW 3,410,65,213.13537059991577
2.3.A,2.3,Fürth 2A,EW 3,448,65,314.00051772068116
2.3.A,2.3,Fürth 2A,EW 3,471,65,201.68217163487571
2.3.A,2.3,Fürth 2A,EW 3,499,65,363.8074964799326
2.3.A,2.3,Fürth 2A,EW 3,528,65,180.30607509477105
2.3.A,2.3,Fürth 2A,EW 3,560,65,269.0058075524513
2.3.A,2.3,Fürth 2A,EW 3,595,65,237.4493570511061
2.3.A,2.3,Fürth 2A,EW 3,626,65,180.08471072868403
2.3.A,2.3,Fürth 2A,EW 3,652,65,202.19227211137573
2.3.A,2.3,Fürth 2A,EW 3,680,65,248.32748998134664
2.3.B,2.3,Fürth 2A,EW 3,92,65,85.7546241530778
2.3.B,2.3,Fürth 2A,EW 3,115,65,288.44737228473434
2.3.B,2.3,Fürth 2A,EW 3,150,65,0.0
2.3.B,2.3,Fürth 2A,EW 3,177,65,549.4648140080631
2.3.B,2.3,Fürth 2A,EW 3,198,65,369.8709546904146
2.3.B,2.3,Fürth 2A,EW 3,228,65,165.0126858896444
2.3.B,2.3,Fürth 2A,EW 3,259,65,375.7419221373127
2.3.B,2.3,Fürth 2A,EW 3,284,65,303.0766683916
2.3.B,2.3,Fürth 2A,EW 3,315,65,335.03013105481676
2.3.B,2.3,Fürth 2A,EW 3,351,65,415.39501438113
2.3.B,2.3,Fürth 2A,EW 3,385,65,312.7012053673506
2.3.B,2.3,Fürth 2A,EW 3,410,65,247.9280717251339
2.3.B,2.3,Fürth 2A,EW 3,448,65,337.82124676575006
2.3.B,2.3,Fürth 2A,EW 3,471,65,235.0311921535592
2.3.B,2.3,Fürth 2A,EW 3,499,65,401.9206626150791
2.3.B,2.3,Fürth 2A,EW 3,528,65,276.9556750707022
2.3.B,2.3,Fürth 2A,EW 3,560,65,425.5176211940365
2.3.B,2.3,Fürth 2A,EW 3,595,65,387.39483036242314
2.3.B,2.3,Fürth 2A,EW 3,626,65,247.8799489740658
2.3.B,2.3,Fürth 2A,EW 3,652,65,194.18465735438647
2.3.B,2.3,Fürth 2A,EW 3,680,65,247.15810867079847
2.3.C,2.3,Fürth 2A,EW 3,92,65,92.39555465431133
2.3.C,2.3,Fürth 2A,EW 3,115,65,318.1871912870811
2.3.C,2.3,Fürth 2A,EW 3,150,65,0.0
2.3.C,2.3,Fürth 2A,EW 3,177,65,548.2136241651122
2.3.C,2.3,Fürth 2A,EW 3,198,65,417.7049033034478
2.3.C,2.3,Fürth 2A,EW 3,228,65,200.19036842168603
2.3.C,2.3,Fürth 2A,EW 3,259,65,301.729233046513
2.3.C,2.3,Fürth 2A,EW 3,284,65,321.363288525182
2.3.C,2.3,Fürth 2A,EW 3,315,65,358.8989826102653
2.3.C,2.3,Fürth 2A,EW 3,351,65,395.3759776159817
2.3.C,2.3,Fürth 2A,EW 3,385,65,309.23637186352966
2.3.C,2.3,Fürth 2A,EW 3,410,65,238.49602544076055
2.3.C,2.3,Fürth 2A,EW 3,448,65,343.59596895120046
2.3.C,2.3,Fürth 2A,EW 3,471,65,228.6789977495637
2.3.C,2.3,Fürth 2A,EW 3,499,65,394.1247877730309
2.3.C,2.3,Fürth 2A,EW 3,528,65,234.39597270593896
2.3.C,2.3,Fürth 2A,EW 3,560,65,373.0326150533967
2.3.C,2.3,Fürth 2A,EW 3,595,65,339.7052497586575
2.3.C,2.3,Fürth 2A,EW 3,626,65,249.98772248631084
2.3.C,2.3,Fürth 2A,EW 3,652,65,215.05065342195033
2.3.C,2.3,Fürth 2A,EW 3,680,65,246.3352107828389
2.3.D,2.3,Fürth 2A,EW 3,92,65,87.77577693001986
2.3.D,2.3,Fürth 2A,EW 3,115,65,293.7889901919489
2.3.D,2.3,Fürth 2A,EW 3,150,65,0.0
2.3.D,2.3,Fürth 2A,EW 3,177,65,654.853493471328
2.3.D,2.3,Fürth 2A,EW 3,198,65,375.9344129008966
2.3.D,2.3,Fürth 2A,EW 3,228,65,126.17767933088632
2.3.D,2.3,Fürth 2A,EW 3,259,65,262.028018292316
2.3.D,2.3,Fürth 2A,EW 3,284,65,254.0877751970636
2.3.D,2.3,Fürth 2A,EW 3,315,65,351.68057981828025
2.3.D,2.3,Fürth 2A,EW 3,351,65,390.3712184848667
2.3.D,2.3,Fürth 2A,EW 3,385,65,336.5219341717311
2.3.D,2.3,Fürth 2A,EW 3,410,65,260.7287056982971
2.3.D,2.3,Fürth 2A,EW 3,448,65,328.8704273421987
2.3.D,2.3,Fürth 2A,EW 3,471,65,180.17133156026233
2.3.D,2.3,Fürth 2A,EW 3,499,65,309.52510812924965
2.3.D,2.3,Fürth 2A,EW 3,528,65,209.89190165473255
2.3.D,2.3,Fürth 2A,EW 3,560,65,342.3688404029317
2.3.D,2.3,Fürth 2A,EW 3,595,65,305.67048103948025
2.3.D,2.3,Fürth 2A,EW 3,626,65,239.0734976593056
2.3.D,2.3,Fürth 2A,EW 3,652,65,198.88864978976784
2.3.D,2.3,Fürth 2A,EW 3,680,65,210.6570523376858
2.4.A,2.4,Fürth 2A,EW 4,92,65,86.23585101389975
2.4.A,2.4,Fürth 2A,EW 4,115,65,304.3759807449305
2.4.A,2.4,Fürth 2A,EW 4,150,65,0.0
2.4.A,2.4,Fürth 2A,EW 4,177,65,559.9074365485288
2.4.A,2.4,Fürth 2A,EW 4,198,65,395.5684683795656
2.4.A,2.4,Fürth 2A,EW 4,228,65,115.3500752632529
2.4.A,2.4,Fürth 2A,EW 4,259,65,311.8349970515675
2.4.A,2.4,Fürth 2A,EW 4,284,65,299.1306080991636
2.4.A,2.4,Fürth 2A,EW 4,315,65,350.33314471388167
2.4.A,2.4,Fürth 2A,EW 4,351,65,353.41299644984656
2.4.A,2.4,Fürth 2A,EW 4,385,65,277.76413622961667
2.4.A,2.4,Fürth 2A,EW 4,410,65,252.4516040676334
2.4.A,2.4,Fürth 2A,EW 4,448,65,312.79745062879834
2.4.A,2.4,Fürth 2A,EW 4,471,65,208.8524516517239
2.4.A,2.4,Fürth 2A,EW 4,499,65,375.7419221373127
2.4.A,2.4,Fürth 2A,EW 4,528,65,225.21416446236236
2.4.A,2.4,Fürth 2A,EW 4,560,65,366.1029484945506
2.4.A,2.4,Fürth 2A,EW 4,595,65,335.30924254641087
2.4.A,2.4,Fürth 2A,EW 4,626,65,229.1361632589205
2.4.A,2.4,Fürth 2A,EW 4,652,65,193.5590624625707
2.4.A,2.4,Fürth 2A,EW 4,680,65,248.515168421686
2.4.B,2.4,Fürth 2A,EW 4,92,65,85.17715193453276
2.4.B,2.4,Fürth 2A,EW 4,115,65,305.5790478368133
2.4.B,2.4,Fürth 2A,EW 4,150,65,0.0
2.4.B,2.4,Fürth 2A,EW 4,177,65,582.4769757506468
2.4.B,2.4,Fürth 2A,EW 4,198,65,412.3151626451652
2.4.B,2.4,Fürth 2A,EW 4,228,65,213.0872479210542
2.4.B,2.4,Fürth 2A,EW 4,259,65,365.6361583729466
2.4.B,2.4,Fürth 2A,EW 4,284,65,237.96667590107705
2.4.B,2.4,Fürth 2A,EW 4,315,65,315.3960755761478
2.4.B,2.4,Fürth 2A,EW 4,351,65,366.8392254648294
2.4.B,2.4,Fürth 2A,EW 4,385,65,288.73610830976594
2.4.B,2.4,Fürth 2A,EW 4,410,65,260.7287056982971
2.4.B,2.4,Fürth 2A,EW 4,448,65,400.3807367470967
2.4.B,2.4,Fürth 2A,EW 4,471,65,278.9190805704314
2.4.B,2.4,Fürth 2A,EW 4,499,65,400.8619637764005
2.4.B,2.4,Fürth 2A,EW 4,528,65,317.0611204043565
2.4.B,2.4,Fürth 2A,EW 4,560,65,456.49178711878824
2.4.B,2.4,Fürth 2A,EW 4,595,65,385.4819535669767
2.4.B,2.4,Fürth 2A,EW 4,626,65,213.4866662013358
2.4.B,2.4,Fürth 2A,EW 4,652,65,157.0002588829931
2.4.B,2.4,Fürth 2A,EW 4,680,65,184.77186023226423
2.4.C,2.4,Fürth 2A,EW 4,92,65,86.52458711113786
2.4.C,2.4,Fürth 2A,EW 4,115,65,311.1612792586798
2.4.C,2.4,Fürth 2A,EW 4,150,65,0.0
2.4.C,2.4,Fürth 2A,EW 4,177,65,779.0100200974787
2.4.C,2.4,Fürth 2A,EW 4,198,65,486.3278517359648
2.4.C,2.4,Fürth 2A,EW 4,228,65,192.68322960466935
2.4.C,2.4,Fürth 2A,EW 4,259,65,364.81807280823153
2.4.C,2.4,Fürth 2A,EW 4,284,65,316.5991426680306
2.4.C,2.4,Fürth 2A,EW 4,315,65,366.9354709669655
2.4.C,2.4,Fürth 2A,EW 4,351,65,411.9301811179975
2.4.C,2.4,Fürth 2A,EW 4,385,65,316.2622838919309
2.4.C,2.4,Fürth 2A,EW 4,410,65,240.5171781936338
2.4.C,2.4,Fürth 2A,EW 4,448,65,360.9201352668632
2.4.C,2.4,Fürth 2A,EW 4,471,65,261.7874047776641
2.4.C,2.4,Fürth 2A,EW 4,499,65,431.9492178831458
2.4.C,2.4,Fürth 2A,EW 4,528,65,288.27413057344
2.4.C,2.4,Fürth 2A,EW 4,560,65,431.3139986203811
2.4.C,2.4,Fürth 2A,EW 4,595,65,353.8172271451433
2.4.C,2.4,Fürth 2A,EW 4,626,65,258.7075530416992
2.4.C,2.4,Fürth 2A,EW 4,652,65,244.55948374362112
2.4.C,2.4,Fürth 2A,EW 4,680,65,282.39353848005294
2.4.D,2.4,Fürth 2A,EW 4,92,65,82.2897908658764
2.4.D,2.4,Fürth 2A,EW 4,115,65,342.2485336061135
2.4.D,2.4,Fürth 2A,EW 4,150,65,0.0
2.4.D,2.4,Fürth 2A,EW 4,177,65,1033.8196355978096
2.4.D,2.4,Fürth 2A,EW 4,198,65,558.993105722366
2.4.D,2.4,Fürth 2A,EW 4,228,65,234.4537199350141
2.4.D,2.4,Fürth 2A,EW 4,259,65,387.6763480353811
2.4.D,2.4,Fürth 2A,EW 4,284,65,314.52986726036465
2.4.D,2.4,Fürth 2A,EW 4,315,65,403.5087112341296
2.4.D,2.4,Fürth 2A,EW 4,351,65,434.0666160418798
2.4.D,2.4,Fürth 2A,EW 4,385,65,315.2998303147
2.4.D,2.4,Fürth 2A,EW 4,410,65,316.64726541909863
2.4.D,2.4,Fürth 2A,EW 4,448,65,363.8074964799326
2.4.D,2.4,Fürth 2A,EW 4,471,65,261.06556447439675
2.4.D,2.4,Fürth 2A,EW 4,499,65,458.36857187556416
2.4.D,2.4,Fürth 2A,EW 4,528,65,393.5473154822793
2.4.D,2.4,Fürth 2A,EW 4,560,65,531.1854122348788
2.4.D,2.4,Fürth 2A,EW 4,595,65,474.913150868902
2.4.D,2.4,Fürth 2A,EW 4,626,65,338.4757152656598
2.4.D,2.4,Fürth 2A,EW 4,652,65,262.0280182808515
2.4.D,2.4,Fürth 2A,EW 4,680,65,275.1558866357783
2.5.A,2.5,Fürth 2A,EW 5,93,65,36.04389084782478
2.5.A,2.5,Fürth 2A,EW 5,115,65,313.71178169564956
2.5.A,2.5,Fürth 2A,EW 5,150,65,0.0
2.5.A,2.5,Fürth 2A,EW 5,177,65,579.0121424875142
2.5.A,2.5,Fürth 2A,EW 5,198,65,425.01955135688064
2.5.A,2.5,Fürth 2A,EW 5,228,65,173.7710145014742
2.5.A,2.5,Fürth 2A,EW 5,259,65,362.17132510981406
2.5.A,2.5,Fürth 2A,EW 5,284,65,288.73610830976594
2.5.A,2.5,Fürth 2A,EW 5,315,65,334.93388555268064
2.5.A,2.5,Fürth 2A,EW 5,351,65,396.4827994464168
2.5.A,2.5,Fürth 2A,EW 5,385,65,331.90215656778383
2.5.A,2.5,Fürth 2A,EW 5,410,65,283.49073566399903
2.5.A,2.5,Fürth 2A,EW 5,448,65,398.45582935194653
2.5.A,2.5,Fürth 2A,EW 5,471,65,239.84346061736565
2.5.A,2.5,Fürth 2A,EW 5,499,65,464.2876620735302
2.5.A,2.5,Fürth 2A,EW 5,528,65,378.74958998736383
2.5.A,2.5,Fürth 2A,EW 5,560,65,464.93731834825945
2.5.A,2.5,Fürth 2A,EW 5,595,65,406.64871649043926
2.5.A,2.5,Fürth 2A,EW 5,626,65,241.32563932847944
2.5.A,2.5,Fürth 2A,EW 5,652,65,192.2741867589043
2.5.A,2.5,Fürth 2A,EW 5,680,65,248.6980345387809
2.5.B,2.5,Fürth 2A,EW 5,93,65,93.69486712798604
2.5.B,2.5,Fürth 2A,EW 5,115,65,301.87360129971717
2.5.B,2.5,Fürth 2A,EW 5,150,65,0.0
2.5.B,2.5,Fürth 2A,EW 5,177,65,595.5663459895301
2.5.B,2.5,Fürth 2A,EW 5,198,65,404.71177832601234
2.5.B,2.5,Fürth 2A,EW 5,228,65,158.80485954630242
2.5.B,2.5,Fürth 2A,EW 5,259,65,308.6107770623985
2.5.B,2.5,Fürth 2A,EW 5,284,65,263.7123121728142
2.5.B,2.5,Fürth 2A,EW 5,315,65,280.55525194054997
2.5.B,2.5,Fürth 2A,EW 5,351,65,365.347422347915
2.5.B,2.5,Fürth 2A,EW 5,385,65,272.85562235994945
2.5.B,2.5,Fürth 2A,EW 5,410,65,237.96667590107705
2.5.B,2.5,Fürth 2A,EW 5,448,65,311.8349970515675
2.5.B,2.5,Fürth 2A,EW 5,471,65,187.96720649858597
2.5.B,2.5,Fürth 2A,EW 5,499,65,403.17185245802995
2.5.B,2.5,Fürth 2A,EW 5,528,65,301.2191327998074
2.5.B,2.5,Fürth 2A,EW 5,560,65,440.5222743579992
2.5.B,2.5,Fürth 2A,EW 5,595,65,335.8915270987911
2.5.B,2.5,Fürth 2A,EW 5,626,65,205.1951276250075
2.5.B,2.5,Fürth 2A,EW 5,652,65,168.6459485858531
2.5.B,2.5,Fürth 2A,EW 5,680,65,205.65229313436427
2.5.C,2.5,Fürth 2A,EW 5,92,65,89.50819356158613
2.5.C,2.5,Fürth 2A,EW 5,115,65,293.0671498886816
2.5.C,2.5,Fürth 2A,EW 5,150,65,0.0
2.5.C,2.5,Fürth 2A,EW 5,177,65,746.0941038570311
2.5.C,2.5,Fürth 2A,EW 5,198,65,469.19617594319755
2.5.C,2.5,Fürth 2A,EW 5,228,65,145.08989441001262
2.5.C,2.5,Fürth 2A,EW 5,259,65,342.15228834466575
2.5.C,2.5,Fürth 2A,EW 5,284,65,334.64514952764904
2.5.C,2.5,Fürth 2A,EW 5,315,65,411.6895678440339
2.5.C,2.5,Fürth 2A,EW 5,351,65,392.77735266863226
2.5.C,2.5,Fürth 2A,EW 5,385,65,317.0322469462663
2.5.C,2.5,Fürth 2A,EW 5,410,65,277.28290944100127
2.5.C,2.5,Fürth 2A,EW 5,448,65,348.889464107347
2.5.C,2.5,Fürth 2A,EW 5,471,65,232.33632180034897
2.5.C,2.5,Fürth 2A,EW 5,499,65,399.7551419459654
2.5.C,2.5,Fürth 2A,EW 5,528,65,302.8264304711475
2.5.C,2.5,Fürth 2A,EW 5,560,65,449.7016764257674
2.5.C,2.5,Fürth 2A,EW 5,595,65,394.49533251672176
2.5.C,2.5,Fürth 2A,EW 5,626,65,257.6007312112642
2.5.C,2.5,Fürth 2A,EW 5,652,65,195.2650116092672
2.5.C,2.5,Fürth 2A,EW 5,680,65,222.05731634875744
2.5.D,2.5,Fürth 2A,EW 5,92,65,61.59703642818461
2.5.D,2.5,Fürth 2A,EW 5,115,65,303.79850869486734
2.5.D,2.5,Fürth 2A,EW 5,150,65,0.0
2.5.D,2.5,Fürth 2A,EW 5,177,65,720.1078541428486
2.5.D,2.5,Fürth 2A,EW 5,198,65,434.7884563451471
2.5.D,2.5,Fürth 2A,EW 5,228,65,171.50924833022444
2.5.D,2.5,Fürth 2A,EW 5,259,65,410.2940099885673
2.5.D,2.5,Fürth 2A,EW 5,284,65,276.7054371502497
2.5.D,2.5,Fürth 2A,EW 5,315,65,369.1009916360792
2.5.D,2.5,Fürth 2A,EW 5,351,65,383.4415517179132
2.5.D,2.5,Fürth 2A,EW 5,385,65,288.0142680064986
2.5.D,2.5,Fürth 2A,EW 5,410,65,264.62664323966544
2.5.D,2.5,Fürth 2A,EW 5,448,65,312.79745062879834
2.5.D,2.5,Fürth 2A,EW 5,471,65,270.64197893976774
2.5.D,2.5,Fürth 2A,EW 5,499,65,410.00527372284733
2.5.D,2.5,Fürth 2A,EW 5,528,65,300.7090323124135
2.5.D,2.5,Fürth 2A,EW 5,560,65,453.55630357866534
2.5.D,2.5,Fürth 2A,EW 5,595,65,391.6224083313717
2.5.D,2.5,Fürth 2A,EW 5,626,65,263.87592947830797
2.5.D,2.5,Fürth 2A,EW 5,652,65,209.54301222812717
2.5.D,2.5,Fürth 2A,EW 5,680,65,251.11379336903545
2.6.A,2.6,Fürth 2A,EW 6,93,65,99.71020273181298
2.6.A,2.6,Fürth 2A,EW 6,115,65,306.6377469161803
2.6.A,2.6,Fürth 2A,EW 6,150,65,0.0
2.6.A,2.6,Fürth 2A,EW 6,177,65,575.5473092243817
2.6.A,2.6,Fürth 2A,EW 6,198,65,410.00527372284733
2.6.A,2.6,Fürth 2A,EW 6,228,65,86.33209637162284
2.6.A,2.6,Fürth 2A,EW 6,259,65,332.62399663036285
2.6.A,2.6,Fürth 2A,EW 6,284,65,284.88629351946565
2.6.A,2.6,Fürth 2A,EW 6,315,65,322.90321439316443
2.6.A,2.6,Fürth 2A,EW 6,351,65,416.6462042240808
2.6.A,2.6,Fürth 2A,EW 6,385,65,302.3548280883326
2.6.A,2.6,Fürth 2A,EW 6,410,65,261.2099327276009
2.6.A,2.6,Fürth 2A,EW 6,448,65,404.23055153739693
2.6.A,2.6,Fürth 2A,EW 6,471,65,250.62294193393103
2.6.A,2.6,Fürth 2A,EW 6,499,65,517.0301245562308
2.6.A,2.6,Fürth 2A,EW 6,528,65,377.20485179613695
2.6.A,2.6,Fürth 2A,EW 6,560,65,588.1891384830204
2.6.A,2.6,Fürth 2A,EW 6,595,65,492.8244141805504
2.6.A,2.6,Fürth 2A,EW 6,626,65,309.698349840544
2.6.A,2.6,Fürth 2A,EW 6,652,65,230.58465607112865
2.6.A,2.6,Fürth 2A,EW 6,680,65,252.69221734159697
2.6.B,2.6,Fürth 2A,EW 6,93,65,112.60708223118118
2.6.B,2.6,Fürth 2A,EW 6,115,65,316.5991426680306
2.6.B,2.6,Fürth 2A,EW 6,150,65,0.0
2.6.B,2.6,Fürth 2A,EW 6,177,65,680.1660256333113
2.6.B,2.6,Fürth 2A,EW 6,198,65,429.6393292015164
2.6.B,2.6,Fürth 2A,EW 6,228,65,206.4463174198207
2.6.B,2.6,Fürth 2A,EW 6,259,65,405.0967598531801
2.6.B,2.6,Fürth 2A,EW 6,284,65,334.26016800048137
2.6.B,2.6,Fürth 2A,EW 6,315,65,362.8450426620133
2.6.B,2.6,Fürth 2A,EW 6,351,65,419.6298106985979
2.6.B,2.6,Fürth 2A,EW 6,385,65,322.90321439316443
2.6.B,2.6,Fürth 2A,EW 6,410,65,292.34530958541427
2.6.B,2.6,Fürth 2A,EW 6,448,65,332.04652458029966
2.6.B,2.6,Fürth 2A,EW 6,471,65,247.6393354594139
2.6.B,2.6,Fürth 2A,EW 6,499,65,471.5060648655153
2.6.B,2.6,Fürth 2A,EW 6,528,65,359.66894566460076
2.6.B,2.6,Fürth 2A,EW 6,560,65,551.1009854311455
2.6.B,2.6,Fürth 2A,EW 6,595,65,447.3484771714923
2.6.B,2.6,Fürth 2A,EW 6,626,65,276.320455623082
2.6.B,2.6,Fürth 2A,EW 6,652,65,217.47603673579175
2.6.B,2.6,Fürth 2A,EW 6,680,65,245.25245032793788
2.6.C,2.6,Fürth 2A,EW 6,92,65,79.06557099705157
2.6.C,2.6,Fürth 2A,EW 6,115,65,290.27603417774833
2.6.C,2.6,Fürth 2A,EW 6,150,65,0.0
2.6.C,2.6,Fürth 2A,EW 6,177,65,409.52404693423193
2.6.C,2.6,Fürth 2A,EW 6,198,65,575.3548184607978
2.6.C,2.6,Fürth 2A,EW 6,228,65,182.7699565316806
2.6.C,2.6,Fürth 2A,EW 6,259,65,384.5964962994163
2.6.C,2.6,Fürth 2A,EW 6,284,65,348.3119920572838
2.6.C,2.6,Fürth 2A,EW 6,315,65,392.68110716649613
2.6.C,2.6,Fürth 2A,EW 6,351,65,435.41405138696666
2.6.C,2.6,Fürth 2A,EW 6,385,65,337.72500126361393
2.6.C,2.6,Fürth 2A,EW 6,410,65,282.14330055960045
2.6.C,2.6,Fürth 2A,EW 6,448,65,421.0734913051326
2.6.C,2.6,Fürth 2A,EW 6,471,65,301.05551549431374
2.6.C,2.6,Fürth 2A,EW 6,499,65,464.86513436428186
2.6.C,2.6,Fürth 2A,EW 6,528,65,314.2218821830435
2.6.C,2.6,Fürth 2A,EW 6,560,65,450.7796245031878
2.6.C,2.6,Fürth 2A,EW 6,595,65,394.2619374431204
2.6.C,2.6,Fürth 2A,EW 6,626,65,302.3211420663096
2.6.C,2.6,Fürth 2A,EW 6,652,65,203.4434618359309
2.6.C,2.6,Fürth 2A,EW 6,680,65,266.5034278837475
2.6.D,2.6,Fürth 2A,EW 6,92,65,69.05605256633973
2.6.D,2.6,Fürth 2A,EW 6,115,65,317.0322469462663
2.6.D,2.6,Fürth 2A,EW 6,150,65,0.0
2.6.D,2.6,Fürth 2A,EW 6,177,65,685.0264169925988
2.6.D,2.6,Fürth 2A,EW 6,198,65,551.3415986521451
2.6.D,2.6,Fürth 2A,EW 6,228,65,154.18508182201094
2.6.D,2.6,Fürth 2A,EW 6,259,65,351.1031077682171
2.6.D,2.6,Fürth 2A,EW 6,284,65,317.6097189963295
2.6.D,2.6,Fürth 2A,EW 6,315,65,410.00527372284733
2.6.D,2.6,Fürth 2A,EW 6,351,65,471.3135741019315
2.6.D,2.6,Fürth 2A,EW 6,385,65,328.19670978999943
2.6.D,2.6,Fürth 2A,EW 6,410,65,323.1919506588844
2.6.D,2.6,Fürth 2A,EW 6,448,65,404.23055153739693
2.6.D,2.6,Fürth 2A,EW 6,471,65,292.00845080931464
2.6.D,2.6,Fürth 2A,EW 6,499,65,453.89316228413264
2.6.D,2.6,Fürth 2A,EW 6,528,65,327.05138985498525
2.6.D,2.6,Fürth 2A,EW 6,560,65,504.9128326529707
2.6.D,2.6,Fürth 2A,EW 6,595,65,427.0407042338558
2.6.D,2.6,Fürth 2A,EW 6,626,65,287.5811639689512
2.6.D,2.6,Fürth 2A,EW 6,652,65,253.98671775132857
2.6.D,2.6,Fürth 2A,EW 6,680,65,275.4205615259643
2.7.A,2.7,Fürth 2A,EW 7,93,65,115.20570720259944
2.7.A,2.7,Fürth 2A,EW 7,115,65,290.27603417774833
2.7.A,2.7,Fürth 2A,EW 7,150,65,0.0
2.7.A,2.7,Fürth 2A,EW 7,177,65,1050.999434141645
2.7.A,2.7,Fürth 2A,EW 7,198,65,459.09041217883146
2.7.A,2.7,Fürth 2A,EW 7,228,65,250.76730994644683
2.7.A,2.7,Fürth 2A,EW 7,259,65,329.83288091942956
2.7.A,2.7,Fürth 2A,EW 7,284,65,288.0142680064986
2.7.A,2.7,Fürth 2A,EW 7,315,65,515.3939531861123
2.7.A,2.7,Fürth 2A,EW 7,351,65,589.0216609904327
2.7.A,2.7,Fürth 2A,EW 7,385,65,425.64514639870026
2.7.A,2.7,Fürth 2A,EW 7,410,65,268.81331680606536
2.7.A,2.7,Fürth 2A,EW 7,448,65,381.1316630362838
2.7.A,2.7,Fürth 2A,EW 7,471,65,278.9672033214995
2.7.A,2.7,Fürth 2A,EW 7,499,65,410.00527372284733
2.7.A,2.7,Fürth 2A,EW 7,528,65,234.1168611348457
2.7.A,2.7,Fürth 2A,EW 7,560,65,425.9098210480712
2.7.A,2.7,Fürth 2A,EW 7,595,65,463.9508033133105
2.7.A,2.7,Fürth 2A,EW 7,626,65,266.35905987123175
2.7.A,2.7,Fürth 2A,EW 7,652,65,239.07349766524032
2.7.A,2.7,Fürth 2A,EW 7,680,65,261.1618099765329
2.7.B,2.7,Fürth 2A,EW 7,93,65,106.06239711173956
2.7.B,2.7,Fürth 2A,EW 7,115,65,262.7017358445153
2.7.B,2.7,Fürth 2A,EW 7,150,65,0.0
2.7.B,2.7,Fürth 2A,EW 7,177,65,678.7223452674649
2.7.B,2.7,Fürth 2A,EW 7,198,65,387.86883879896504
2.7.B,2.7,Fürth 2A,EW 7,228,65,193.693805981106
2.7.B,2.7,Fürth 2A,EW 7,259,65,348.2157465551477
2.7.B,2.7,Fürth 2A,EW 7,284,65,307.5039552319634
2.7.B,2.7,Fürth 2A,EW 7,315,65,314.72235802394846
2.7.B,2.7,Fürth 2A,EW 7,351,65,377.9555657981827
2.7.B,2.7,Fürth 2A,EW 7,385,65,320.9301842469463
2.7.B,2.7,Fürth 2A,EW 7,410,65,292.2009415728985
2.7.B,2.7,Fürth 2A,EW 7,448,65,346.48332992358144
2.7.B,2.7,Fürth 2A,EW 7,471,65,256.5901548829653
2.7.B,2.7,Fürth 2A,EW 7,499,65,397.2527622600638
2.7.B,2.7,Fürth 2A,EW 7,528,65,320.8916861423672
2.7.B,2.7,Fürth 2A,EW 7,560,65,479.31637650774854
2.7.B,2.7,Fürth 2A,EW 7,595,65,424.26883746904355
2.7.B,2.7,Fürth 2A,EW 7,626,65,239.4103564594741
2.7.B,2.7,Fürth 2A,EW 7,652,65,194.19187576515225
2.7.B,2.7,Fürth 2A,EW 7,680,65,235.91664954570072
2.7.C,2.7,Fürth 2A,EW 7,92,65,70.74034651904446
2.7.C,2.7,Fürth 2A,EW 7,115,65,279.11157133401525
2.7.C,2.7,Fürth 2A,EW 7,150,65,0.0
2.7.C,2.7,Fürth 2A,EW 7,177,65,662.6493685540646
2.7.C,2.7,Fürth 2A,EW 7,198,65,363.9999872435165
2.7.C,2.7,Fürth 2A,EW 7,228,65,106.83236006979962
2.7.C,2.7,Fürth 2A,EW 7,259,65,318.81278632890064
2.7.C,2.7,Fürth 2A,EW 7,284,65,314.52986726036465
2.7.C,2.7,Fürth 2A,EW 7,315,65,377.61870702208313
2.7.C,2.7,Fürth 2A,EW 7,351,65,397.87835730188334
2.7.C,2.7,Fürth 2A,EW 7,385,65,318.33155929959685
2.7.C,2.7,Fürth 2A,EW 7,410,65,275.9354740959143
2.7.C,2.7,Fürth 2A,EW 7,448,65,382.5753434021301
2.7.C,2.7,Fürth 2A,EW 7,471,65,271.55630976593056
2.7.C,2.7,Fürth 2A,EW 7,499,65,398.45582935194653
2.7.C,2.7,Fürth 2A,EW 7,528,65,282.11442686082194
2.7.C,2.7,Fürth 2A,EW 7,560,65,433.5372666191855
2.7.C,2.7,Fürth 2A,EW 7,595,65,401.2998801269897
2.7.C,2.7,Fürth 2A,EW 7,626,65,260.7190812924965
2.7.C,2.7,Fürth 2A,EW 7,652,65,216.2922187668345
2.7.C,2.7,Fürth 2A,EW 7,680,65,254.79999109453036
2.7.D,2.7,Fürth 2A,EW 7,92,65,90.95187411998316
2.7.D,2.7,Fürth 2A,EW 7,115,65,293.7889901919489
2.7.D,2.7,Fürth 2A,EW 7,150,65,0.0
2.7.D,2.7,Fürth 2A,EW 7,177,65,826.2664964197605
2.7.D,2.7,Fürth 2A,EW 7,198,65,422.2765583970155
2.7.D,2.7,Fürth 2A,EW 7,228,65,210.199886828329
2.7.D,2.7,Fürth 2A,EW 7,259,65,323.7694227089476
2.7.D,2.7,Fürth 2A,EW 7,284,65,292.34530958541427
2.7.D,2.7,Fürth 2A,EW 7,315,65,326.56053841988086
2.7.D,2.7,Fürth 2A,EW 7,351,65,355.7228853721644
2.7.D,2.7,Fürth 2A,EW 7,385,65,277.18666393886514
2.7.D,2.7,Fürth 2A,EW 7,410,65,290.1797889163006
2.7.D,2.7,Fürth 2A,EW 7,448,65,383.77841049401286
2.7.D,2.7,Fürth 2A,EW 7,471,65,242.5383310668512
2.7.D,2.7,Fürth 2A,EW 7,499,65,441.28501883386485
2.7.D,2.7,Fürth 2A,EW 7,528,65,333.06672555508754
2.7.D,2.7,Fürth 2A,EW 7,560,65,452.16555795507406
2.7.D,2.7,Fürth 2A,EW 7,595,65,390.0319535709472
2.7.D,2.7,Fürth 2A,EW 7,626,65,249.4679975931163
2.7.D,2.7,Fürth 2A,EW 7,652,65,205.1181313425797
2.7.D,2.7,Fürth 2A,EW 7,680,65,225.44515335459417
2.8.A,L.1,Fürth 2B,Basanite,150,107,128.9687950418196
2.8.A,L.1,Fürth 2B,Basanite,177,107,378.9661421264817
2.8.A,L.1,Fürth 2B,Basanite,198,107,278.1491177567844
2.8.A,L.1,Fürth 2B,Basanite,228,107,113.56953592875624
2.8.A,L.1,Fürth 2B,Basanite,259,107,309.91008965641737
2.8.A,L.1,Fürth 2B,Basanite,284,107,258.7075530416992
2.8.A,L.1,Fürth 2B,Basanite,315,107,277.18666393886514
2.8.A,L.1,Fürth 2B,Basanite,351,107,375.9344129008966
2.8.A,L.1,Fürth 2B,Basanite,385,107,322.1813740898971
2.8.A,L.1,Fürth 2B,Basanite,415,107,350.8143715024971
2.8.A,L.1,Fürth 2B,Basanite,448,107,320.7376937240508
2.8.A,L.1,Fürth 2B,Basanite,469,107,330.31410794873335
2.8.A,L.1,Fürth 2B,Basanite,498,107,539.9365225344485
2.8.A,L.1,Fürth 2B,Basanite,528,107,343.51416017810936
2.8.B,L.1,Fürth 2B,Basanite,150,107,109.71972113845598
2.8.B,L.1,Fürth 2B,Basanite,177,107,278.2453630182322
2.8.B,L.1,Fürth 2B,Basanite,198,107,218.28449786389072
2.8.B,L.1,Fürth 2B,Basanite,228,107,53.41618003489981
2.8.B,L.1,Fürth 2B,Basanite,259,107,237.19671296708583
2.8.B,L.1,Fürth 2B,Basanite,284,107,280.4590064384139
2.8.B,L.1,Fürth 2B,Basanite,315,107,298.74562681268424
2.8.B,L.1,Fürth 2B,Basanite,351,107,334.93388555268064
2.8.B,L.1,Fürth 2B,Basanite,385,107,262.74985859558336
2.8.B,L.1,Fürth 2B,Basanite,415,107,238.2072893435225
2.8.B,L.1,Fürth 2B,Basanite,448,107,320.7376937240508
2.8.B,L.1,Fürth 2B,Basanite,469,107,170.9317761116794
2.8.B,L.1,Fürth 2B,Basanite,498,107,519.0993999638968
2.8.B,L.1,Fürth 2B,Basanite,528,107,275.9739722004934
2.8.B,L.1,Fürth 2B,Basanite,560,107,258.091582624776
2.8.B,L.1,Fürth 2B,Basanite,595,107,405.2844383404685
2.8.B,L.1,Fürth 2B,Basanite,626,107,317.0322469462663
2.8.B,L.1,Fürth 2B,Basanite,652,107,283.1683136602919
2.8.B,L.1,Fürth 2B,Basanite,680,107,296.5945426319273
2.8.B,L.1,Fürth 2B,Basanite,714,107,454.18189830916424
2.8.B,L.1,Fürth 2B,Basanite,742,107,
2.8.B,L.1,Fürth 2B,Basanite,770,107,323.75017365665803
2.8.B,L.1,Fürth 2B,Basanite,798,107,364.5967085865575
2.8.B,L.1,Fürth 2B,Basanite,833,107,457.35799554726515
2.8.B,L.1,Fürth 2B,Basanite,868,107,740.7043631987484
2.8.B,L.1,Fürth 2B,Basanite,892,107,628.140591371322
2.8.B,L.1,Fürth 2B,Basanite,925,107,446.48708105180816
2.8.B,L.1,Fürth 2B,Basanite,955,107,495.9620131175161
2.8.B,L.1,Fürth 2B,Basanite,987,107,293.5868750225645
2.8.B,L.1,Fürth 2B,Basanite,1023,107,
2.8.B,L.1,Fürth 2B,Basanite,1043,107,
2.8.C,L.1,Fürth 2B,Basanite,150,107,98.74774903423793
2.8.C,L.1,Fürth 2B,Basanite,177,107,288.73610830976594
2.8.C,L.1,Fürth 2B,Basanite,198,107,215.97460898971056
2.8.C,L.1,Fürth 2B,Basanite,228,107,129.49814455743424
2.8.C,L.1,Fürth 2B,Basanite,259,107,246.38814561646308
2.8.C,L.1,Fürth 2B,Basanite,284,107,230.7001505265058
2.8.C,L.1,Fürth 2B,Basanite,315,107,304.90533028461397
2.8.C,L.1,Fürth 2B,Basanite,351,107,346.9164342018172
2.8.C,L.1,Fürth 2B,Basanite,385,107,274.29930296648416
2.8.C,L.1,Fürth 2B,Basanite,415,107,304.52034875744624
2.8.C,L.1,Fürth 2B,Basanite,448,107,327.5229922378001
2.8.C,L.1,Fürth 2B,Basanite,469,107,313.2786774174138
2.8.C,L.1,Fürth 2B,Basanite,498,107,589.0216609904327
2.8.C,L.1,Fürth 2B,Basanite,528,107,454.3936381250376
2.8.C,L.1,Fürth 2B,Basanite,560,107,617.1253088352718
2.8.C,L.1,Fürth 2B,Basanite,595,107,639.1654984365315
2.8.C,L.1,Fürth 2B,Basanite,626,107,400.2267443287803
2.8.C,L.1,Fürth 2B,Basanite,652,107,365.0298127035612
2.8.C,L.1,Fürth 2B,Basanite,680,107,353.93272158372946
2.8.C,L.1,Fürth 2B,Basanite,714,107,368.6967612973103
2.8.C,L.1,Fürth 2B,Basanite,742,107,
2.8.C,L.1,Fürth 2B,Basanite,770,107,375.22700956736264
2.8.C,L.1,Fürth 2B,Basanite,798,107,391.59834695228346
2.8.C,L.1,Fürth 2B,Basanite,833,107,522.9732761297311
2.8.C,L.1,Fürth 2B,Basanite,868,107,694.6990764787291
2.8.C,L.1,Fürth 2B,Basanite,892,107,726.7054742162585
2.8.C,L.1,Fürth 2B,Basanite,925,107,451.7468904266201
2.8.C,L.1,Fürth 2B,Basanite,955,107,328.19670978999943
2.8.C,L.1,Fürth 2B,Basanite,987,107,147.92913279980746
2.8.C,L.1,Fürth 2B,Basanite,1023,107,
2.8.C,L.1,Fürth 2B,Basanite,1043,107,
2.8.D,L.1,Fürth 2B,Basanite,150,107,98.17027681569289
2.8.D,L.1,Fürth 2B,Basanite,177,107,326.7049064323966
2.8.D,L.1,Fürth 2B,Basanite,198,107,418.8598476442626
2.8.D,L.1,Fürth 2B,Basanite,228,107,270.2569974126
2.8.D,L.1,Fürth 2B,Basanite,259,107,274.49179348937963
2.8.D,L.1,Fürth 2B,Basanite,284,107,265.63721956796434
2.8.D,L.1,Fürth 2B,Basanite,315,107,531.9962794391961
2.8.D,L.1,Fürth 2B,Basanite,351,107,418.2823755941994
2.8.D,L.1,Fürth 2B,Basanite,385,107,343.45160069799624
2.8.D,L.1,Fürth 2B,Basanite,415,107,377.2818482459835
2.8.D,L.1,Fürth 2B,Basanite,448,107,413.8550885131475
2.8.D,L.1,Fürth 2B,Basanite,469,107,346.48332992358144
2.8.D,L.1,Fürth 2B,Basanite,498,107,529.5901452554306
2.8.D,L.1,Fürth 2B,Basanite,528,107,383.1046929418136
2.8.D,L.1,Fürth 2B,Basanite,560,107,445.6738076515303
2.8.D,L.1,Fürth 2B,Basanite,595,107,411.5163260432013
2.8.D,L.1,Fürth 2B,Basanite,626,107,289.1595879415127
2.8.D,L.1,Fürth 2B,Basanite,652,107,238.0003617940122
2.8.D,L.1,Fürth 2B,Basanite,680,107,252.84621000060173
2.9.A,L.2,Fürth 2B,Basanite 200,150,107,134.1660449846561
2.9.A,L.2,Fürth 2B,Basanite 200,177,107,222.32680336963716
2.9.A,L.2,Fürth 2B,Basanite 200,198,107,379.3992464047175
2.9.A,L.2,Fürth 2B,Basanite 200,228,107,365.0105635718154
2.9.A,L.2,Fürth 2B,Basanite 200,259,107,471.21732859979534
2.9.A,L.2,Fürth 2B,Basanite 200,284,107,337.72500126361393
2.9.A,L.2,Fürth 2B,Basanite 200,315,107,351.295598531801
2.9.A,L.2,Fürth 2B,Basanite 200,351,107,414.4806835549672
2.9.A,L.2,Fürth 2B,Basanite 200,385,107,381.6128898248992
2.9.A,L.2,Fürth 2B,Basanite 200,415,107,420.2072829893495
2.9.A,L.2,Fürth 2B,Basanite 200,448,107,433.58538925326434
2.9.A,L.2,Fürth 2B,Basanite 200,469,107,315.2998303147
2.9.A,L.2,Fürth 2B,Basanite 200,498,107,541.2839576388471
2.9.A,L.2,Fürth 2B,Basanite 200,528,107,383.51373584451534
2.9.A,L.2,Fürth 2B,Basanite 200,560,107,516.089326034261
2.9.A,L.2,Fürth 2B,Basanite 200,595,107,581.7286679782052
2.9.A,L.2,Fürth 2B,Basanite 200,626,107,384.32700908598594
2.9.A,L.2,Fürth 2B,Basanite 200,652,107,298.92849291092693
2.9.A,L.2,Fürth 2B,Basanite 200,680,107,295.8486411938143
2.9.B,L.2,Fürth 2B,Basanite 200,150,107,91.43310095673628
2.9.B,L.2,Fürth 2B,Basanite 200,177,107,288.44737228473434
2.9.B,L.2,Fürth 2B,Basanite 200,198,107,240.85403694566455
2.9.B,L.2,Fürth 2B,Basanite 200,228,107,133.54045008724952
2.9.B,L.2,Fürth 2B,Basanite 200,259,107,338.7837003429809
2.9.B,L.2,Fürth 2B,Basanite 200,284,107,265.25223804079667
2.9.B,L.2,Fürth 2B,Basanite 200,315,107,132.04864684999097
2.9.B,L.2,Fürth 2B,Basanite 200,351,107,
2.9.B,L.2,Fürth 2B,Basanite 200,385,107,534.2580456104458
2.9.B,L.2,Fürth 2B,Basanite 200,415,107,293.3558859137132
2.9.B,L.2,Fürth 2B,Basanite 200,448,107,374.87571382152953
2.9.B,L.2,Fürth 2B,Basanite 200,469,107,184.2136370900776
2.9.B,L.2,Fürth 2B,Basanite 200,498,107,617.6065356519646
2.9.B,L.2,Fürth 2B,Basanite 200,528,107,383.8794683193934
2.9.B,L.2,Fürth 2B,Basanite 200,560,107,404.12949400869417
2.9.B,L.2,Fürth 2B,Basanite 200,595,107,444.1699738298321
2.9.B,L.2,Fürth 2B,Basanite 200,626,107,346.3774600156447
2.9.B,L.2,Fürth 2B,Basanite 200,652,107,280.90654751459334
2.9.B,L.2,Fürth 2B,Basanite 200,680,107,299.5059652205307
2.9.B,L.2,Fürth 2B,Basanite 200,714,107,373.3117265780131
2.9.C,L.2,Fürth 2B,Basanite 200,150,107,99.46958931343642
2.9.C,L.2,Fürth 2B,Basanite 200,177,107,232.14383106083395
2.9.C,L.2,Fürth 2B,Basanite 200,198,107,217.22579880859257
2.9.C,L.2,Fürth 2B,Basanite 200,228,107,41.81861300920633
2.9.C,L.2,Fürth 2B,Basanite 200,259,107,207.69750721463384
2.9.C,L.2,Fürth 2B,Basanite 200,284,107,203.5589563511643
2.9.C,L.2,Fürth 2B,Basanite 200,315,107,246.38814561646308
2.9.C,L.2,Fürth 2B,Basanite 200,351,107,292.00845080931464
2.9.C,L.2,Fürth 2B,Basanite 200,385,107,271.50818725555087
2.9.C,L.2,Fürth 2B,Basanite 200,415,107,276.320455623082
2.9.C,L.2,Fürth 2B,Basanite 200,448,107,250.9598007100307
2.9.C,L.2,Fürth 2B,Basanite 200,469,107,497.2035785546664
2.9.C,L.2,Fürth 2B,Basanite 200,498,107,505.2881894217462
2.9.C,L.2,Fürth 2B,Basanite 200,528,107,416.3574681990493
2.9.C,L.2,Fürth 2B,Basanite 200,560,107,539.8763692176268
2.9.C,L.2,Fürth 2B,Basanite 200,595,107,559.7438196048355
2.9.C,L.2,Fürth 2B,Basanite 200,626,107,401.53568132859976
2.9.C,L.2,Fürth 2B,Basanite 200,652,107,333.9882748420768
2.9.C,L.2,Fürth 2B,Basanite 200,680,107,370.3714305313195
2.9.C,L.2,Fürth 2B,Basanite 200,714,107,382.3732282327457
3.0.A,A.0,Farmer 1,Control,35,-2,
3.0.A,A.0,Farmer 1,Control,57,-2,
3.0.A,A.0,Farmer 1,Control,87,-2,
3.0.A,A.0,Farmer 1,Control,115,-2,
3.0.A,A.0,Farmer 1,Control,149,-2,
3.0.A,A.0,Farmer 1,Control,177,-2,
3.0.A,A.0,Farmer 1,Control,198,-2,
3.0.A,A.0,Farmer 1,Control,226,-2,
3.0.A,A.0,Farmer 1,Control,255,-2,0.0
3.0.A,A.0,Farmer 1,Control,284,-2,47.54521249172634
3.0.A,A.0,Farmer 1,Control,315,-2,11.356953592875623
3.0.A,A.0,Farmer 1,Control,354,-2,46.48651343642818
3.0.A,A.0,Farmer 1,Control,385,-2,11.549444332390635
3.0.A,A.0,Farmer 1,Control,410,-2,202.69274801131235
3.0.A,A.0,Farmer 1,Control,450,-2,72.18402707744148
3.0.A,A.0,Farmer 1,Control,470,-2,19.249073886515436
3.0.A,A.0,Farmer 1,Control,499,-2,58.51718462001323
3.0.A,A.0,Farmer 1,Control,527,-2,
3.0.B,A.0,Farmer 1,Control,35,-2,
3.0.B,A.0,Farmer 1,Control,57,-2,
3.0.B,A.0,Farmer 1,Control,87,-2,
3.0.B,A.0,Farmer 1,Control,115,-2,
3.0.B,A.0,Farmer 1,Control,149,-2,
3.0.B,A.0,Farmer 1,Control,177,-2,
3.0.B,A.0,Farmer 1,Control,198,-2,
3.0.B,A.0,Farmer 1,Control,226,-2,
3.0.B,A.0,Farmer 1,Control,255,-2,0.0
3.0.B,A.0,Farmer 1,Control,284,-2,28.584874709669656
3.0.B,A.0,Farmer 1,Control,315,-2,60.63458273060954
3.0.B,A.0,Farmer 1,Control,354,-2,40.423055153739696
3.0.B,A.0,Farmer 1,Control,385,-2,20.788999795414885
3.0.B,A.0,Farmer 1,Control,410,-2,74.59016130934472
3.0.B,A.0,Farmer 1,Control,450,-2,82.57852696311451
3.0.B,A.0,Farmer 1,Control,470,-2,34.35959689512004
3.0.B,A.0,Farmer 1,Control,499,-2,65.8318326975149
3.0.B,A.0,Farmer 1,Control,527,-2,
3.0.C,A.0,Farmer 1,Control,35,-2,
3.0.C,A.0,Farmer 1,Control,57,-2,
3.0.C,A.0,Farmer 1,Control,87,-2,
3.0.C,A.0,Farmer 1,Control,115,-2,
3.0.C,A.0,Farmer 1,Control,149,-2,
3.0.C,A.0,Farmer 1,Control,177,-2,
3.0.C,A.0,Farmer 1,Control,198,-2,
3.0.C,A.0,Farmer 1,Control,226,-2,
3.0.C,A.0,Farmer 1,Control,255,-2,0.0
3.0.C,A.0,Farmer 1,Control,284,-2,4.138550885131476
3.0.C,A.0,Farmer 1,Control,315,-2,20.788999795414885
3.0.C,A.0,Farmer 1,Control,354,-2,48.17080738913292
3.0.C,A.0,Farmer 1,Control,382,-2,125.6002071123413
3.0.C,A.0,Farmer 1,Control,410,-2,92.58804539382636
3.0.C,A.0,Farmer 1,Control,450,-2,114.33949888681629
3.0.C,A.0,Farmer 1,Control,470,-2,97.59280459714785
3.0.C,A.0,Farmer 1,Control,499,-2,199.90163230037908
3.0.C,A.0,Farmer 1,Control,527,-2,
3.0.D,A.0,Farmer 1,Control,35,-2,
3.0.D,A.0,Farmer 1,Control,57,-2,
3.0.D,A.0,Farmer 1,Control,87,-2,
3.0.D,A.0,Farmer 1,Control,115,-2,
3.0.D,A.0,Farmer 1,Control,149,-2,
3.0.D,A.0,Farmer 1,Control,177,-2,
3.0.D,A.0,Farmer 1,Control,198,-2,
3.0.D,A.0,Farmer 1,Control,226,-2,
3.0.D,A.0,Farmer 1,Control,255,-2,0.0
3.0.D,A.0,Farmer 1,Control,284,-2,21.655208120825563
3.0.D,A.0,Farmer 1,Control,315,-2,24.35007846440821
3.0.D,A.0,Farmer 1,Control,354,-2,73.62770761176967
3.0.D,A.0,Farmer 1,Control,382,-2,92.39555465431133
3.0.D,A.0,Farmer 1,Control,410,-2,49.75885598411457
3.0.D,A.0,Farmer 1,Control,450,-2,187.678470377279
3.0.D,A.0,Farmer 1,Control,470,-2,68.62294840844815
3.0.D,A.0,Farmer 1,Control,499,-2,132.433628329021
3.0.D,A.0,Farmer 1,Control,527,-2,
3.1.A,A.1,Farmer 1,Basanite,35,-2,
3.1.A,A.1,Farmer 1,Basanite,57,-2,
3.1.A,A.1,Farmer 1,Basanite,87,-2,
3.1.A,A.1,Farmer 1,Basanite,115,-2,
3.1.A,A.1,Farmer 1,Basanite,149,-2,
3.1.A,A.1,Farmer 1,Basanite,177,-2,
3.1.A,A.1,Farmer 1,Basanite,198,-2,
3.1.A,A.1,Farmer 1,Basanite,226,-2,
3.1.A,A.1,Farmer 1,Basanite,255,-2,0.0
3.1.A,A.1,Farmer 1,Basanite,284,-2,40.904282014561645
3.1.A,A.1,Farmer 1,Basanite,315,-2,12.704388764666946
3.1.A,A.1,Farmer 1,Basanite,354,-2,49.18138378963836
3.1.A,A.1,Farmer 1,Basanite,385,-2,43.117925506949874
3.1.A,A.1,Farmer 1,Basanite,410,-2,56.44790916420964
3.1.A,A.1,Farmer 1,Basanite,450,-2,121.26916548528791
3.1.A,A.1,Farmer 1,Basanite,470,-2,35.41829595041819
3.1.A,A.1,Farmer 1,Basanite,499,-2,116.07191551838255
3.1.A,A.1,Farmer 1,Basanite,527,-2,
3.1.B,A.1,Farmer 1,Basanite,35,-2,
3.1.B,A.1,Farmer 1,Basanite,57,-2,
3.1.B,A.1,Farmer 1,Basanite,87,-2,
3.1.B,A.1,Farmer 1,Basanite,115,-2,
3.1.B,A.1,Farmer 1,Basanite,149,-2,
3.1.B,A.1,Farmer 1,Basanite,177,-2,
3.1.B,A.1,Farmer 1,Basanite,198,-2,
3.1.B,A.1,Farmer 1,Basanite,226,-2,
3.1.B,A.1,Farmer 1,Basanite,255,-2,0.0
3.1.B,A.1,Farmer 1,Basanite,284,-2,20.404018318791746
3.1.B,A.1,Farmer 1,Basanite,315,-2,50.81755506348156
3.1.B,A.1,Farmer 1,Basanite,354,-2,64.09941604187978
3.1.B,A.1,Farmer 1,Basanite,385,-2,181.08566257897587
3.1.B,A.1,Farmer 1,Basanite,410,-2,45.0428329021
3.1.B,A.1,Farmer 1,Basanite,450,-2,163.61712803417774
3.1.B,A.1,Farmer 1,Basanite,470,-2,132.818609808051
3.1.B,A.1,Farmer 1,Basanite,499,-2,97.40031385763282
3.1.B,A.1,Farmer 1,Basanite,527,-2,
3.1.C,A.1,Farmer 1,Basanite,35,-2,
3.1.C,A.1,Farmer 1,Basanite,57,-2,
3.1.C,A.1,Farmer 1,Basanite,87,-2,
3.1.C,A.1,Farmer 1,Basanite,115,-2,
3.1.C,A.1,Farmer 1,Basanite,149,-2,
3.1.C,A.1,Farmer 1,Basanite,177,-2,
3.1.C,A.1,Farmer 1,Basanite,198,-2,
3.1.C,A.1,Farmer 1,Basanite,226,-2,
3.1.C,A.1,Farmer 1,Basanite,255,-2,0.0
3.1.C,A.1,Farmer 1,Basanite,284,-2,25.40877751970636
3.1.C,A.1,Farmer 1,Basanite,315,-2,47.64145787351826
3.1.C,A.1,Farmer 1,Basanite,354,-2,77.28503166255491
3.1.C,A.1,Farmer 1,Basanite,382,-2,112.22210075215116
3.1.C,A.1,Farmer 1,Basanite,410,-2,70.06662894277633
3.1.C,A.1,Farmer 1,Basanite,450,-2,129.93124873939465
3.1.C,A.1,Farmer 1,Basanite,470,-2,79.25806171249774
3.1.C,A.1,Farmer 1,Basanite,499,-2,205.86884520127563
3.1.C,A.1,Farmer 1,Basanite,527,-2,
3.1.D,A.1,Farmer 1,Basanite,35,-2,
3.1.D,A.1,Farmer 1,Basanite,57,-2,
3.1.D,A.1,Farmer 1,Basanite,87,-2,
3.1.D,A.1,Farmer 1,Basanite,115,-2,
3.1.D,A.1,Farmer 1,Basanite,149,-2,
3.1.D,A.1,Farmer 1,Basanite,177,-2,
3.1.D,A.1,Farmer 1,Basanite,198,-2,
3.1.D,A.1,Farmer 1,Basanite,226,-2,
3.1.D,A.1,Farmer 1,Basanite,255,-2,0.0
3.1.D,A.1,Farmer 1,Basanite,284,-2,11.068217483603105
3.1.D,A.1,Farmer 1,Basanite,315,-2,39.268110716649616
3.1.D,A.1,Farmer 1,Basanite,354,-2,73.14648077501656
3.1.D,A.1,Farmer 1,Basanite,382,-2,92.1068185330044
3.1.D,A.1,Farmer 1,Basanite,410,-2,88.01639034839641
3.1.D,A.1,Farmer 1,Basanite,450,-2,172.0385978458391
3.1.D,A.1,Farmer 1,Basanite,470,-2,103.22315872194476
3.1.D,A.1,Farmer 1,Basanite,499,-2,252.6440948312173
3.1.D,A.1,Farmer 1,Basanite,527,-2,
3.11.A,L.0,Fürth 2B,Control,150,107,152.45266516637582
3.11.A,L.0,Fürth 2B,Control,177,107,531.2744393120422
3.11.A,L.0,Fürth 2B,Control,211,107,536.6641800348998
3.11.A,L.0,Fürth 2B,Control,255,107,400.3807367470967
3.11.A,L.0,Fürth 2B,Control,284,107,277.18666393886514
3.11.A,L.0,Fürth 2B,Control,315,107,440.75566929418136
3.11.A,L.0,Fürth 2B,Control,354,107,457.9354678380167
3.11.A,L.0,Fürth 2B,Control,385,107,268.52458078103376
3.11.A,L.0,Fürth 2B,Control,410,107,339.2168046212167
3.11.A,L.0,Fürth 2B,Control,448,107,343.0184966604489
3.11.A,L.0,Fürth 2B,Control,469,107,216.5520812082556
3.11.A,L.0,Fürth 2B,Control,498,107,352.5467881340634
3.11.A,L.0,Fürth 2B,Control,528,107,336.0166460075817
3.11.A,L.0,Fürth 2B,Control,559,107,477.8799144152276
3.11.A,L.0,Fürth 2B,Control,595,107,419.2231740176906
3.11.A,L.0,Fürth 2B,Control,626,107,275.4542473072989
3.11.A,L.0,Fürth 2B,Control,652,107,47.208353711176585
3.11.A,L.0,Fürth 2B,Control,680,107,271.1039566760936
3.11.B,L.0,Fürth 2B,Control,150,107,175.16657235694083
3.11.B,L.0,Fürth 2B,Control,177,107,560.1480500152214
3.11.B,L.0,Fürth 2B,Control,211,107,551.1972306396293
3.11.B,L.0,Fürth 2B,Control,255,107,436.56899572778144
3.11.B,L.0,Fürth 2B,Control,284,107,296.2913698778506
3.11.B,L.0,Fürth 2B,Control,315,107,334.64514952764904
3.11.B,L.0,Fürth 2B,Control,354,107,377.2818482459835
3.11.B,L.0,Fürth 2B,Control,385,107,277.18666393886514
3.11.B,L.0,Fürth 2B,Control,410,107,356.5890936879475
3.11.B,L.0,Fürth 2B,Control,448,107,428.4843846200132
3.11.B,L.0,Fürth 2B,Control,469,107,218.6694793429207
3.11.B,L.0,Fürth 2B,Control,498,107,353.8461007280823
3.11.B,L.0,Fürth 2B,Control,528,107,387.68597268187017
3.11.B,L.0,Fürth 2B,Control,559,107,510.19670341806864
3.11.B,L.0,Fürth 2B,Control,595,107,438.8788846607043
3.11.B,L.0,Fürth 2B,Control,626,107,229.6895741500692
3.11.B,L.0,Fürth 2B,Control,652,107,50.92583110647394
3.11.B,L.0,Fürth 2B,Control,680,107,351.96931608400024
3.11.C,L.0,Fürth 2B,Control,150,107,196.9180258499308
3.11.C,L.0,Fürth 2B,Control,177,107,522.9732760640889
3.11.C,L.0,Fürth 2B,Control,211,107,366.6948574523136
3.11.C,L.0,Fürth 2B,Control,255,107,418.8598476442626
3.11.C,L.0,Fürth 2B,Control,284,107,240.37281015704917
3.11.C,L.0,Fürth 2B,Control,315,107,394.1247877730309
3.11.C,L.0,Fürth 2B,Control,354,107,424.0089750285817
3.11.C,L.0,Fürth 2B,Control,385,107,257.26387243516456
3.11.C,L.0,Fürth 2B,Control,410,107,396.33843119321256
3.11.C,L.0,Fürth 2B,Control,448,107,402.78687117155056
3.11.C,L.0,Fürth 2B,Control,469,107,229.06397922859375
3.11.C,L.0,Fürth 2B,Control,498,107,425.01955135688064
3.11.C,L.0,Fürth 2B,Control,528,107,409.71653769781574
3.11.C,L.0,Fürth 2B,Control,559,107,499.7540807719822
3.11.C,L.0,Fürth 2B,Control,595,107,539.2868662658852
3.11.C,L.0,Fürth 2B,Control,626,107,375.8766657440279
3.11.C,L.0,Fürth 2B,Control,652,107,59.364143861286976
3.11.C,L.0,Fürth 2B,Control,680,107,207.3029012094591
3.11.C,L.0,Fürth 2B,Control,742,107,
3.11.C,L.0,Fürth 2B,Control,771,107,257.26387243516456
3.11.C,L.0,Fürth 2B,Control,798,107,279.48692821469405
3.11.C,L.0,Fürth 2B,Control,833,107,345.34763463505624
3.11.C,L.0,Fürth 2B,Control,868,107,567.1547130392923
3.11.C,L.0,Fürth 2B,Control,892,107,471.7514904627234
3.11.C,L.0,Fürth 2B,Control,925,107,274.8382769119682
3.11.C,L.0,Fürth 2B,Control,955,107,396.57904470786445
3.11.C,L.0,Fürth 2B,Control,987,107,399.6973947890968
3.11.C,L.0,Fürth 2B,Control,1023,107,
3.11.C,L.0,Fürth 2B,Control,1043,107,
3.11.D,L.0,Fürth 2B,Control,150,107,164.09835487093088
3.11.D,L.0,Fürth 2B,Control,177,107,436.7133638633157
3.11.D,L.0,Fürth 2B,Control,211,107,267.65837246525064
3.11.D,L.0,Fürth 2B,Control,255,107,340.1311354473795
3.11.D,L.0,Fürth 2B,Control,284,107,275.9354740959143
3.11.D,L.0,Fürth 2B,Control,315,107,208.99681971237737
3.11.D,L.0,Fürth 2B,Control,354,107,407.93599831518145
3.11.D,L.0,Fürth 2B,Control,385,107,505.2881894217462
3.11.D,L.0,Fürth 2B,Control,410,107,293.3558859137132
3.11.D,L.0,Fürth 2B,Control,448,107,428.7731208857332
3.11.D,L.0,Fürth 2B,Control,469,107,266.1184463565798
3.11.D,L.0,Fürth 2B,Control,498,107,488.4452498946988
3.11.D,L.0,Fürth 2B,Control,528,107,372.5562004934111
3.11.D,L.0,Fürth 2B,Control,559,107,542.1501659464558
3.11.D,L.0,Fürth 2B,Control,595,107,547.1934233040972
3.11.D,L.0,Fürth 2B,Control,626,107,325.7376405319213
3.11.D,L.0,Fürth 2B,Control,652,107,41.12324022896789
3.11.D,L.0,Fürth 2B,Control,680,107,265.0597475179012
3.11.D,L.0,Fürth 2B,Control,742,107,
3.11.D,L.0,Fürth 2B,Control,771,107,119.97466526265116
3.11.D,L.0,Fürth 2B,Control,798,107,135.1284986822312
3.11.D,L.0,Fürth 2B,Control,833,107,215.75805692279917
3.11.D,L.0,Fürth 2B,Control,868,107,211.61469376015404
3.11.D,L.0,Fürth 2B,Control,892,107,166.0040131897226
3.11.D,L.0,Fürth 2B,Control,925,107,119.78698679824298
3.11.D,L.0,Fürth 2B,Control,955,107,192.15388006498583
3.11.D,L.0,Fürth 2B,Control,987,107,88.42543315482278
3.11.D,L.0,Fürth 2B,Control,1023,107,
3.11.D,L.0,Fürth 2B,Control,1043,107,
3.2.A,B.0,Farmer 2,Control,35,-2,
3.2.A,B.0,Farmer 2,Control,57,-2,
3.2.A,B.0,Farmer 2,Control,87,-2,
3.2.A,B.0,Farmer 2,Control,115,-2,
3.2.A,B.0,Farmer 2,Control,149,-2,
3.2.A,B.0,Farmer 2,Control,177,-2,
3.2.A,B.0,Farmer 2,Control,198,-2,
3.2.A,B.0,Farmer 2,Control,226,-2,
3.2.A,B.0,Farmer 2,Control,255,-2,86.81332323244479
3.2.A,B.0,Farmer 2,Control,284,-2,155.91749847764606
3.2.A,B.0,Farmer 2,Control,315,-2,425.5970236476322
3.2.A,B.0,Farmer 2,Control,354,-2,206.59068548047415
3.2.A,B.0,Farmer 2,Control,385,-2,171.50924833022444
3.2.A,B.0,Farmer 2,Control,410,-2,220.20940525904084
3.2.A,B.0,Farmer 2,Control,450,-2,286.3299741260003
3.2.A,B.0,Farmer 2,Control,470,-2,161.69222063902762
3.2.A,B.0,Farmer 2,Control,499,-2,285.2712750466334
3.2.A,B.0,Farmer 2,Control,527,-2,
3.2.B,B.0,Farmer 2,Control,35,-2,
3.2.B,B.0,Farmer 2,Control,57,-2,
3.2.B,B.0,Farmer 2,Control,87,-2,
3.2.B,B.0,Farmer 2,Control,115,-2,
3.2.B,B.0,Farmer 2,Control,149,-2,
3.2.B,B.0,Farmer 2,Control,177,-2,
3.2.B,B.0,Farmer 2,Control,198,-2,
3.2.B,B.0,Farmer 2,Control,226,-2,
3.2.B,B.0,Farmer 2,Control,255,-2,14.629296152596424
3.2.B,B.0,Farmer 2,Control,284,-2,110.87466557554606
3.2.B,B.0,Farmer 2,Control,315,-2,187.870961116794
3.2.B,B.0,Farmer 2,Control,354,-2,117.03436921595762
3.2.B,B.0,Farmer 2,Control,385,-2,215.97460898971056
3.2.B,B.0,Farmer 2,Control,410,-2,117.61184143450268
3.2.B,B.0,Farmer 2,Control,450,-2,144.36805413081413
3.2.B,B.0,Farmer 2,Control,470,-2,109.71972113845598
3.2.B,B.0,Farmer 2,Control,499,-2,173.43415570130574
3.2.B,B.0,Farmer 2,Control,527,-2,
3.2.C,B.0,Farmer 2,Control,35,-2,
3.2.C,B.0,Farmer 2,Control,57,-2,
3.2.C,B.0,Farmer 2,Control,87,-2,
3.2.C,B.0,Farmer 2,Control,115,-2,
3.2.C,B.0,Farmer 2,Control,149,-2,
3.2.C,B.0,Farmer 2,Control,177,-2,
3.2.C,B.0,Farmer 2,Control,198,-2,
3.2.C,B.0,Farmer 2,Control,226,-2,
3.2.C,B.0,Farmer 2,Control,255,-2,167.46694280040916
3.2.C,B.0,Farmer 2,Control,284,-2,220.20940525904084
3.2.C,B.0,Farmer 2,Control,315,-2,200.19036842168603
3.2.C,B.0,Farmer 2,Control,354,-2,270.69010145014744
3.2.C,B.0,Farmer 2,Control,382,-2,228.2458935916722
3.2.C,B.0,Farmer 2,Control,410,-2,261.2099327276009
3.2.C,B.0,Farmer 2,Control,450,-2,485.076661893014
3.2.C,B.0,Farmer 2,Control,470,-2,287.2924277032312
3.2.C,B.0,Farmer 2,Control,499,-2,72.6652539141946
3.2.C,B.0,Farmer 2,Control,527,-2,
3.2.D,B.0,Farmer 2,Control,35,-2,
3.2.D,B.0,Farmer 2,Control,57,-2,
3.2.D,B.0,Farmer 2,Control,87,-2,
3.2.D,B.0,Farmer 2,Control,115,-2,
3.2.D,B.0,Farmer 2,Control,149,-2,
3.2.D,B.0,Farmer 2,Control,177,-2,
3.2.D,B.0,Farmer 2,Control,198,-2,
3.2.D,B.0,Farmer 2,Control,226,-2,
3.2.D,B.0,Farmer 2,Control,255,-2,83.54098066068957
3.2.D,B.0,Farmer 2,Control,284,-2,177.86144271015104
3.2.D,B.0,Farmer 2,Control,315,-2,107.60232302785968
3.2.D,B.0,Farmer 2,Control,354,-2,172.0385978458391
3.2.D,B.0,Farmer 2,Control,382,-2,320.25646669474696
3.2.D,B.0,Farmer 2,Control,410,-2,166.88947058186412
3.2.D,B.0,Farmer 2,Control,450,-2,198.939178602804
3.2.D,B.0,Farmer 2,Control,470,-2,105.38867953547144
3.2.D,B.0,Farmer 2,Control,499,-2,208.41934749383236
3.2.D,B.0,Farmer 2,Control,527,-2,
3.3.A,B.1,Farmer 2,Basanite,35,-2,
3.3.A,B.1,Farmer 2,Basanite,57,-2,
3.3.A,B.1,Farmer 2,Basanite,87,-2,
3.3.A,B.1,Farmer 2,Basanite,115,-2,
3.3.A,B.1,Farmer 2,Basanite,149,-2,
3.3.A,B.1,Farmer 2,Basanite,177,-2,
3.3.A,B.1,Farmer 2,Basanite,198,-2,
3.3.A,B.1,Farmer 2,Basanite,226,-2,
3.3.A,B.1,Farmer 2,Basanite,255,-2,145.5229985679042
3.3.A,B.1,Farmer 2,Basanite,284,-2,238.2072893435225
3.3.A,B.1,Farmer 2,Basanite,315,-2,281.08460148023346
3.3.A,B.1,Farmer 2,Basanite,354,-2,255.6277013057344
3.3.A,B.1,Farmer 2,Basanite,385,-2,317.6097189963295
3.3.A,B.1,Farmer 2,Basanite,410,-2,254.66524748781512
3.3.A,B.1,Farmer 2,Basanite,450,-2,212.70226644202415
3.3.A,B.1,Farmer 2,Basanite,470,-2,127.81385060472952
3.3.A,B.1,Farmer 2,Basanite,499,-2,269.9682611468801
3.3.A,B.1,Farmer 2,Basanite,527,-2,
3.3.B,B.1,Farmer 2,Basanite,35,-2,
3.3.B,B.1,Farmer 2,Basanite,57,-2,
3.3.B,B.1,Farmer 2,Basanite,87,-2,
3.3.B,B.1,Farmer 2,Basanite,115,-2,
3.3.B,B.1,Farmer 2,Basanite,149,-2,
3.3.B,B.1,Farmer 2,Basanite,177,-2,
3.3.B,B.1,Farmer 2,Basanite,198,-2,
3.3.B,B.1,Farmer 2,Basanite,226,-2,
3.3.B,B.1,Farmer 2,Basanite,255,-2,225.2622871412239
3.3.B,B.1,Farmer 2,Basanite,284,-2,304.13536747096697
3.3.B,B.1,Farmer 2,Basanite,315,-2,400.4769822492328
3.3.B,B.1,Farmer 2,Basanite,354,-2,359.95768168963235
3.3.B,B.1,Farmer 2,Basanite,385,-2,551.774702930381
3.3.B,B.1,Farmer 2,Basanite,410,-2,260.7287056982971
3.3.B,B.1,Farmer 2,Basanite,450,-2,572.6599480113124
3.3.B,B.1,Farmer 2,Basanite,470,-2,275.6467380708827
3.3.B,B.1,Farmer 2,Basanite,499,-2,81.85668670798484
3.3.B,B.1,Farmer 2,Basanite,527,-2,
3.3.C,B.1,Farmer 2,Basanite,35,-2,
3.3.C,B.1,Farmer 2,Basanite,57,-2,
3.3.C,B.1,Farmer 2,Basanite,87,-2,
3.3.C,B.1,Farmer 2,Basanite,115,-2,
3.3.C,B.1,Farmer 2,Basanite,149,-2,
3.3.C,B.1,Farmer 2,Basanite,177,-2,
3.3.C,B.1,Farmer 2,Basanite,198,-2,
3.3.C,B.1,Farmer 2,Basanite,226,-2,
3.3.C,B.1,Farmer 2,Basanite,255,-2,148.55472772128286
3.3.C,B.1,Farmer 2,Basanite,284,-2,198.65044250556593
3.3.C,B.1,Farmer 2,Basanite,315,-2,235.22368289307417
3.3.C,B.1,Farmer 2,Basanite,354,-2,343.30723268548047
3.3.C,B.1,Farmer 2,Basanite,382,-2,216.5520812082556
3.3.C,B.1,Farmer 2,Basanite,410,-2,307.02272844334794
3.3.C,B.1,Farmer 2,Basanite,450,-2,346.48332992358144
3.3.C,B.1,Farmer 2,Basanite,470,-2,316.93600144413017
3.3.C,B.1,Farmer 2,Basanite,499,-2,134.74351720320115
3.3.C,B.1,Farmer 2,Basanite,527,-2,
3.3.D,B.1,Farmer 2,Basanite,35,-2,
3.3.D,B.1,Farmer 2,Basanite,57,-2,
3.3.D,B.1,Farmer 2,Basanite,87,-2,
3.3.D,B.1,Farmer 2,Basanite,115,-2,
3.3.D,B.1,Farmer 2,Basanite,149,-2,
3.3.D,B.1,Farmer 2,Basanite,177,-2,
3.3.D,B.1,Farmer 2,Basanite,198,-2,
3.3.D,B.1,Farmer 2,Basanite,226,-2,
3.3.D,B.1,Farmer 2,Basanite,255,-2,275.8873515855346
3.3.D,B.1,Farmer 2,Basanite,284,-2,308.3220410373669
3.3.D,B.1,Farmer 2,Basanite,315,-2,351.1031077682171
3.3.D,B.1,Farmer 2,Basanite,354,-2,260.7287056982971
3.3.D,B.1,Farmer 2,Basanite,382,-2,293.0671498886816
3.3.D,B.1,Farmer 2,Basanite,410,-2,346.72394343823333
3.3.D,B.1,Farmer 2,Basanite,450,-2,484.3548215897467
3.3.D,B.1,Farmer 2,Basanite,470,-2,252.06662254046572
3.3.D,B.1,Farmer 2,Basanite,499,-2,187.38973428004093
3.3.D,B.1,Farmer 2,Basanite,527,-2,
3.4.A,C.0,Farmer 3,Control,35,-2,
3.4.A,C.0,Farmer 3,Control,57,-2,
3.4.A,C.0,Farmer 3,Control,87,-2,
3.4.A,C.0,Farmer 3,Control,115,-2,
3.4.A,C.0,Farmer 3,Control,149,-2,
3.4.A,C.0,Farmer 3,Control,177,-2,
3.4.A,C.0,Farmer 3,Control,198,-2,
3.4.A,C.0,Farmer 3,Control,226,-2,
3.4.A,C.0,Farmer 3,Control,255,-2,172.47170200373066
3.4.A,C.0,Farmer 3,Control,284,-2,301.87360129971717
3.4.A,C.0,Farmer 3,Control,315,-2,474.2971805764487
3.4.A,C.0,Farmer 3,Control,354,-2,500.66841181779887
3.4.A,C.0,Farmer 3,Control,385,-2,433.5372665021963
3.4.A,C.0,Farmer 3,Control,410,-2,314.3854992478489
3.4.A,C.0,Farmer 3,Control,450,-2,469.9180162464648
3.4.A,C.0,Farmer 3,Control,470,-2,257.26387243516456
3.4.A,C.0,Farmer 3,Control,499,-2,470.0623842589806
3.4.A,C.0,Farmer 3,Control,527,-2,
3.4.B,C.0,Farmer 3,Control,35,-2,
3.4.B,C.0,Farmer 3,Control,57,-2,
3.4.B,C.0,Farmer 3,Control,87,-2,
3.4.B,C.0,Farmer 3,Control,115,-2,
3.4.B,C.0,Farmer 3,Control,149,-2,
3.4.B,C.0,Farmer 3,Control,177,-2,
3.4.B,C.0,Farmer 3,Control,198,-2,
3.4.B,C.0,Farmer 3,Control,226,-2,
3.4.B,C.0,Farmer 3,Control,255,-2,252.3553585654973
3.4.B,C.0,Farmer 3,Control,284,-2,483.3923680125158
3.4.B,C.0,Farmer 3,Control,315,-2,535.8942169805644
3.4.B,C.0,Farmer 3,Control,354,-2,646.3839010770804
3.4.B,C.0,Farmer 3,Control,385,-2,595.5663459895301
3.4.B,C.0,Farmer 3,Control,410,-2,483.6329812864793
3.4.B,C.0,Farmer 3,Control,450,-2,673.0438683434622
3.4.B,C.0,Farmer 3,Control,470,-2,338.54308706901736
3.4.B,C.0,Farmer 3,Control,499,-2,544.508177628016
3.4.B,C.0,Farmer 3,Control,527,-2,
3.4.C,C.0,Farmer 3,Control,35,-2,
3.4.C,C.0,Farmer 3,Control,57,-2,
3.4.C,C.0,Farmer 3,Control,87,-2,
3.4.C,C.0,Farmer 3,Control,115,-2,
3.4.C,C.0,Farmer 3,Control,149,-2,
3.4.C,C.0,Farmer 3,Control,177,-2,
3.4.C,C.0,Farmer 3,Control,198,-2,
3.4.C,C.0,Farmer 3,Control,226,-2,
3.4.C,C.0,Farmer 3,Control,255,-2,270.64197893976774
3.4.C,C.0,Farmer 3,Control,284,-2,443.4986622540466
3.4.C,C.0,Farmer 3,Control,315,-2,398.07084806546726
3.4.C,C.0,Farmer 3,Control,354,-2,690.5605256633974
3.4.C,C.0,Farmer 3,Control,382,-2,705.0935765088152
3.4.C,C.0,Farmer 3,Control,410,-2,525.2591036765148
3.4.C,C.0,Farmer 3,Control,450,-2,635.2194382333473
3.4.C,C.0,Farmer 3,Control,470,-2,284.549434743366
3.4.C,C.0,Farmer 3,Control,499,-2,686.758833624165
3.4.C,C.0,Farmer 3,Control,527,-2,
3.4.D,C.0,Farmer 3,Control,35,-2,
3.4.D,C.0,Farmer 3,Control,57,-2,
3.4.D,C.0,Farmer 3,Control,87,-2,
3.4.D,C.0,Farmer 3,Control,115,-2,
3.4.D,C.0,Farmer 3,Control,149,-2,
3.4.D,C.0,Farmer 3,Control,177,-2,
3.4.D,C.0,Farmer 3,Control,198,-2,
3.4.D,C.0,Farmer 3,Control,226,-2,
3.4.D,C.0,Farmer 3,Control,255,-2,313.71178169564956
3.4.D,C.0,Farmer 3,Control,284,-2,515.3939531861123
3.4.D,C.0,Farmer 3,Control,315,-2,522.4679879655815
3.4.D,C.0,Farmer 3,Control,354,-2,722.8989698537819
3.4.D,C.0,Farmer 3,Control,382,-2,634.8344567061796
3.4.D,C.0,Farmer 3,Control,410,-2,576.4135175401649
3.4.D,C.0,Farmer 3,Control,450,-2,653.024831578314
3.4.D,C.0,Farmer 3,Control,470,-2,340.32362621096337
3.4.D,C.0,Farmer 3,Control,499,-2,680.4547618990313
3.4.D,C.0,Farmer 3,Control,527,-2,
3.5.A,C.1,Farmer 3,Basanite,35,-2,
3.5.A,C.1,Farmer 3,Basanite,57,-2,
3.5.A,C.1,Farmer 3,Basanite,87,-2,
3.5.A,C.1,Farmer 3,Basanite,115,-2,
3.5.A,C.1,Farmer 3,Basanite,149,-2,
3.5.A,C.1,Farmer 3,Basanite,177,-2,
3.5.A,C.1,Farmer 3,Basanite,198,-2,
3.5.A,C.1,Farmer 3,Basanite,226,-2,
3.5.A,C.1,Farmer 3,Basanite,255,-2,388.0613295625489
3.5.A,C.1,Farmer 3,Basanite,284,-2,467.7524953366629
3.5.A,C.1,Farmer 3,Basanite,315,-2,595.9513275166977
3.5.A,C.1,Farmer 3,Basanite,354,-2,818.5668670798484
3.5.A,C.1,Farmer 3,Basanite,385,-2,497.39606931825017
3.5.A,C.1,Farmer 3,Basanite,410,-2,425.93388242373186
3.5.A,C.1,Farmer 3,Basanite,450,-2,576.0285360129972
3.5.A,C.1,Farmer 3,Basanite,470,-2,336.66630218424694
3.5.A,C.1,Farmer 3,Basanite,499,-2,509.33049497563024
3.5.A,C.1,Farmer 3,Basanite,527,-2,
3.5.B,C.1,Farmer 3,Basanite,35,-2,
3.5.B,C.1,Farmer 3,Basanite,57,-2,
3.5.B,C.1,Farmer 3,Basanite,87,-2,
3.5.B,C.1,Farmer 3,Basanite,115,-2,
3.5.B,C.1,Farmer 3,Basanite,149,-2,
3.5.B,C.1,Farmer 3,Basanite,177,-2,
3.5.B,C.1,Farmer 3,Basanite,198,-2,
3.5.B,C.1,Farmer 3,Basanite,226,-2,
3.5.B,C.1,Farmer 3,Basanite,255,-2,379.01426487754975
3.5.B,C.1,Farmer 3,Basanite,284,-2,431.6604818581142
3.5.B,C.1,Farmer 3,Basanite,315,-2,516.7413885311993
3.5.B,C.1,Farmer 3,Basanite,354,-2,639.3579890486792
3.5.B,C.1,Farmer 3,Basanite,385,-2,574.0073833563993
3.5.B,C.1,Farmer 3,Basanite,410,-2,463.7101900234671
3.5.B,C.1,Farmer 3,Basanite,450,-2,534.1618003489981
3.5.B,C.1,Farmer 3,Basanite,470,-2,268.13959925386604
3.5.B,C.1,Farmer 3,Basanite,499,-2,657.7408546843974
3.5.B,C.1,Farmer 3,Basanite,527,-2,
3.5.C,C.1,Farmer 3,Basanite,35,-2,
3.5.C,C.1,Farmer 3,Basanite,57,-2,
3.5.C,C.1,Farmer 3,Basanite,87,-2,
3.5.C,C.1,Farmer 3,Basanite,115,-2,
3.5.C,C.1,Farmer 3,Basanite,149,-2,
3.5.C,C.1,Farmer 3,Basanite,177,-2,
3.5.C,C.1,Farmer 3,Basanite,198,-2,
3.5.C,C.1,Farmer 3,Basanite,226,-2,
3.5.C,C.1,Farmer 3,Basanite,255,-2,317.6097189963295
3.5.C,C.1,Farmer 3,Basanite,284,-2,422.9502759492147
3.5.C,C.1,Farmer 3,Basanite,315,-2,464.86513436428186
3.5.C,C.1,Farmer 3,Basanite,354,-2,615.0079107046151
3.5.C,C.1,Farmer 3,Basanite,382,-2,843.1094361874963
3.5.C,C.1,Farmer 3,Basanite,410,-2,709.0396365605632
3.5.C,C.1,Farmer 3,Basanite,450,-2,754.5636962512787
3.5.C,C.1,Farmer 3,Basanite,470,-2,390.85244527348215
3.5.C,C.1,Farmer 3,Basanite,499,-2,775.5451868343462
3.5.C,C.1,Farmer 3,Basanite,527,-2,
3.5.D,C.1,Farmer 3,Basanite,35,-2,
3.5.D,C.1,Farmer 3,Basanite,57,-2,
3.5.D,C.1,Farmer 3,Basanite,87,-2,
3.5.D,C.1,Farmer 3,Basanite,115,-2,
3.5.D,C.1,Farmer 3,Basanite,149,-2,
3.5.D,C.1,Farmer 3,Basanite,177,-2,
3.5.D,C.1,Farmer 3,Basanite,198,-2,
3.5.D,C.1,Farmer 3,Basanite,226,-2,
3.5.D,C.1,Farmer 3,Basanite,255,-2,250.911678199651
3.5.D,C.1,Farmer 3,Basanite,284,-2,414.336315301763
3.5.D,C.1,Farmer 3,Basanite,315,-2,471.7948008905469
3.5.D,C.1,Farmer 3,Basanite,354,-2,721.8402707744148
3.5.D,C.1,Farmer 3,Basanite,382,-2,544.3638093748119
3.5.D,C.1,Farmer 3,Basanite,410,-2,535.2686221794332
3.5.D,C.1,Farmer 3,Basanite,450,-2,505.2881894217462
3.5.D,C.1,Farmer 3,Basanite,470,-2,253.4621803959323
3.5.D,C.1,Farmer 3,Basanite,499,-2,621.9856999819484
3.5.D,C.1,Farmer 3,Basanite,527,-2,
3.6.A,D.0,Farmer 4,Control,35,15,
3.6.A,D.0,Farmer 4,Control,57,15,
3.6.A,D.0,Farmer 4,Control,87,15,
3.6.A,D.0,Farmer 4,Control,115,15,
3.6.A,D.0,Farmer 4,Control,149,15,
3.6.A,D.0,Farmer 4,Control,177,15,
3.6.A,D.0,Farmer 4,Control,198,15,
3.6.A,D.0,Farmer 4,Control,226,15,
3.6.A,D.0,Farmer 4,Control,255,15,246.24377760394728
3.6.A,D.0,Farmer 4,Control,284,15,300.67053420783435
3.6.A,D.0,Farmer 4,Control,315,15,467.9449861002467
3.6.A,D.0,Farmer 4,Control,354,15,398.45582935194653
3.6.A,D.0,Farmer 4,Control,385,15,434.98094710873096
3.6.A,D.0,Farmer 4,Control,410,15,357.070320476563
3.6.A,D.0,Farmer 4,Control,450,15,468.4743356399302
3.6.A,D.0,Farmer 4,Control,470,15,185.94605374571273
3.6.A,D.0,Farmer 4,Control,499,15,417.7049033034478
3.6.A,D.0,Farmer 4,Control,527,15,
3.6.B,D.0,Farmer 4,Control,35,15,
3.6.B,D.0,Farmer 4,Control,57,15,
3.6.B,D.0,Farmer 4,Control,87,15,
3.6.B,D.0,Farmer 4,Control,115,15,
3.6.B,D.0,Farmer 4,Control,149,15,
3.6.B,D.0,Farmer 4,Control,177,15,
3.6.B,D.0,Farmer 4,Control,198,15,
3.6.B,D.0,Farmer 4,Control,226,15,
3.6.B,D.0,Farmer 4,Control,255,15,323.7694227089476
3.6.B,D.0,Farmer 4,Control,284,15,340.70860773813104
3.6.B,D.0,Farmer 4,Control,315,15,412.8926349359168
3.6.B,D.0,Farmer 4,Control,354,15,457.35799554726515
3.6.B,D.0,Farmer 4,Control,385,15,577.8571981466996
3.6.B,D.0,Farmer 4,Control,410,15,415.53938239364584
3.6.B,D.0,Farmer 4,Control,450,15,522.1311291894818
3.6.B,D.0,Farmer 4,Control,470,15,255.6277013057344
3.6.B,D.0,Farmer 4,Control,499,15,447.3003545339671
3.6.B,D.0,Farmer 4,Control,527,15,
3.6.C,D.0,Farmer 4,Control,35,15,
3.6.C,D.0,Farmer 4,Control,57,15,
3.6.C,D.0,Farmer 4,Control,87,15,
3.6.C,D.0,Farmer 4,Control,115,15,
3.6.C,D.0,Farmer 4,Control,149,15,
3.6.C,D.0,Farmer 4,Control,177,15,
3.6.C,D.0,Farmer 4,Control,198,15,
3.6.C,D.0,Farmer 4,Control,226,15,
3.6.C,D.0,Farmer 4,Control,255,15,369.1009916360792
3.6.C,D.0,Farmer 4,Control,284,15,403.2199752090981
3.6.C,D.0,Farmer 4,Control,315,15,367.56106576809674
3.6.C,D.0,Farmer 4,Control,354,15,527.2321338227331
3.6.C,D.0,Farmer 4,Control,382,15,415.77999590829774
3.6.C,D.0,Farmer 4,Control,410,15,519.1475227149648
3.6.C,D.0,Farmer 4,Control,450,15,538.0116151392983
3.6.C,D.0,Farmer 4,Control,470,15,267.65837246525064
3.6.C,D.0,Farmer 4,Control,499,15,446.3379007160479
3.6.C,D.0,Farmer 4,Control,527,15,
3.6.D,D.0,Farmer 4,Control,35,15,
3.6.D,D.0,Farmer 4,Control,57,15,
3.6.D,D.0,Farmer 4,Control,87,15,
3.6.D,D.0,Farmer 4,Control,115,15,
3.6.D,D.0,Farmer 4,Control,149,15,
3.6.D,D.0,Farmer 4,Control,177,15,
3.6.D,D.0,Farmer 4,Control,198,15,
3.6.D,D.0,Farmer 4,Control,226,15,
3.6.D,D.0,Farmer 4,Control,255,15,427.4738082917143
3.6.D,D.0,Farmer 4,Control,284,15,477.3770323124135
3.6.D,D.0,Farmer 4,Control,315,15,449.65836596666463
3.6.D,D.0,Farmer 4,Control,354,15,622.5150495216319
3.6.D,D.0,Farmer 4,Control,382,15,616.1628550454299
3.6.D,D.0,Farmer 4,Control,410,15,400.3807367470967
3.6.D,D.0,Farmer 4,Control,450,15,682.8608960827968
3.6.D,D.0,Farmer 4,Control,470,15,299.8524484024309
3.6.D,D.0,Farmer 4,Control,499,15,536.6160572838318
3.6.D,D.0,Farmer 4,Control,527,15,
3.7.A,D.1,Farmer 4,Basanite,35,15,
3.7.A,D.1,Farmer 4,Basanite,57,15,
3.7.A,D.1,Farmer 4,Basanite,87,15,
3.7.A,D.1,Farmer 4,Basanite,115,15,
3.7.A,D.1,Farmer 4,Basanite,149,15,
3.7.A,D.1,Farmer 4,Basanite,177,15,
3.7.A,D.1,Farmer 4,Basanite,198,15,
3.7.A,D.1,Farmer 4,Basanite,226,15,
3.7.A,D.1,Farmer 4,Basanite,255,15,286.4262193874481
3.7.A,D.1,Farmer 4,Basanite,284,15,346.86831145074916
3.7.A,D.1,Farmer 4,Basanite,315,15,360.9201352668632
3.7.A,D.1,Farmer 4,Basanite,354,15,460.1972340092664
3.7.A,D.1,Farmer 4,Basanite,385,15,292.2009415728985
3.7.A,D.1,Farmer 4,Basanite,410,15,358.51400108309764
3.7.A,D.1,Farmer 4,Basanite,450,15,491.140120103496
3.7.A,D.1,Farmer 4,Basanite,470,15,229.54520608941573
3.7.A,D.1,Farmer 4,Basanite,499,15,442.63245393826344
3.7.A,D.1,Farmer 4,Basanite,527,15,
3.7.B,D.1,Farmer 4,Basanite,35,15,
3.7.B,D.1,Farmer 4,Basanite,57,15,
3.7.B,D.1,Farmer 4,Basanite,87,15,
3.7.B,D.1,Farmer 4,Basanite,115,15,
3.7.B,D.1,Farmer 4,Basanite,149,15,
3.7.B,D.1,Farmer 4,Basanite,177,15,
3.7.B,D.1,Farmer 4,Basanite,198,15,
3.7.B,D.1,Farmer 4,Basanite,226,15,
3.7.B,D.1,Farmer 4,Basanite,255,15,386.42515819243033
3.7.B,D.1,Farmer 4,Basanite,284,15,434.45159756904746
3.7.B,D.1,Farmer 4,Basanite,315,15,572.6599480113124
3.7.B,D.1,Farmer 4,Basanite,354,15,508.36804139839944
3.7.B,D.1,Farmer 4,Basanite,385,15,602.736626030447
3.7.B,D.1,Farmer 4,Basanite,410,15,441.6218776099645
3.7.B,D.1,Farmer 4,Basanite,450,15,661.6869147361454
3.7.B,D.1,Farmer 4,Basanite,470,15,247.73558096155003
3.7.B,D.1,Farmer 4,Basanite,499,15,520.4949578193634
3.7.B,D.1,Farmer 4,Basanite,527,15,
3.7.C,D.1,Farmer 4,Basanite,35,15,
3.7.C,D.1,Farmer 4,Basanite,57,15,
3.7.C,D.1,Farmer 4,Basanite,87,15,
3.7.C,D.1,Farmer 4,Basanite,115,15,
3.7.C,D.1,Farmer 4,Basanite,149,15,
3.7.C,D.1,Farmer 4,Basanite,177,15,
3.7.C,D.1,Farmer 4,Basanite,198,15,
3.7.C,D.1,Farmer 4,Basanite,226,15,
3.7.C,D.1,Farmer 4,Basanite,255,15,253.99152993561583
3.7.C,D.1,Farmer 4,Basanite,284,15,387.9650840604127
3.7.C,D.1,Farmer 4,Basanite,315,15,323.6731774474998
3.7.C,D.1,Farmer 4,Basanite,354,15,440.6113012816656
3.7.C,D.1,Farmer 4,Basanite,382,15,294.0296037066009
3.7.C,D.1,Farmer 4,Basanite,410,15,430.64990552981527
3.7.C,D.1,Farmer 4,Basanite,450,15,273.52933991214877
3.7.C,D.1,Farmer 4,Basanite,470,15,233.9724930741922
3.7.C,D.1,Farmer 4,Basanite,499,15,476.99205078524574
3.7.C,D.1,Farmer 4,Basanite,527,15,
3.7.D,D.1,Farmer 4,Basanite,35,15,
3.7.D,D.1,Farmer 4,Basanite,57,15,
3.7.D,D.1,Farmer 4,Basanite,87,15,
3.7.D,D.1,Farmer 4,Basanite,115,15,
3.7.D,D.1,Farmer 4,Basanite,149,15,
3.7.D,D.1,Farmer 4,Basanite,177,15,
3.7.D,D.1,Farmer 4,Basanite,198,15,
3.7.D,D.1,Farmer 4,Basanite,226,15,
3.7.D,D.1,Farmer 4,Basanite,255,15,377.2818482459835
3.7.D,D.1,Farmer 4,Basanite,284,15,418.6673571213672
3.7.D,D.1,Farmer 4,Basanite,315,15,335.2226218184006
3.7.D,D.1,Farmer 4,Basanite,354,15,515.8751802154161
3.7.D,D.1,Farmer 4,Basanite,382,15,569.3876055117636
3.7.D,D.1,Farmer 4,Basanite,410,15,391.81489885071306
3.7.D,D.1,Farmer 4,Basanite,450,15,519.7249950057163
3.7.D,D.1,Farmer 4,Basanite,470,15,235.3199282507973
3.7.D,D.1,Farmer 4,Basanite,499,15,478.1469953667489
3.7.D,D.1,Farmer 4,Basanite,527,15,
3.8.A,E.0,Farmer 5,Control,35,15,
3.8.A,E.0,Farmer 5,Control,57,15,
3.8.A,E.0,Farmer 5,Control,87,15,
3.8.A,E.0,Farmer 5,Control,115,15,
3.8.A,E.0,Farmer 5,Control,149,15,
3.8.A,E.0,Farmer 5,Control,177,15,
3.8.A,E.0,Farmer 5,Control,198,15,
3.8.A,E.0,Farmer 5,Control,226,15,
3.8.A,E.0,Farmer 5,Control,255,15,147.92913279980746
3.8.A,E.0,Farmer 5,Control,284,15,135.1284986822312
3.8.A,E.0,Farmer 5,Control,315,15,190.2289726698357
3.8.A,E.0,Farmer 5,Control,354,15,187.10099815873397
3.8.A,E.0,Farmer 5,Control,385,15,252.6440948312173
3.8.A,E.0,Farmer 5,Control,410,15,147.25541522353933
3.8.A,E.0,Farmer 5,Control,450,15,204.42516466694744
3.8.A,E.0,Farmer 5,Control,470,15,121.65414696431796
3.8.A,E.0,Farmer 5,Control,499,15,225.50290055960045
3.8.A,E.0,Farmer 5,Control,527,15,
3.8.B,E.0,Farmer 5,Control,35,15,
3.8.B,E.0,Farmer 5,Control,57,15,
3.8.B,E.0,Farmer 5,Control,87,15,
3.8.B,E.0,Farmer 5,Control,115,15,
3.8.B,E.0,Farmer 5,Control,149,15,
3.8.B,E.0,Farmer 5,Control,177,15,
3.8.B,E.0,Farmer 5,Control,198,15,
3.8.B,E.0,Farmer 5,Control,226,15,
3.8.B,E.0,Farmer 5,Control,255,15,90.95187411998316
3.8.B,E.0,Farmer 5,Control,284,15,171.50924833022444
3.8.B,E.0,Farmer 5,Control,315,15,181.90374821589745
3.8.B,E.0,Farmer 5,Control,354,15,175.16657235694083
3.8.B,E.0,Farmer 5,Control,385,15,207.26440305674228
3.8.B,E.0,Farmer 5,Control,410,15,142.92437359648594
3.8.B,E.0,Farmer 5,Control,450,15,272.85562235994945
3.8.B,E.0,Farmer 5,Control,470,15,121.0766747457729
3.8.B,E.0,Farmer 5,Control,499,15,238.59227082255248
3.8.B,E.0,Farmer 5,Control,527,15,
3.8.C,E.0,Farmer 5,Control,35,15,
3.8.C,E.0,Farmer 5,Control,57,15,
3.8.C,E.0,Farmer 5,Control,87,15,
3.8.C,E.0,Farmer 5,Control,115,15,
3.8.C,E.0,Farmer 5,Control,149,15,
3.8.C,E.0,Farmer 5,Control,177,15,
3.8.C,E.0,Farmer 5,Control,198,15,
3.8.C,E.0,Farmer 5,Control,226,15,
3.8.C,E.0,Farmer 5,Control,255,15,106.06239711173956
3.8.C,E.0,Farmer 5,Control,284,15,209.09306509416933
3.8.C,E.0,Farmer 5,Control,315,15,160.681644262591
3.8.C,E.0,Farmer 5,Control,354,15,232.91379401889404
3.8.C,E.0,Farmer 5,Control,382,15,215.58962751068057
3.8.C,E.0,Farmer 5,Control,410,15,185.27233614537576
3.8.C,E.0,Farmer 5,Control,450,15,276.2242103616343
3.8.C,E.0,Farmer 5,Control,470,15,138.8820680907395
3.8.C,E.0,Farmer 5,Control,499,15,246.00316432998375
3.8.C,E.0,Farmer 5,Control,527,15,
3.8.D,E.0,Farmer 5,Control,35,15,
3.8.D,E.0,Farmer 5,Control,57,15,
3.8.D,E.0,Farmer 5,Control,87,15,
3.8.D,E.0,Farmer 5,Control,115,15,
3.8.D,E.0,Farmer 5,Control,149,15,
3.8.D,E.0,Farmer 5,Control,177,15,
3.8.D,E.0,Farmer 5,Control,198,15,
3.8.D,E.0,Farmer 5,Control,226,15,
3.8.D,E.0,Farmer 5,Control,255,15,252.54784932908117
3.8.D,E.0,Farmer 5,Control,284,15,199.42040546362597
3.8.D,E.0,Farmer 5,Control,315,15,204.04018318791745
3.8.D,E.0,Farmer 5,Control,354,15,212.70226644202415
3.8.D,E.0,Farmer 5,Control,382,15,213.95345623683733
3.8.D,E.0,Farmer 5,Control,410,15,194.80062771526565
3.8.D,E.0,Farmer 5,Control,450,15,530.3119855586979
3.8.D,E.0,Farmer 5,Control,470,15,155.19565819844755
3.8.D,E.0,Farmer 5,Control,499,15,229.06397922859375
3.8.D,E.0,Farmer 5,Control,527,15,
3.9.A,E.1,Farmer 5,Basanite,35,15,
3.9.A,E.1,Farmer 5,Basanite,57,15,
3.9.A,E.1,Farmer 5,Basanite,87,15,
3.9.A,E.1,Farmer 5,Basanite,115,15,
3.9.A,E.1,Farmer 5,Basanite,149,15,
3.9.A,E.1,Farmer 5,Basanite,177,15,
3.9.A,E.1,Farmer 5,Basanite,198,15,
3.9.A,E.1,Farmer 5,Basanite,226,15,
3.9.A,E.1,Farmer 5,Basanite,255,15,161.88471137854265
3.9.A,E.1,Farmer 5,Basanite,284,15,181.03753990011433
3.9.A,E.1,Farmer 5,Basanite,315,15,193.93441939948252
3.9.A,E.1,Farmer 5,Basanite,354,15,166.88947058186412
3.9.A,E.1,Farmer 5,Basanite,385,15,177.33209317046754
3.9.A,E.1,Farmer 5,Basanite,410,15,168.4293964979842
3.9.A,E.1,Farmer 5,Basanite,450,15,230.98888662374392
3.9.A,E.1,Farmer 5,Basanite,470,15,110.87466557554606
3.9.A,E.1,Farmer 5,Basanite,499,15,212.79851179974727
3.9.A,E.1,Farmer 5,Basanite,527,15,
3.9.B,E.1,Farmer 5,Basanite,35,15,
3.9.B,E.1,Farmer 5,Basanite,57,15,
3.9.B,E.1,Farmer 5,Basanite,87,15,
3.9.B,E.1,Farmer 5,Basanite,115,15,
3.9.B,E.1,Farmer 5,Basanite,149,15,
3.9.B,E.1,Farmer 5,Basanite,177,15,
3.9.B,E.1,Farmer 5,Basanite,198,15,
3.9.B,E.1,Farmer 5,Basanite,226,15,
3.9.B,E.1,Farmer 5,Basanite,255,15,135.1284986822312
3.9.B,E.1,Farmer 5,Basanite,284,15,195.1856091942957
3.9.B,E.1,Farmer 5,Basanite,315,15,222.32680336963716
3.9.B,E.1,Farmer 5,Basanite,354,15,207.26440305674228
3.9.B,E.1,Farmer 5,Basanite,385,15,258.9962890667308
3.9.B,E.1,Farmer 5,Basanite,410,15,186.2829125218124
3.9.B,E.1,Farmer 5,Basanite,450,15,282.2395458210482
3.9.B,E.1,Farmer 5,Basanite,470,15,151.5864568505927
3.9.B,E.1,Farmer 5,Basanite,499,15,258.5150622781154
3.9.B,E.1,Farmer 5,Basanite,527,15,
3.9.C,E.1,Farmer 5,Basanite,35,15,
3.9.C,E.1,Farmer 5,Basanite,57,15,
3.9.C,E.1,Farmer 5,Basanite,87,15,
3.9.C,E.1,Farmer 5,Basanite,115,15,
3.9.C,E.1,Farmer 5,Basanite,149,15,
3.9.C,E.1,Farmer 5,Basanite,177,15,
3.9.C,E.1,Farmer 5,Basanite,198,15,
3.9.C,E.1,Farmer 5,Basanite,226,15,
3.9.C,E.1,Farmer 5,Basanite,255,15,168.62188723749924
3.9.C,E.1,Farmer 5,Basanite,284,15,184.79110930862265
3.9.C,E.1,Farmer 5,Basanite,315,15,161.21099380227452
3.9.C,E.1,Farmer 5,Basanite,354,15,167.32257473975568
3.9.C,E.1,Farmer 5,Basanite,382,15,299.900571153499
3.9.C,E.1,Farmer 5,Basanite,410,15,179.3051232444792
3.9.C,E.1,Farmer 5,Basanite,450,15,255.6277013057344
3.9.C,E.1,Farmer 5,Basanite,470,15,118.38180439256274
3.9.C,E.1,Farmer 5,Basanite,499,15,256.5901548829653
3.9.C,E.1,Farmer 5,Basanite,527,15,
3.9.D,E.1,Farmer 5,Basanite,35,15,
3.9.D,E.1,Farmer 5,Basanite,57,15,
3.9.D,E.1,Farmer 5,Basanite,87,15,
3.9.D,E.1,Farmer 5,Basanite,115,15,
3.9.D,E.1,Farmer 5,Basanite,149,15,
3.9.D,E.1,Farmer 5,Basanite,177,15,
3.9.D,E.1,Farmer 5,Basanite,198,15,
3.9.D,E.1,Farmer 5,Basanite,226,15,
3.9.D,E.1,Farmer 5,Basanite,255,15,93.16551761237136
3.9.D,E.1,Farmer 5,Basanite,284,15,191.72077590709424
3.9.D,E.1,Farmer 5,Basanite,315,15,117.22685995547263
3.9.D,E.1,Farmer 5,Basanite,354,15,200.23849110054755
3.9.D,E.1,Farmer 5,Basanite,382,15,189.26651897226063
3.9.D,E.1,Farmer 5,Basanite,410,15,205.0026368854925
3.9.D,E.1,Farmer 5,Basanite,450,15,211.7398127444491
3.9.D,E.1,Farmer 5,Basanite,470,15,106.06239711173956
3.9.D,E.1,Farmer 5,Basanite,499,15,210.199886828329
3.9.D,E.1,Farmer 5,Basanite,527,15,
4.0.A,F.0,Farmer 6,Control,35,-2,
4.0.A,F.0,Farmer 6,Control,57,-2,
4.0.A,F.0,Farmer 6,Control,87,-2,
4.0.A,F.0,Farmer 6,Control,112,-2,
4.0.A,F.0,Farmer 6,Control,149,-2,
4.0.A,F.0,Farmer 6,Control,177,-2,
4.0.A,F.0,Farmer 6,Control,198,-2,
4.0.A,F.0,Farmer 6,Control,226,-2,
4.0.A,F.0,Farmer 6,Control,255,-2,343.59596895120046
4.0.A,F.0,Farmer 6,Control,281,-2,320.7376937240508
4.0.A,F.0,Farmer 6,Control,309,-2,628.0010354413623
4.0.A,F.0,Farmer 6,Control,347,-2,576.8947443287802
4.0.A,F.0,Farmer 6,Control,382,-2,524.9222449004152
4.0.A,F.0,Farmer 6,Control,420,-2,769.0967470966965
4.0.A,F.0,Farmer 6,Control,450,-2,548.3579924183164
4.0.A,F.0,Farmer 6,Control,471,-2,379.59173692761294
4.0.A,F.0,Farmer 6,Control,527,-2,
4.0.B,F.0,Farmer 6,Control,35,-2,
4.0.B,F.0,Farmer 6,Control,57,-2,
4.0.B,F.0,Farmer 6,Control,87,-2,
4.0.B,F.0,Farmer 6,Control,112,-2,
4.0.B,F.0,Farmer 6,Control,149,-2,
4.0.B,F.0,Farmer 6,Control,177,-2,
4.0.B,F.0,Farmer 6,Control,198,-2,
4.0.B,F.0,Farmer 6,Control,226,-2,
4.0.B,F.0,Farmer 6,Control,255,-2,
4.0.B,F.0,Farmer 6,Control,281,-2,473.86407629821286
4.0.B,F.0,Farmer 6,Control,309,-2,490.4664025512967
4.0.B,F.0,Farmer 6,Control,347,-2,655.2384749984957
4.0.B,F.0,Farmer 6,Control,382,-2,475.45212491726335
4.0.B,F.0,Farmer 6,Control,420,-2,668.3278452373788
4.0.B,F.0,Farmer 6,Control,450,-2,672.9476230820145
4.0.B,F.0,Farmer 6,Control,471,-2,331.18031626451653
4.0.B,F.0,Farmer 6,Control,527,-2,
4.0.C,F.0,Farmer 6,Control,35,-2,
4.0.C,F.0,Farmer 6,Control,57,-2,
4.0.C,F.0,Farmer 6,Control,87,-2,
4.0.C,F.0,Farmer 6,Control,112,-2,
4.0.C,F.0,Farmer 6,Control,149,-2,
4.0.C,F.0,Farmer 6,Control,177,-2,
4.0.C,F.0,Farmer 6,Control,198,-2,
4.0.C,F.0,Farmer 6,Control,226,-2,
4.0.C,F.0,Farmer 6,Control,248,-2,
4.0.C,F.0,Farmer 6,Control,281,-2,373.04705192851554
4.0.C,F.0,Farmer 6,Control,309,-2,655.2865977495637
4.0.C,F.0,Farmer 6,Control,347,-2,954.6096966123112
4.0.C,F.0,Farmer 6,Control,382,-2,612.1205494915457
4.0.C,F.0,Farmer 6,Control,420,-2,640.60917889163
4.0.C,F.0,Farmer 6,Control,450,-2,540.5139948252
4.0.C,F.0,Farmer 6,Control,471,-2,405.6742321439316
4.0.C,F.0,Farmer 6,Control,527,-2,
4.0.D,F.0,Farmer 6,Control,35,-2,
4.0.D,F.0,Farmer 6,Control,57,-2,
4.0.D,F.0,Farmer 6,Control,87,-2,
4.0.D,F.0,Farmer 6,Control,112,-2,
4.0.D,F.0,Farmer 6,Control,149,-2,
4.0.D,F.0,Farmer 6,Control,177,-2,
4.0.D,F.0,Farmer 6,Control,198,-2,
4.0.D,F.0,Farmer 6,Control,226,-2,
4.0.D,F.0,Farmer 6,Control,248,-2,
4.0.D,F.0,Farmer 6,Control,281,-2,476.3183332330466
4.0.D,F.0,Farmer 6,Control,309,-2,628.2897717070822
4.0.D,F.0,Farmer 6,Control,347,-2,1022.5108047415608
4.0.D,F.0,Farmer 6,Control,382,-2,513.4209232805825
4.0.D,F.0,Farmer 6,Control,420,-2,657.4521186593659
4.0.D,F.0,Farmer 6,Control,450,-2,298.36064528551657
4.0.D,F.0,Farmer 6,Control,471,-2,196.77365778927737
4.0.D,F.0,Farmer 6,Control,527,-2,
4.1.A,F.1,Farmer 6,Basanite,35,-2,
4.1.A,F.1,Farmer 6,Basanite,57,-2,
4.1.A,F.1,Farmer 6,Basanite,87,-2,
4.1.A,F.1,Farmer 6,Basanite,112,-2,
4.1.A,F.1,Farmer 6,Basanite,149,-2,
4.1.A,F.1,Farmer 6,Basanite,177,-2,
4.1.A,F.1,Farmer 6,Basanite,198,-2,
4.1.A,F.1,Farmer 6,Basanite,226,-2,
4.1.A,F.1,Farmer 6,Basanite,255,-2,272.4706408327817
4.1.A,F.1,Farmer 6,Basanite,281,-2,441.76624562248026
4.1.A,F.1,Farmer 6,Basanite,309,-2,644.4589936819303
4.1.A,F.1,Farmer 6,Basanite,347,-2,858.2199590829773
4.1.A,F.1,Farmer 6,Basanite,382,-2,565.4415454600156
4.1.A,F.1,Farmer 6,Basanite,420,-2,662.3606322883446
4.1.A,F.1,Farmer 6,Basanite,450,-2,489.1189674468981
4.1.A,F.1,Farmer 6,Basanite,471,-2,377.9555657981827
4.1.A,F.1,Farmer 6,Basanite,527,-2,
4.1.B,F.1,Farmer 6,Basanite,35,-2,
4.1.B,F.1,Farmer 6,Basanite,57,-2,
4.1.B,F.1,Farmer 6,Basanite,87,-2,
4.1.B,F.1,Farmer 6,Basanite,112,-2,
4.1.B,F.1,Farmer 6,Basanite,149,-2,
4.1.B,F.1,Farmer 6,Basanite,177,-2,
4.1.B,F.1,Farmer 6,Basanite,198,-2,
4.1.B,F.1,Farmer 6,Basanite,226,-2,
4.1.B,F.1,Farmer 6,Basanite,255,-2,
4.1.B,F.1,Farmer 6,Basanite,281,-2,482.1411781695649
4.1.B,F.1,Farmer 6,Basanite,309,-2,572.8524387748962
4.1.B,F.1,Farmer 6,Basanite,347,-2,814.8132975509958
4.1.B,F.1,Farmer 6,Basanite,382,-2,503.12266875263253
4.1.B,F.1,Farmer 6,Basanite,420,-2,620.7826328900655
4.1.B,F.1,Farmer 6,Basanite,450,-2,606.3458273060954
4.1.B,F.1,Farmer 6,Basanite,471,-2,386.37703568205063
4.1.B,F.1,Farmer 6,Basanite,527,-2,
4.1.C,F.1,Farmer 6,Basanite,35,-2,
4.1.C,F.1,Farmer 6,Basanite,57,-2,
4.1.C,F.1,Farmer 6,Basanite,87,-2,
4.1.C,F.1,Farmer 6,Basanite,112,-2,
4.1.C,F.1,Farmer 6,Basanite,149,-2,
4.1.C,F.1,Farmer 6,Basanite,177,-2,
4.1.C,F.1,Farmer 6,Basanite,198,-2,
4.1.C,F.1,Farmer 6,Basanite,226,-2,
4.1.C,F.1,Farmer 6,Basanite,248,-2,
4.1.C,F.1,Farmer 6,Basanite,281,-2,373.4801559660629
4.1.C,F.1,Farmer 6,Basanite,309,-2,181.18190793669896
4.1.C,F.1,Farmer 6,Basanite,347,-2,913.3685559901318
4.1.C,F.1,Farmer 6,Basanite,382,-2,251.970377279018
4.1.C,F.1,Farmer 6,Basanite,420,-2,603.4584663337145
4.1.C,F.1,Farmer 6,Basanite,450,-2,448.69591238943383
4.1.C,F.1,Farmer 6,Basanite,471,-2,267.08090017449905
4.1.C,F.1,Farmer 6,Basanite,527,-2,
4.1.D,F.1,Farmer 6,Basanite,35,-2,
4.1.D,F.1,Farmer 6,Basanite,57,-2,
4.1.D,F.1,Farmer 6,Basanite,87,-2,
4.1.D,F.1,Farmer 6,Basanite,112,-2,
4.1.D,F.1,Farmer 6,Basanite,149,-2,
4.1.D,F.1,Farmer 6,Basanite,177,-2,
4.1.D,F.1,Farmer 6,Basanite,198,-2,
4.1.D,F.1,Farmer 6,Basanite,226,-2,
4.1.D,F.1,Farmer 6,Basanite,248,-2,
4.1.D,F.1,Farmer 6,Basanite,281,-2,229.06397922859375
4.1.D,F.1,Farmer 6,Basanite,309,-2,571.6974944340815
4.1.D,F.1,Farmer 6,Basanite,347,-2,938.3923518863952
4.1.D,F.1,Farmer 6,Basanite,382,-2,452.3532364161502
4.1.D,F.1,Farmer 6,Basanite,420,-2,525.2109809254467
4.1.D,F.1,Farmer 6,Basanite,450,-2,648.5975444972622
4.1.D,F.1,Farmer 6,Basanite,471,-2,432.7191809374812
4.1.D,F.1,Farmer 6,Basanite,527,-2,
4.2.A,G.0,Farmer 7,Control,35,-2,
4.2.A,G.0,Farmer 7,Control,57,-2,
4.2.A,G.0,Farmer 7,Control,87,-2,
4.2.A,G.0,Farmer 7,Control,112,-2,
4.2.A,G.0,Farmer 7,Control,149,-2,
4.2.A,G.0,Farmer 7,Control,177,-2,
4.2.A,G.0,Farmer 7,Control,198,-2,
4.2.A,G.0,Farmer 7,Control,226,-2,
4.2.A,G.0,Farmer 7,Control,255,-2,272.4706408327817
4.2.A,G.0,Farmer 7,Control,281,-2,292.77841386365003
4.2.A,G.0,Farmer 7,Control,309,-2,240.6134235754257
4.2.A,G.0,Farmer 7,Control,347,-2,234.59808797159877
4.2.A,G.0,Farmer 7,Control,382,-2,360.5832764907636
4.2.A,G.0,Farmer 7,Control,420,-2,370.54467224261384
4.2.A,G.0,Farmer 7,Control,450,-2,289.6985618869968
4.2.A,G.0,Farmer 7,Control,471,-2,187.293488898249
4.2.A,G.0,Farmer 7,Control,527,-2,
4.2.B,G.0,Farmer 7,Control,35,-2,
4.2.B,G.0,Farmer 7,Control,57,-2,
4.2.B,G.0,Farmer 7,Control,87,-2,
4.2.B,G.0,Farmer 7,Control,112,-2,
4.2.B,G.0,Farmer 7,Control,149,-2,
4.2.B,G.0,Farmer 7,Control,177,-2,
4.2.B,G.0,Farmer 7,Control,198,-2,
4.2.B,G.0,Farmer 7,Control,226,-2,
4.2.B,G.0,Farmer 7,Control,255,-2,
4.2.B,G.0,Farmer 7,Control,281,-2,301.729233046513
4.2.B,G.0,Farmer 7,Control,309,-2,232.4325671580721
4.2.B,G.0,Farmer 7,Control,347,-2,311.73875154943136
4.2.B,G.0,Farmer 7,Control,382,-2,256.7826456465491
4.2.B,G.0,Farmer 7,Control,420,-2,234.64621067452916
4.2.B,G.0,Farmer 7,Control,450,-2,298.841872074132
4.2.B,G.0,Farmer 7,Control,471,-2,251.0560462121668
4.2.B,G.0,Farmer 7,Control,527,-2,
4.2.C,G.0,Farmer 7,Control,35,-2,
4.2.C,G.0,Farmer 7,Control,57,-2,
4.2.C,G.0,Farmer 7,Control,87,-2,
4.2.C,G.0,Farmer 7,Control,112,-2,
4.2.C,G.0,Farmer 7,Control,149,-2,
4.2.C,G.0,Farmer 7,Control,177,-2,
4.2.C,G.0,Farmer 7,Control,198,-2,
4.2.C,G.0,Farmer 7,Control,226,-2,
4.2.C,G.0,Farmer 7,Control,248,-2,
4.2.C,G.0,Farmer 7,Control,281,-2,278.1972402671641
4.2.C,G.0,Farmer 7,Control,309,-2,233.8281250135387
4.2.C,G.0,Farmer 7,Control,347,-2,417.2236765148324
4.2.C,G.0,Farmer 7,Control,382,-2,171.50924833022444
4.2.C,G.0,Farmer 7,Control,420,-2,396.33843119321256
4.2.C,G.0,Farmer 7,Control,450,-2,290.0835434141645
4.2.C,G.0,Farmer 7,Control,471,-2,219.2469515614658
4.2.C,G.0,Farmer 7,Control,527,-2,
4.2.D,G.0,Farmer 7,Control,35,-2,
4.2.D,G.0,Farmer 7,Control,57,-2,
4.2.D,G.0,Farmer 7,Control,87,-2,
4.2.D,G.0,Farmer 7,Control,112,-2,
4.2.D,G.0,Farmer 7,Control,149,-2,
4.2.D,G.0,Farmer 7,Control,177,-2,
4.2.D,G.0,Farmer 7,Control,198,-2,
4.2.D,G.0,Farmer 7,Control,226,-2,
4.2.D,G.0,Farmer 7,Control,248,-2,
4.2.D,G.0,Farmer 7,Control,281,-2,279.11157133401525
4.2.D,G.0,Farmer 7,Control,309,-2,243.50078464408207
4.2.D,G.0,Farmer 7,Control,347,-2,280.1702704133823
4.2.D,G.0,Farmer 7,Control,382,-2,194.0306647572056
4.2.D,G.0,Farmer 7,Control,420,-2,332.04652458029966
4.2.D,G.0,Farmer 7,Control,450,-2,209.622414609784
4.2.D,G.0,Farmer 7,Control,471,-2,203.4627109693724
4.2.D,G.0,Farmer 7,Control,527,-2,
4.3.A,G.1,Farmer 7,Basanite,35,-2,
4.3.A,G.1,Farmer 7,Basanite,57,-2,
4.3.A,G.1,Farmer 7,Basanite,87,-2,
4.3.A,G.1,Farmer 7,Basanite,112,-2,
4.3.A,G.1,Farmer 7,Basanite,149,-2,
4.3.A,G.1,Farmer 7,Basanite,177,-2,
4.3.A,G.1,Farmer 7,Basanite,198,-2,
4.3.A,G.1,Farmer 7,Basanite,226,-2,
4.3.A,G.1,Farmer 7,Basanite,255,-2,222.32680336963716
4.3.A,G.1,Farmer 7,Basanite,281,-2,177.86144271015104
4.3.A,G.1,Farmer 7,Basanite,309,-2,189.41088703291413
4.3.A,G.1,Farmer 7,Basanite,347,-2,247.6393354594139
4.3.A,G.1,Farmer 7,Basanite,382,-2,216.9370626872856
4.3.A,G.1,Farmer 7,Basanite,420,-2,304.3759807449305
4.3.A,G.1,Farmer 7,Basanite,450,-2,330.12161718514955
4.3.A,G.1,Farmer 7,Basanite,471,-2,227.09094915458212
4.3.A,G.1,Farmer 7,Basanite,527,-2,
4.3.B,G.1,Farmer 7,Basanite,35,-2,
4.3.B,G.1,Farmer 7,Basanite,57,-2,
4.3.B,G.1,Farmer 7,Basanite,87,-2,
4.3.B,G.1,Farmer 7,Basanite,112,-2,
4.3.B,G.1,Farmer 7,Basanite,149,-2,
4.3.B,G.1,Farmer 7,Basanite,177,-2,
4.3.B,G.1,Farmer 7,Basanite,198,-2,
4.3.B,G.1,Farmer 7,Basanite,226,-2,
4.3.B,G.1,Farmer 7,Basanite,255,-2,
4.3.B,G.1,Farmer 7,Basanite,281,-2,205.86884520127563
4.3.B,G.1,Farmer 7,Basanite,309,-2,362.17132510981406
4.3.B,G.1,Farmer 7,Basanite,347,-2,338.7837003429809
4.3.B,G.1,Farmer 7,Basanite,382,-2,546.1924715085144
4.3.B,G.1,Farmer 7,Basanite,420,-2,285.84874709669657
4.3.B,G.1,Farmer 7,Basanite,450,-2,368.6678875985318
4.3.B,G.1,Farmer 7,Basanite,471,-2,170.8355307298875
4.3.B,G.1,Farmer 7,Basanite,527,-2,
4.3.C,G.1,Farmer 7,Basanite,35,-2,
4.3.C,G.1,Farmer 7,Basanite,57,-2,
4.3.C,G.1,Farmer 7,Basanite,87,-2,
4.3.C,G.1,Farmer 7,Basanite,112,-2,
4.3.C,G.1,Farmer 7,Basanite,149,-2,
4.3.C,G.1,Farmer 7,Basanite,177,-2,
4.3.C,G.1,Farmer 7,Basanite,198,-2,
4.3.C,G.1,Farmer 7,Basanite,226,-2,
4.3.C,G.1,Farmer 7,Basanite,248,-2,
4.3.C,G.1,Farmer 7,Basanite,281,-2,176.70649827306093
4.3.C,G.1,Farmer 7,Basanite,309,-2,202.98148413261927
4.3.C,G.1,Farmer 7,Basanite,347,-2,221.36434969613092
4.3.C,G.1,Farmer 7,Basanite,382,-2,130.60496631566278
4.3.C,G.1,Farmer 7,Basanite,420,-2,353.41299644984656
4.3.C,G.1,Farmer 7,Basanite,450,-2,254.18402069919972
4.3.C,G.1,Farmer 7,Basanite,471,-2,240.18031941753412
4.3.C,G.1,Farmer 7,Basanite,527,-2,
4.3.D,G.1,Farmer 7,Basanite,35,-2,
4.3.D,G.1,Farmer 7,Basanite,57,-2,
4.3.D,G.1,Farmer 7,Basanite,87,-2,
4.3.D,G.1,Farmer 7,Basanite,112,-2,
4.3.D,G.1,Farmer 7,Basanite,149,-2,
4.3.D,G.1,Farmer 7,Basanite,177,-2,
4.3.D,G.1,Farmer 7,Basanite,198,-2,
4.3.D,G.1,Farmer 7,Basanite,226,-2,
4.3.D,G.1,Farmer 7,Basanite,248,-2,
4.3.D,G.1,Farmer 7,Basanite,281,-2,351.1031077682171
4.3.D,G.1,Farmer 7,Basanite,309,-2,216.5520812082556
4.3.D,G.1,Farmer 7,Basanite,347,-2,322.22949684096517
4.3.D,G.1,Farmer 7,Basanite,382,-2,190.5658314700042
4.3.D,G.1,Farmer 7,Basanite,420,-2,232.4325671580721
4.3.D,G.1,Farmer 7,Basanite,450,-2,232.33632180034897
4.3.D,G.1,Farmer 7,Basanite,471,-2,199.42040546362597
4.3.D,G.1,Farmer 7,Basanite,527,-2,
4.4.A,H.0,Farmer 8,Control,35,-2,
4.4.A,H.0,Farmer 8,Control,57,-2,
4.4.A,H.0,Farmer 8,Control,87,-2,
4.4.A,H.0,Farmer 8,Control,112,-2,
4.4.A,H.0,Farmer 8,Control,149,-2,
4.4.A,H.0,Farmer 8,Control,177,-2,
4.4.A,H.0,Farmer 8,Control,198,-2,
4.4.A,H.0,Farmer 8,Control,226,-2,
4.4.A,H.0,Farmer 8,Control,255,-2,152.78952396654432
4.4.A,H.0,Farmer 8,Control,281,-2,441.76624562248026
4.4.A,H.0,Farmer 8,Control,309,-2,476.4145787351826
4.4.A,H.0,Farmer 8,Control,347,-2,569.3876055117636
4.4.A,H.0,Farmer 8,Control,382,-2,307.60020073409953
4.4.A,H.0,Farmer 8,Control,420,-2,357.3590567422829
4.4.A,H.0,Farmer 8,Control,450,-2,330.9878255009327
4.4.A,H.0,Farmer 8,Control,471,-2,243.3082938804982
4.4.A,H.0,Farmer 8,Control,499,-2,383.29718370539746
4.4.A,H.0,Farmer 8,Control,527,-2,
4.4.B,H.0,Farmer 8,Control,35,-2,
4.4.B,H.0,Farmer 8,Control,57,-2,
4.4.B,H.0,Farmer 8,Control,87,-2,
4.4.B,H.0,Farmer 8,Control,112,-2,
4.4.B,H.0,Farmer 8,Control,149,-2,
4.4.B,H.0,Farmer 8,Control,177,-2,
4.4.B,H.0,Farmer 8,Control,198,-2,
4.4.B,H.0,Farmer 8,Control,226,-2,
4.4.B,H.0,Farmer 8,Control,255,-2,
4.4.B,H.0,Farmer 8,Control,281,-2,540.5621173355797
4.4.B,H.0,Farmer 8,Control,309,-2,539.9365225344485
4.4.B,H.0,Farmer 8,Control,347,-2,513.3728005295144
4.4.B,H.0,Farmer 8,Control,382,-2,384.0190240086648
4.4.B,H.0,Farmer 8,Control,420,-2,413.277616222396
4.4.B,H.0,Farmer 8,Control,450,-2,493.1612730007822
4.4.B,H.0,Farmer 8,Control,471,-2,265.8778330826163
4.4.B,H.0,Farmer 8,Control,499,-2,476.4145787351826
4.4.B,H.0,Farmer 8,Control,527,-2,
4.4.C,H.0,Farmer 8,Control,35,-2,
4.4.C,H.0,Farmer 8,Control,57,-2,
4.4.C,H.0,Farmer 8,Control,87,-2,
4.4.C,H.0,Farmer 8,Control,112,-2,
4.4.C,H.0,Farmer 8,Control,149,-2,
4.4.C,H.0,Farmer 8,Control,177,-2,
4.4.C,H.0,Farmer 8,Control,198,-2,
4.4.C,H.0,Farmer 8,Control,226,-2,
4.4.C,H.0,Farmer 8,Control,248,-2,
4.4.C,H.0,Farmer 8,Control,281,-2,487.2421828028161
4.4.C,H.0,Farmer 8,Control,309,-2,459.37914820386305
4.4.C,H.0,Farmer 8,Control,347,-2,710.9645439557133
4.4.C,H.0,Farmer 8,Control,382,-2,403.5087112341296
4.4.C,H.0,Farmer 8,Control,420,-2,422.2765583970155
4.4.C,H.0,Farmer 8,Control,450,-2,481.03435633912994
4.4.C,H.0,Farmer 8,Control,471,-2,319.5827491425477
4.4.C,H.0,Farmer 8,Control,499,-2,486.0391157109332
4.4.C,H.0,Farmer 8,Control,527,-2,
4.4.D,H.0,Farmer 8,Control,35,-2,
4.4.D,H.0,Farmer 8,Control,57,-2,
4.4.D,H.0,Farmer 8,Control,87,-2,
4.4.D,H.0,Farmer 8,Control,112,-2,
4.4.D,H.0,Farmer 8,Control,149,-2,
4.4.D,H.0,Farmer 8,Control,177,-2,
4.4.D,H.0,Farmer 8,Control,198,-2,
4.4.D,H.0,Farmer 8,Control,226,-2,
4.4.D,H.0,Farmer 8,Control,248,-2,
4.4.D,H.0,Farmer 8,Control,281,-2,376.60813045309584
4.4.D,H.0,Farmer 8,Control,309,-2,407.64726229014985
4.4.D,H.0,Farmer 8,Control,347,-2,686.6144656116493
4.4.D,H.0,Farmer 8,Control,382,-2,407.64726229014985
4.4.D,H.0,Farmer 8,Control,420,-2,91.81808243576629
4.4.D,H.0,Farmer 8,Control,450,-2,412.3632853962332
4.4.D,H.0,Farmer 8,Control,471,-2,258.7075530416992
4.4.D,H.0,Farmer 8,Control,499,-2,392.68110716649613
4.4.D,H.0,Farmer 8,Control,527,-2,
4.5.A,H.1,Farmer 8,Basanite,35,-2,
4.5.A,H.1,Farmer 8,Basanite,57,-2,
4.5.A,H.1,Farmer 8,Basanite,87,-2,
4.5.A,H.1,Farmer 8,Basanite,112,-2,
4.5.A,H.1,Farmer 8,Basanite,149,-2,
4.5.A,H.1,Farmer 8,Basanite,177,-2,
4.5.A,H.1,Farmer 8,Basanite,198,-2,
4.5.A,H.1,Farmer 8,Basanite,226,-2,
4.5.A,H.1,Farmer 8,Basanite,255,-2,152.74140128768278
4.5.A,H.1,Farmer 8,Basanite,281,-2,384.5002507972802
4.5.A,H.1,Farmer 8,Basanite,309,-2,364.9624408207473
4.5.A,H.1,Farmer 8,Basanite,347,-2,631.0808874180155
4.5.A,H.1,Farmer 8,Basanite,382,-2,341.91167483001385
4.5.A,H.1,Farmer 8,Basanite,420,-2,358.0327742944822
4.5.A,H.1,Farmer 8,Basanite,450,-2,306.25276562970095
4.5.A,H.1,Farmer 8,Basanite,471,-2,179.25700056561766
4.5.A,H.1,Farmer 8,Basanite,499,-2,349.3706911366508
4.5.A,H.1,Farmer 8,Basanite,527,-2,
4.5.B,H.1,Farmer 8,Basanite,35,-2,
4.5.B,H.1,Farmer 8,Basanite,57,-2,
4.5.B,H.1,Farmer 8,Basanite,87,-2,
4.5.B,H.1,Farmer 8,Basanite,112,-2,
4.5.B,H.1,Farmer 8,Basanite,149,-2,
4.5.B,H.1,Farmer 8,Basanite,177,-2,
4.5.B,H.1,Farmer 8,Basanite,198,-2,
4.5.B,H.1,Farmer 8,Basanite,226,-2,
4.5.B,H.1,Farmer 8,Basanite,255,-2,
4.5.B,H.1,Farmer 8,Basanite,281,-2,364.9624408207473
4.5.B,H.1,Farmer 8,Basanite,309,-2,389.7937461941152
4.5.B,H.1,Farmer 8,Basanite,347,-2,599.1274247547987
4.5.B,H.1,Farmer 8,Basanite,382,-2,339.2649271315963
4.5.B,H.1,Farmer 8,Basanite,420,-2,430.4092920151634
4.5.B,H.1,Farmer 8,Basanite,450,-2,336.4738114206631
4.5.B,H.1,Farmer 8,Basanite,471,-2,250.911678199651
4.5.B,H.1,Farmer 8,Basanite,499,-2,391.7186535892653
4.5.B,H.1,Farmer 8,Basanite,527,-2,
4.5.C,H.1,Farmer 8,Basanite,35,-2,
4.5.C,H.1,Farmer 8,Basanite,57,-2,
4.5.C,H.1,Farmer 8,Basanite,87,-2,
4.5.C,H.1,Farmer 8,Basanite,112,-2,
4.5.C,H.1,Farmer 8,Basanite,149,-2,
4.5.C,H.1,Farmer 8,Basanite,177,-2,
4.5.C,H.1,Farmer 8,Basanite,198,-2,
4.5.C,H.1,Farmer 8,Basanite,226,-2,
4.5.C,H.1,Farmer 8,Basanite,248,-2,
4.5.C,H.1,Farmer 8,Basanite,281,-2,384.8852323244479
4.5.C,H.1,Farmer 8,Basanite,309,-2,344.31780901377937
4.5.C,H.1,Farmer 8,Basanite,347,-2,631.3696234430471
4.5.C,H.1,Farmer 8,Basanite,382,-2,279.3521848486672
4.5.C,H.1,Farmer 8,Basanite,420,-2,323.5769319453637
4.5.C,H.1,Farmer 8,Basanite,450,-2,294.51083049521634
4.5.C,H.1,Farmer 8,Basanite,471,-2,214.38656039472892
4.5.C,H.1,Farmer 8,Basanite,499,-2,395.3759776159817
4.5.C,H.1,Farmer 8,Basanite,527,-2,
4.5.D,H.1,Farmer 8,Basanite,35,-2,
4.5.D,H.1,Farmer 8,Basanite,57,-2,
4.5.D,H.1,Farmer 8,Basanite,87,-2,
4.5.D,H.1,Farmer 8,Basanite,112,-2,
4.5.D,H.1,Farmer 8,Basanite,149,-2,
4.5.D,H.1,Farmer 8,Basanite,177,-2,
4.5.D,H.1,Farmer 8,Basanite,198,-2,
4.5.D,H.1,Farmer 8,Basanite,226,-2,
4.5.D,H.1,Farmer 8,Basanite,248,-2,
4.5.D,H.1,Farmer 8,Basanite,281,-2,352.83552439978337
4.5.D,H.1,Farmer 8,Basanite,309,-2,273.14435838498105
4.5.D,H.1,Farmer 8,Basanite,347,-2,560.3405408267646
4.5.D,H.1,Farmer 8,Basanite,382,-2,289.6985618869968
4.5.D,H.1,Farmer 8,Basanite,420,-2,220.88312283530897
4.5.D,H.1,Farmer 8,Basanite,450,-2,308.37016354774653
4.5.D,H.1,Farmer 8,Basanite,471,-2,178.9201417654492
4.5.D,H.1,Farmer 8,Basanite,499,-2,334.93388555268064
4.5.D,H.1,Farmer 8,Basanite,527,-2,
4.6.A,I.0,Farmer 9,Control,35,-2,
4.6.A,I.0,Farmer 9,Control,57,-2,
4.6.A,I.0,Farmer 9,Control,87,-2,
4.6.A,I.0,Farmer 9,Control,112,-2,
4.6.A,I.0,Farmer 9,Control,149,-2,
4.6.A,I.0,Farmer 9,Control,177,-2,
4.6.A,I.0,Farmer 9,Control,198,-2,
4.6.A,I.0,Farmer 9,Control,226,-2,
4.6.A,I.0,Farmer 9,Control,255,-2,26.274985859558335
4.6.A,I.0,Farmer 9,Control,281,-2,241.6721227510681
4.6.A,I.0,Farmer 9,Control,309,-2,333.15334617004635
4.6.A,I.0,Farmer 9,Control,347,-2,544.2675641133642
4.6.A,I.0,Farmer 9,Control,382,-2,119.5367488296528
4.6.A,I.0,Farmer 9,Control,420,-2,380.4098227330164
4.6.A,I.0,Farmer 9,Control,450,-2,305.38655731391776
4.6.A,I.0,Farmer 9,Control,471,-2,155.19565819844755
4.6.A,I.0,Farmer 9,Control,499,-2,264.09729369998195
4.6.A,I.0,Farmer 9,Control,527,-2,
4.6.B,I.0,Farmer 9,Control,35,-2,
4.6.B,I.0,Farmer 9,Control,57,-2,
4.6.B,I.0,Farmer 9,Control,87,-2,
4.6.B,I.0,Farmer 9,Control,112,-2,
4.6.B,I.0,Farmer 9,Control,149,-2,
4.6.B,I.0,Farmer 9,Control,177,-2,
4.6.B,I.0,Farmer 9,Control,198,-2,
4.6.B,I.0,Farmer 9,Control,226,-2,
4.6.B,I.0,Farmer 9,Control,255,-2,
4.6.B,I.0,Farmer 9,Control,281,-2,215.83024095312595
4.6.B,I.0,Farmer 9,Control,309,-2,395.2316096034658
4.6.B,I.0,Farmer 9,Control,347,-2,527.4246243456284
4.6.B,I.0,Farmer 9,Control,382,-2,437.4352040435646
4.6.B,I.0,Farmer 9,Control,420,-2,321.0745525001504
4.6.B,I.0,Farmer 9,Control,450,-2,235.8011550875504
4.6.B,I.0,Farmer 9,Control,471,-2,163.90586413141585
4.6.B,I.0,Farmer 9,Control,499,-2,302.78793212588005
4.6.B,I.0,Farmer 9,Control,527,-2,
4.6.C,I.0,Farmer 9,Control,35,-2,
4.6.C,I.0,Farmer 9,Control,57,-2,
4.6.C,I.0,Farmer 9,Control,87,-2,
4.6.C,I.0,Farmer 9,Control,112,-2,
4.6.C,I.0,Farmer 9,Control,149,-2,
4.6.C,I.0,Farmer 9,Control,177,-2,
4.6.C,I.0,Farmer 9,Control,198,-2,
4.6.C,I.0,Farmer 9,Control,226,-2,
4.6.C,I.0,Farmer 9,Control,248,-2,
4.6.C,I.0,Farmer 9,Control,281,-2,160.72976694145254
4.6.C,I.0,Farmer 9,Control,309,-2,363.8074964799326
4.6.C,I.0,Farmer 9,Control,347,-2,653.2654450929659
4.6.C,I.0,Farmer 9,Control,382,-2,360.3426632168
4.6.C,I.0,Farmer 9,Control,420,-2,385.4627046151994
4.6.C,I.0,Farmer 9,Control,450,-2,272.1337820566821
4.6.C,I.0,Farmer 9,Control,471,-2,167.7556789217161
4.6.C,I.0,Farmer 9,Control,499,-2,314.14488573319693
4.6.C,I.0,Farmer 9,Control,527,-2,
4.6.D,I.0,Farmer 9,Control,35,-2,
4.6.D,I.0,Farmer 9,Control,57,-2,
4.6.D,I.0,Farmer 9,Control,87,-2,
4.6.D,I.0,Farmer 9,Control,112,-2,
4.6.D,I.0,Farmer 9,Control,149,-2,
4.6.D,I.0,Farmer 9,Control,177,-2,
4.6.D,I.0,Farmer 9,Control,198,-2,
4.6.D,I.0,Farmer 9,Control,226,-2,
4.6.D,I.0,Farmer 9,Control,248,-2,
4.6.D,I.0,Farmer 9,Control,281,-2,240.85403694566455
4.6.D,I.0,Farmer 9,Control,309,-2,372.469579637764
4.6.D,I.0,Farmer 9,Control,347,-2,583.5356748300138
4.6.D,I.0,Farmer 9,Control,382,-2,273.14435838498105
4.6.D,I.0,Farmer 9,Control,420,-2,400.8619637764005
4.6.D,I.0,Farmer 9,Control,450,-2,371.2183897948131
4.6.D,I.0,Farmer 9,Control,471,-2,212.70226644202415
4.6.D,I.0,Farmer 9,Control,499,-2,406.6366857211625
4.6.D,I.0,Farmer 9,Control,527,-2,
4.7.A,I.1,Farmer 9,Basanite,35,-2,
4.7.A,I.1,Farmer 9,Basanite,57,-2,
4.7.A,I.1,Farmer 9,Basanite,87,-2,
4.7.A,I.1,Farmer 9,Basanite,112,-2,
4.7.A,I.1,Farmer 9,Basanite,149,-2,
4.7.A,I.1,Farmer 9,Basanite,177,-2,
4.7.A,I.1,Farmer 9,Basanite,198,-2,
4.7.A,I.1,Farmer 9,Basanite,226,-2,
4.7.A,I.1,Farmer 9,Basanite,255,-2,2.8873610830976593
4.7.A,I.1,Farmer 9,Basanite,281,-2,146.10047078644925
4.7.A,I.1,Farmer 9,Basanite,309,-2,173.24166496179072
4.7.A,I.1,Farmer 9,Basanite,347,-2,380.3616999819484
4.7.A,I.1,Farmer 9,Basanite,382,-2,270.2569974126
4.7.A,I.1,Farmer 9,Basanite,420,-2,240.6134235754257
4.7.A,I.1,Farmer 9,Basanite,450,-2,259.67000661893013
4.7.A,I.1,Farmer 9,Basanite,471,-2,141.62506112281125
4.7.A,I.1,Farmer 9,Basanite,499,-2,245.23320127564836
4.7.A,I.1,Farmer 9,Basanite,527,-2,
4.7.B,I.1,Farmer 9,Basanite,35,-2,
4.7.B,I.1,Farmer 9,Basanite,57,-2,
4.7.B,I.1,Farmer 9,Basanite,87,-2,
4.7.B,I.1,Farmer 9,Basanite,112,-2,
4.7.B,I.1,Farmer 9,Basanite,149,-2,
4.7.B,I.1,Farmer 9,Basanite,177,-2,
4.7.B,I.1,Farmer 9,Basanite,198,-2,
4.7.B,I.1,Farmer 9,Basanite,226,-2,
4.7.B,I.1,Farmer 9,Basanite,255,-2,
4.7.B,I.1,Farmer 9,Basanite,281,-2,276.2242103616343
4.7.B,I.1,Farmer 9,Basanite,309,-2,397.30088501113187
4.7.B,I.1,Farmer 9,Basanite,347,-2,615.0079107046151
4.7.B,I.1,Farmer 9,Basanite,382,-2,177.86144271015104
4.7.B,I.1,Farmer 9,Basanite,420,-2,432.911671701065
4.7.B,I.1,Farmer 9,Basanite,450,-2,384.981477826584
4.7.B,I.1,Farmer 9,Basanite,471,-2,188.92966019616105
4.7.B,I.1,Farmer 9,Basanite,499,-2,315.2998303147
4.7.B,I.1,Farmer 9,Basanite,527,-2,
4.7.C,I.1,Farmer 9,Basanite,35,-2,
4.7.C,I.1,Farmer 9,Basanite,57,-2,
4.7.C,I.1,Farmer 9,Basanite,87,-2,
4.7.C,I.1,Farmer 9,Basanite,112,-2,
4.7.C,I.1,Farmer 9,Basanite,149,-2,
4.7.C,I.1,Farmer 9,Basanite,177,-2,
4.7.C,I.1,Farmer 9,Basanite,198,-2,
4.7.C,I.1,Farmer 9,Basanite,226,-2,
4.7.C,I.1,Farmer 9,Basanite,248,-2,
4.7.C,I.1,Farmer 9,Basanite,281,-2,144.36805413081413
4.7.C,I.1,Farmer 9,Basanite,309,-2,251.4891502497142
4.7.C,I.1,Farmer 9,Basanite,347,-2,440.3225650159456
4.7.C,I.1,Farmer 9,Basanite,382,-2,258.7075530416992
4.7.C,I.1,Farmer 9,Basanite,420,-2,360.9201352668632
4.7.C,I.1,Farmer 9,Basanite,450,-2,296.43573789036645
4.7.C,I.1,Farmer 9,Basanite,471,-2,139.89264446717613
4.7.C,I.1,Farmer 9,Basanite,499,-2,332.479628617847
4.7.C,I.1,Farmer 9,Basanite,527,-2,
4.7.D,I.1,Farmer 9,Basanite,35,-2,
4.7.D,I.1,Farmer 9,Basanite,57,-2,
4.7.D,I.1,Farmer 9,Basanite,87,-2,
4.7.D,I.1,Farmer 9,Basanite,112,-2,
4.7.D,I.1,Farmer 9,Basanite,149,-2,
4.7.D,I.1,Farmer 9,Basanite,177,-2,
4.7.D,I.1,Farmer 9,Basanite,198,-2,
4.7.D,I.1,Farmer 9,Basanite,226,-2,
4.7.D,I.1,Farmer 9,Basanite,248,-2,
4.7.D,I.1,Farmer 9,Basanite,281,-2,194.8968730970576
4.7.D,I.1,Farmer 9,Basanite,309,-2,378.82177411396594
4.7.D,I.1,Farmer 9,Basanite,347,-2,541.8133071785306
4.7.D,I.1,Farmer 9,Basanite,382,-2,248.3130530116132
4.7.D,I.1,Farmer 9,Basanite,420,-2,234.83870141404415
4.7.D,I.1,Farmer 9,Basanite,450,-2,392.44049389253263
4.7.D,I.1,Farmer 9,Basanite,471,-2,119.63299418737589
4.7.D,I.1,Farmer 9,Basanite,499,-2,226.3691088994524
4.7.D,I.1,Farmer 9,Basanite,527,-2,
4.8.A,J.0,Farmer 10,Control,35,15,
4.8.A,J.0,Farmer 10,Control,57,15,
4.8.A,J.0,Farmer 10,Control,87,15,
4.8.A,J.0,Farmer 10,Control,112,15,
4.8.A,J.0,Farmer 10,Control,149,15,
4.8.A,J.0,Farmer 10,Control,177,15,
4.8.A,J.0,Farmer 10,Control,198,15,
4.8.A,J.0,Farmer 10,Control,226,15,
4.8.A,J.0,Farmer 10,Control,255,15,5.534108743004994
4.8.A,J.0,Farmer 10,Control,281,15,15.014277631626452
4.8.A,J.0,Farmer 10,Control,309,15,21.173981274444913
4.8.A,J.0,Farmer 10,Control,347,15,3.9941828316986583
4.8.A,J.0,Farmer 10,Control,382,15,4.042305515373969
4.8.A,J.0,Farmer 10,Control,420,15,22.95452060894157
4.8.A,J.0,Farmer 10,Control,450,15,20.115282209519226
4.8.A,J.0,Farmer 10,Control,471,15,17.468534552018774
4.8.A,J.0,Farmer 10,Control,499,15,32.91591633672303
4.8.A,J.0,Farmer 10,Control,527,15,
4.8.B,J.0,Farmer 10,Control,35,15,
4.8.B,J.0,Farmer 10,Control,57,15,
4.8.B,J.0,Farmer 10,Control,87,15,
4.8.B,J.0,Farmer 10,Control,112,15,
4.8.B,J.0,Farmer 10,Control,149,15,
4.8.B,J.0,Farmer 10,Control,177,15,
4.8.B,J.0,Farmer 10,Control,198,15,
4.8.B,J.0,Farmer 10,Control,226,15,
4.8.B,J.0,Farmer 10,Control,255,15,
4.8.B,J.0,Farmer 10,Control,281,15,31.76097189963295
4.8.B,J.0,Farmer 10,Control,309,15,23.098888662374392
4.8.B,J.0,Farmer 10,Control,347,15,33.782124676575
4.8.B,J.0,Farmer 10,Control,382,15,56.30354110355617
4.8.B,J.0,Farmer 10,Control,420,15,34.40771957398159
4.8.B,J.0,Farmer 10,Control,450,15,24.92755068295325
4.8.B,J.0,Farmer 10,Control,471,15,37.53569408508334
4.8.B,J.0,Farmer 10,Control,499,15,27.911157133401527
4.8.B,J.0,Farmer 10,Control,527,15,
4.8.C,J.0,Farmer 10,Control,35,15,
4.8.C,J.0,Farmer 10,Control,57,15,
4.8.C,J.0,Farmer 10,Control,87,15,
4.8.C,J.0,Farmer 10,Control,112,15,
4.8.C,J.0,Farmer 10,Control,149,15,
4.8.C,J.0,Farmer 10,Control,177,15,
4.8.C,J.0,Farmer 10,Control,198,15,
4.8.C,J.0,Farmer 10,Control,226,15,
4.8.C,J.0,Farmer 10,Control,248,15,
4.8.C,J.0,Farmer 10,Control,281,15,8.084611030747938
4.8.C,J.0,Farmer 10,Control,309,15,19.585932679463266
4.8.C,J.0,Farmer 10,Control,347,15,25.264409483121728
4.8.C,J.0,Farmer 10,Control,382,15,14.629296152596424
4.8.C,J.0,Farmer 10,Control,420,15,5.5822314266803055
4.8.C,J.0,Farmer 10,Control,450,15,26.563721956796435
4.8.C,J.0,Farmer 10,Control,471,15,14.292437359648591
4.8.C,J.0,Farmer 10,Control,499,15,29.64357378903664
4.8.C,J.0,Farmer 10,Control,527,15,
4.8.D,J.0,Farmer 10,Control,35,15,
4.8.D,J.0,Farmer 10,Control,57,15,
4.8.D,J.0,Farmer 10,Control,87,15,
4.8.D,J.0,Farmer 10,Control,112,15,
4.8.D,J.0,Farmer 10,Control,149,15,
4.8.D,J.0,Farmer 10,Control,177,15,
4.8.D,J.0,Farmer 10,Control,198,15,
4.8.D,J.0,Farmer 10,Control,226,15,
4.8.D,J.0,Farmer 10,Control,248,15,
4.8.D,J.0,Farmer 10,Control,281,15,32.09783069980143
4.8.D,J.0,Farmer 10,Control,309,15,45.47593705999157
4.8.D,J.0,Farmer 10,Control,347,15,157.8424058727962
4.8.D,J.0,Farmer 10,Control,382,15,163.47275997352426
4.8.D,J.0,Farmer 10,Control,420,15,24.542569203923215
4.8.D,J.0,Farmer 10,Control,450,15,16.842939649798424
4.8.D,J.0,Farmer 10,Control,471,15,8.421469826102653
4.8.D,J.0,Farmer 10,Control,499,15,20.355895635116433
4.8.D,J.0,Farmer 10,Control,527,15,
4.9.A,J.1,Farmer 10,Basanite,35,15,
4.9.A,J.1,Farmer 10,Basanite,57,15,
4.9.A,J.1,Farmer 10,Basanite,87,15,
4.9.A,J.1,Farmer 10,Basanite,112,15,
4.9.A,J.1,Farmer 10,Basanite,149,15,
4.9.A,J.1,Farmer 10,Basanite,177,15,
4.9.A,J.1,Farmer 10,Basanite,198,15,
4.9.A,J.1,Farmer 10,Basanite,226,15,
4.9.A,J.1,Farmer 10,Basanite,255,15,10.971972113845595
4.9.A,J.1,Farmer 10,Basanite,281,15,25.9862497382514
4.9.A,J.1,Farmer 10,Basanite,309,15,25.023796040676334
4.9.A,J.1,Farmer 10,Basanite,347,15,35.94764549010169
4.9.A,J.1,Farmer 10,Basanite,382,15,9.143310095673629
4.9.A,J.1,Farmer 10,Basanite,420,15,9.624536942054275
4.9.A,J.1,Farmer 10,Basanite,450,15,23.098888662374392
4.9.A,J.1,Farmer 10,Basanite,471,15,20.788999795414885
4.9.A,J.1,Farmer 10,Basanite,499,15,31.279745062879833
4.9.A,J.1,Farmer 10,Basanite,527,15,
4.9.B,J.1,Farmer 10,Basanite,35,15,
4.9.B,J.1,Farmer 10,Basanite,57,15,
4.9.B,J.1,Farmer 10,Basanite,87,15,
4.9.B,J.1,Farmer 10,Basanite,112,15,
4.9.B,J.1,Farmer 10,Basanite,149,15,
4.9.B,J.1,Farmer 10,Basanite,177,15,
4.9.B,J.1,Farmer 10,Basanite,198,15,
4.9.B,J.1,Farmer 10,Basanite,226,15,
4.9.B,J.1,Farmer 10,Basanite,255,15,
4.9.B,J.1,Farmer 10,Basanite,281,15,26.948703435826463
4.9.B,J.1,Farmer 10,Basanite,309,15,24.25383310668512
4.9.B,J.1,Farmer 10,Basanite,347,15,3.2723425597208013
4.9.B,J.1,Farmer 10,Basanite,382,15,30.31729136530477
4.9.B,J.1,Farmer 10,Basanite,420,15,15.014277631626452
4.9.B,J.1,Farmer 10,Basanite,450,15,27.622421036163423
4.9.B,J.1,Farmer 10,Basanite,471,15,23.38762477164691
4.9.B,J.1,Farmer 10,Basanite,499,15,33.782124676575
4.9.B,J.1,Farmer 10,Basanite,527,15,
4.9.C,J.1,Farmer 10,Basanite,35,15,
4.9.C,J.1,Farmer 10,Basanite,57,15,
4.9.C,J.1,Farmer 10,Basanite,87,15,
4.9.C,J.1,Farmer 10,Basanite,112,15,
4.9.C,J.1,Farmer 10,Basanite,149,15,
4.9.C,J.1,Farmer 10,Basanite,177,15,
4.9.C,J.1,Farmer 10,Basanite,198,15,
4.9.C,J.1,Farmer 10,Basanite,226,15,
4.9.C,J.1,Farmer 10,Basanite,248,15,
4.9.C,J.1,Farmer 10,Basanite,281,15,25.9862497382514
4.9.C,J.1,Farmer 10,Basanite,309,15,13.811210513267945
4.9.C,J.1,Farmer 10,Basanite,347,15,44.994710199169624
4.9.C,J.1,Farmer 10,Basanite,382,15,10.82760406041278
4.9.C,J.1,Farmer 10,Basanite,420,15,5.101004580299657
4.9.C,J.1,Farmer 10,Basanite,450,15,16.93918501955593
4.9.C,J.1,Farmer 10,Basanite,471,15,20.452141002467055
4.9.C,J.1,Farmer 10,Basanite,499,15,25.023796040676334
4.9.C,J.1,Farmer 10,Basanite,527,15,
4.9.D,J.1,Farmer 10,Basanite,35,15,
4.9.D,J.1,Farmer 10,Basanite,57,15,
4.9.D,J.1,Farmer 10,Basanite,87,15,
4.9.D,J.1,Farmer 10,Basanite,112,15,
4.9.D,J.1,Farmer 10,Basanite,149,15,
4.9.D,J.1,Farmer 10,Basanite,177,15,
4.9.D,J.1,Farmer 10,Basanite,198,15,
4.9.D,J.1,Farmer 10,Basanite,226,15,
4.9.D,J.1,Farmer 10,Basanite,248,15,
4.9.D,J.1,Farmer 10,Basanite,281,15,17.516657235694083
4.9.D,J.1,Farmer 10,Basanite,309,15,13.137492927372284
4.9.D,J.1,Farmer 10,Basanite,347,15,22.52141644623623
4.9.D,J.1,Farmer 10,Basanite,382,15,27.718666393886515
4.9.D,J.1,Farmer 10,Basanite,420,15,7.507138814609784
4.9.D,J.1,Farmer 10,Basanite,450,15,24.83130530116132
4.9.D,J.1,Farmer 10,Basanite,471,15,20.788999795414885
4.9.D,J.1,Farmer 10,Basanite,499,15,36.28450426620134
4.9.D,J.1,Farmer 10,Basanite,527,15,
5.0.A,5.0,LUFA 6S A,Control,35,-3,648.4531764847463
5.0.A,5.0,LUFA 6S A,Control,59,-3,288.3511267825982
5.0.A,5.0,LUFA 6S A,Control,85,-3,0.0
5.0.A,5.0,LUFA 6S A,Control,114,-3,1101.8169892291955
5.0.A,5.0,LUFA 6S A,Control,149,-3,423.2630733497804
5.0.A,5.0,LUFA 6S A,Control,178,-3,1149.9396738672604
5.0.A,5.0,LUFA 6S A,Control,198,-3,687.0956924002647
5.0.A,5.0,LUFA 6S A,Control,227,-3,400.8619637764005
5.0.A,5.0,LUFA 6S A,Control,255,-3,269.4870343582646
5.0.A,5.0,LUFA 6S A,Control,281,-3,517.944455623082
5.0.A,5.0,LUFA 6S A,Control,309,-3,446.5785142306998
5.0.A,5.0,LUFA 6S A,Control,350,-3,728.4812012756483
5.0.A,5.0,LUFA 6S A,Control,382,-3,462.555245441964
5.0.A,5.0,LUFA 6S A,Control,410,-3,355.7228853721644
5.0.A,5.0,LUFA 6S A,Control,449,-3,487.9640231060834
5.0.A,5.0,LUFA 6S A,Control,471,-3,345.6171216077983
5.0.A,5.0,LUFA 6S A,Control,501,-3,468.9074399181659
5.0.A,5.0,LUFA 6S A,Control,529,-3,299.83801167338584
5.0.A,5.0,LUFA 6S A,Control,556,-3,608.7639923254244
5.0.A,5.0,LUFA 6S A,Control,596,-3,675.1420174499068
5.0.A,5.0,LUFA 6S A,Control,626,-3,329.534520372407
5.0.A,5.0,LUFA 6S A,Control,652,-3,424.57682267284434
5.0.A,5.0,LUFA 6S A,Control,682,-3,630.6574077862688
5.0.B,5.0,LUFA 6S A,Control,35,-3,597.2025173596486
5.0.B,5.0,LUFA 6S A,Control,58,-3,272.1337820566821
5.0.B,5.0,LUFA 6S A,Control,85,-3,0.0
5.0.B,5.0,LUFA 6S A,Control,114,-3,1296.2326354172933
5.0.B,5.0,LUFA 6S A,Control,149,-3,746.1422266080991
5.0.B,5.0,LUFA 6S A,Control,178,-3,1071.403452433961
5.0.B,5.0,LUFA 6S A,Control,198,-3,703.361159877249
5.0.B,5.0,LUFA 6S A,Control,227,-3,446.6747594921475
5.0.B,5.0,LUFA 6S A,Control,255,-3,261.4986687526325
5.0.B,5.0,LUFA 6S A,Control,281,-3,565.4415454600156
5.0.B,5.0,LUFA 6S A,Control,309,-3,432.6229356760334
5.0.B,5.0,LUFA 6S A,Control,350,-3,699.4150995848125
5.0.B,5.0,LUFA 6S A,Control,350,-3,401.5693671099344
5.0.B,5.0,LUFA 6S A,Control,382,-3,466.74191900836394
5.0.B,5.0,LUFA 6S A,Control,410,-3,436.52087297671335
5.0.B,5.0,LUFA 6S A,Control,449,-3,636.663118839882
5.0.B,5.0,LUFA 6S A,Control,471,-3,345.6171216077983
5.0.B,5.0,LUFA 6S A,Control,501,-3,387.9650840604127
5.0.B,5.0,LUFA 6S A,Control,529,-3,281.8064417835008
5.0.B,5.0,LUFA 6S A,Control,556,-3,606.2640189737572
5.0.B,5.0,LUFA 6S A,Control,596,-3,599.6760233467717
5.0.B,5.0,LUFA 6S A,Control,626,-3,292.06138579636297
5.0.B,5.0,LUFA 6S A,Control,652,-3,381.03541753414765
5.0.B,5.0,LUFA 6S A,Control,682,-3,427.4304978638907
5.0.B,5.0,LUFA 6S A,Control,742,-3,73.30047335320664
5.0.B,5.0,LUFA 6S A,Control,771,-3,278.13949311029546
5.0.B,5.0,LUFA 6S A,Control,798,-3,224.50194873337745
5.0.B,5.0,LUFA 6S A,Control,833,-3,233.4960785125459
5.0.B,5.0,LUFA 6S A,Control,868,-3,453.16169733437624
5.0.B,5.0,LUFA 6S A,Control,899,-3,432.3149503580239
5.0.B,5.0,LUFA 6S A,Control,926,-3,260.28116493170467
5.0.B,5.0,LUFA 6S A,Control,955,-3,409.76466044888383
5.0.B,5.0,LUFA 6S A,Control,987,-3,358.26376316264515
5.0.B,5.0,LUFA 6S A,Control,1016,-3,
5.0.B,5.0,LUFA 6S A,Control,1043,-3,
5.0.B,5.0,LUFA 6S A,Control,1079,-3,
5.0.B,5.0,LUFA 6S A,Control,1107,-3,
5.0.B,5.0,LUFA 6S A,Control,1142,-3,
5.0.B,5.0,LUFA 6S A,Control,1171,-3,
5.0.C,5.0,LUFA 6S A,Control,32,-3,618.4727439677478
5.0.C,5.0,LUFA 6S A,Control,58,-3,281.5177055177809
5.0.C,5.0,LUFA 6S A,Control,85,-3,0.0
5.0.C,5.0,LUFA 6S A,Control,114,-3,0.0
5.0.C,5.0,LUFA 6S A,Control,149,-3,424.2495885432336
5.0.C,5.0,LUFA 6S A,Control,178,-3,927.7091158312776
5.0.C,5.0,LUFA 6S A,Control,198,-3,504.9994533967146
5.0.C,5.0,LUFA 6S A,Control,226,-3,286.3299741260003
5.0.C,5.0,LUFA 6S A,Control,255,-3,170.0655677718274
5.0.C,5.0,LUFA 6S A,Control,281,-3,436.23213695168175
5.0.C,5.0,LUFA 6S A,Control,309,-3,424.4420790661292
5.0.C,5.0,LUFA 6S A,Control,350,-3,634.9788247186955
5.0.C,5.0,LUFA 6S A,Control,382,-3,465.8275879415127
5.0.C,5.0,LUFA 6S A,Control,410,-3,326.9455199470485
5.0.C,5.0,LUFA 6S A,Control,449,-3,510.1004580299656
5.0.C,5.0,LUFA 6S A,Control,471,-3,281.8064417835008
5.0.C,5.0,LUFA 6S A,Control,501,-3,1091.4224891991094
5.0.C,5.0,LUFA 6S A,Control,529,-3,244.1552531439918
5.0.C,5.0,LUFA 6S A,Control,556,-3,552.535041300065
5.0.C,5.0,LUFA 6S A,Control,596,-3,609.3727442084361
5.0.C,5.0,LUFA 6S A,Control,626,-3,330.77849180921123
5.0.C,5.0,LUFA 6S A,Control,652,-3,408.8118312774536
5.0.C,5.0,LUFA 6S A,Control,682,-3,433.58057693001984
5.0.D,5.0,LUFA 6S A,Control,32,-3,623.6699938624465
5.0.D,5.0,LUFA 6S A,Control,58,-3,249.03489331488055
5.0.D,5.0,LUFA 6S A,Control,85,-3,0.0
5.0.D,5.0,LUFA 6S A,Control,114,-3,0.0
5.0.D,5.0,LUFA 6S A,Control,149,-3,1183.8180439256273
5.0.D,5.0,LUFA 6S A,Control,178,-3,1190.4589744268608
5.0.D,5.0,LUFA 6S A,Control,198,-3,593.9301748600999
5.0.D,5.0,LUFA 6S A,Control,226,-3,398.45582935194653
5.0.D,5.0,LUFA 6S A,Control,255,-3,298.07190901979664
5.0.D,5.0,LUFA 6S A,Control,281,-3,461.9777731512125
5.0.D,5.0,LUFA 6S A,Control,309,-3,565.0084411817799
5.0.D,5.0,LUFA 6S A,Control,350,-3,457.35799554726515
5.0.D,5.0,LUFA 6S A,Control,350,-3,461.9777731512125
5.0.D,5.0,LUFA 6S A,Control,382,-3,515.2014626632168
5.0.D,5.0,LUFA 6S A,Control,410,-3,420.1110374872134
5.0.D,5.0,LUFA 6S A,Control,449,-3,548.5986056922799
5.0.D,5.0,LUFA 6S A,Control,471,-3,338.30247355436546
5.0.D,5.0,LUFA 6S A,Control,501,-3,444.6536068355497
5.0.D,5.0,LUFA 6S A,Control,529,-3,289.46757301883383
5.0.D,5.0,LUFA 6S A,Control,556,-3,614.0526753141456
5.0.D,5.0,LUFA 6S A,Control,596,-3,658.5926262711354
5.0.D,5.0,LUFA 6S A,Control,626,-3,301.65704914941
5.0.D,5.0,LUFA 6S A,Control,652,-3,457.58898441542806
5.0.D,5.0,LUFA 6S A,Control,682,-3,469.8554567663518
5.0.D,5.0,LUFA 6S A,Control,742,-3,103.94499897152112
5.0.D,5.0,LUFA 6S A,Control,771,-3,343.0088720139599
5.0.D,5.0,LUFA 6S A,Control,798,-3,268.17809735844514
5.0.D,5.0,LUFA 6S A,Control,833,-3,326.9358953005596
5.0.D,5.0,LUFA 6S A,Control,868,-3,483.5367360250316
5.0.D,5.0,LUFA 6S A,Control,899,-3,371.1558303147
5.0.D,5.0,LUFA 6S A,Control,926,-3,168.28502843733077
5.0.D,5.0,LUFA 6S A,Control,955,-3,313.71178169564956
5.0.D,5.0,LUFA 6S A,Control,987,-3,335.8722779950659
5.0.D,5.0,LUFA 6S A,Control,1016,-3,
5.0.D,5.0,LUFA 6S A,Control,1043,-3,
5.0.D,5.0,LUFA 6S A,Control,1079,-3,
5.0.D,5.0,LUFA 6S A,Control,1107,-3,
5.0.D,5.0,LUFA 6S A,Control,1142,-3,
5.0.D,5.0,LUFA 6S A,Control,1171,-3,
5.10.A,5.8,LUFA 6S A,Steel Slag,35,1,
5.10.A,5.8,LUFA 6S A,Steel Slag,59,1,804.611288525182
5.10.A,5.8,LUFA 6S A,Steel Slag,85,1,0.0
5.10.A,5.8,LUFA 6S A,Steel Slag,114,1,1166.493877369276
5.10.A,5.8,LUFA 6S A,Steel Slag,149,1,0.0
5.10.A,5.8,LUFA 6S A,Steel Slag,178,1,1953.1554045369755
5.10.A,5.8,LUFA 6S A,Steel Slag,198,1,1262.2580200974787
5.10.A,5.8,LUFA 6S A,Steel Slag,227,1,1273.3262374390756
5.10.A,5.8,LUFA 6S A,Steel Slag,255,1,1136.8984263794453
5.10.A,5.8,LUFA 6S A,Steel Slag,281,1,1153.5969978939768
5.10.A,5.8,LUFA 6S A,Steel Slag,309,1,1068.3236006979962
5.10.A,5.8,LUFA 6S A,Steel Slag,350,1,1421.1591248570912
5.10.A,5.8,LUFA 6S A,Steel Slag,350,1,691.4363586256694
5.10.A,5.8,LUFA 6S A,Steel Slag,382,1,866.3526929418135
5.10.A,5.8,LUFA 6S A,Steel Slag,410,1,685.1707850051146
5.10.A,5.8,LUFA 6S A,Steel Slag,449,1,878.2389960888139
5.10.A,5.8,LUFA 6S A,Steel Slag,471,1,400.5251047596125
5.10.A,5.8,LUFA 6S A,Steel Slag,501,1,776.9888674408809
5.10.A,5.8,LUFA 6S A,Steel Slag,529,1,675.6424932908117
5.10.A,5.8,LUFA 6S A,Steel Slag,556,1,1121.9563326460432
5.10.A,5.8,LUFA 6S A,Steel Slag,596,1,1211.8543199951862
5.10.A,5.8,LUFA 6S A,Steel Slag,626,1,774.7367256949166
5.10.A,5.8,LUFA 6S A,Steel Slag,652,1,723.8181130031891
5.10.B,5.8,LUFA 6S A,Steel Slag,35,1,
5.10.B,5.8,LUFA 6S A,Steel Slag,59,1,925.5917176725436
5.10.B,5.8,LUFA 6S A,Steel Slag,85,1,0.0
5.10.B,5.8,LUFA 6S A,Steel Slag,114,1,1292.6715568927132
5.10.B,5.8,LUFA 6S A,Steel Slag,149,1,0.0
5.10.B,5.8,LUFA 6S A,Steel Slag,178,1,2158.92800433239
5.10.B,5.8,LUFA 6S A,Steel Slag,198,1,1218.081395511162
5.10.B,5.8,LUFA 6S A,Steel Slag,227,1,972.1744766833142
5.10.B,5.8,LUFA 6S A,Steel Slag,255,1,731.5129302605451
5.10.B,5.8,LUFA 6S A,Steel Slag,281,1,1084.6853134364285
5.10.B,5.8,LUFA 6S A,Steel Slag,309,1,1007.5927726096636
5.10.B,5.8,LUFA 6S A,Steel Slag,350,1,1672.8407661110778
5.10.B,5.8,LUFA 6S A,Steel Slag,350,1,420.0099799025212
5.10.B,5.8,LUFA 6S A,Steel Slag,382,1,807.7392630122149
5.10.B,5.8,LUFA 6S A,Steel Slag,410,1,673.0438683434622
5.10.B,5.8,LUFA 6S A,Steel Slag,449,1,952.8291572296768
5.10.B,5.8,LUFA 6S A,Steel Slag,471,1,533.9211868343463
5.10.B,5.8,LUFA 6S A,Steel Slag,501,1,659.0882897887959
5.10.B,5.8,LUFA 6S A,Steel Slag,529,1,237.60094349840543
5.10.B,5.8,LUFA 6S A,Steel Slag,556,1,652.4232980017406
5.10.B,5.8,LUFA 6S A,Steel Slag,596,1,662.0333981587339
5.10.B,5.8,LUFA 6S A,Steel Slag,626,1,429.9473142925404
5.10.B,5.8,LUFA 6S A,Steel Slag,652,1,564.7967013659064
5.10.C,5.8,LUFA 6S A,Steel Slag,35,1,
5.10.C,5.8,LUFA 6S A,Steel Slag,59,1,1068.3236006979962
5.10.C,5.8,LUFA 6S A,Steel Slag,85,1,0.0
5.10.C,5.8,LUFA 6S A,Steel Slag,114,1,958.3632661411638
5.10.C,5.8,LUFA 6S A,Steel Slag,149,1,0.0
5.10.C,5.8,LUFA 6S A,Steel Slag,178,1,1787.5171237739935
5.10.C,5.8,LUFA 6S A,Steel Slag,198,1,1362.8344311932126
5.10.C,5.8,LUFA 6S A,Steel Slag,227,1,996.1395735002104
5.10.C,5.8,LUFA 6S A,Steel Slag,255,1,916.4484077260964
5.10.C,5.8,LUFA 6S A,Steel Slag,281,1,1102.6831975449786
5.10.C,5.8,LUFA 6S A,Steel Slag,309,1,1012.7418997532944
5.10.C,5.8,LUFA 6S A,Steel Slag,350,1,1211.4885877609966
5.10.C,5.8,LUFA 6S A,Steel Slag,350,1,481.4433992418316
5.10.C,5.8,LUFA 6S A,Steel Slag,382,1,693.7366229014983
5.10.C,5.8,LUFA 6S A,Steel Slag,410,1,677.5674006859618
5.10.C,5.8,LUFA 6S A,Steel Slag,449,1,693.4478866357783
5.10.C,5.8,LUFA 6S A,Steel Slag,471,1,481.4674606173657
5.10.C,5.8,LUFA 6S A,Steel Slag,501,1,756.9698304350442
5.10.C,5.8,LUFA 6S A,Steel Slag,529,1,469.83139539081776
5.10.C,5.8,LUFA 6S A,Steel Slag,556,1,1019.1301862123208
5.10.C,5.8,LUFA 6S A,Steel Slag,596,1,910.2117078043204
5.10.C,5.8,LUFA 6S A,Steel Slag,626,1,428.3833270807454
5.10.C,5.8,LUFA 6S A,Steel Slag,652,1,656.6436575004512
5.2.A,5.2,LUFA 6S A,Basanite,35,-3,749.9439186473313
5.2.A,5.2,LUFA 6S A,Basanite,59,-3,336.08883013418375
5.2.A,5.2,LUFA 6S A,Basanite,85,-3,0.0
5.2.A,5.2,LUFA 6S A,Basanite,114,-3,1133.289225103797
5.2.A,5.2,LUFA 6S A,Basanite,149,-3,0.0
5.2.A,5.2,LUFA 6S A,Basanite,178,-3,1719.8085064083275
5.2.A,5.2,LUFA 6S A,Basanite,198,-3,725.2088585354112
5.2.A,5.2,LUFA 6S A,Basanite,227,-3,693.2072733618148
5.2.A,5.2,LUFA 6S A,Basanite,255,-3,376.8006212166797
5.2.A,5.2,LUFA 6S A,Basanite,281,-3,632.9095493110295
5.2.A,5.2,LUFA 6S A,Basanite,309,-3,528.0983421385162
5.2.A,5.2,LUFA 6S A,Basanite,350,-3,487.9640231060834
5.2.A,5.2,LUFA 6S A,Basanite,350,-3,393.84567615379984
5.2.A,5.2,LUFA 6S A,Basanite,382,-3,442.5362086768157
5.2.A,5.2,LUFA 6S A,Basanite,410,-3,412.3632853962332
5.2.A,5.2,LUFA 6S A,Basanite,449,-3,577.7128298934954
5.2.A,5.2,LUFA 6S A,Basanite,471,-3,402.3056441422467
5.2.A,5.2,LUFA 6S A,Basanite,501,-3,590.7540776219989
5.2.A,5.2,LUFA 6S A,Basanite,529,-3,204.19417577471572
5.2.A,5.2,LUFA 6S A,Basanite,556,-3,613.3958007406059
5.2.A,5.2,LUFA 6S A,Basanite,596,-3,655.8448209880257
5.2.A,5.2,LUFA 6S A,Basanite,626,-3,323.1438278305273
5.2.A,5.2,LUFA 6S A,Basanite,652,-3,397.7339890486792
5.2.A,5.2,LUFA 6S A,Basanite,682,-3,448.4360498224923
5.2.A,5.2,LUFA 6S A,Basanite,742,-3,80.36488347474743
5.2.A,5.2,LUFA 6S A,Basanite,771,-3,236.27275739815877
5.2.A,5.2,LUFA 6S A,Basanite,798,-3,216.4558358505325
5.2.A,5.2,LUFA 6S A,Basanite,833,-3,325.60770924845053
5.2.A,5.2,LUFA 6S A,Basanite,868,-3,458.0605867982429
5.2.A,5.2,LUFA 6S A,Basanite,899,-3,473.7870800890547
5.2.A,5.2,LUFA 6S A,Basanite,926,-3,273.2598529394067
5.2.A,5.2,LUFA 6S A,Basanite,955,-3,395.5299702749864
5.2.A,5.2,LUFA 6S A,Basanite,987,-3,321.2092958661773
5.2.A,5.2,LUFA 6S A,Basanite,1016,-3,
5.2.A,5.2,LUFA 6S A,Basanite,1043,-3,
5.2.A,5.2,LUFA 6S A,Basanite,1079,-3,
5.2.A,5.2,LUFA 6S A,Basanite,1107,-3,
5.2.A,5.2,LUFA 6S A,Basanite,1142,-3,
5.2.A,5.2,LUFA 6S A,Basanite,1171,-3,
5.2.B,5.2,LUFA 6S A,Basanite,35,-3,
5.2.B,5.2,LUFA 6S A,Basanite,59,-3,275.7911060833985
5.2.B,5.2,LUFA 6S A,Basanite,85,-3,0.0
5.2.B,5.2,LUFA 6S A,Basanite,114,-3,1034.8302121667969
5.2.B,5.2,LUFA 6S A,Basanite,149,-3,0.0
5.2.B,5.2,LUFA 6S A,Basanite,178,-3,2040.498077140622
5.2.B,5.2,LUFA 6S A,Basanite,198,-3,808.55734857693
5.2.B,5.2,LUFA 6S A,Basanite,227,-3,678.5298545038811
5.2.B,5.2,LUFA 6S A,Basanite,255,-3,362.123202358746
5.2.B,5.2,LUFA 6S A,Basanite,281,-3,493.06502749864615
5.2.B,5.2,LUFA 6S A,Basanite,309,-3,625.209919730429
5.2.B,5.2,LUFA 6S A,Basanite,350,-3,667.0285328840483
5.2.B,5.2,LUFA 6S A,Basanite,350,-3,357.9605901678801
5.2.B,5.2,LUFA 6S A,Basanite,382,-3,436.1358914495457
5.2.B,5.2,LUFA 6S A,Basanite,410,-3,543.1126197725495
5.2.B,5.2,LUFA 6S A,Basanite,449,-3,680.6953751729948
5.2.B,5.2,LUFA 6S A,Basanite,471,-3,357.74403826945064
5.2.B,5.2,LUFA 6S A,Basanite,501,-3,687.1919376617125
5.2.B,5.2,LUFA 6S A,Basanite,529,-3,268.3898371743186
5.2.B,5.2,LUFA 6S A,Basanite,556,-3,637.6015110765402
5.2.B,5.2,LUFA 6S A,Basanite,596,-3,779.5586186894519
5.2.B,5.2,LUFA 6S A,Basanite,626,-3,302.06127967974385
5.2.B,5.2,LUFA 6S A,Basanite,652,-3,572.5444536975751
5.2.B,5.2,LUFA 6S A,Basanite,682,-3,359.13959612491726
5.2.B,5.2,LUFA 6S A,Basanite,742,-3,
5.2.B,5.2,LUFA 6S A,Basanite,771,-3,440.2696301823214
5.2.B,5.2,LUFA 6S A,Basanite,798,-3,315.5645049641976
5.2.B,5.2,LUFA 6S A,Basanite,833,-3,357.72960129971716
5.2.B,5.2,LUFA 6S A,Basanite,868,-3,725.3628511944161
5.2.B,5.2,LUFA 6S A,Basanite,899,-3,640.849792406282
5.2.B,5.2,LUFA 6S A,Basanite,926,-3,369.1154286058126
5.2.B,5.2,LUFA 6S A,Basanite,955,-3,493.5221931524159
5.2.B,5.2,LUFA 6S A,Basanite,987,-3,392.045887718876
5.2.B,5.2,LUFA 6S A,Basanite,1016,-3,
5.2.B,5.2,LUFA 6S A,Basanite,1043,-3,
5.2.B,5.2,LUFA 6S A,Basanite,1079,-3,
5.2.C,5.2,LUFA 6S A,Basanite,32,-3,738.923823816114
5.2.C,5.2,LUFA 6S A,Basanite,59,-3,325.5499620915819
5.2.C,5.2,LUFA 6S A,Basanite,85,-3,0.0
5.2.C,5.2,LUFA 6S A,Basanite,114,-3,898.3542783560985
5.2.C,5.2,LUFA 6S A,Basanite,149,-3,0.0
5.2.C,5.2,LUFA 6S A,Basanite,178,-3,1858.0168568505928
5.2.C,5.2,LUFA 6S A,Basanite,198,-3,753.6012426740477
5.2.C,5.2,LUFA 6S A,Basanite,226,-3,335.655725855948
5.2.C,5.2,LUFA 6S A,Basanite,255,-3,258.7075530416992
5.2.C,5.2,LUFA 6S A,Basanite,281,-3,417.7049033034478
5.2.C,5.2,LUFA 6S A,Basanite,309,-3,458.1279583609122
5.2.C,5.2,LUFA 6S A,Basanite,350,-3,686.0369933208977
5.2.C,5.2,LUFA 6S A,Basanite,382,-3,406.5404404597148
5.2.C,5.2,LUFA 6S A,Basanite,410,-3,380.3616999819484
5.2.C,5.2,LUFA 6S A,Basanite,449,-3,610.4362556110476
5.2.C,5.2,LUFA 6S A,Basanite,471,-3,363.9037417413803
5.2.C,5.2,LUFA 6S A,Basanite,501,-3,548.1173789036645
5.2.C,5.2,LUFA 6S A,Basanite,529,-3,231.07550745532225
5.2.C,5.2,LUFA 6S A,Basanite,556,-3,513.2356508281716
5.2.C,5.2,LUFA 6S A,Basanite,596,-3,546.2887167699621
5.2.C,5.2,LUFA 6S A,Basanite,626,-3,335.39586334299213
5.2.C,5.2,LUFA 6S A,Basanite,652,-3,436.1647651483242
5.2.C,5.2,LUFA 6S A,Basanite,682,-3,416.89644214453335
5.2.D,5.2,LUFA 6S A,Basanite,32,-3,858.9899221373126
5.2.D,5.2,LUFA 6S A,Basanite,58,-3,365.97301714904626
5.2.D,5.2,LUFA 6S A,Basanite,85,-3,0.0
5.2.D,5.2,LUFA 6S A,Basanite,114,-3,1083.1935103195135
5.2.D,5.2,LUFA 6S A,Basanite,149,-3,0.0
5.2.D,5.2,LUFA 6S A,Basanite,178,-3,1920.095120043324
5.2.D,5.2,LUFA 6S A,Basanite,198,-3,800.8577189963296
5.2.D,5.2,LUFA 6S A,Basanite,226,-3,420.1110374872134
5.2.D,5.2,LUFA 6S A,Basanite,255,-3,254.66524748781512
5.2.D,5.2,LUFA 6S A,Basanite,281,-3,638.6842714964799
5.2.D,5.2,LUFA 6S A,Basanite,309,-3,503.9407543173476
5.2.D,5.2,LUFA 6S A,Basanite,350,-3,728.9143053131958
5.2.D,5.2,LUFA 6S A,Basanite,382,-3,461.9777731512125
5.2.D,5.2,LUFA 6S A,Basanite,410,-3,455.2405973885312
5.2.D,5.2,LUFA 6S A,Basanite,449,-3,589.502887779048
5.2.D,5.2,LUFA 6S A,Basanite,471,-3,311.7868743004994
5.2.D,5.2,LUFA 6S A,Basanite,501,-3,524.7297541368313
5.2.D,5.2,LUFA 6S A,Basanite,529,-3,155.62876235633914
5.2.D,5.2,LUFA 6S A,Basanite,556,-3,636.2155778248645
5.2.D,5.2,LUFA 6S A,Basanite,596,-3,705.2235077922859
5.2.D,5.2,LUFA 6S A,Basanite,626,-3,316.967281247304
5.2.D,5.2,LUFA 6S A,Basanite,652,-3,397.4163793248691
5.2.D,5.2,LUFA 6S A,Basanite,682,-3,461.8622788374752
5.6.A,5.6,LUFA 6S A,Metabasalt,35,-3,795.6123463505627
5.6.A,5.6,LUFA 6S A,Metabasalt,59,-3,376.07878115410074
5.6.A,5.6,LUFA 6S A,Metabasalt,85,-3,0.0
5.6.A,5.6,LUFA 6S A,Metabasalt,114,-3,354.18295950418195
5.6.A,5.6,LUFA 6S A,Metabasalt,149,-3,0.0
5.6.A,5.6,LUFA 6S A,Metabasalt,178,-3,1585.1612344906432
5.6.A,5.6,LUFA 6S A,Metabasalt,198,-3,756.9698304350442
5.6.A,5.6,LUFA 6S A,Metabasalt,227,-3,566.5002445393826
5.6.A,5.6,LUFA 6S A,Metabasalt,255,-3,337.82124676575006
5.6.A,5.6,LUFA 6S A,Metabasalt,281,-3,644.8439752090981
5.6.A,5.6,LUFA 6S A,Metabasalt,309,-3,571.3125129069138
5.6.A,5.6,LUFA 6S A,Metabasalt,350,-3,838.4896583428606
5.6.A,5.6,LUFA 6S A,Metabasalt,350,-3,346.3582109633552
5.6.A,5.6,LUFA 6S A,Metabasalt,382,-3,469.19617594319755
5.6.A,5.6,LUFA 6S A,Metabasalt,410,-3,486.2797289848968
5.6.A,5.6,LUFA 6S A,Metabasalt,449,-3,649.6562435766292
5.6.A,5.6,LUFA 6S A,Metabasalt,471,-3,389.0719058908478
5.6.A,5.6,LUFA 6S A,Metabasalt,501,-3,603.4584663337145
5.6.A,5.6,LUFA 6S A,Metabasalt,529,-3,259.16953077802515
5.6.A,5.6,LUFA 6S A,Metabasalt,556,-3,727.8989166882748
5.6.A,5.6,LUFA 6S A,Metabasalt,596,-3,895.9962666827125
5.6.A,5.6,LUFA 6S A,Metabasalt,626,-3,484.4630976259622
5.6.A,5.6,LUFA 6S A,Metabasalt,652,-3,439.56222660809914
5.6.A,5.6,LUFA 6S A,Metabasalt,682,-3,422.6952257055177
5.6.A,5.6,LUFA 6S A,Metabasalt,742,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,771,-3,521.7317109332691
5.6.A,5.6,LUFA 6S A,Metabasalt,798,-3,479.792791142668
5.6.A,5.6,LUFA 6S A,Metabasalt,833,-3,553.8969134123594
5.6.A,5.6,LUFA 6S A,Metabasalt,868,-3,753.6012426740477
5.6.A,5.6,LUFA 6S A,Metabasalt,899,-3,510.24001371923697
5.6.A,5.6,LUFA 6S A,Metabasalt,926,-3,223.135264480414
5.6.A,5.6,LUFA 6S A,Metabasalt,955,-3,490.83213502617485
5.6.A,5.6,LUFA 6S A,Metabasalt,987,-3,524.2292782959264
5.6.A,5.6,LUFA 6S A,Metabasalt,1016,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,1043,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,1079,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,1107,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,1142,-3,
5.6.A,5.6,LUFA 6S A,Metabasalt,1171,-3,
5.6.B,5.6,LUFA 6S A,Metabasalt,35,-3,
5.6.B,5.6,LUFA 6S A,Metabasalt,59,-3,397.4933757747157
5.6.B,5.6,LUFA 6S A,Metabasalt,85,-3,0.0
5.6.B,5.6,LUFA 6S A,Metabasalt,114,-3,858.9899221373126
5.6.B,5.6,LUFA 6S A,Metabasalt,149,-3,0.0
5.6.B,5.6,LUFA 6S A,Metabasalt,178,-3,1289.2067233888922
5.6.B,5.6,LUFA 6S A,Metabasalt,198,-3,703.361159877249
5.6.B,5.6,LUFA 6S A,Metabasalt,227,-3,514.5277448703291
5.6.B,5.6,LUFA 6S A,Metabasalt,255,-3,296.8207194175341
5.6.B,5.6,LUFA 6S A,Metabasalt,281,-3,674.487548949997
5.6.B,5.6,LUFA 6S A,Metabasalt,309,-3,595.5663459895301
5.6.B,5.6,LUFA 6S A,Metabasalt,350,-3,726.0750668511944
5.6.B,5.6,LUFA 6S A,Metabasalt,382,-3,437.91643083218
5.6.B,5.6,LUFA 6S A,Metabasalt,410,-3,434.0666160418798
5.6.B,5.6,LUFA 6S A,Metabasalt,449,-3,649.6562435766292
5.6.B,5.6,LUFA 6S A,Metabasalt,471,-3,420.2072829893495
5.6.B,5.6,LUFA 6S A,Metabasalt,501,-3,662.3606322883446
5.6.B,5.6,LUFA 6S A,Metabasalt,529,-3,321.2285449184668
5.6.B,5.6,LUFA 6S A,Metabasalt,556,-3,742.8458224828995
5.6.B,5.6,LUFA 6S A,Metabasalt,596,-3,742.9180066189301
5.6.B,5.6,LUFA 6S A,Metabasalt,626,-3,381.2952801214706
5.6.B,5.6,LUFA 6S A,Metabasalt,652,-3,331.16106721222695
5.6.B,5.6,LUFA 6S A,Metabasalt,682,-3,364.7795747036524
5.6.C,5.6,LUFA 6S A,Metabasalt,32,-3,943.2046204946146
5.6.C,5.6,LUFA 6S A,Metabasalt,59,-3,471.6023101269631
5.6.C,5.6,LUFA 6S A,Metabasalt,85,-3,0.0
5.6.C,5.6,LUFA 6S A,Metabasalt,114,-3,0.0
5.6.C,5.6,LUFA 6S A,Metabasalt,149,-3,0.0
5.6.C,5.6,LUFA 6S A,Metabasalt,178,-3,1740.8862422528432
5.6.C,5.6,LUFA 6S A,Metabasalt,198,-3,688.6356182682472
5.6.C,5.6,LUFA 6S A,Metabasalt,226,-3,452.7382177026294
5.6.C,5.6,LUFA 6S A,Metabasalt,255,-3,567.943924905229
5.6.C,5.6,LUFA 6S A,Metabasalt,281,-3,544.5563001383957
5.6.C,5.6,LUFA 6S A,Metabasalt,309,-3,628.0010354413623
5.6.C,5.6,LUFA 6S A,Metabasalt,350,-3,757.3548119622119
5.6.C,5.6,LUFA 6S A,Metabasalt,382,-3,525.2109809254467
5.6.C,5.6,LUFA 6S A,Metabasalt,410,-3,412.8926349359168
5.6.C,5.6,LUFA 6S A,Metabasalt,449,-3,667.2210236476323
5.6.C,5.6,LUFA 6S A,Metabasalt,471,-3,427.3294402791985
5.6.C,5.6,LUFA 6S A,Metabasalt,501,-3,437.3389587821168
5.6.C,5.6,LUFA 6S A,Metabasalt,529,-3,308.8706396293399
5.6.C,5.6,LUFA 6S A,Metabasalt,556,-3,759.4890531170912
5.6.C,5.6,LUFA 6S A,Metabasalt,596,-3,673.929325711535
5.6.C,5.6,LUFA 6S A,Metabasalt,626,-3,295.2807933986481
5.6.C,5.6,LUFA 6S A,Metabasalt,652,-3,624.2907765810216
5.6.C,5.6,LUFA 6S A,Metabasalt,682,-3,388.8409170226848
5.6.D,5.6,LUFA 6S A,Metabasalt,32,-3,869.6731581924304
5.6.D,5.6,LUFA 6S A,Metabasalt,58,-3,375.3569408508334
5.6.D,5.6,LUFA 6S A,Metabasalt,85,-3,0.0
5.6.D,5.6,LUFA 6S A,Metabasalt,114,-3,0.0
5.6.D,5.6,LUFA 6S A,Metabasalt,149,-3,0.0
5.6.D,5.6,LUFA 6S A,Metabasalt,178,-3,1550.9941284072447
5.6.D,5.6,LUFA 6S A,Metabasalt,198,-3,318.7646635778326
5.6.D,5.6,LUFA 6S A,Metabasalt,226,-3,651.8217644864311
5.6.D,5.6,LUFA 6S A,Metabasalt,255,-3,360.3426632168
5.6.D,5.6,LUFA 6S A,Metabasalt,281,-3,522.0348836873458
5.6.D,5.6,LUFA 6S A,Metabasalt,309,-3,538.9740687165292
5.6.D,5.6,LUFA 6S A,Metabasalt,350,-3,711.7345070100488
5.6.D,5.6,LUFA 6S A,Metabasalt,350,-3,522.0060102292557
5.6.D,5.6,LUFA 6S A,Metabasalt,382,-3,522.997337505265
5.6.D,5.6,LUFA 6S A,Metabasalt,410,-3,323.3844411817799
5.6.D,5.6,LUFA 6S A,Metabasalt,449,-3,597.683744148264
5.6.D,5.6,LUFA 6S A,Metabasalt,471,-3,331.90215656778383
5.6.D,5.6,LUFA 6S A,Metabasalt,501,-3,558.993105722366
5.6.D,5.6,LUFA 6S A,Metabasalt,529,-3,186.75932710752753
5.6.D,5.6,LUFA 6S A,Metabasalt,556,-3,609.8876569911724
5.6.D,5.6,LUFA 6S A,Metabasalt,596,-3,700.8780292436368
5.6.D,5.6,LUFA 6S A,Metabasalt,626,-3,393.6194995958379
5.6.D,5.6,LUFA 6S A,Metabasalt,652,-3,602.8809940429629
5.6.D,5.6,LUFA 6S A,Metabasalt,682,-3,558.3578862747457
5.6.D,5.6,LUFA 6S A,Metabasalt,742,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,771,-3,395.8572044045971
5.6.D,5.6,LUFA 6S A,Metabasalt,798,-3,323.6828018533004
5.6.D,5.6,LUFA 6S A,Metabasalt,833,-3,374.1490614357061
5.6.D,5.6,LUFA 6S A,Metabasalt,868,-3,647.6158418677418
5.6.D,5.6,LUFA 6S A,Metabasalt,899,-3,525.7307060593297
5.6.D,5.6,LUFA 6S A,Metabasalt,926,-3,235.9214618208075
5.6.D,5.6,LUFA 6S A,Metabasalt,955,-3,435.6739137132198
5.6.D,5.6,LUFA 6S A,Metabasalt,987,-3,417.8588959624526
5.6.D,5.6,LUFA 6S A,Metabasalt,1016,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,1043,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,1079,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,1107,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,1142,-3,
5.6.D,5.6,LUFA 6S A,Metabasalt,1171,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,35,-3,721.8402707744148
5.7.A,5.7,LUFA 6S A,Peridotite,59,-3,359.0914733738492
5.7.A,5.7,LUFA 6S A,Peridotite,85,-3,0.0
5.7.A,5.7,LUFA 6S A,Peridotite,114,-3,1473.4203605511764
5.7.A,5.7,LUFA 6S A,Peridotite,149,-3,0.0
5.7.A,5.7,LUFA 6S A,Peridotite,178,-3,1858.883065166376
5.7.A,5.7,LUFA 6S A,Peridotite,198,-3,721.8402707744148
5.7.A,5.7,LUFA 6S A,Peridotite,227,-3,707.4034651904446
5.7.A,5.7,LUFA 6S A,Peridotite,255,-3,322.22949684096517
5.7.A,5.7,LUFA 6S A,Peridotite,281,-3,515.3939531861123
5.7.A,5.7,LUFA 6S A,Peridotite,309,-3,559.5705777724291
5.7.A,5.7,LUFA 6S A,Peridotite,350,-3,743.7842149347133
5.7.A,5.7,LUFA 6S A,Peridotite,350,-3,439.4659813466514
5.7.A,5.7,LUFA 6S A,Peridotite,382,-3,412.6520214212648
5.7.A,5.7,LUFA 6S A,Peridotite,410,-3,501.24588386786206
5.7.A,5.7,LUFA 6S A,Peridotite,449,-3,742.7736386064144
5.7.A,5.7,LUFA 6S A,Peridotite,471,-3,400.1882459835128
5.7.A,5.7,LUFA 6S A,Peridotite,501,-3,702.2062152957458
5.7.A,5.7,LUFA 6S A,Peridotite,529,-3,251.31590853841988
5.7.A,5.7,LUFA 6S A,Peridotite,556,-3,672.002012077374
5.7.A,5.7,LUFA 6S A,Peridotite,596,-3,666.7397966183283
5.7.A,5.7,LUFA 6S A,Peridotite,626,-3,362.6718010935983
5.7.A,5.7,LUFA 6S A,Peridotite,652,-3,409.8897794091101
5.7.A,5.7,LUFA 6S A,Peridotite,682,-3,464.05667320536736
5.7.A,5.7,LUFA 6S A,Peridotite,742,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,771,-3,435.1012537457128
5.7.A,5.7,LUFA 6S A,Peridotite,798,-3,427.3438770082436
5.7.A,5.7,LUFA 6S A,Peridotite,833,-3,558.3434493050123
5.7.A,5.7,LUFA 6S A,Peridotite,868,-3,771.474007822372
5.7.A,5.7,LUFA 6S A,Peridotite,899,-3,721.1328672001925
5.7.A,5.7,LUFA 6S A,Peridotite,926,-3,446.9057483603104
5.7.A,5.7,LUFA 6S A,Peridotite,955,-3,615.8308085925747
5.7.A,5.7,LUFA 6S A,Peridotite,987,-3,521.341917082857
5.7.A,5.7,LUFA 6S A,Peridotite,1016,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,1043,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,1079,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,1107,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,1142,-3,
5.7.A,5.7,LUFA 6S A,Peridotite,1171,-3,
5.7.B,5.7,LUFA 6S A,Peridotite,32,-3,773.0428071484446
5.7.B,5.7,LUFA 6S A,Peridotite,59,-3,411.6414450929659
5.7.B,5.7,LUFA 6S A,Peridotite,85,-3,0.0
5.7.B,5.7,LUFA 6S A,Peridotite,114,-3,0.0
5.7.B,5.7,LUFA 6S A,Peridotite,149,-3,0.0
5.7.B,5.7,LUFA 6S A,Peridotite,178,-3,1118.708051507311
5.7.B,5.7,LUFA 6S A,Peridotite,198,-3,273.14435838498105
5.7.B,5.7,LUFA 6S A,Peridotite,227,-3,632.0433409952464
5.7.B,5.7,LUFA 6S A,Peridotite,255,-3,212.94287986040075
5.7.B,5.7,LUFA 6S A,Peridotite,281,-3,554.0845918526987
5.7.B,5.7,LUFA 6S A,Peridotite,309,-3,664.2855396834948
5.7.B,5.7,LUFA 6S A,Peridotite,350,-3,680.2622711354473
5.7.B,5.7,LUFA 6S A,Peridotite,382,-3,481.22684710271375
5.7.B,5.7,LUFA 6S A,Peridotite,410,-3,420.0629149768337
5.7.B,5.7,LUFA 6S A,Peridotite,449,-3,487.9640231060834
5.7.B,5.7,LUFA 6S A,Peridotite,471,-3,415.1062783560984
5.7.B,5.7,LUFA 6S A,Peridotite,501,-3,606.3458273060954
5.7.B,5.7,LUFA 6S A,Peridotite,529,-3,388.46556014200615
5.7.B,5.7,LUFA 6S A,Peridotite,556,-3,627.871104351699
5.7.B,5.7,LUFA 6S A,Peridotite,596,-3,781.2429128106384
5.7.B,5.7,LUFA 6S A,Peridotite,626,-3,297.23457445953835
5.7.B,5.7,LUFA 6S A,Peridotite,652,-3,335.77122041037364
5.7.B,5.7,LUFA 6S A,Peridotite,682,-3,678.2699919369396
5.7.C,5.7,LUFA 6S A,Peridotite,32,-3,605.5758644924483
5.7.C,5.7,LUFA 6S A,Peridotite,59,-3,349.0819548709309
5.7.C,5.7,LUFA 6S A,Peridotite,85,-3,0.0
5.7.C,5.7,LUFA 6S A,Peridotite,114,-3,1238.677904567062
5.7.C,5.7,LUFA 6S A,Peridotite,149,-3,0.0
5.7.C,5.7,LUFA 6S A,Peridotite,178,-3,2106.7148913893734
5.7.C,5.7,LUFA 6S A,Peridotite,198,-3,792.2918810999458
5.7.C,5.7,LUFA 6S A,Peridotite,226,-3,683.823349900716
5.7.C,5.7,LUFA 6S A,Peridotite,255,-3,362.8450426620133
5.7.C,5.7,LUFA 6S A,Peridotite,281,-3,413.8550885131475
5.7.C,5.7,LUFA 6S A,Peridotite,309,-3,488.3971271436308
5.7.C,5.7,LUFA 6S A,Peridotite,350,-3,630.5996603887118
5.7.C,5.7,LUFA 6S A,Peridotite,350,-3,467.2375827667128
5.7.C,5.7,LUFA 6S A,Peridotite,382,-3,452.3532364161502
5.7.C,5.7,LUFA 6S A,Peridotite,410,-3,497.5885598411456
5.7.C,5.7,LUFA 6S A,Peridotite,449,-3,649.6562435766292
5.7.C,5.7,LUFA 6S A,Peridotite,471,-3,355.0491678199651
5.7.C,5.7,LUFA 6S A,Peridotite,501,-3,695.2765487694808
5.7.C,5.7,LUFA 6S A,Peridotite,529,-3,296.5897305493712
5.7.C,5.7,LUFA 6S A,Peridotite,556,-3,610.9872604897853
5.7.C,5.7,LUFA 6S A,Peridotite,596,-3,613.2081222696913
5.7.C,5.7,LUFA 6S A,Peridotite,626,-3,356.68533906498334
5.7.C,5.7,LUFA 6S A,Peridotite,652,-3,426.2611165533426
5.7.C,5.7,LUFA 6S A,Peridotite,682,-3,489.9129916360792
5.7.C,5.7,LUFA 6S A,Peridotite,742,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,771,-3,342.4506490161863
5.7.C,5.7,LUFA 6S A,Peridotite,798,-3,259.43420566821106
5.7.C,5.7,LUFA 6S A,Peridotite,833,-3,211.00834793910585
5.7.C,5.7,LUFA 6S A,Peridotite,868,-3,447.704584872736
5.7.C,5.7,LUFA 6S A,Peridotite,899,-3,338.58639749684096
5.7.C,5.7,LUFA 6S A,Peridotite,926,-3,151.0667318611228
5.7.C,5.7,LUFA 6S A,Peridotite,955,-3,268.00485564715086
5.7.C,5.7,LUFA 6S A,Peridotite,987,-3,289.6793128347072
5.7.C,5.7,LUFA 6S A,Peridotite,1016,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,1043,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,1079,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,1107,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,1142,-3,
5.7.C,5.7,LUFA 6S A,Peridotite,1171,-3,
5.7.D,5.7,LUFA 6S A,Peridotite,32,-3,713.178187375895
5.7.D,5.7,LUFA 6S A,Peridotite,58,-3,384.0190240086648
5.7.D,5.7,LUFA 6S A,Peridotite,85,-3,0.0
5.7.D,5.7,LUFA 6S A,Peridotite,114,-3,1639.0586413141584
5.7.D,5.7,LUFA 6S A,Peridotite,149,-3,0.0
5.7.D,5.7,LUFA 6S A,Peridotite,178,-3,1948.0062771526568
5.7.D,5.7,LUFA 6S A,Peridotite,198,-3,844.8418528190624
5.7.D,5.7,LUFA 6S A,Peridotite,226,-3,596.4806770563812
5.7.D,5.7,LUFA 6S A,Peridotite,255,-3,303.1729136530477
5.7.D,5.7,LUFA 6S A,Peridotite,281,-3,649.7524890787653
5.7.D,5.7,LUFA 6S A,Peridotite,309,-3,630.8883966544316
5.7.D,5.7,LUFA 6S A,Peridotite,350,-3,488.54149515614654
5.7.D,5.7,LUFA 6S A,Peridotite,382,-3,495.47116192310006
5.7.D,5.7,LUFA 6S A,Peridotite,410,-3,372.469579637764
5.7.D,5.7,LUFA 6S A,Peridotite,449,-3,623.6699938624465
5.7.D,5.7,LUFA 6S A,Peridotite,471,-3,305.3384345628497
5.7.D,5.7,LUFA 6S A,Peridotite,501,-3,606.3458273060954
5.7.D,5.7,LUFA 6S A,Peridotite,529,-3,245.67111763644024
5.7.D,5.7,LUFA 6S A,Peridotite,556,-3,845.9005517592386
5.7.D,5.7,LUFA 6S A,Peridotite,596,-3,854.7021909862207
5.7.D,5.7,LUFA 6S A,Peridotite,626,-3,371.7573639661898
5.7.D,5.7,LUFA 6S A,Peridotite,652,-3,156.23029592634936
5.7.D,5.7,LUFA 6S A,Peridotite,682,-3,110.20094799927791
5.8.A,5.8,LUFA 6S A,Steel Slag,35,-3,837.3347140020459
5.8.A,5.8,LUFA 6S A,Steel Slag,59,-3,320.6895709729827
5.8.A,5.8,LUFA 6S A,Steel Slag,85,-3,0.0
5.8.A,5.8,LUFA 6S A,Steel Slag,114,-3,411.1602183043504
5.8.A,5.8,LUFA 6S A,Steel Slag,149,-3,0.0
5.8.A,5.8,LUFA 6S A,Steel Slag,178,-3,1917.4964950959743
5.8.A,5.8,LUFA 6S A,Steel Slag,198,-3,999.4119162404476
5.8.A,5.8,LUFA 6S A,Steel Slag,227,-3,947.2469258078104
5.8.A,5.8,LUFA 6S A,Steel Slag,255,-3,381.1316630362838
5.8.A,5.8,LUFA 6S A,Steel Slag,281,-3,1108.7466557554606
5.8.A,5.8,LUFA 6S A,Steel Slag,309,-3,1074.0983228834466
5.8.A,5.8,LUFA 6S A,Steel Slag,350,-3,1122.750357061195
5.8.A,5.8,LUFA 6S A,Steel Slag,350,-3,572.7946916180275
5.8.A,5.8,LUFA 6S A,Steel Slag,382,-3,846.7667602142127
5.8.A,5.8,LUFA 6S A,Steel Slag,410,-3,629.0597345207293
5.8.A,5.8,LUFA 6S A,Steel Slag,449,-3,1059.1802905108611
5.8.A,5.8,LUFA 6S A,Steel Slag,471,-3,551.774702930381
5.8.A,5.8,LUFA 6S A,Steel Slag,501,-3,751.8688260424815
5.8.A,5.8,LUFA 6S A,Steel Slag,529,-3,448.5996668872976
5.8.A,5.8,LUFA 6S A,Steel Slag,556,-3,1005.5162786502924
5.8.A,5.8,LUFA 6S A,Steel Slag,596,-3,1114.1171473614536
5.8.A,5.8,LUFA 6S A,Steel Slag,626,-3,510.81267373537304
5.8.A,5.8,LUFA 6S A,Steel Slag,652,-3,564.7678279078164
5.8.A,5.8,LUFA 6S A,Steel Slag,682,-3,630.3831084902822
5.8.A,5.8,LUFA 6S A,Steel Slag,742,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,771,-3,460.4378472832301
5.8.A,5.8,LUFA 6S A,Steel Slag,798,-3,407.0746023226427
5.8.A,5.8,LUFA 6S A,Steel Slag,833,-3,498.38258427101505
5.8.A,5.8,LUFA 6S A,Steel Slag,868,-3,802.9173699981948
5.8.A,5.8,LUFA 6S A,Steel Slag,899,-3,667.1969622720982
5.8.A,5.8,LUFA 6S A,Steel Slag,926,-3,434.4130994644684
5.8.A,5.8,LUFA 6S A,Steel Slag,955,-3,690.9551315963656
5.8.A,5.8,LUFA 6S A,Steel Slag,987,-3,499.0947999277934
5.8.A,5.8,LUFA 6S A,Steel Slag,1016,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,1043,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,1079,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,1107,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,1142,-3,
5.8.A,5.8,LUFA 6S A,Steel Slag,1171,-3,
5.8.B,5.8,LUFA 6S A,Steel Slag,35,-3,683.9195951621638
5.8.B,5.8,LUFA 6S A,Steel Slag,58,-3,166.023262266081
5.8.B,5.8,LUFA 6S A,Steel Slag,85,-3,0.0
5.8.B,5.8,LUFA 6S A,Steel Slag,114,-3,328.5816913171671
5.8.B,5.8,LUFA 6S A,Steel Slag,149,-3,0.0
5.8.B,5.8,LUFA 6S A,Steel Slag,178,-3,2056.8597901197422
5.8.B,5.8,LUFA 6S A,Steel Slag,198,-3,907.9788150911606
5.8.B,5.8,LUFA 6S A,Steel Slag,227,-3,813.0327581683614
5.8.B,5.8,LUFA 6S A,Steel Slag,255,-3,396.5309219567965
5.8.B,5.8,LUFA 6S A,Steel Slag,281,-3,1149.8915513568809
5.8.B,5.8,LUFA 6S A,Steel Slag,309,-3,976.8904997893976
5.8.B,5.8,LUFA 6S A,Steel Slag,350,-3,1261.7767933088633
5.8.B,5.8,LUFA 6S A,Steel Slag,382,-3,677.3749099223779
5.8.B,5.8,LUFA 6S A,Steel Slag,410,-3,569.0507467356639
5.8.B,5.8,LUFA 6S A,Steel Slag,449,-3,943.2046204946146
5.8.B,5.8,LUFA 6S A,Steel Slag,471,-3,556.2982352728804
5.8.B,5.8,LUFA 6S A,Steel Slag,501,-3,857.9793458090137
5.8.B,5.8,LUFA 6S A,Steel Slag,529,-3,200.97958043203565
5.8.B,5.8,LUFA 6S A,Steel Slag,556,-3,747.0758065177575
5.8.B,5.8,LUFA 6S A,Steel Slag,596,-3,933.1421669173836
5.8.B,5.8,LUFA 6S A,Steel Slag,626,-3,487.5237004231073
5.8.B,5.8,LUFA 6S A,Steel Slag,652,-3,576.4135175401649
5.8.B,5.8,LUFA 6S A,Steel Slag,682,-3,610.893421024129
5.8.C,5.8,LUFA 6S A,Steel Slag,32,-3,826.6514779469281
5.8.C,5.8,LUFA 6S A,Steel Slag,58,-3,558.6562469462663
5.8.C,5.8,LUFA 6S A,Steel Slag,85,-3,0.0
5.8.C,5.8,LUFA 6S A,Steel Slag,114,-3,52.13130436247667
5.8.C,5.8,LUFA 6S A,Steel Slag,149,-3,0.0
5.8.C,5.8,LUFA 6S A,Steel Slag,178,-3,2121.6329237619593
5.8.C,5.8,LUFA 6S A,Steel Slag,198,-3,1163.6065163968951
5.8.C,5.8,LUFA 6S A,Steel Slag,226,-3,841.6657555809614
5.8.C,5.8,LUFA 6S A,Steel Slag,255,-3,480.74562031409835
5.8.C,5.8,LUFA 6S A,Steel Slag,281,-3,1171.2099004753595
5.8.C,5.8,LUFA 6S A,Steel Slag,309,-3,936.0343404536976
5.8.C,5.8,LUFA 6S A,Steel Slag,350,-3,1061.1051979060112
5.8.C,5.8,LUFA 6S A,Steel Slag,350,-3,529.7585746434803
5.8.C,5.8,LUFA 6S A,Steel Slag,382,-3,874.4854265599615
5.8.C,5.8,LUFA 6S A,Steel Slag,410,-3,843.1094361874963
5.8.C,5.8,LUFA 6S A,Steel Slag,449,-3,1143.394988868163
5.8.C,5.8,LUFA 6S A,Steel Slag,471,-3,520.3024670557795
5.8.C,5.8,LUFA 6S A,Steel Slag,501,-3,963.5123932847944
5.8.C,5.8,LUFA 6S A,Steel Slag,529,-3,676.7974378723148
5.8.C,5.8,LUFA 6S A,Steel Slag,556,-3,1104.0186020983645
5.8.C,5.8,LUFA 6S A,Steel Slag,596,-3,1035.840788495096
5.8.C,5.8,LUFA 6S A,Steel Slag,626,-3,557.924782027791
5.8.C,5.8,LUFA 6S A,Steel Slag,652,-3,597.8281121607798
5.8.C,5.8,LUFA 6S A,Steel Slag,682,-3,501.1785123051928
5.8.C,5.8,LUFA 6S A,Steel Slag,742,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,771,-3,483.77253721643905
5.8.C,5.8,LUFA 6S A,Steel Slag,798,-3,534.5852799807449
5.8.C,5.8,LUFA 6S A,Steel Slag,833,-3,444.1868167759793
5.8.C,5.8,LUFA 6S A,Steel Slag,868,-3,876.4343953306457
5.8.C,5.8,LUFA 6S A,Steel Slag,899,-3,736.219328960828
5.8.C,5.8,LUFA 6S A,Steel Slag,926,-3,552.2366806667068
5.8.C,5.8,LUFA 6S A,Steel Slag,955,-3,502.26608484265
5.8.C,5.8,LUFA 6S A,Steel Slag,987,-3,365.4773536313857
5.8.C,5.8,LUFA 6S A,Steel Slag,1016,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,1043,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,1079,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,1107,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,1142,-3,
5.8.C,5.8,LUFA 6S A,Steel Slag,1171,-3,
5.8.D,5.8,LUFA 6S A,Steel Slag,32,-3,1054.6086354172933
5.8.D,5.8,LUFA 6S A,Steel Slag,58,-3,612.1205494915457
5.8.D,5.8,LUFA 6S A,Steel Slag,85,-3,0.0
5.8.D,5.8,LUFA 6S A,Steel Slag,114,-3,54.73955386003971
5.8.D,5.8,LUFA 6S A,Steel Slag,149,-3,0.0
5.8.D,5.8,LUFA 6S A,Steel Slag,178,-3,1978.227323184307
5.8.D,5.8,LUFA 6S A,Steel Slag,198,-3,1042.818577772429
5.8.D,5.8,LUFA 6S A,Steel Slag,226,-3,962.7905529815272
5.8.D,5.8,LUFA 6S A,Steel Slag,255,-3,487.9640231060834
5.8.D,5.8,LUFA 6S A,Steel Slag,281,-3,1154.9444332390638
5.8.D,5.8,LUFA 6S A,Steel Slag,309,-3,971.3082683675312
5.8.D,5.8,LUFA 6S A,Steel Slag,350,-3,1246.3294113965942
5.8.D,5.8,LUFA 6S A,Steel Slag,382,-3,830.020065948613
5.8.D,5.8,LUFA 6S A,Steel Slag,410,-3,675.8831068054636
5.8.D,5.8,LUFA 6S A,Steel Slag,449,-3,996.1395735002104
5.8.D,5.8,LUFA 6S A,Steel Slag,471,-3,518.1369463866658
5.8.D,5.8,LUFA 6S A,Steel Slag,501,-3,1020.9708788735784
5.8.D,5.8,LUFA 6S A,Steel Slag,529,-3,666.9419120284012
5.8.D,5.8,LUFA 6S A,Steel Slag,556,-3,1024.3274361904223
5.8.D,5.8,LUFA 6S A,Steel Slag,596,-3,1017.7803449064324
5.8.D,5.8,LUFA 6S A,Steel Slag,626,-3,500.9186497384856
5.8.D,5.8,LUFA 6S A,Steel Slag,652,-3,567.5733803477947
5.8.D,5.8,LUFA 6S A,Steel Slag,682,-3,539.2339312834707
6.0.A,6.0,LUFA 2.2 A,Control,32,-3,63.81067992057284
6.0.A,6.0,LUFA 2.2 A,Control,59,-3,206.1575813225826
6.0.A,6.0,LUFA 2.2 A,Control,85,-3,153.89634572477286
6.0.A,6.0,LUFA 2.2 A,Control,114,-3,159.38233176484746
6.0.A,6.0,LUFA 2.2 A,Control,149,-3,97.44843653649436
6.0.A,6.0,LUFA 2.2 A,Control,178,-3,141.67318380167276
6.0.A,6.0,LUFA 2.2 A,Control,203,-3,24.25383310668512
6.0.A,6.0,LUFA 2.2 A,Control,227,-3,1.9249073886515435
6.0.A,6.0,LUFA 2.2 A,Control,254,-3,31.03913164450328
6.0.A,6.0,LUFA 2.2 A,Control,281,-3,63.32945308381973
6.0.A,6.0,LUFA 2.2 A,Control,309,-3,127.04388764666948
6.0.A,6.0,LUFA 2.2 A,Control,351,-3,85.03278389794812
6.0.A,6.0,LUFA 2.2 A,Control,381,-3,40.423055153739696
6.0.A,6.0,LUFA 2.2 A,Control,413,-3,0.0
6.0.A,6.0,LUFA 2.2 A,Control,449,-3,145.5229985679042
6.0.A,6.0,LUFA 2.2 A,Control,471,-3,49.75885598411457
6.0.A,6.0,LUFA 2.2 A,Control,501,-3,135.1284986822312
6.0.A,6.0,LUFA 2.2 A,Control,528,-3,48.04568842890667
6.0.A,6.0,LUFA 2.2 A,Control,563,-3,153.77603899151572
6.0.A,6.0,LUFA 2.2 A,Control,598,-3,77.51120827967988
6.0.A,6.0,LUFA 2.2 A,Control,626,-3,44.5038588122029
6.0.A,6.0,LUFA 2.2 A,Control,653,-3,48.45954351043985
6.0.A,6.0,LUFA 2.2 A,Control,680,-3,61.24092857572657
6.0.B,6.0,LUFA 2.2 A,Control,28,-3,62.36699938624466
6.0.B,6.0,LUFA 2.2 A,Control,59,-3,145.5229985679042
6.0.B,6.0,LUFA 2.2 A,Control,85,-3,71.60655485889644
6.0.B,6.0,LUFA 2.2 A,Control,114,-3,175.98465799386244
6.0.B,6.0,LUFA 2.2 A,Control,149,-3,104.04124435886636
6.0.B,6.0,LUFA 2.2 A,Control,178,-3,118.57429513207776
6.0.B,6.0,LUFA 2.2 A,Control,203,-3,71.94341365906492
6.0.B,6.0,LUFA 2.2 A,Control,227,-3,12.415652655394428
6.0.B,6.0,LUFA 2.2 A,Control,254,-3,29.258592310006616
6.0.B,6.0,LUFA 2.2 A,Control,281,-3,127.04388764666948
6.0.B,6.0,LUFA 2.2 A,Control,309,-3,164.6758270894759
6.0.B,6.0,LUFA 2.2 A,Control,351,-3,303.4135271676996
6.0.B,6.0,LUFA 2.2 A,Control,381,-3,53.897406871652926
6.0.B,6.0,LUFA 2.2 A,Control,413,-3,0.0
6.0.B,6.0,LUFA 2.2 A,Control,449,-3,182.86620191347257
6.0.B,6.0,LUFA 2.2 A,Control,471,-3,63.13696234430471
6.0.B,6.0,LUFA 2.2 A,Control,501,-3,118.23743633190928
6.0.B,6.0,LUFA 2.2 A,Control,528,-3,47.81469953667489
6.0.B,6.0,LUFA 2.2 A,Control,563,-3,86.7074533245081
6.0.B,6.0,LUFA 2.2 A,Control,598,-3,157.64510285817437
6.0.B,6.0,LUFA 2.2 A,Control,626,-3,140.38349585414284
6.0.B,6.0,LUFA 2.2 A,Control,653,-3,129.69063529694927
6.0.B,6.0,LUFA 2.2 A,Control,680,-3,129.20940846019616
6.0.B,6.0,LUFA 2.2 A,Control,716,-3,135.03225330043924
6.0.B,6.0,LUFA 2.2 A,Control,742,-3,24.94679975931163
6.0.B,6.0,LUFA 2.2 A,Control,771,-3,74.63828398820627
6.0.B,6.0,LUFA 2.2 A,Control,798,-3,75.42749602262471
6.0.B,6.0,LUFA 2.2 A,Control,833,-3,79.26768626271135
6.0.B,6.0,LUFA 2.2 A,Control,868,-3,73.32934696431795
6.0.B,6.0,LUFA 2.2 A,Control,899,-3,62.338125783741496
6.0.B,6.0,LUFA 2.2 A,Control,926,-3,32.95441448943979
6.0.B,6.0,LUFA 2.2 A,Control,955,-3,95.87482476683314
6.0.B,6.0,LUFA 2.2 A,Control,987,-3,76.52950550574643
6.0.B,6.0,LUFA 2.2 A,Control,1016,-3,
6.0.B,6.0,LUFA 2.2 A,Control,1043,-3,
6.0.B,6.0,LUFA 2.2 A,Control,1079,-3,
6.0.B,6.0,LUFA 2.2 A,Control,1107,-3,
6.0.B,6.0,LUFA 2.2 A,Control,1142,-3,
6.0.B,6.0,LUFA 2.2 A,Control,1171,-3,
6.0.C,6.0,LUFA 2.2 A,Control,28,-3,51.97249950057164
6.0.C,6.0,LUFA 2.2 A,Control,59,-3,71.3178187375895
6.0.C,6.0,LUFA 2.2 A,Control,85,-3,75.36012426740477
6.0.C,6.0,LUFA 2.2 A,Control,114,-3,101.44261938744808
6.0.C,6.0,LUFA 2.2 A,Control,149,-3,80.07614737348817
6.0.C,6.0,LUFA 2.2 A,Control,178,-3,66.69804101329802
6.0.C,6.0,LUFA 2.2 A,Control,203,-3,64.91750167880137
6.0.C,6.0,LUFA 2.2 A,Control,227,-3,4.619777731512125
6.0.C,6.0,LUFA 2.2 A,Control,254,-3,41.57799959082977
6.0.C,6.0,LUFA 2.2 A,Control,281,-3,61.30830033094651
6.0.C,6.0,LUFA 2.2 A,Control,309,-3,48.45954351043985
6.0.C,6.0,LUFA 2.2 A,Control,350,-3,47.11210833383477
6.0.C,6.0,LUFA 2.2 A,Control,381,-3,35.37017327155665
6.0.C,6.0,LUFA 2.2 A,Control,413,-3,0.0
6.0.C,6.0,LUFA 2.2 A,Control,449,-3,33.49338855526807
6.0.C,6.0,LUFA 2.2 A,Control,471,-3,26.563721956796435
6.0.C,6.0,LUFA 2.2 A,Control,501,-3,25.023796040676334
6.0.C,6.0,LUFA 2.2 A,Control,528,-3,21.544525947409593
6.0.C,6.0,LUFA 2.2 A,Control,563,-3,50.288205523798055
6.0.C,6.0,LUFA 2.2 A,Control,598,-3,55.50951681809976
6.0.C,6.0,LUFA 2.2 A,Control,626,-3,2.473505994343823
6.0.C,6.0,LUFA 2.2 A,Control,653,-3,29.00835434141645
6.0.C,6.0,LUFA 2.2 A,Control,680,-3,28.161395101991697
6.0.D,6.0,LUFA 2.2 A,Control,28,-3,67.56424932908118
6.0.D,6.0,LUFA 2.2 A,Control,85,-3,112.60708223118118
6.0.D,6.0,LUFA 2.2 A,Control,114,-3,164.43521367109935
6.0.D,6.0,LUFA 2.2 A,Control,149,-3,80.17239273121126
6.0.D,6.0,LUFA 2.2 A,Control,178,-3,163.61712803417774
6.0.D,6.0,LUFA 2.2 A,Control,203,-3,131.37492927372284
6.0.D,6.0,LUFA 2.2 A,Control,227,-3,23.82072893435225
6.0.D,6.0,LUFA 2.2 A,Control,254,-3,92.58804539382636
6.0.D,6.0,LUFA 2.2 A,Control,281,-3,224.9735510439858
6.0.D,6.0,LUFA 2.2 A,Control,309,-3,152.64515590589085
6.0.D,6.0,LUFA 2.2 A,Control,350,-3,113.08830906793428
6.0.D,6.0,LUFA 2.2 A,Control,381,-3,67.56424932908118
6.0.D,6.0,LUFA 2.2 A,Control,413,-3,309.23637186352966
6.0.D,6.0,LUFA 2.2 A,Control,449,-3,129.3537765208496
6.0.D,6.0,LUFA 2.2 A,Control,471,-3,46.294022696913174
6.0.D,6.0,LUFA 2.2 A,Control,501,-3,84.21469826102653
6.0.D,6.0,LUFA 2.2 A,Control,528,-3,33.39714319754498
6.0.D,6.0,LUFA 2.2 A,Control,563,-3,139.63278197244117
6.0.D,6.0,LUFA 2.2 A,Control,598,-3,93.63711992297972
6.0.D,6.0,LUFA 2.2 A,Control,626,-3,33.844684156688125
6.0.D,6.0,LUFA 2.2 A,Control,653,-3,41.808988483061555
6.0.D,6.0,LUFA 2.2 A,Control,680,-3,48.67609557735123
6.0.D,6.0,LUFA 2.2 A,Control,716,-3,29.162346928214692
6.0.D,6.0,LUFA 2.2 A,Control,742,-3,27.083446946266317
6.0.D,6.0,LUFA 2.2 A,Control,771,-3,24.9852979120284
6.0.D,6.0,LUFA 2.2 A,Control,798,-3,23.50311921535592
6.0.D,6.0,LUFA 2.2 A,Control,833,-3,28.989105265058065
6.0.D,6.0,LUFA 2.2 A,Control,868,-3,58.20919942234791
6.0.D,6.0,LUFA 2.2 A,Control,899,-3,53.84928419279138
6.0.D,6.0,LUFA 2.2 A,Control,926,-3,15.375197766411938
6.0.D,6.0,LUFA 2.2 A,Control,955,-3,30.54346798242975
6.0.D,6.0,LUFA 2.2 A,Control,987,-3,49.191008315783144
6.0.D,6.0,LUFA 2.2 A,Control,1016,-3,
6.0.D,6.0,LUFA 2.2 A,Control,1043,-3,
6.0.D,6.0,LUFA 2.2 A,Control,1079,-3,
6.0.D,6.0,LUFA 2.2 A,Control,1107,-3,
6.0.D,6.0,LUFA 2.2 A,Control,1142,-3,
6.0.D,6.0,LUFA 2.2 A,Control,1171,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,32,-3,66.313059534268
6.2.A,6.2,LUFA 2.2 A,Basanite,57,-3,79.88365662998855
6.2.A,6.2,LUFA 2.2 A,Basanite,85,-3,85.03278389794812
6.2.A,6.2,LUFA 2.2 A,Basanite,114,-3,147.64039670256935
6.2.A,6.2,LUFA 2.2 A,Basanite,149,-3,76.90005018352488
6.2.A,6.2,LUFA 2.2 A,Basanite,178,-3,142.63563749924785
6.2.A,6.2,LUFA 2.2 A,Basanite,203,-3,85.7546241530778
6.2.A,6.2,LUFA 2.2 A,Basanite,227,-3,28.29613861243155
6.2.A,6.2,LUFA 2.2 A,Basanite,254,-3,48.5076661893014
6.2.A,6.2,LUFA 2.2 A,Basanite,281,-3,42.44420790661291
6.2.A,6.2,LUFA 2.2 A,Basanite,309,-3,111.74087391539804
6.2.A,6.2,LUFA 2.2 A,Basanite,351,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,381,-3,53.41618003489981
6.2.A,6.2,LUFA 2.2 A,Basanite,413,-3,0.0
6.2.A,6.2,LUFA 2.2 A,Basanite,449,-3,71.94341365906492
6.2.A,6.2,LUFA 2.2 A,Basanite,471,-3,50.52881894217462
6.2.A,6.2,LUFA 2.2 A,Basanite,501,-3,190.99893562789575
6.2.A,6.2,LUFA 2.2 A,Basanite,528,-3,69.85488912690293
6.2.A,6.2,LUFA 2.2 A,Basanite,563,-3,98.7669981105963
6.2.A,6.2,LUFA 2.2 A,Basanite,598,-3,85.29264639268307
6.2.A,6.2,LUFA 2.2 A,Basanite,626,-3,24.455948372344903
6.2.A,6.2,LUFA 2.2 A,Basanite,653,-3,29.523267079848367
6.2.A,6.2,LUFA 2.2 A,Basanite,680,-3,26.414541645105
6.2.A,6.2,LUFA 2.2 A,Basanite,716,-3,30.01411846681509
6.2.A,6.2,LUFA 2.2 A,Basanite,742,-3,60.63458273060954
6.2.A,6.2,LUFA 2.2 A,Basanite,771,-3,24.205710403754736
6.2.A,6.2,LUFA 2.2 A,Basanite,798,-3,28.902484433479753
6.2.A,6.2,LUFA 2.2 A,Basanite,833,-3,35.29798924122992
6.2.A,6.2,LUFA 2.2 A,Basanite,868,-3,77.91543881099945
6.2.A,6.2,LUFA 2.2 A,Basanite,899,-3,61.751029014982855
6.2.A,6.2,LUFA 2.2 A,Basanite,926,-3,39.84077068415669
6.2.A,6.2,LUFA 2.2 A,Basanite,955,-3,57.593229075154944
6.2.A,6.2,LUFA 2.2 A,Basanite,987,-3,52.31898282688489
6.2.A,6.2,LUFA 2.2 A,Basanite,1016,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,1043,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,1079,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,1107,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,1142,-3,
6.2.A,6.2,LUFA 2.2 A,Basanite,1171,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,32,-3,76.7556821228714
6.2.B,6.2,LUFA 2.2 A,Basanite,57,-3,88.54573987307596
6.2.B,6.2,LUFA 2.2 A,Basanite,85,-3,106.11051979060112
6.2.B,6.2,LUFA 2.2 A,Basanite,114,-3,175.55155383597088
6.2.B,6.2,LUFA 2.2 A,Basanite,149,-3,109.14224891991094
6.2.B,6.2,LUFA 2.2 A,Basanite,178,-3,137.4383875323425
6.2.B,6.2,LUFA 2.2 A,Basanite,203,-3,96.29349212347311
6.2.B,6.2,LUFA 2.2 A,Basanite,227,-3,41.57799959082977
6.2.B,6.2,LUFA 2.2 A,Basanite,254,-3,50.81755506348156
6.2.B,6.2,LUFA 2.2 A,Basanite,281,-3,80.07614737348817
6.2.B,6.2,LUFA 2.2 A,Basanite,309,-3,66.313059534268
6.2.B,6.2,LUFA 2.2 A,Basanite,351,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,381,-3,166.31199836331908
6.2.B,6.2,LUFA 2.2 A,Basanite,413,-3,288.0142680064986
6.2.B,6.2,LUFA 2.2 A,Basanite,449,-3,163.61712803417774
6.2.B,6.2,LUFA 2.2 A,Basanite,471,-3,68.71919376617124
6.2.B,6.2,LUFA 2.2 A,Basanite,501,-3,90.95187411998316
6.2.B,6.2,LUFA 2.2 A,Basanite,528,-3,57.02538138275468
6.2.B,6.2,LUFA 2.2 A,Basanite,563,-3,348.7932188458993
6.2.B,6.2,LUFA 2.2 A,Basanite,598,-3,135.6963463505626
6.2.B,6.2,LUFA 2.2 A,Basanite,626,-3,92.39555465431133
6.2.B,6.2,LUFA 2.2 A,Basanite,653,-3,124.53188350682952
6.2.B,6.2,LUFA 2.2 A,Basanite,680,-3,54.67699437992658
6.2.B,6.2,LUFA 2.2 A,Basanite,716,-3,39.989950995848126
6.2.B,6.2,LUFA 2.2 A,Basanite,742,-3,15.086461657139418
6.2.B,6.2,LUFA 2.2 A,Basanite,771,-3,40.08619637764005
6.2.B,6.2,LUFA 2.2 A,Basanite,798,-3,27.6320455623082
6.2.B,6.2,LUFA 2.2 A,Basanite,833,-3,38.53664591130634
6.2.B,6.2,LUFA 2.2 A,Basanite,868,-3,117.11136552139116
6.2.B,6.2,LUFA 2.2 A,Basanite,899,-3,109.3106783320296
6.2.B,6.2,LUFA 2.2 A,Basanite,926,-3,55.77419158794151
6.2.B,6.2,LUFA 2.2 A,Basanite,955,-3,129.58476538901257
6.2.B,6.2,LUFA 2.2 A,Basanite,987,-3,59.93680380287623
6.2.B,6.2,LUFA 2.2 A,Basanite,1016,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,1043,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,1079,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,1107,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,1142,-3,
6.2.B,6.2,LUFA 2.2 A,Basanite,1171,-3,
6.2.C,6.2,LUFA 2.2 A,Basanite,32,-3,74.10893444852277
6.2.C,6.2,LUFA 2.2 A,Basanite,57,-3,122.32786454305798
6.2.C,6.2,LUFA 2.2 A,Basanite,85,-3,170.9317761116794
6.2.C,6.2,LUFA 2.2 A,Basanite,114,-3,187.678470377279
6.2.C,6.2,LUFA 2.2 A,Basanite,149,-3,104.42622583789635
6.2.C,6.2,LUFA 2.2 A,Basanite,178,-3,150.14277631626453
6.2.C,6.2,LUFA 2.2 A,Basanite,203,-3,110.87466557554606
6.2.C,6.2,LUFA 2.2 A,Basanite,227,-3,55.34108743004994
6.2.C,6.2,LUFA 2.2 A,Basanite,254,-3,39.701214898610026
6.2.C,6.2,LUFA 2.2 A,Basanite,281,-3,90.71126067753777
6.2.C,6.2,LUFA 2.2 A,Basanite,309,-3,89.31570282207112
6.2.C,6.2,LUFA 2.2 A,Basanite,351,-3,90.95187411998316
6.2.C,6.2,LUFA 2.2 A,Basanite,381,-3,59.19090219628137
6.2.C,6.2,LUFA 2.2 A,Basanite,413,-3,68.1417215476262
6.2.C,6.2,LUFA 2.2 A,Basanite,449,-3,156.39872531439917
6.2.C,6.2,LUFA 2.2 A,Basanite,471,-3,54.57112447198989
6.2.C,6.2,LUFA 2.2 A,Basanite,501,-3,129.93124873939465
6.2.C,6.2,LUFA 2.2 A,Basanite,528,-3,82.62183739093808
6.2.C,6.2,LUFA 2.2 A,Basanite,563,-3,264.669953667489
6.2.C,6.2,LUFA 2.2 A,Basanite,598,-3,243.50078464408207
6.2.C,6.2,LUFA 2.2 A,Basanite,626,-3,143.1649870148625
6.2.C,6.2,LUFA 2.2 A,Basanite,653,-3,121.77445367350622
6.2.C,6.2,LUFA 2.2 A,Basanite,680,-3,90.20116023828147
6.2.D,6.2,LUFA 2.2 A,Basanite,28,-3,59.7683744148264
6.2.D,6.2,LUFA 2.2 A,Basanite,57,-3,101.61104876627712
6.2.D,6.2,LUFA 2.2 A,Basanite,85,-3,123.19407285636922
6.2.D,6.2,LUFA 2.2 A,Basanite,114,-3,165.78264884770442
6.2.D,6.2,LUFA 2.2 A,Basanite,149,-3,98.74774903423793
6.2.D,6.2,LUFA 2.2 A,Basanite,178,-3,125.6002071123413
6.2.D,6.2,LUFA 2.2 A,Basanite,203,-3,85.08090657680968
6.2.D,6.2,LUFA 2.2 A,Basanite,227,-3,11.549444332390635
6.2.D,6.2,LUFA 2.2 A,Basanite,254,-3,23.77260624827005
6.2.D,6.2,LUFA 2.2 A,Basanite,281,-3,72.18402707744148
6.2.D,6.2,LUFA 2.2 A,Basanite,309,-3,54.57112447198989
6.2.D,6.2,LUFA 2.2 A,Basanite,351,-3,64.6768882604248
6.2.D,6.2,LUFA 2.2 A,Basanite,381,-3,49.66261062639148
6.2.D,6.2,LUFA 2.2 A,Basanite,413,-3,29.354837667729704
6.2.D,6.2,LUFA 2.2 A,Basanite,449,-3,99.61395735002104
6.2.D,6.2,LUFA 2.2 A,Basanite,471,-3,42.44420790661291
6.2.D,6.2,LUFA 2.2 A,Basanite,501,-3,54.667369829712975
6.2.D,6.2,LUFA 2.2 A,Basanite,528,-3,43.66652409892291
6.2.D,6.2,LUFA 2.2 A,Basanite,563,-3,95.93738424694628
6.2.D,6.2,LUFA 2.2 A,Basanite,598,-3,87.27048871773272
6.2.D,6.2,LUFA 2.2 A,Basanite,626,-3,37.39613829953667
6.2.D,6.2,LUFA 2.2 A,Basanite,653,-3,38.113166303628375
6.2.D,6.2,LUFA 2.2 A,Basanite,680,-3,47.72807870509657
6.4.A,K.0,Bramstedt,Control,93,65,25.9862497382514
6.4.A,K.0,Bramstedt,Control,113,65,32.55499620275039
6.4.A,K.0,Bramstedt,Control,178,65,54.85986056922799
6.4.A,K.0,Bramstedt,Control,203,65,23.82072893435225
6.4.A,K.0,Bramstedt,Control,227,65,6.15970364281846
6.4.A,K.0,Bramstedt,Control,254,65,10.971972113845595
6.4.A,K.0,Bramstedt,Control,281,65,9.04706472591612
6.4.A,K.0,Bramstedt,Control,309,65,5.101004580299657
6.4.A,K.0,Bramstedt,Control,351,65,18.671601667970396
6.4.A,K.0,Bramstedt,Control,381,65,6.063458273060954
6.4.A,K.0,Bramstedt,Control,413,65,158.80485954630242
6.4.A,K.0,Bramstedt,Control,449,65,0.0
6.4.A,K.0,Bramstedt,Control,471,65,6.9777892845538245
6.4.A,K.0,Bramstedt,Control,501,65,12.126916548528792
6.4.A,K.0,Bramstedt,Control,528,65,11.886303125338468
6.4.A,K.0,Bramstedt,Control,563,65,34.47509132920152
6.4.A,K.0,Bramstedt,Control,598,65,13.51284986822312
6.4.A,K.0,Bramstedt,Control,626,65,13.575409357963776
6.4.A,K.0,Bramstedt,Control,653,65,17.71396024309525
6.4.A,K.0,Bramstedt,Control,680,65,17.92088778867561
6.4.B,K.0,Bramstedt,Control,93,65,19.249073886515436
6.4.B,K.0,Bramstedt,Control,113,65,25.336593502773933
6.4.B,K.0,Bramstedt,Control,178,65,41.57799959429569
6.4.B,K.0,Bramstedt,Control,203,65,23.291379401889404
6.4.B,K.0,Bramstedt,Control,227,65,19.48968730970576
6.4.B,K.0,Bramstedt,Control,254,65,33.20465245802996
6.4.B,K.0,Bramstedt,Control,281,65,12.030671178771286
6.4.B,K.0,Bramstedt,Control,309,65,15.591749847764603
6.4.B,K.0,Bramstedt,Control,351,65,37.72818482459835
6.4.B,K.0,Bramstedt,Control,381,65,4.138550885131476
6.4.B,K.0,Bramstedt,Control,413,65,2.117398127444491
6.4.B,K.0,Bramstedt,Control,449,65,2.4061342357542572
6.4.B,K.0,Bramstedt,Control,471,65,5.197249950057164
6.4.B,K.0,Bramstedt,Control,501,65,36.380749647993255
6.4.B,K.0,Bramstedt,Control,528,65,10.269380918226126
6.4.B,K.0,Bramstedt,Control,563,65,25.163351826223
6.4.B,K.0,Bramstedt,Control,598,65,11.453198962633133
6.4.B,K.0,Bramstedt,Control,626,65,6.448439752090981
6.4.B,K.0,Bramstedt,Control,653,65,15.591749847764603
6.4.B,K.0,Bramstedt,Control,680,65,17.824642418918106
6.4.C,K.0,Bramstedt,Control,93,65,20.0190368421686
6.4.C,K.0,Bramstedt,Control,113,65,32.72342559720801
6.4.C,K.0,Bramstedt,Control,178,65,47.54521248998803
6.4.C,K.0,Bramstedt,Control,203,65,23.38762477164691
6.4.C,K.0,Bramstedt,Control,227,65,13.57059709007762
6.4.C,K.0,Bramstedt,Control,254,65,41.38550885131476
6.4.C,K.0,Bramstedt,Control,281,65,28.873610830976595
6.4.C,K.0,Bramstedt,Control,309,65,27.285562235994945
6.4.C,K.0,Bramstedt,Control,351,65,34.69645567121969
6.4.C,K.0,Bramstedt,Control,381,65,10.586990637222456
6.4.C,K.0,Bramstedt,Control,413,65,2.454256920392322
6.4.C,K.0,Bramstedt,Control,449,65,23.38762477164691
6.4.C,K.0,Bramstedt,Control,471,65,2.8873610830976593
6.4.C,K.0,Bramstedt,Control,501,65,19.634055363138575
6.4.C,K.0,Bramstedt,Control,528,65,13.897831344846258
6.4.C,K.0,Bramstedt,Control,563,65,24.13833864853481
6.4.C,K.0,Bramstedt,Control,598,65,17.92088778867561
6.4.C,K.0,Bramstedt,Control,626,65,11.34732905469643
6.4.C,K.0,Bramstedt,Control,653,65,9.119248753835972
6.4.C,K.0,Bramstedt,Control,680,65,9.744843653649436
6.4.D,K.0,Bramstedt,Control,93,65,17.32416649617907
6.4.D,K.0,Bramstedt,Control,113,65,20.211527578975872
6.4.D,K.0,Bramstedt,Control,178,65,35.610786686767014
6.4.D,K.0,Bramstedt,Control,203,65,21.55896275106805
6.4.D,K.0,Bramstedt,Control,227,65,4.716023101269631
6.4.D,K.0,Bramstedt,Control,254,65,23.82072893435225
6.4.D,K.0,Bramstedt,Control,281,65,51.87625411877971
6.4.D,K.0,Bramstedt,Control,309,65,5.5822314266803055
6.4.D,K.0,Bramstedt,Control,351,65,6.15970364281846
6.4.D,K.0,Bramstedt,Control,381,65,2.165520812082556
6.4.D,K.0,Bramstedt,Control,413,65,14.148069306215776
6.4.D,K.0,Bramstedt,Control,449,65,10.394499897707442
6.4.D,K.0,Bramstedt,Control,471,65,3.753569408508334
6.4.D,K.0,Bramstedt,Control,501,65,12.511898025151934
6.4.D,K.0,Bramstedt,Control,528,65,8.055737421024128
6.4.D,K.0,Bramstedt,Control,563,65,33.68587929478308
6.4.D,K.0,Bramstedt,Control,598,65,19.865044250556593
6.4.D,K.0,Bramstedt,Control,626,65,1.9585932679463265
6.4.D,K.0,Bramstedt,Control,653,65,10.803542718575123
6.4.D,K.0,Bramstedt,Control,680,65,7.536012426740477
6.5.A,K.1,Bramstedt,Basanite,93,65,25.264409483121728
6.5.A,K.1,Bramstedt,Basanite,113,65,37.05446723501093
6.5.A,K.1,Bramstedt,Basanite,178,65,91.43310095673628
6.5.A,K.1,Bramstedt,Basanite,203,65,35.65890936879475
6.5.A,K.1,Bramstedt,Basanite,227,65,20.355895635116433
6.5.A,K.1,Bramstedt,Basanite,254,65,61.30830033094651
6.5.A,K.1,Bramstedt,Basanite,281,65,17.709147975209095
6.5.A,K.1,Bramstedt,Basanite,309,65,35.7070320476563
6.5.A,K.1,Bramstedt,Basanite,351,65,23.58011550875504
6.5.A,K.1,Bramstedt,Basanite,381,65,11.30883090679343
6.5.A,K.1,Bramstedt,Basanite,413,65,129.93124873939465
6.5.A,K.1,Bramstedt,Basanite,449,65,0.0
6.5.A,K.1,Bramstedt,Basanite,471,65,15.880485954630243
6.5.A,K.1,Bramstedt,Basanite,501,65,21.222103958120226
6.5.A,K.1,Bramstedt,Basanite,528,65,9.398360324929298
6.5.A,K.1,Bramstedt,Basanite,563,65,27.37218306757326
6.5.A,K.1,Bramstedt,Basanite,598,65,13.137492927372284
6.5.A,K.1,Bramstedt,Basanite,626,65,14.128820232264276
6.5.A,K.1,Bramstedt,Basanite,653,65,21.424219235814427
6.5.A,K.1,Bramstedt,Basanite,680,65,15.784240587279616
6.5.B,K.1,Bramstedt,Basanite,93,65,16.93918501955593
6.5.B,K.1,Bramstedt,Basanite,113,65,26.63590598983152
6.5.B,K.1,Bramstedt,Basanite,178,65,51.97249949235419
6.5.B,K.1,Bramstedt,Basanite,203,65,23.82072893435225
6.5.B,K.1,Bramstedt,Basanite,227,65,24.542569203923215
6.5.B,K.1,Bramstedt,Basanite,254,65,19.05658314700042
6.5.B,K.1,Bramstedt,Basanite,281,65,19.634055363138575
6.5.B,K.1,Bramstedt,Basanite,309,65,20.404018318791746
6.5.B,K.1,Bramstedt,Basanite,351,65,28.873610830976595
6.5.B,K.1,Bramstedt,Basanite,381,65,4.138550885131476
6.5.B,K.1,Bramstedt,Basanite,413,65,2.5505022901498284
6.5.B,K.1,Bramstedt,Basanite,449,65,3.368587929478308
6.5.B,K.1,Bramstedt,Basanite,471,65,15.591749847764603
6.5.B,K.1,Bramstedt,Basanite,501,65,56.30354110355617
6.5.B,K.1,Bramstedt,Basanite,528,65,23.584927779048076
6.5.B,K.1,Bramstedt,Basanite,563,65,33.92649271315963
6.5.B,K.1,Bramstedt,Basanite,598,65,22.63691088994524
6.5.B,K.1,Bramstedt,Basanite,626,65,7.608196452253445
6.5.B,K.1,Bramstedt,Basanite,653,65,20.59650905830675
6.5.B,K.1,Bramstedt,Basanite,680,65,15.327075080329744
6.5.C,K.1,Bramstedt,Basanite,93,65,21.655208120825563
6.5.C,K.1,Bramstedt,Basanite,113,65,25.649390951628593
6.5.C,K.1,Bramstedt,Basanite,178,65,43.166048186887146
6.5.C,K.1,Bramstedt,Basanite,203,65,25.938127059389856
6.5.C,K.1,Bramstedt,Basanite,227,65,17.32416649617907
6.5.C,K.1,Bramstedt,Basanite,254,65,20.64463174198207
6.5.C,K.1,Bramstedt,Basanite,281,65,30.991008965641736
6.5.C,K.1,Bramstedt,Basanite,309,65,43.117925506949874
6.5.C,K.1,Bramstedt,Basanite,351,65,23.58011550875504
6.5.C,K.1,Bramstedt,Basanite,381,65,4.427286994403995
6.5.C,K.1,Bramstedt,Basanite,413,65,11.790057755580962
6.5.C,K.1,Bramstedt,Basanite,449,65,18.527233614537575
6.5.C,K.1,Bramstedt,Basanite,471,65,6.496562435766291
6.5.C,K.1,Bramstedt,Basanite,501,65,24.25383310668512
6.5.C,K.1,Bramstedt,Basanite,528,65,8.050925153137975
6.5.C,K.1,Bramstedt,Basanite,563,65,40.85615931163127
6.5.C,K.1,Bramstedt,Basanite,598,65,13.474351720320117
6.5.C,K.1,Bramstedt,Basanite,626,65,11.284769564955774
6.5.C,K.1,Bramstedt,Basanite,653,65,12.617767933088633
6.5.C,K.1,Bramstedt,Basanite,680,65,18.257746581623444
6.5.D,K.1,Bramstedt,Basanite,93,65,21.655208120825563
6.5.D,K.1,Bramstedt,Basanite,113,65,31.76097189963295
6.5.D,K.1,Bramstedt,Basanite,178,65,46.75118819088357
6.5.D,K.1,Bramstedt,Basanite,203,65,31.664726541909864
6.5.D,K.1,Bramstedt,Basanite,227,65,14.484928099163607
6.5.D,K.1,Bramstedt,Basanite,254,65,46.19777731512125
6.5.D,K.1,Bramstedt,Basanite,281,65,47.59333517058788
6.5.D,K.1,Bramstedt,Basanite,309,65,39.701214898610026
6.5.D,K.1,Bramstedt,Basanite,351,65,24.06134235754257
6.5.D,K.1,Bramstedt,Basanite,381,65,9.239555465431131
6.5.D,K.1,Bramstedt,Basanite,413,65,32.48281217883146
6.5.D,K.1,Bramstedt,Basanite,449,65,8.854573986401107
6.5.D,K.1,Bramstedt,Basanite,471,65,6.352194382333473
6.5.D,K.1,Bramstedt,Basanite,501,65,9.528291572296768
6.5.D,K.1,Bramstedt,Basanite,528,65,5.235748097960165
6.5.D,K.1,Bramstedt,Basanite,563,65,20.399206050905587
6.5.D,K.1,Bramstedt,Basanite,598,65,10.490745267464948
6.5.D,K.1,Bramstedt,Basanite,626,65,8.219354548408448
6.5.D,K.1,Bramstedt,Basanite,653,65,12.531147099103435
6.5.D,K.1,Bramstedt,Basanite,680,65,13.305922322642758
6.6.A,6.6,LUFA 2.2 A,Metabasalt,32,-3,14.436805413081412
6.6.A,6.6,LUFA 2.2 A,Metabasalt,57,-3,69.39291135763712
6.6.A,6.6,LUFA 2.2 A,Metabasalt,85,-3,0.0
6.6.A,6.6,LUFA 2.2 A,Metabasalt,114,-3,172.13484322763102
6.6.A,6.6,LUFA 2.2 A,Metabasalt,149,-3,103.75250823755944
6.6.A,6.6,LUFA 2.2 A,Metabasalt,178,-3,259.862497382514
6.6.A,6.6,LUFA 2.2 A,Metabasalt,203,-3,256.5901548829653
6.6.A,6.6,LUFA 2.2 A,Metabasalt,227,-3,140.08513520669115
6.6.A,6.6,LUFA 2.2 A,Metabasalt,254,-3,198.0729702870209
6.6.A,6.6,LUFA 2.2 A,Metabasalt,281,-3,214.8196645526205
6.6.A,6.6,LUFA 2.2 A,Metabasalt,309,-3,300.95927023286595
6.6.A,6.6,LUFA 2.2 A,Metabasalt,351,-3,599.9455103195138
6.6.A,6.6,LUFA 2.2 A,Metabasalt,381,-3,279.3521848486672
6.6.A,6.6,LUFA 2.2 A,Metabasalt,413,-3,409.6202921956796
6.6.A,6.6,LUFA 2.2 A,Metabasalt,449,-3,405.7704774053794
6.6.A,6.6,LUFA 2.2 A,Metabasalt,471,-3,138.160227811541
6.6.A,6.6,LUFA 2.2 A,Metabasalt,501,-3,422.4209264095312
6.6.A,6.6,LUFA 2.2 A,Metabasalt,528,-3,166.46599097418616
6.6.A,6.6,LUFA 2.2 A,Metabasalt,563,-3,289.1932737228473
6.6.A,6.6,LUFA 2.2 A,Metabasalt,598,-3,235.66641157711052
6.6.A,6.6,LUFA 2.2 A,Metabasalt,626,-3,140.06588613033276
6.6.A,6.6,LUFA 2.2 A,Metabasalt,653,-3,182.2983542451411
6.6.A,6.6,LUFA 2.2 A,Metabasalt,680,-3,238.7222020578855
6.6.A,6.6,LUFA 2.2 A,Metabasalt,716,-3,444.92309380829175
6.6.A,6.6,LUFA 2.2 A,Metabasalt,742,-3,328.54319297189966
6.6.A,6.6,LUFA 2.2 A,Metabasalt,771,-3,468.7726963114507
6.6.A,6.6,LUFA 2.2 A,Metabasalt,798,-3,275.3387527528732
6.6.A,6.6,LUFA 2.2 A,Metabasalt,833,-3,185.87868196642395
6.6.A,6.6,LUFA 2.2 A,Metabasalt,868,-3,169.56027958360912
6.6.A,6.6,LUFA 2.2 A,Metabasalt,899,-3,246.4458930140201
6.6.A,6.6,LUFA 2.2 A,Metabasalt,926,-3,82.82876493170467
6.6.A,6.6,LUFA 2.2 A,Metabasalt,955,-3,124.73399877248931
6.6.A,6.6,LUFA 2.2 A,Metabasalt,987,-3,104.08455476262108
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1016,-3,
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1043,-3,
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1079,-3,
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1107,-3,
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1142,-3,
6.6.A,6.6,LUFA 2.2 A,Metabasalt,1171,-3,
6.6.B,6.6,LUFA 2.2 A,Metabasalt,32,-3,85.89899221373128
6.6.B,6.6,LUFA 2.2 A,Metabasalt,57,-3,72.56900853948174
6.6.B,6.6,LUFA 2.2 A,Metabasalt,85,-3,35.08143715024971
6.6.B,6.6,LUFA 2.2 A,Metabasalt,114,-3,86.62083249292978
6.6.B,6.6,LUFA 2.2 A,Metabasalt,149,-3,31.279745062879833
6.6.B,6.6,LUFA 2.2 A,Metabasalt,178,-3,242.39396281364705
6.6.B,6.6,LUFA 2.2 A,Metabasalt,203,-3,328.8704273421987
6.6.B,6.6,LUFA 2.2 A,Metabasalt,227,-3,132.433628329021
6.6.B,6.6,LUFA 2.2 A,Metabasalt,254,-3,185.80168568505923
6.6.B,6.6,LUFA 2.2 A,Metabasalt,281,-3,337.82124676575006
6.6.B,6.6,LUFA 2.2 A,Metabasalt,309,-3,230.26704634454535
6.6.B,6.6,LUFA 2.2 A,Metabasalt,351,-3,413.8550885131475
6.6.B,6.6,LUFA 2.2 A,Metabasalt,381,-3,155.19565819844755
6.6.B,6.6,LUFA 2.2 A,Metabasalt,413,-3,255.5314558035983
6.6.B,6.6,LUFA 2.2 A,Metabasalt,449,-3,291.6234692821469
6.6.B,6.6,LUFA 2.2 A,Metabasalt,471,-3,85.7546241530778
6.6.B,6.6,LUFA 2.2 A,Metabasalt,501,-3,0.6255949012575966
6.6.B,6.6,LUFA 2.2 A,Metabasalt,528,-3,99.17122866598471
6.6.B,6.6,LUFA 2.2 A,Metabasalt,563,-3,186.1866671640893
6.6.B,6.6,LUFA 2.2 A,Metabasalt,598,-3,108.56477672543474
6.6.B,6.6,LUFA 2.2 A,Metabasalt,626,-3,80.51406380648656
6.6.B,6.6,LUFA 2.2 A,Metabasalt,653,-3,97.231884469583
6.6.B,6.6,LUFA 2.2 A,Metabasalt,680,-3,114.12775907094289
6.6.C,6.6,LUFA 2.2 A,Metabasalt,32,-3,73.09835807208616
6.6.C,6.6,LUFA 2.2 A,Metabasalt,57,-3,93.16551760544
6.6.C,6.6,LUFA 2.2 A,Metabasalt,114,-3,194.8968730970576
6.6.C,6.6,LUFA 2.2 A,Metabasalt,149,-3,101.05763790841807
6.6.C,6.6,LUFA 2.2 A,Metabasalt,178,-3,506.73187002828087
6.6.C,6.6,LUFA 2.2 A,Metabasalt,203,-3,382.8640796678501
6.6.C,6.6,LUFA 2.2 A,Metabasalt,227,-3,180.17133156026233
6.6.C,6.6,LUFA 2.2 A,Metabasalt,254,-3,246.8693726457669
6.6.C,6.6,LUFA 2.2 A,Metabasalt,281,-3,271.02696022624707
6.6.C,6.6,LUFA 2.2 A,Metabasalt,309,-3,264.62664323966544
6.6.C,6.6,LUFA 2.2 A,Metabasalt,351,-3,354.5198182802816
6.6.C,6.6,LUFA 2.2 A,Metabasalt,381,-3,254.18402069919972
6.6.C,6.6,LUFA 2.2 A,Metabasalt,413,-3,176.032780672724
6.6.C,6.6,LUFA 2.2 A,Metabasalt,449,-3,335.03013105481676
6.6.C,6.6,LUFA 2.2 A,Metabasalt,471,-3,153.99259108249595
6.6.C,6.6,LUFA 2.2 A,Metabasalt,501,-3,245.570060051748
6.6.C,6.6,LUFA 2.2 A,Metabasalt,528,-3,86.30322276911969
6.6.C,6.6,LUFA 2.2 A,Metabasalt,563,-3,205.47423919610085
6.6.C,6.6,LUFA 2.2 A,Metabasalt,598,-3,122.93902263674109
6.6.C,6.6,LUFA 2.2 A,Metabasalt,626,-3,99.13273051326796
6.6.C,6.6,LUFA 2.2 A,Metabasalt,653,-3,103.75250823755944
6.6.C,6.6,LUFA 2.2 A,Metabasalt,680,-3,105.50417396955292
6.6.C,6.6,LUFA 2.2 A,Metabasalt,716,-3,139.1130569829713
6.6.C,6.6,LUFA 2.2 A,Metabasalt,742,-3,38.113166303628375
6.6.C,6.6,LUFA 2.2 A,Metabasalt,771,-3,86.23585101389975
6.6.C,6.6,LUFA 2.2 A,Metabasalt,798,-3,92.49661228714122
6.6.C,6.6,LUFA 2.2 A,Metabasalt,833,-3,101.13463418978276
6.6.C,6.6,LUFA 2.2 A,Metabasalt,868,-3,195.43584716288584
6.6.C,6.6,LUFA 2.2 A,Metabasalt,899,-3,165.4939127263975
6.6.C,6.6,LUFA 2.2 A,Metabasalt,926,-3,86.10110747939105
6.6.C,6.6,LUFA 2.2 A,Metabasalt,955,-3,153.60760960346593
6.6.C,6.6,LUFA 2.2 A,Metabasalt,987,-3,120.7783140983212
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1016,-3,
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1043,-3,
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1079,-3,
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1107,-3,
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1142,-3,
6.6.C,6.6,LUFA 2.2 A,Metabasalt,1171,-3,
6.6.D,6.6,LUFA 2.2 A,Metabasalt,32,-3,68.71919376617124
6.6.D,6.6,LUFA 2.2 A,Metabasalt,57,-3,53.704916138084336
6.6.D,6.6,LUFA 2.2 A,Metabasalt,85,-3,93.8392351886395
6.6.D,6.6,LUFA 2.2 A,Metabasalt,114,-3,148.12162353932246
6.6.D,6.6,LUFA 2.2 A,Metabasalt,149,-3,101.05763790841807
6.6.D,6.6,LUFA 2.2 A,Metabasalt,178,-3,371.6033713219808
6.6.D,6.6,LUFA 2.2 A,Metabasalt,203,-3,396.5309219567965
6.6.D,6.6,LUFA 2.2 A,Metabasalt,227,-3,200.96033135567723
6.6.D,6.6,LUFA 2.2 A,Metabasalt,254,-3,287.1480596907154
6.6.D,6.6,LUFA 2.2 A,Metabasalt,281,-3,279.3521848486672
6.6.D,6.6,LUFA 2.2 A,Metabasalt,309,-3,554.0845918526987
6.6.D,6.6,LUFA 2.2 A,Metabasalt,351,-3,521.1686753715627
6.6.D,6.6,LUFA 2.2 A,Metabasalt,381,-3,217.51453490583063
6.6.D,6.6,LUFA 2.2 A,Metabasalt,413,-3,399.12954690414585
6.6.D,6.6,LUFA 2.2 A,Metabasalt,449,-3,323.3844411817799
6.6.D,6.6,LUFA 2.2 A,Metabasalt,471,-3,157.21681095132078
6.6.D,6.6,LUFA 2.2 A,Metabasalt,501,-3,307.5039552319634
6.6.D,6.6,LUFA 2.2 A,Metabasalt,528,-3,146.00422542872616
6.6.D,6.6,LUFA 2.2 A,Metabasalt,563,-3,281.34446380648654
6.6.D,6.6,LUFA 2.2 A,Metabasalt,598,-3,165.56128448161743
6.6.D,6.6,LUFA 2.2 A,Metabasalt,626,-3,108.10279894097118
6.6.D,6.6,LUFA 2.2 A,Metabasalt,653,-3,114.85922387628618
6.6.D,6.6,LUFA 2.2 A,Metabasalt,680,-3,134.53177738732774
6.7.A,6.7,LUFA 2.2 A,Peridotite,32,-3,79.54679783380467
6.7.A,6.7,LUFA 2.2 A,Peridotite,59,-3,169.3918501955593
6.7.A,6.7,LUFA 2.2 A,Peridotite,85,-3,64.6768882604248
6.7.A,6.7,LUFA 2.2 A,Peridotite,114,-3,71.17345070100487
6.7.A,6.7,LUFA 2.2 A,Peridotite,149,-3,13.137492927372284
6.7.A,6.7,LUFA 2.2 A,Peridotite,178,-3,68.71919376617124
6.7.A,6.7,LUFA 2.2 A,Peridotite,203,-3,39.845582935194656
6.7.A,6.7,LUFA 2.2 A,Peridotite,227,-3,9.38392351886395
6.7.A,6.7,LUFA 2.2 A,Peridotite,254,-3,19.634055363138575
6.7.A,6.7,LUFA 2.2 A,Peridotite,281,-3,42.44420790661291
6.7.A,6.7,LUFA 2.2 A,Peridotite,309,-3,21.173981274444913
6.7.A,6.7,LUFA 2.2 A,Peridotite,351,-3,36.57324038750828
6.7.A,6.7,LUFA 2.2 A,Peridotite,381,-3,18.7678470377279
6.7.A,6.7,LUFA 2.2 A,Peridotite,413,-3,13.23373829712979
6.7.A,6.7,LUFA 2.2 A,Peridotite,449,-3,28.584874709669656
6.7.A,6.7,LUFA 2.2 A,Peridotite,471,-3,17.75727065888441
6.7.A,6.7,LUFA 2.2 A,Peridotite,501,-3,22.906397922859377
6.7.A,6.7,LUFA 2.2 A,Peridotite,528,-3,13.23373829712979
6.7.A,6.7,LUFA 2.2 A,Peridotite,563,-3,53.77710016246464
6.7.A,6.7,LUFA 2.2 A,Peridotite,598,-3,22.136434969613095
6.7.A,6.7,LUFA 2.2 A,Peridotite,626,-3,14.21544106384259
6.7.A,6.7,LUFA 2.2 A,Peridotite,653,-3,26.79471084902822
6.7.A,6.7,LUFA 2.2 A,Peridotite,680,-3,23.993970597508874
6.7.B,6.7,LUFA 2.2 A,Peridotite,32,-3,66.16869147361454
6.7.B,6.7,LUFA 2.2 A,Peridotite,59,-3,163.71337339190083
6.7.B,6.7,LUFA 2.2 A,Peridotite,85,-3,65.63934195799987
6.7.B,6.7,LUFA 2.2 A,Peridotite,114,-3,68.81543914796318
6.7.B,6.7,LUFA 2.2 A,Peridotite,149,-3,31.90533996028642
6.7.B,6.7,LUFA 2.2 A,Peridotite,178,-3,46.05340927853661
6.7.B,6.7,LUFA 2.2 A,Peridotite,203,-3,59.72025173596486
6.7.B,6.7,LUFA 2.2 A,Peridotite,227,-3,27.5261756543715
6.7.B,6.7,LUFA 2.2 A,Peridotite,254,-3,35.99576816896323
6.7.B,6.7,LUFA 2.2 A,Peridotite,281,-3,36.81385380588483
6.7.B,6.7,LUFA 2.2 A,Peridotite,309,-3,36.380749647993255
6.7.B,6.7,LUFA 2.2 A,Peridotite,351,-3,45.23532364161502
6.7.B,6.7,LUFA 2.2 A,Peridotite,381,-3,24.83130530116132
6.7.B,6.7,LUFA 2.2 A,Peridotite,413,-3,34.0708607738131
6.7.B,6.7,LUFA 2.2 A,Peridotite,449,-3,40.423055153739696
6.7.B,6.7,LUFA 2.2 A,Peridotite,471,-3,21.173981274444913
6.7.B,6.7,LUFA 2.2 A,Peridotite,501,-3,24.63881456164631
6.7.B,6.7,LUFA 2.2 A,Peridotite,528,-3,15.89492276069559
6.7.B,6.7,LUFA 2.2 A,Peridotite,563,-3,50.93304949756303
6.7.B,6.7,LUFA 2.2 A,Peridotite,598,-3,23.214383106083396
6.7.B,6.7,LUFA 2.2 A,Peridotite,626,-3,14.956530409771949
6.7.B,6.7,LUFA 2.2 A,Peridotite,653,-3,20.692754428064266
6.7.B,6.7,LUFA 2.2 A,Peridotite,680,-3,23.965096987785063
6.7.B,6.7,LUFA 2.2 A,Peridotite,716,-3,23.329877549792407
6.7.B,6.7,LUFA 2.2 A,Peridotite,742,-3,4.100052737228474
6.7.B,6.7,LUFA 2.2 A,Peridotite,771,-3,13.859333196943258
6.7.B,6.7,LUFA 2.2 A,Peridotite,798,-3,9.720782311811782
6.7.B,6.7,LUFA 2.2 A,Peridotite,833,-3,18.979586851194416
6.7.B,6.7,LUFA 2.2 A,Peridotite,868,-3,29.9900571153499
6.7.B,6.7,LUFA 2.2 A,Peridotite,899,-3,38.85425563511643
6.7.B,6.7,LUFA 2.2 A,Peridotite,926,-3,19.749549806847583
6.7.B,6.7,LUFA 2.2 A,Peridotite,955,-3,27.237439557133403
6.7.B,6.7,LUFA 2.2 A,Peridotite,987,-3,45.11982918346471
6.7.B,6.7,LUFA 2.2 A,Peridotite,1016,-3,
6.7.B,6.7,LUFA 2.2 A,Peridotite,1043,-3,
6.7.B,6.7,LUFA 2.2 A,Peridotite,1079,-3,
6.7.B,6.7,LUFA 2.2 A,Peridotite,1107,-3,
6.7.B,6.7,LUFA 2.2 A,Peridotite,1142,-3,
6.7.B,6.7,LUFA 2.2 A,Peridotite,1171,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,32,-3,53.99365225344485
6.7.C,6.7,LUFA 2.2 A,Peridotite,59,-3,110.39343873879297
6.7.C,6.7,LUFA 2.2 A,Peridotite,85,-3,56.30354110355617
6.7.C,6.7,LUFA 2.2 A,Peridotite,114,-3,102.02009158192428
6.7.C,6.7,LUFA 2.2 A,Peridotite,149,-3,55.629823527288046
6.7.C,6.7,LUFA 2.2 A,Peridotite,178,-3,127.04388764666948
6.7.C,6.7,LUFA 2.2 A,Peridotite,203,-3,114.33949888681629
6.7.C,6.7,LUFA 2.2 A,Peridotite,227,-3,22.23268033696372
6.7.C,6.7,LUFA 2.2 A,Peridotite,254,-3,82.77101770262952
6.7.C,6.7,LUFA 2.2 A,Peridotite,281,-3,101.63511012696311
6.7.C,6.7,LUFA 2.2 A,Peridotite,309,-3,548.9835872194476
6.7.C,6.7,LUFA 2.2 A,Peridotite,351,-3,1152.1051947770625
6.7.C,6.7,LUFA 2.2 A,Peridotite,381,-3,296.2432471267826
6.7.C,6.7,LUFA 2.2 A,Peridotite,413,-3,277.04229592634937
6.7.C,6.7,LUFA 2.2 A,Peridotite,449,-3,303.1729136530477
6.7.C,6.7,LUFA 2.2 A,Peridotite,471,-3,42.059226451651725
6.7.C,6.7,LUFA 2.2 A,Peridotite,501,-3,176.032780672724
6.7.C,6.7,LUFA 2.2 A,Peridotite,528,-3,26.67921641494675
6.7.C,6.7,LUFA 2.2 A,Peridotite,563,-3,91.16361391178772
6.7.C,6.7,LUFA 2.2 A,Peridotite,598,-3,75.09063722245622
6.7.C,6.7,LUFA 2.2 A,Peridotite,626,-3,167.46694280040916
6.7.C,6.7,LUFA 2.2 A,Peridotite,653,-3,323.4133148805584
6.7.C,6.7,LUFA 2.2 A,Peridotite,680,-3,512.6509602262471
6.7.C,6.7,LUFA 2.2 A,Peridotite,716,-3,249.52574474998497
6.7.C,6.7,LUFA 2.2 A,Peridotite,742,-3,32.36731774474998
6.7.C,6.7,LUFA 2.2 A,Peridotite,771,-3,87.31379914555629
6.7.C,6.7,LUFA 2.2 A,Peridotite,798,-3,60.28809940429628
6.7.C,6.7,LUFA 2.2 A,Peridotite,833,-3,51.19291199229797
6.7.C,6.7,LUFA 2.2 A,Peridotite,868,-3,126.65409389253264
6.7.C,6.7,LUFA 2.2 A,Peridotite,899,-3,127.18825570732292
6.7.C,6.7,LUFA 2.2 A,Peridotite,926,-3,61.54410147421626
6.7.C,6.7,LUFA 2.2 A,Peridotite,955,-3,74.44579324869126
6.7.C,6.7,LUFA 2.2 A,Peridotite,987,-3,53.79153696371622
6.7.C,6.7,LUFA 2.2 A,Peridotite,1016,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,1043,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,1079,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,1107,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,1142,-3,
6.7.C,6.7,LUFA 2.2 A,Peridotite,1171,-3,
6.7.D,6.7,LUFA 2.2 A,Peridotite,32,-3,43.406661604187974
6.7.D,6.7,LUFA 2.2 A,Peridotite,59,-3,131.66366537096096
6.7.D,6.7,LUFA 2.2 A,Peridotite,85,-3,85.17715193453276
6.7.D,6.7,LUFA 2.2 A,Peridotite,114,-3,127.765727925868
6.7.D,6.7,LUFA 2.2 A,Peridotite,149,-3,71.17345070100487
6.7.D,6.7,LUFA 2.2 A,Peridotite,178,-3,122.61660066189302
6.7.D,6.7,LUFA 2.2 A,Peridotite,203,-3,95.09042498345268
6.7.D,6.7,LUFA 2.2 A,Peridotite,227,-3,36.81385380588483
6.7.D,6.7,LUFA 2.2 A,Peridotite,254,-3,84.88841583729466
6.7.D,6.7,LUFA 2.2 A,Peridotite,281,-3,84.88841583729466
6.7.D,6.7,LUFA 2.2 A,Peridotite,309,-3,342.15228834466575
6.7.D,6.7,LUFA 2.2 A,Peridotite,351,-3,388.0613295625489
6.7.D,6.7,LUFA 2.2 A,Peridotite,381,-3,97.68904997893978
6.7.D,6.7,LUFA 2.2 A,Peridotite,413,-3,247.59121294903423
6.7.D,6.7,LUFA 2.2 A,Peridotite,449,-3,187.38973428004093
6.7.D,6.7,LUFA 2.2 A,Peridotite,471,-3,31.08725432336482
6.7.D,6.7,LUFA 2.2 A,Peridotite,501,-3,184.9354773452073
6.7.D,6.7,LUFA 2.2 A,Peridotite,528,-3,157.99639845959445
6.7.D,6.7,LUFA 2.2 A,Peridotite,563,-3,464.8458853119923
6.7.D,6.7,LUFA 2.2 A,Peridotite,598,-3,457.9835903483964
6.7.D,6.7,LUFA 2.2 A,Peridotite,626,-3,113.85827202599434
6.7.D,6.7,LUFA 2.2 A,Peridotite,653,-3,69.12342432155965
6.7.D,6.7,LUFA 2.2 A,Peridotite,680,-3,43.88788846500993
6.8.A,6.8,LUFA 2.2 A,Steel Slag,32,-3,117.22685995547263
6.8.A,6.8,LUFA 2.2 A,Steel Slag,59,-3,610.6768691256995
6.8.A,6.8,LUFA 2.2 A,Steel Slag,85,-3,180.60443574222276
6.8.A,6.8,LUFA 2.2 A,Steel Slag,114,-3,760.8196452253445
6.8.A,6.8,LUFA 2.2 A,Steel Slag,149,-3,0.0
6.8.A,6.8,LUFA 2.2 A,Steel Slag,178,-3,656.3934195799989
6.8.A,6.8,LUFA 2.2 A,Steel Slag,203,-3,756.9698304350442
6.8.A,6.8,LUFA 2.2 A,Steel Slag,227,-3,416.3574681990493
6.8.A,6.8,LUFA 2.2 A,Steel Slag,254,-3,451.0058010710632
6.8.A,6.8,LUFA 2.2 A,Steel Slag,281,-3,575.5473092243817
6.8.A,6.8,LUFA 2.2 A,Steel Slag,309,-3,488.54149515614654
6.8.A,6.8,LUFA 2.2 A,Steel Slag,351,-3,976.50551826223
6.8.A,6.8,LUFA 2.2 A,Steel Slag,381,-3,545.7112447198989
6.8.A,6.8,LUFA 2.2 A,Steel Slag,413,-3,606.3458273060954
6.8.A,6.8,LUFA 2.2 A,Steel Slag,449,-3,557.934406642999
6.8.A,6.8,LUFA 2.2 A,Steel Slag,471,-3,229.73769682893072
6.8.A,6.8,LUFA 2.2 A,Steel Slag,501,-3,488.4452498946988
6.8.A,6.8,LUFA 2.2 A,Steel Slag,528,-3,305.48280257536555
6.8.A,6.8,LUFA 2.2 A,Steel Slag,563,-3,570.9082823274565
6.8.A,6.8,LUFA 2.2 A,Steel Slag,598,-3,281.5128931945364
6.8.A,6.8,LUFA 2.2 A,Steel Slag,626,-3,235.3006791744389
6.8.A,6.8,LUFA 2.2 A,Steel Slag,653,-3,248.7413449666045
6.8.A,6.8,LUFA 2.2 A,Steel Slag,680,-3,301.2191327998074
6.8.A,6.8,LUFA 2.2 A,Steel Slag,716,-3,319.0726486551537
6.8.A,6.8,LUFA 2.2 A,Steel Slag,742,-3,138.65107919850774
6.8.A,6.8,LUFA 2.2 A,Steel Slag,771,-3,250.1754009266502
6.8.A,6.8,LUFA 2.2 A,Steel Slag,798,-3,247.58158830254527
6.8.A,6.8,LUFA 2.2 A,Steel Slag,833,-3,266.3494354654311
6.8.A,6.8,LUFA 2.2 A,Steel Slag,868,-3,615.3688308562489
6.8.A,6.8,LUFA 2.2 A,Steel Slag,899,-3,408.5423440640231
6.8.A,6.8,LUFA 2.2 A,Steel Slag,926,-3,220.38264692219747
6.8.A,6.8,LUFA 2.2 A,Steel Slag,955,-3,433.970370780432
6.8.A,6.8,LUFA 2.2 A,Steel Slag,987,-3,345.8240491004272
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1016,-3,
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1043,-3,
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1079,-3,
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1107,-3,
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1142,-3,
6.8.A,6.8,LUFA 2.2 A,Steel Slag,1171,-3,
6.8.B,6.8,LUFA 2.2 A,Steel Slag,32,-3,128.9687950418196
6.8.B,6.8,LUFA 2.2 A,Steel Slag,59,-3,475.9814744569469
6.8.B,6.8,LUFA 2.2 A,Steel Slag,85,-3,207.88999795414887
6.8.B,6.8,LUFA 2.2 A,Steel Slag,114,-3,1317.0216352367772
6.8.B,6.8,LUFA 2.2 A,Steel Slag,149,-3,0.0
6.8.B,6.8,LUFA 2.2 A,Steel Slag,178,-3,924.918000120344
6.8.B,6.8,LUFA 2.2 A,Steel Slag,203,-3,972.8000714844454
6.8.B,6.8,LUFA 2.2 A,Steel Slag,227,-3,618.4727439677478
6.8.B,6.8,LUFA 2.2 A,Steel Slag,254,-3,599.0311792526626
6.8.B,6.8,LUFA 2.2 A,Steel Slag,281,-3,746.3828398820626
6.8.B,6.8,LUFA 2.2 A,Steel Slag,309,-3,774.3902424935314
6.8.B,6.8,LUFA 2.2 A,Steel Slag,351,-3,1089.11260051748
6.8.B,6.8,LUFA 2.2 A,Steel Slag,381,-3,439.1676206751308
6.8.B,6.8,LUFA 2.2 A,Steel Slag,413,-3,534.1618003489981
6.8.B,6.8,LUFA 2.2 A,Steel Slag,449,-3,394.0285425115831
6.8.B,6.8,LUFA 2.2 A,Steel Slag,471,-3,200.7678406161622
6.8.B,6.8,LUFA 2.2 A,Steel Slag,501,-3,557.934406642999
6.8.B,6.8,LUFA 2.2 A,Steel Slag,528,-3,196.1191892893676
6.8.B,6.8,LUFA 2.2 A,Steel Slag,563,-3,299.3471602382815
6.8.B,6.8,LUFA 2.2 A,Steel Slag,598,-3,399.37978506528674
6.8.B,6.8,LUFA 2.2 A,Steel Slag,626,-3,230.85414311330405
6.8.B,6.8,LUFA 2.2 A,Steel Slag,653,-3,265.3003607918647
6.8.B,6.8,LUFA 2.2 A,Steel Slag,680,-3,313.0669376015404
6.8.C,6.8,LUFA 2.2 A,Steel Slag,32,-3,210.05551876767555
6.8.C,6.8,LUFA 2.2 A,Steel Slag,59,-3,1299.793714182562
6.8.C,6.8,LUFA 2.2 A,Steel Slag,85,-3,80.84611030747939
6.8.C,6.8,LUFA 2.2 A,Steel Slag,114,-3,917.940210843011
6.8.C,6.8,LUFA 2.2 A,Steel Slag,149,-3,0.0
6.8.C,6.8,LUFA 2.2 A,Steel Slag,178,-3,598.0687256754317
6.8.C,6.8,LUFA 2.2 A,Steel Slag,203,-3,719.530381852097
6.8.C,6.8,LUFA 2.2 A,Steel Slag,227,-3,739.1644370900776
6.8.C,6.8,LUFA 2.2 A,Steel Slag,254,-3,571.6974944340815
6.8.C,6.8,LUFA 2.2 A,Steel Slag,281,-3,575.162327697214
6.8.C,6.8,LUFA 2.2 A,Steel Slag,309,-3,704.9492082556111
6.8.C,6.8,LUFA 2.2 A,Steel Slag,351,-3,1311.6800173295626
6.8.C,6.8,LUFA 2.2 A,Steel Slag,381,-3,486.2797289848968
6.8.C,6.8,LUFA 2.2 A,Steel Slag,413,-3,662.6493685540646
6.8.C,6.8,LUFA 2.2 A,Steel Slag,449,-3,373.4801559660629
6.8.C,6.8,LUFA 2.2 A,Steel Slag,471,-3,239.0734976593056
6.8.C,6.8,LUFA 2.2 A,Steel Slag,501,-3,772.2728443347975
6.8.C,6.8,LUFA 2.2 A,Steel Slag,528,-3,271.80173560382696
6.8.C,6.8,LUFA 2.2 A,Steel Slag,563,-3,563.6513814308923
6.8.C,6.8,LUFA 2.2 A,Steel Slag,598,-3,439.2975519586016
6.8.C,6.8,LUFA 2.2 A,Steel Slag,626,-3,321.1322996570191
6.8.C,6.8,LUFA 2.2 A,Steel Slag,653,-3,284.53981009687703
6.8.C,6.8,LUFA 2.2 A,Steel Slag,680,-3,330.31410794873335
6.8.C,6.8,LUFA 2.2 A,Steel Slag,716,-3,464.0903592273903
6.8.C,6.8,LUFA 2.2 A,Steel Slag,742,-3,185.94605374571273
6.8.C,6.8,LUFA 2.2 A,Steel Slag,771,-3,327.58073939466874
6.8.C,6.8,LUFA 2.2 A,Steel Slag,798,-3,226.4461051808171
6.8.C,6.8,LUFA 2.2 A,Steel Slag,833,-3,362.8161692039232
6.8.C,6.8,LUFA 2.2 A,Steel Slag,868,-3,559.5705777724291
6.8.C,6.8,LUFA 2.2 A,Steel Slag,899,-3,386.28079017991456
6.8.C,6.8,LUFA 2.2 A,Steel Slag,926,-3,173.72289182261267
6.8.C,6.8,LUFA 2.2 A,Steel Slag,955,-3,343.59596895120046
6.8.C,6.8,LUFA 2.2 A,Steel Slag,987,-3,309.91008965641737
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1016,-3,
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1043,-3,
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1079,-3,
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1107,-3,
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1142,-3,
6.8.C,6.8,LUFA 2.2 A,Steel Slag,1171,-3,
6.8.D,6.8,LUFA 2.2 A,Steel Slag,32,-3,159.76731324387748
6.8.D,6.8,LUFA 2.2 A,Steel Slag,59,-3,1428.1850368854923
6.8.D,6.8,LUFA 2.2 A,Steel Slag,85,-3,197.30300732896083
6.8.D,6.8,LUFA 2.2 A,Steel Slag,114,-3,799.0771796136952
6.8.D,6.8,LUFA 2.2 A,Steel Slag,149,-3,0.0
6.8.D,6.8,LUFA 2.2 A,Steel Slag,178,-3,682.2834237920453
6.8.D,6.8,LUFA 2.2 A,Steel Slag,203,-3,839.0190078825441
6.8.D,6.8,LUFA 2.2 A,Steel Slag,227,-3,764.4769692520609
6.8.D,6.8,LUFA 2.2 A,Steel Slag,254,-3,556.2982352728804
6.8.D,6.8,LUFA 2.2 A,Steel Slag,281,-3,777.4219714784284
6.8.D,6.8,LUFA 2.2 A,Steel Slag,309,-3,1193.4425809013778
6.8.D,6.8,LUFA 2.2 A,Steel Slag,351,-3,1538.001003429809
6.8.D,6.8,LUFA 2.2 A,Steel Slag,381,-3,517.3188605812625
6.8.D,6.8,LUFA 2.2 A,Steel Slag,413,-3,667.5578824237318
6.8.D,6.8,LUFA 2.2 A,Steel Slag,449,-3,800.8577189963296
6.8.D,6.8,LUFA 2.2 A,Steel Slag,471,-3,446.0491646910163
6.8.D,6.8,LUFA 2.2 A,Steel Slag,501,-3,770.3479369396474
6.8.D,6.8,LUFA 2.2 A,Steel Slag,528,-3,201.20575704916055
6.8.D,6.8,LUFA 2.2 A,Steel Slag,563,-3,321.5365302364763
6.8.D,6.8,LUFA 2.2 A,Steel Slag,598,-3,426.5787262771526
6.8.D,6.8,LUFA 2.2 A,Steel Slag,626,-3,352.08481039773756
6.8.D,6.8,LUFA 2.2 A,Steel Slag,653,-3,300.6994076659245
6.8.D,6.8,LUFA 2.2 A,Steel Slag,680,-3,376.2375858956616
7.0.A,7.0,LUFA 2.1,Control,35,-3,
7.0.A,7.0,LUFA 2.1,Control,57,-3,
7.0.A,7.0,LUFA 2.1,Control,85,-3,27.429930296648415
7.0.A,7.0,LUFA 2.1,Control,113,-3,86.23585101389975
7.0.A,7.0,LUFA 2.1,Control,148,-3,73.14648077501656
7.0.A,7.0,LUFA 2.1,Control,176,-3,117.90057755580962
7.0.A,7.0,LUFA 2.1,Control,199,-3,76.22633258318791
7.0.A,7.0,LUFA 2.1,Control,227,-3,8.950819356158613
7.0.A,7.0,LUFA 2.1,Control,254,-3,7.795874923882303
7.0.A,7.0,LUFA 2.1,Control,280,-3,25.023796040676334
7.0.A,7.0,LUFA 2.1,Control,308,-3,34.35959689512004
7.0.A,7.0,LUFA 2.1,Control,357,-3,146.10047078644925
7.0.A,7.0,LUFA 2.1,Control,381,-3,31.76097189963295
7.0.A,7.0,LUFA 2.1,Control,413,-3,27.911157133401527
7.0.A,7.0,LUFA 2.1,Control,442,-3,84.21469826102653
7.0.A,7.0,LUFA 2.1,Control,470,-3,17.805393344966603
7.0.A,7.0,LUFA 2.1,Control,501,-3,45.47593705999157
7.0.A,7.0,LUFA 2.1,Control,528,-3,21.58302409531259
7.0.A,7.0,LUFA 2.1,Control,563,-3,46.688628702087975
7.0.A,7.0,LUFA 2.1,Control,598,-3,43.502906985979905
7.0.A,7.0,LUFA 2.1,Control,623,-3,29.54732840724472
7.0.A,7.0,LUFA 2.1,Control,652,-3,31.005445766893317
7.0.A,7.0,LUFA 2.1,Control,680,-3,63.37757576268127
7.0.A,7.0,LUFA 2.1,Control,716,-3,90.20116023828147
7.0.A,7.0,LUFA 2.1,Control,742,-3,48.70015693936035
7.0.A,7.0,LUFA 2.1,Control,771,-3,30.90438813406342
7.0.A,7.0,LUFA 2.1,Control,798,-3,35.99576816896323
7.0.A,7.0,LUFA 2.1,Control,833,-3,44.42205025573139
7.0.A,7.0,LUFA 2.1,Control,868,-3,46.3565821770263
7.0.A,7.0,LUFA 2.1,Control,899,-3,60.047485985919735
7.0.A,7.0,LUFA 2.1,Control,926,-3,16.717820670317106
7.0.A,7.0,LUFA 2.1,Control,955,-3,51.10629116071965
7.0.A,7.0,LUFA 2.1,Control,987,-3,82.75658090137794
7.0.A,7.0,LUFA 2.1,Control,1016,-3,
7.0.A,7.0,LUFA 2.1,Control,1043,-3,
7.0.A,7.0,LUFA 2.1,Control,1079,-3,
7.0.A,7.0,LUFA 2.1,Control,1107,-3,
7.0.A,7.0,LUFA 2.1,Control,1142,-3,
7.0.A,7.0,LUFA 2.1,Control,1171,-3,
7.0.B,7.0,LUFA 2.1,Control,35,-3,
7.0.B,7.0,LUFA 2.1,Control,57,-3,
7.0.B,7.0,LUFA 2.1,Control,85,-3,31.18349970515675
7.0.B,7.0,LUFA 2.1,Control,113,-3,90.95187411998316
7.0.B,7.0,LUFA 2.1,Control,148,-3,58.757798038389794
7.0.B,7.0,LUFA 2.1,Control,176,-3,80.79798762861785
7.0.B,7.0,LUFA 2.1,Control,199,-3,66.40930491605992
7.0.B,7.0,LUFA 2.1,Control,227,-3,3.176097189963295
7.0.B,7.0,LUFA 2.1,Control,254,-3,5.534108743004994
7.0.B,7.0,LUFA 2.1,Control,280,-3,22.0883122835309
7.0.B,7.0,LUFA 2.1,Control,308,-3,51.97249950057164
7.0.B,7.0,LUFA 2.1,Control,357,-3,192.49073886515436
7.0.B,7.0,LUFA 2.1,Control,381,-3,73.00211271436308
7.0.B,7.0,LUFA 2.1,Control,413,-3,2.7911157133401527
7.0.B,7.0,LUFA 2.1,Control,442,-3,37.53569408508334
7.0.B,7.0,LUFA 2.1,Control,470,-3,25.023796040676334
7.0.B,7.0,LUFA 2.1,Control,501,-3,30.31729136530477
7.0.B,7.0,LUFA 2.1,Control,528,-3,11.66493877369276
7.0.B,7.0,LUFA 2.1,Control,563,-3,34.93706911366508
7.0.B,7.0,LUFA 2.1,Control,598,-3,48.70015692881641
7.0.B,7.0,LUFA 2.1,Control,623,-3,33.92168046212167
7.0.B,7.0,LUFA 2.1,Control,652,-3,25.51464742764306
7.0.B,7.0,LUFA 2.1,Control,680,-3,20.865996091220893
7.0.B,7.0,LUFA 2.1,Control,716,-3,6.188577254949155
7.0.B,7.0,LUFA 2.1,Control,742,-3,4.059148455786966
7.0.B,7.0,LUFA 2.1,Control,771,-3,10.923849430170288
7.0.B,7.0,LUFA 2.1,Control,798,-3,21.60708543715025
7.0.B,7.0,LUFA 2.1,Control,833,-3,15.389634572477284
7.0.B,7.0,LUFA 2.1,Control,868,-3,28.373134917865094
7.0.B,7.0,LUFA 2.1,Control,899,-3,13.974827640652263
7.0.B,7.0,LUFA 2.1,Control,926,-3,9.528291572296768
7.0.B,7.0,LUFA 2.1,Control,955,-3,19.345319253866055
7.0.B,7.0,LUFA 2.1,Control,987,-3,14.484928099163607
7.0.B,7.0,LUFA 2.1,Control,1016,-3,
7.0.B,7.0,LUFA 2.1,Control,1043,-3,
7.0.B,7.0,LUFA 2.1,Control,1079,-3,
7.0.B,7.0,LUFA 2.1,Control,1107,-3,
7.0.B,7.0,LUFA 2.1,Control,1142,-3,
7.0.B,7.0,LUFA 2.1,Control,1171,-3,
7.0.C,7.0,LUFA 2.1,Control,35,-3,
7.0.C,7.0,LUFA 2.1,Control,57,-3,
7.0.C,7.0,LUFA 2.1,Control,85,-3,27.285562235994945
7.0.C,7.0,LUFA 2.1,Control,113,-3,93.8392351886395
7.0.C,7.0,LUFA 2.1,Control,148,-3,65.44685121848487
7.0.C,7.0,LUFA 2.1,Control,176,-3,101.63511012696311
7.0.C,7.0,LUFA 2.1,Control,199,-3,93.8392351886395
7.0.C,7.0,LUFA 2.1,Control,227,-3,11.790057755580962
7.0.C,7.0,LUFA 2.1,Control,254,-3,0.0
7.0.C,7.0,LUFA 2.1,Control,280,-3,14.244314675973282
7.0.C,7.0,LUFA 2.1,Control,308,-3,63.81067992057284
7.0.C,7.0,LUFA 2.1,Control,357,-3,165.06080856850593
7.0.C,7.0,LUFA 2.1,Control,381,-3,41.38550885131476
7.0.C,7.0,LUFA 2.1,Control,413,-3,64.19566139960286
7.0.C,7.0,LUFA 2.1,Control,442,-3,52.54997171911667
7.0.C,7.0,LUFA 2.1,Control,470,-3,37.53569408508334
7.0.C,7.0,LUFA 2.1,Control,501,-3,55.5817008484265
7.0.C,7.0,LUFA 2.1,Control,528,-3,32.2373864853481
7.0.C,7.0,LUFA 2.1,Control,563,-3,133.51157646067753
7.0.C,7.0,LUFA 2.1,Control,598,-3,65.74521186593658
7.0.C,7.0,LUFA 2.1,Control,623,-3,40.928343341958
7.0.C,7.0,LUFA 2.1,Control,652,-3,140.62892154762622
7.0.C,7.0,LUFA 2.1,Control,680,-3,50.6539379264697
7.0.D,7.0,LUFA 2.1,Control,35,-3,
7.0.D,7.0,LUFA 2.1,Control,57,-3,
7.0.D,7.0,LUFA 2.1,Control,85,-3,53.99365225344485
7.0.D,7.0,LUFA 2.1,Control,113,-3,116.16816090017448
7.0.D,7.0,LUFA 2.1,Control,148,-3,73.91644370900777
7.0.D,7.0,LUFA 2.1,Control,176,-3,67.51612665021962
7.0.D,7.0,LUFA 2.1,Control,199,-3,64.4843975209098
7.0.D,7.0,LUFA 2.1,Control,227,-3,7.314648077501654
7.0.D,7.0,LUFA 2.1,Control,254,-3,1.3955578566700764
7.0.D,7.0,LUFA 2.1,Control,280,-3,16.45795817317528
7.0.D,7.0,LUFA 2.1,Control,308,-3,115.3500752632529
7.0.D,7.0,LUFA 2.1,Control,357,-3,83.15599918165954
7.0.D,7.0,LUFA 2.1,Control,381,-3,33.92649271315963
7.0.D,7.0,LUFA 2.1,Control,413,-3,2.7911157133401527
7.0.D,7.0,LUFA 2.1,Control,442,-3,21.895821544015885
7.0.D,7.0,LUFA 2.1,Control,470,-3,15.303013738492089
7.0.D,7.0,LUFA 2.1,Control,501,-3,19.8746687863289
7.0.D,7.0,LUFA 2.1,Control,528,-3,3.4359596895120044
7.0.D,7.0,LUFA 2.1,Control,563,-3,24.79280717251339
7.0.D,7.0,LUFA 2.1,Control,598,-3,24.484821974848067
7.0.D,7.0,LUFA 2.1,Control,623,-3,7.391644370900776
7.0.D,7.0,LUFA 2.1,Control,652,-3,25.726387243516456
7.0.D,7.0,LUFA 2.1,Control,680,-3,20.865996091220893
7.2.A,7.2,LUFA 2.1,Basanite,35,-3,
7.2.A,7.2,LUFA 2.1,Basanite,57,-3,
7.2.A,7.2,LUFA 2.1,Basanite,85,-3,
7.2.A,7.2,LUFA 2.1,Basanite,113,-3,
7.2.A,7.2,LUFA 2.1,Basanite,148,-3,17.805393344966603
7.2.A,7.2,LUFA 2.1,Basanite,176,-3,127.04388764666948
7.2.A,7.2,LUFA 2.1,Basanite,199,-3,91.62559169625128
7.2.A,7.2,LUFA 2.1,Basanite,227,-3,26.708090017449905
7.2.A,7.2,LUFA 2.1,Basanite,254,-3,17.32416649617907
7.2.A,7.2,LUFA 2.1,Basanite,280,-3,74.10893444852277
7.2.A,7.2,LUFA 2.1,Basanite,308,-3,47.59333517058788
7.2.A,7.2,LUFA 2.1,Basanite,357,-3,59.43151561465792
7.2.A,7.2,LUFA 2.1,Basanite,381,-3,19.634055363138575
7.2.A,7.2,LUFA 2.1,Basanite,413,-3,36.91009916360792
7.2.A,7.2,LUFA 2.1,Basanite,442,-3,20.211527579276733
7.2.A,7.2,LUFA 2.1,Basanite,470,-3,53.41618003489981
7.2.A,7.2,LUFA 2.1,Basanite,501,-3,2.935483766772971
7.2.A,7.2,LUFA 2.1,Basanite,528,-3,63.290954931102945
7.2.A,7.2,LUFA 2.1,Basanite,563,-3,126.36054552018771
7.2.A,7.2,LUFA 2.1,Basanite,598,-3,81.20221818400626
7.2.A,7.2,LUFA 2.1,Basanite,623,-3,56.0918013117516
7.2.A,7.2,LUFA 2.1,Basanite,652,-3,135.2151195138095
7.2.A,7.2,LUFA 2.1,Basanite,680,-3,65.63934195799987
7.2.B,7.2,LUFA 2.1,Basanite,35,-3,
7.2.B,7.2,LUFA 2.1,Basanite,57,-3,
7.2.B,7.2,LUFA 2.1,Basanite,85,-3,
7.2.B,7.2,LUFA 2.1,Basanite,113,-3,
7.2.B,7.2,LUFA 2.1,Basanite,148,-3,10.394499897707442
7.2.B,7.2,LUFA 2.1,Basanite,176,-3,108.85351282267284
7.2.B,7.2,LUFA 2.1,Basanite,199,-3,93.50237638847102
7.2.B,7.2,LUFA 2.1,Basanite,227,-3,8.084611030747938
7.2.B,7.2,LUFA 2.1,Basanite,254,-3,22.906397922859377
7.2.B,7.2,LUFA 2.1,Basanite,280,-3,58.757798038389794
7.2.B,7.2,LUFA 2.1,Basanite,308,-3,49.66261062639148
7.2.B,7.2,LUFA 2.1,Basanite,357,-3,85.03278389794812
7.2.B,7.2,LUFA 2.1,Basanite,381,-3,25.40877751970636
7.2.B,7.2,LUFA 2.1,Basanite,413,-3,28.873610830976595
7.2.B,7.2,LUFA 2.1,Basanite,442,-3,22.906397922859377
7.2.B,7.2,LUFA 2.1,Basanite,470,-3,33.20465245802996
7.2.B,7.2,LUFA 2.1,Basanite,501,-3,46.9677402731813
7.2.B,7.2,LUFA 2.1,Basanite,528,-3,36.481807280823155
7.2.B,7.2,LUFA 2.1,Basanite,563,-3,64.21491047596125
7.2.B,7.2,LUFA 2.1,Basanite,598,-3,69.53727940309284
7.2.B,7.2,LUFA 2.1,Basanite,623,-3,45.64917872314821
7.2.B,7.2,LUFA 2.1,Basanite,652,-3,49.42199720801492
7.2.B,7.2,LUFA 2.1,Basanite,680,-3,44.35949077561827
7.2.B,7.2,LUFA 2.1,Basanite,716,-3,51.93400134785487
7.2.B,7.2,LUFA 2.1,Basanite,742,-3,24.749496748630087
7.2.B,7.2,LUFA 2.1,Basanite,771,-3,25.466524748781517
7.2.B,7.2,LUFA 2.1,Basanite,798,-3,42.33352574763825
7.2.B,7.2,LUFA 2.1,Basanite,833,-3,70.13881297310307
7.2.B,7.2,LUFA 2.1,Basanite,868,-3,102.51575524399784
7.2.B,7.2,LUFA 2.1,Basanite,899,-3,85.89899221373128
7.2.B,7.2,LUFA 2.1,Basanite,926,-3,42.84843846200132
7.2.B,7.2,LUFA 2.1,Basanite,955,-3,51.43352541067453
7.2.B,7.2,LUFA 2.1,Basanite,987,-3,37.78593202960467
7.2.B,7.2,LUFA 2.1,Basanite,1016,-3,
7.2.B,7.2,LUFA 2.1,Basanite,1043,-3,
7.2.B,7.2,LUFA 2.1,Basanite,1079,-3,
7.2.B,7.2,LUFA 2.1,Basanite,1107,-3,
7.2.B,7.2,LUFA 2.1,Basanite,1142,-3,
7.2.B,7.2,LUFA 2.1,Basanite,1171,-3,
7.2.C,7.2,LUFA 2.1,Basanite,35,-3,
7.2.C,7.2,LUFA 2.1,Basanite,57,-3,
7.2.C,7.2,LUFA 2.1,Basanite,85,-3,
7.2.C,7.2,LUFA 2.1,Basanite,113,-3,
7.2.C,7.2,LUFA 2.1,Basanite,148,-3,44.03225650159456
7.2.C,7.2,LUFA 2.1,Basanite,176,-3,145.13801708887416
7.2.C,7.2,LUFA 2.1,Basanite,199,-3,131.37492927372284
7.2.C,7.2,LUFA 2.1,Basanite,227,-3,28.777365449184668
7.2.C,7.2,LUFA 2.1,Basanite,254,-3,20.211527579276733
7.2.C,7.2,LUFA 2.1,Basanite,280,-3,52.35748097960166
7.2.C,7.2,LUFA 2.1,Basanite,308,-3,72.5690085564715
7.2.C,7.2,LUFA 2.1,Basanite,357,-3,103.75250823755944
7.2.C,7.2,LUFA 2.1,Basanite,381,-3,37.05446722426139
7.2.C,7.2,LUFA 2.1,Basanite,413,-3,73.82019835128467
7.2.C,7.2,LUFA 2.1,Basanite,442,-3,48.12268471027138
7.2.C,7.2,LUFA 2.1,Basanite,470,-3,26.948703435826463
7.2.C,7.2,LUFA 2.1,Basanite,501,-3,68.33421228714121
7.2.C,7.2,LUFA 2.1,Basanite,528,-3,14.080697546182082
7.2.C,7.2,LUFA 2.1,Basanite,563,-3,52.28048467416812
7.2.C,7.2,LUFA 2.1,Basanite,598,-3,70.50454537577471
7.2.C,7.2,LUFA 2.1,Basanite,623,-3,48.77715323424995
7.2.C,7.2,LUFA 2.1,Basanite,652,-3,67.99735351104157
7.2.C,7.2,LUFA 2.1,Basanite,680,-3,166.04732361754617
7.2.C,7.2,LUFA 2.1,Basanite,716,-3,186.10967085865576
7.2.C,7.2,LUFA 2.1,Basanite,742,-3,59.74912533942115
7.2.C,7.2,LUFA 2.1,Basanite,771,-3,82.98275751850291
7.2.C,7.2,LUFA 2.1,Basanite,798,-3,61.029188759853184
7.2.C,7.2,LUFA 2.1,Basanite,833,-3,66.53442387628617
7.2.C,7.2,LUFA 2.1,Basanite,868,-3,98.8921170948914
7.2.C,7.2,LUFA 2.1,Basanite,899,-3,83.32924084481617
7.2.C,7.2,LUFA 2.1,Basanite,926,-3,21.539713677116552
7.2.C,7.2,LUFA 2.1,Basanite,955,-3,48.5076661893014
7.2.C,7.2,LUFA 2.1,Basanite,987,-3,60.53833737288645
7.2.C,7.2,LUFA 2.1,Basanite,1016,-3,
7.2.C,7.2,LUFA 2.1,Basanite,1043,-3,
7.2.C,7.2,LUFA 2.1,Basanite,1079,-3,
7.2.C,7.2,LUFA 2.1,Basanite,1107,-3,
7.2.C,7.2,LUFA 2.1,Basanite,1142,-3,
7.2.C,7.2,LUFA 2.1,Basanite,1171,-3,
7.2.D,7.2,LUFA 2.1,Basanite,35,-3,
7.2.D,7.2,LUFA 2.1,Basanite,57,-3,
7.2.D,7.2,LUFA 2.1,Basanite,85,-3,
7.2.D,7.2,LUFA 2.1,Basanite,113,-3,
7.2.D,7.2,LUFA 2.1,Basanite,148,-3,48.796402310608336
7.2.D,7.2,LUFA 2.1,Basanite,176,-3,152.74140128768278
7.2.D,7.2,LUFA 2.1,Basanite,199,-3,175.84028993320896
7.2.D,7.2,LUFA 2.1,Basanite,227,-3,49.325751826223
7.2.D,7.2,LUFA 2.1,Basanite,254,-3,45.52405973885312
7.2.D,7.2,LUFA 2.1,Basanite,280,-3,42.73294402791985
7.2.D,7.2,LUFA 2.1,Basanite,308,-3,76.99629554124797
7.2.D,7.2,LUFA 2.1,Basanite,357,-3,438.3014123593477
7.2.D,7.2,LUFA 2.1,Basanite,381,-3,98.459012912931
7.2.D,7.2,LUFA 2.1,Basanite,413,-3,85.7546241530778
7.2.D,7.2,LUFA 2.1,Basanite,442,-3,90.4706472591612
7.2.D,7.2,LUFA 2.1,Basanite,470,-3,32.33844411817799
7.2.D,7.2,LUFA 2.1,Basanite,501,-3,88.35324912449606
7.2.D,7.2,LUFA 2.1,Basanite,528,-3,53.31031012696311
7.2.D,7.2,LUFA 2.1,Basanite,563,-3,56.26023069980143
7.2.D,7.2,LUFA 2.1,Basanite,598,-3,50.066841181779886
7.2.D,7.2,LUFA 2.1,Basanite,623,-3,44.234371791323184
7.2.D,7.2,LUFA 2.1,Basanite,652,-3,50.23045829472291
7.2.D,7.2,LUFA 2.1,Basanite,680,-3,69.52765487694806
7.6.A,7.6,LUFA 2.1,Metabasalt,35,-3,
7.6.A,7.6,LUFA 2.1,Metabasalt,57,-3,
7.6.A,7.6,LUFA 2.1,Metabasalt,85,-3,
7.6.A,7.6,LUFA 2.1,Metabasalt,113,-3,
7.6.A,7.6,LUFA 2.1,Metabasalt,148,-3,44.75409678079306
7.6.A,7.6,LUFA 2.1,Metabasalt,176,-3,386.42515819243033
7.6.A,7.6,LUFA 2.1,Metabasalt,199,-3,322.22949684096517
7.6.A,7.6,LUFA 2.1,Metabasalt,227,-3,164.5795817317528
7.6.A,7.6,LUFA 2.1,Metabasalt,254,-3,103.2712814008063
7.6.A,7.6,LUFA 2.1,Metabasalt,280,-3,189.98835925145917
7.6.A,7.6,LUFA 2.1,Metabasalt,308,-3,166.023262266081
7.6.A,7.6,LUFA 2.1,Metabasalt,357,-3,342.0079203321499
7.6.A,7.6,LUFA 2.1,Metabasalt,381,-3,92.1068185330044
7.6.A,7.6,LUFA 2.1,Metabasalt,413,-3,190.80644488838075
7.6.A,7.6,LUFA 2.1,Metabasalt,442,-3,154.76255404055598
7.6.A,7.6,LUFA 2.1,Metabasalt,470,-3,149.7096721343041
7.6.A,7.6,LUFA 2.1,Metabasalt,501,-3,284.549434743366
7.6.A,7.6,LUFA 2.1,Metabasalt,528,-3,131.20168761056624
7.6.A,7.6,LUFA 2.1,Metabasalt,563,-3,267.9471084902822
7.6.A,7.6,LUFA 2.1,Metabasalt,598,-3,207.0334141645105
7.6.A,7.6,LUFA 2.1,Metabasalt,623,-3,122.7128460196161
7.6.A,7.6,LUFA 2.1,Metabasalt,652,-3,156.03299291172755
7.6.A,7.6,LUFA 2.1,Metabasalt,680,-3,92.66504167519103
7.6.B,7.6,LUFA 2.1,Metabasalt,35,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,57,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,85,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,113,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,148,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,176,-3,168.4293964979842
7.6.B,7.6,LUFA 2.1,Metabasalt,199,-3,265.63721956796434
7.6.B,7.6,LUFA 2.1,Metabasalt,227,-3,129.69063529694927
7.6.B,7.6,LUFA 2.1,Metabasalt,254,-3,82.57852696311451
7.6.B,7.6,LUFA 2.1,Metabasalt,280,-3,138.59333196943257
7.6.B,7.6,LUFA 2.1,Metabasalt,308,-3,136.28344311932125
7.6.B,7.6,LUFA 2.1,Metabasalt,357,-3,244.84821974848063
7.6.B,7.6,LUFA 2.1,Metabasalt,381,-3,85.4658880558397
7.6.B,7.6,LUFA 2.1,Metabasalt,413,-3,240.7096689331488
7.6.B,7.6,LUFA 2.1,Metabasalt,442,-3,139.55578566700763
7.6.B,7.6,LUFA 2.1,Metabasalt,470,-3,137.4383875323425
7.6.B,7.6,LUFA 2.1,Metabasalt,501,-3,166.023262266081
7.6.B,7.6,LUFA 2.1,Metabasalt,528,-3,114.4598055960046
7.6.B,7.6,LUFA 2.1,Metabasalt,563,-3,232.22082736626749
7.6.B,7.6,LUFA 2.1,Metabasalt,598,-3,199.61289620314096
7.6.B,7.6,LUFA 2.1,Metabasalt,623,-3,98.6611282026596
7.6.B,7.6,LUFA 2.1,Metabasalt,652,-3,126.77921287682771
7.6.B,7.6,LUFA 2.1,Metabasalt,680,-3,111.74087391539804
7.6.B,7.6,LUFA 2.1,Metabasalt,716,-3,152.54891054816775
7.6.B,7.6,LUFA 2.1,Metabasalt,742,-3,48.36329813914035
7.6.B,7.6,LUFA 2.1,Metabasalt,771,-3,122.47223260123954
7.6.B,7.6,LUFA 2.1,Metabasalt,798,-3,121.23547960767796
7.6.B,7.6,LUFA 2.1,Metabasalt,833,-3,147.60189854985256
7.6.B,7.6,LUFA 2.1,Metabasalt,868,-3,223.17376263313076
7.6.B,7.6,LUFA 2.1,Metabasalt,899,-3,179.97884082074734
7.6.B,7.6,LUFA 2.1,Metabasalt,926,-3,81.61607326553944
7.6.B,7.6,LUFA 2.1,Metabasalt,955,-3,117.65996413743306
7.6.B,7.6,LUFA 2.1,Metabasalt,987,-3,126.062184872736
7.6.B,7.6,LUFA 2.1,Metabasalt,1016,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,1043,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,1079,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,1107,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,1142,-3,
7.6.B,7.6,LUFA 2.1,Metabasalt,1171,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,35,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,57,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,85,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,113,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,148,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,176,-3,133.39608202659605
7.6.C,7.6,LUFA 2.1,Metabasalt,199,-3,332.62399663036285
7.6.C,7.6,LUFA 2.1,Metabasalt,227,-3,109.6234757807329
7.6.C,7.6,LUFA 2.1,Metabasalt,254,-3,65.8318326975149
7.6.C,7.6,LUFA 2.1,Metabasalt,280,-3,106.30301053011614
7.6.C,7.6,LUFA 2.1,Metabasalt,308,-3,156.39872531439917
7.6.C,7.6,LUFA 2.1,Metabasalt,357,-3,214.8196645526205
7.6.C,7.6,LUFA 2.1,Metabasalt,381,-3,53.70491613213792
7.6.C,7.6,LUFA 2.1,Metabasalt,413,-3,73.3870941933931
7.6.C,7.6,LUFA 2.1,Metabasalt,442,-3,105.86990637222456
7.6.C,7.6,LUFA 2.1,Metabasalt,470,-3,108.56477672543474
7.6.C,7.6,LUFA 2.1,Metabasalt,501,-3,158.5161234490643
7.6.C,7.6,LUFA 2.1,Metabasalt,528,-3,144.42580135988928
7.6.C,7.6,LUFA 2.1,Metabasalt,563,-3,
7.6.C,7.6,LUFA 2.1,Metabasalt,598,-3,460.44747192971903
7.6.C,7.6,LUFA 2.1,Metabasalt,623,-3,235.8974004693423
7.6.C,7.6,LUFA 2.1,Metabasalt,652,-3,339.11093471328
7.6.C,7.6,LUFA 2.1,Metabasalt,680,-3,388.27788170166673
7.6.D,7.6,LUFA 2.1,Metabasalt,35,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,57,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,85,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,113,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,148,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,176,-3,192.7313522835309
7.6.D,7.6,LUFA 2.1,Metabasalt,199,-3,275.261756543715
7.6.D,7.6,LUFA 2.1,Metabasalt,227,-3,125.45583905168782
7.6.D,7.6,LUFA 2.1,Metabasalt,254,-3,87.8238996088814
7.6.D,7.6,LUFA 2.1,Metabasalt,280,-3,192.4426161622239
7.6.D,7.6,LUFA 2.1,Metabasalt,308,-3,210.34425488898248
7.6.D,7.6,LUFA 2.1,Metabasalt,357,-3,364.9624408207473
7.6.D,7.6,LUFA 2.1,Metabasalt,381,-3,122.32786454058608
7.6.D,7.6,LUFA 2.1,Metabasalt,413,-3,266.07032384620015
7.6.D,7.6,LUFA 2.1,Metabasalt,442,-3,192.49073886515436
7.6.D,7.6,LUFA 2.1,Metabasalt,470,-3,198.939178602804
7.6.D,7.6,LUFA 2.1,Metabasalt,501,-3,221.31622699320056
7.6.D,7.6,LUFA 2.1,Metabasalt,528,-3,190.04129420542753
7.6.D,7.6,LUFA 2.1,Metabasalt,563,-3,294.25096792827486
7.6.D,7.6,LUFA 2.1,Metabasalt,598,-3,313.39417197183946
7.6.D,7.6,LUFA 2.1,Metabasalt,623,-3,138.4778375353511
7.6.D,7.6,LUFA 2.1,Metabasalt,652,-3,161.4034845417895
7.6.D,7.6,LUFA 2.1,Metabasalt,680,-3,199.776513316084
7.6.D,7.6,LUFA 2.1,Metabasalt,716,-3,221.6771471448342
7.6.D,7.6,LUFA 2.1,Metabasalt,742,-3,86.39224973457466
7.6.D,7.6,LUFA 2.1,Metabasalt,771,-3,184.7333620795475
7.6.D,7.6,LUFA 2.1,Metabasalt,798,-3,156.33135355917923
7.6.D,7.6,LUFA 2.1,Metabasalt,833,-3,174.74309272519403
7.6.D,7.6,LUFA 2.1,Metabasalt,868,-3,248.3323023045911
7.6.D,7.6,LUFA 2.1,Metabasalt,899,-3,220.88312283530897
7.6.D,7.6,LUFA 2.1,Metabasalt,926,-3,105.59079480113122
7.6.D,7.6,LUFA 2.1,Metabasalt,955,-3,207.26440305674228
7.6.D,7.6,LUFA 2.1,Metabasalt,987,-3,200.8159633190926
7.6.D,7.6,LUFA 2.1,Metabasalt,1016,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,1043,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,1079,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,1107,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,1142,-3,
7.6.D,7.6,LUFA 2.1,Metabasalt,1171,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,35,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,57,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,85,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,113,-3,94.89793424393766
7.7.A,7.7,LUFA 2.1,Peridotite,148,-3,80.84611030747939
7.7.A,7.7,LUFA 2.1,Peridotite,176,-3,134.74351720320115
7.7.A,7.7,LUFA 2.1,Peridotite,199,-3,74.78265204885973
7.7.A,7.7,LUFA 2.1,Peridotite,227,-3,6.737175858956616
7.7.A,7.7,LUFA 2.1,Peridotite,254,-3,22.0883122835309
7.7.A,7.7,LUFA 2.1,Peridotite,280,-3,72.42464049581804
7.7.A,7.7,LUFA 2.1,Peridotite,308,-3,54.57112447198989
7.7.A,7.7,LUFA 2.1,Peridotite,357,-3,103.94499897707443
7.7.A,7.7,LUFA 2.1,Peridotite,381,-3,45.0428329021
7.7.A,7.7,LUFA 2.1,Peridotite,413,-3,8.806451302725796
7.7.A,7.7,LUFA 2.1,Peridotite,442,-3,57.554730922438175
7.7.A,7.7,LUFA 2.1,Peridotite,470,-3,68.81543914796318
7.7.A,7.7,LUFA 2.1,Peridotite,501,-3,63.23320772609663
7.7.A,7.7,LUFA 2.1,Peridotite,528,-3,40.423055153739696
7.7.A,7.7,LUFA 2.1,Peridotite,563,-3,87.58328619050484
7.7.A,7.7,LUFA 2.1,Peridotite,598,-3,78.08386822311812
7.7.A,7.7,LUFA 2.1,Peridotite,623,-3,43.74352040435646
7.7.A,7.7,LUFA 2.1,Peridotite,652,-3,47.55483704193995
7.7.A,7.7,LUFA 2.1,Peridotite,680,-3,40.827285709128105
7.7.A,7.7,LUFA 2.1,Peridotite,716,-3,20.904494239123892
7.7.A,7.7,LUFA 2.1,Peridotite,742,-3,2.945108304305547
7.7.A,7.7,LUFA 2.1,Peridotite,771,-3,30.89476358384981
7.7.A,7.7,LUFA 2.1,Peridotite,798,-3,21.75626575846922
7.7.A,7.7,LUFA 2.1,Peridotite,833,-3,33.28164873939467
7.7.A,7.7,LUFA 2.1,Peridotite,868,-3,56.30354110355617
7.7.A,7.7,LUFA 2.1,Peridotite,899,-3,69.13304887177327
7.7.A,7.7,LUFA 2.1,Peridotite,926,-3,38.01692092183645
7.7.A,7.7,LUFA 2.1,Peridotite,955,-3,106.06239711173956
7.7.A,7.7,LUFA 2.1,Peridotite,987,-3,107.72744201215475
7.7.A,7.7,LUFA 2.1,Peridotite,1016,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,1043,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,1079,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,1107,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,1142,-3,
7.7.A,7.7,LUFA 2.1,Peridotite,1171,-3,
7.7.B,7.7,LUFA 2.1,Peridotite,35,-3,
7.7.B,7.7,LUFA 2.1,Peridotite,57,-3,
7.7.B,7.7,LUFA 2.1,Peridotite,85,-3,
7.7.B,7.7,LUFA 2.1,Peridotite,113,-3,75.07138814609785
7.7.B,7.7,LUFA 2.1,Peridotite,148,-3,95.28291572296769
7.7.B,7.7,LUFA 2.1,Peridotite,176,-3,175.4071857753174
7.7.B,7.7,LUFA 2.1,Peridotite,199,-3,112.89581832841928
7.7.B,7.7,LUFA 2.1,Peridotite,227,-3,36.958221866538295
7.7.B,7.7,LUFA 2.1,Peridotite,254,-3,26.467476599073347
7.7.B,7.7,LUFA 2.1,Peridotite,280,-3,60.34584663337144
7.7.B,7.7,LUFA 2.1,Peridotite,308,-3,103.46377214032132
7.7.B,7.7,LUFA 2.1,Peridotite,357,-3,495.1824256573801
7.7.B,7.7,LUFA 2.1,Peridotite,381,-3,126.46641542812444
7.7.B,7.7,LUFA 2.1,Peridotite,413,-3,145.5711212708346
7.7.B,7.7,LUFA 2.1,Peridotite,442,-3,48.12268471027138
7.7.B,7.7,LUFA 2.1,Peridotite,470,-3,50.04759210542151
7.7.B,7.7,LUFA 2.1,Peridotite,501,-3,52.934953174077854
7.7.B,7.7,LUFA 2.1,Peridotite,528,-3,22.43479561465792
7.7.B,7.7,LUFA 2.1,Peridotite,563,-3,122.13537380107104
7.7.B,7.7,LUFA 2.1,Peridotite,598,-3,120.30189951260606
7.7.B,7.7,LUFA 2.1,Peridotite,623,-3,68.32458776099645
7.7.B,7.7,LUFA 2.1,Peridotite,652,-3,94.35896017810938
7.7.B,7.7,LUFA 2.1,Peridotite,680,-3,87.37154637463144
7.7.C,7.7,LUFA 2.1,Peridotite,35,-3,
7.7.C,7.7,LUFA 2.1,Peridotite,57,-3,
7.7.C,7.7,LUFA 2.1,Peridotite,85,-3,
7.7.C,7.7,LUFA 2.1,Peridotite,113,-3,126.27392468860944
7.7.C,7.7,LUFA 2.1,Peridotite,148,-3,87.92014496660448
7.7.C,7.7,LUFA 2.1,Peridotite,176,-3,90.85562873819124
7.7.C,7.7,LUFA 2.1,Peridotite,199,-3,83.15599918165954
7.7.C,7.7,LUFA 2.1,Peridotite,227,-3,19.922791472411095
7.7.C,7.7,LUFA 2.1,Peridotite,254,-3,8.469592509777964
7.7.C,7.7,LUFA 2.1,Peridotite,280,-3,63.13696234430471
7.7.C,7.7,LUFA 2.1,Peridotite,308,-3,99.46958931343642
7.7.C,7.7,LUFA 2.1,Peridotite,357,-3,188.6409240748541
7.7.C,7.7,LUFA 2.1,Peridotite,381,-3,55.43733278777303
7.7.C,7.7,LUFA 2.1,Peridotite,413,-3,116.93812385823456
7.7.C,7.7,LUFA 2.1,Peridotite,442,-3,50.52881894217462
7.7.C,7.7,LUFA 2.1,Peridotite,470,-3,41.28926349359167
7.7.C,7.7,LUFA 2.1,Peridotite,501,-3,121.75039232204104
7.7.C,7.7,LUFA 2.1,Peridotite,528,-3,51.74151060833984
7.7.C,7.7,LUFA 2.1,Peridotite,563,-3,423.77317383717434
7.7.C,7.7,LUFA 2.1,Peridotite,598,-3,437.3004604368494
7.7.C,7.7,LUFA 2.1,Peridotite,623,-3,207.50501647511885
7.7.C,7.7,LUFA 2.1,Peridotite,652,-3,129.7580070521692
7.7.C,7.7,LUFA 2.1,Peridotite,680,-3,91.74589840543956
7.7.D,7.7,LUFA 2.1,Peridotite,35,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,57,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,85,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,113,-3,170.16181315361936
7.7.D,7.7,LUFA 2.1,Peridotite,148,-3,111.64462853360612
7.7.D,7.7,LUFA 2.1,Peridotite,176,-3,158.32363270954932
7.7.D,7.7,LUFA 2.1,Peridotite,199,-3,124.63775341476624
7.7.D,7.7,LUFA 2.1,Peridotite,227,-3,49.373874505084544
7.7.D,7.7,LUFA 2.1,Peridotite,254,-3,3.079851822612672
7.7.D,7.7,LUFA 2.1,Peridotite,280,-3,61.16393227029304
7.7.D,7.7,LUFA 2.1,Peridotite,308,-3,79.40242977315121
7.7.D,7.7,LUFA 2.1,Peridotite,357,-3,112.89581832841928
7.7.D,7.7,LUFA 2.1,Peridotite,381,-3,38.73876120103496
7.7.D,7.7,LUFA 2.1,Peridotite,413,-3,51.97249950057164
7.7.D,7.7,LUFA 2.1,Peridotite,442,-3,33.68587929478308
7.7.D,7.7,LUFA 2.1,Peridotite,470,-3,38.73876120103496
7.7.D,7.7,LUFA 2.1,Peridotite,501,-3,65.8318326975149
7.7.D,7.7,LUFA 2.1,Peridotite,528,-3,47.81469953667489
7.7.D,7.7,LUFA 2.1,Peridotite,563,-3,166.31199836331908
7.7.D,7.7,LUFA 2.1,Peridotite,598,-3,75.28794023707805
7.7.D,7.7,LUFA 2.1,Peridotite,623,-3,35.45679410313497
7.7.D,7.7,LUFA 2.1,Peridotite,652,-3,136.74542087971597
7.7.D,7.7,LUFA 2.1,Peridotite,680,-3,74.78265204885973
7.7.D,7.7,LUFA 2.1,Peridotite,716,-3,66.79909864612792
7.7.D,7.7,LUFA 2.1,Peridotite,742,-3,22.74278079533565
7.7.D,7.7,LUFA 2.1,Peridotite,771,-3,27.651294638666585
7.7.D,7.7,LUFA 2.1,Peridotite,798,-3,32.7426746735664
7.7.D,7.7,LUFA 2.1,Peridotite,833,-3,53.897406871652926
7.7.D,7.7,LUFA 2.1,Peridotite,868,-3,86.52458711113786
7.7.D,7.7,LUFA 2.1,Peridotite,899,-3,52.83870781635477
7.7.D,7.7,LUFA 2.1,Peridotite,926,-3,33.82062280522293
7.7.D,7.7,LUFA 2.1,Peridotite,955,-3,103.48302121667967
7.7.D,7.7,LUFA 2.1,Peridotite,987,-3,89.4360095312594
7.7.D,7.7,LUFA 2.1,Peridotite,1016,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,1043,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,1079,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,1107,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,1142,-3,
7.7.D,7.7,LUFA 2.1,Peridotite,1171,-3,
7.8.A,7.8,LUFA 2.1,Steel Slag,35,-3,
7.8.A,7.8,LUFA 2.1,Steel Slag,57,-3,
7.8.A,7.8,LUFA 2.1,Steel Slag,85,-3,
7.8.A,7.8,LUFA 2.1,Steel Slag,113,-3,429.2543476743486
7.8.A,7.8,LUFA 2.1,Steel Slag,148,-3,0.0
7.8.A,7.8,LUFA 2.1,Steel Slag,176,-3,1045.946552259462
7.8.A,7.8,LUFA 2.1,Steel Slag,199,-3,1155.1369237619592
7.8.A,7.8,LUFA 2.1,Steel Slag,227,-3,431.9010953727661
7.8.A,7.8,LUFA 2.1,Steel Slag,254,-3,372.85456116493174
7.8.A,7.8,LUFA 2.1,Steel Slag,280,-3,646.7688826042481
7.8.A,7.8,LUFA 2.1,Steel Slag,308,-3,738.7313330525302
7.8.A,7.8,LUFA 2.1,Steel Slag,357,-3,1114.3770099283952
7.8.A,7.8,LUFA 2.1,Steel Slag,381,-3,528.4833234249954
7.8.A,7.8,LUFA 2.1,Steel Slag,413,-3,677.5192781755821
7.8.A,7.8,LUFA 2.1,Steel Slag,442,-3,519.7249950057163
7.8.A,7.8,LUFA 2.1,Steel Slag,470,-3,606.2977047957157
7.8.A,7.8,LUFA 2.1,Steel Slag,501,-3,954.032224562248
7.8.A,7.8,LUFA 2.1,Steel Slag,528,-3,371.8343601901438
7.8.A,7.8,LUFA 2.1,Steel Slag,563,-3,1028.0737871111378
7.8.A,7.8,LUFA 2.1,Steel Slag,598,-3,880.1639034839642
7.8.A,7.8,LUFA 2.1,Steel Slag,623,-3,430.6787792285937
7.8.A,7.8,LUFA 2.1,Steel Slag,652,-3,555.4801497081653
7.8.A,7.8,LUFA 2.1,Steel Slag,680,-3,524.4987652686683
7.8.B,7.8,LUFA 2.1,Steel Slag,35,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,57,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,85,-3,205.58010910403752
7.8.B,7.8,LUFA 2.1,Steel Slag,113,-3,354.18295950418195
7.8.B,7.8,LUFA 2.1,Steel Slag,148,-3,0.0
7.8.B,7.8,LUFA 2.1,Steel Slag,176,-3,896.0443894337806
7.8.B,7.8,LUFA 2.1,Steel Slag,199,-3,1030.7879066129128
7.8.B,7.8,LUFA 2.1,Steel Slag,227,-3,491.8138378963836
7.8.B,7.8,LUFA 2.1,Steel Slag,254,-3,423.0946439617306
7.8.B,7.8,LUFA 2.1,Steel Slag,280,-3,620.205160599314
7.8.B,7.8,LUFA 2.1,Steel Slag,308,-3,680.9841114387146
7.8.B,7.8,LUFA 2.1,Steel Slag,357,-3,1015.6292609663636
7.8.B,7.8,LUFA 2.1,Steel Slag,381,-3,341.86355231963415
7.8.B,7.8,LUFA 2.1,Steel Slag,413,-3,538.6853326914977
7.8.B,7.8,LUFA 2.1,Steel Slag,442,-3,450.4283290210001
7.8.B,7.8,LUFA 2.1,Steel Slag,470,-3,587.0967535952825
7.8.B,7.8,LUFA 2.1,Steel Slag,501,-3,794.9867515494313
7.8.B,7.8,LUFA 2.1,Steel Slag,528,-3,648.1596281364704
7.8.B,7.8,LUFA 2.1,Steel Slag,563,-3,999.4552266682712
7.8.B,7.8,LUFA 2.1,Steel Slag,598,-3,1094.2857890366447
7.8.B,7.8,LUFA 2.1,Steel Slag,623,-3,460.5340927853661
7.8.B,7.8,LUFA 2.1,Steel Slag,652,-3,522.160002647572
7.8.B,7.8,LUFA 2.1,Steel Slag,680,-3,470.1345683855827
7.8.B,7.8,LUFA 2.1,Steel Slag,716,-3,531.5054282447801
7.8.B,7.8,LUFA 2.1,Steel Slag,742,-3,251.71773310613167
7.8.B,7.8,LUFA 2.1,Steel Slag,771,-3,367.464820506649
7.8.B,7.8,LUFA 2.1,Steel Slag,798,-3,366.4734932306396
7.8.B,7.8,LUFA 2.1,Steel Slag,833,-3,456.23192466454054
7.8.B,7.8,LUFA 2.1,Steel Slag,868,-3,605.6528607016066
7.8.B,7.8,LUFA 2.1,Steel Slag,899,-3,566.9766590047536
7.8.B,7.8,LUFA 2.1,Steel Slag,926,-3,288.0623907575666
7.8.B,7.8,LUFA 2.1,Steel Slag,955,-3,457.1847538359709
7.8.B,7.8,LUFA 2.1,Steel Slag,987,-3,408.6193405138696
7.8.B,7.8,LUFA 2.1,Steel Slag,1016,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,1043,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,1079,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,1107,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,1142,-3,
7.8.B,7.8,LUFA 2.1,Steel Slag,1171,-3,
7.8.C,7.8,LUFA 2.1,Steel Slag,35,-3,
7.8.C,7.8,LUFA 2.1,Steel Slag,57,-3,
7.8.C,7.8,LUFA 2.1,Steel Slag,85,-3,
7.8.C,7.8,LUFA 2.1,Steel Slag,113,-3,169.34372751669775
7.8.C,7.8,LUFA 2.1,Steel Slag,148,-3,0.0
7.8.C,7.8,LUFA 2.1,Steel Slag,176,-3,855.4769661231121
7.8.C,7.8,LUFA 2.1,Steel Slag,199,-3,1053.4055683254107
7.8.C,7.8,LUFA 2.1,Steel Slag,227,-3,563.0354110355618
7.8.C,7.8,LUFA 2.1,Steel Slag,254,-3,325.69433010409773
7.8.C,7.8,LUFA 2.1,Steel Slag,280,-3,526.8471522955653
7.8.C,7.8,LUFA 2.1,Steel Slag,308,-3,697.2977014260786
7.8.C,7.8,LUFA 2.1,Steel Slag,357,-3,1138.197738973464
7.8.C,7.8,LUFA 2.1,Steel Slag,381,-3,439.6007249533666
7.8.C,7.8,LUFA 2.1,Steel Slag,413,-3,619.338952283531
7.8.C,7.8,LUFA 2.1,Steel Slag,442,-3,501.1015158553463
7.8.C,7.8,LUFA 2.1,Steel Slag,470,-3,486.76095577351225
7.8.C,7.8,LUFA 2.1,Steel Slag,501,-3,703.9386319273121
7.8.C,7.8,LUFA 2.1,Steel Slag,528,-3,332.4074447319333
7.8.C,7.8,LUFA 2.1,Steel Slag,563,-3,548.5408585354113
7.8.C,7.8,LUFA 2.1,Steel Slag,598,-3,509.70585185630904
7.8.C,7.8,LUFA 2.1,Steel Slag,623,-3,456.1212424333594
7.8.C,7.8,LUFA 2.1,Steel Slag,652,-3,560.3886635778326
7.8.C,7.8,LUFA 2.1,Steel Slag,680,-3,452.1655577351224
7.8.D,7.8,LUFA 2.1,Steel Slag,35,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,57,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,85,-3,12.632204736747097
7.8.D,7.8,LUFA 2.1,Steel Slag,113,-3,458.36857187556416
7.8.D,7.8,LUFA 2.1,Steel Slag,148,-3,0.0
7.8.D,7.8,LUFA 2.1,Steel Slag,176,-3,751.1951084902821
7.8.D,7.8,LUFA 2.1,Steel Slag,199,-3,988.0549626331308
7.8.D,7.8,LUFA 2.1,Steel Slag,227,-3,293.64462217943316
7.8.D,7.8,LUFA 2.1,Steel Slag,254,-3,414.4806835549672
7.8.D,7.8,LUFA 2.1,Steel Slag,280,-3,484.3548215897467
7.8.D,7.8,LUFA 2.1,Steel Slag,308,-3,565.4415454600156
7.8.D,7.8,LUFA 2.1,Steel Slag,357,-3,831.5599918165955
7.8.D,7.8,LUFA 2.1,Steel Slag,381,-3,323.3844411817799
7.8.D,7.8,LUFA 2.1,Steel Slag,413,-3,459.66788446958304
7.8.D,7.8,LUFA 2.1,Steel Slag,442,-3,424.4420790661292
7.8.D,7.8,LUFA 2.1,Steel Slag,470,-3,407.59913953908176
7.8.D,7.8,LUFA 2.1,Steel Slag,501,-3,482.3817916842168
7.8.D,7.8,LUFA 2.1,Steel Slag,528,-3,348.2253712016367
7.8.D,7.8,LUFA 2.1,Steel Slag,563,-3,675.6424932908117
7.8.D,7.8,LUFA 2.1,Steel Slag,598,-3,563.0931584331187
7.8.D,7.8,LUFA 2.1,Steel Slag,623,-3,291.0459972320837
7.8.D,7.8,LUFA 2.1,Steel Slag,652,-3,375.18369913953904
7.8.D,7.8,LUFA 2.1,Steel Slag,680,-3,388.2923184307118
7.8.D,7.8,LUFA 2.1,Steel Slag,716,-3,483.796598591973
7.8.D,7.8,LUFA 2.1,Steel Slag,742,-3,219.09295897090257
7.8.D,7.8,LUFA 2.1,Steel Slag,771,-3,341.57481605391416
7.8.D,7.8,LUFA 2.1,Steel Slag,798,-3,404.31717239304413
7.8.D,7.8,LUFA 2.1,Steel Slag,833,-3,435.6161665563512
7.8.D,7.8,LUFA 2.1,Steel Slag,868,-3,551.4859669053493
7.8.D,7.8,LUFA 2.1,Steel Slag,899,-3,521.659526806667
7.8.D,7.8,LUFA 2.1,Steel Slag,926,-3,228.23626906552735
7.8.D,7.8,LUFA 2.1,Steel Slag,955,-3,395.80908165352906
7.8.D,7.8,LUFA 2.1,Steel Slag,987,-3,373.9132602442986
7.8.D,7.8,LUFA 2.1,Steel Slag,1016,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,1043,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,1079,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,1107,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,1142,-3,
7.8.D,7.8,LUFA 2.1,Steel Slag,1171,-3,
9.0.A,9.0,Fürth 1,Control,30,-3,337.46032661411635
9.0.A,9.0,Fürth 1,Control,58,-3,324.9243672904507
9.0.A,9.0,Fürth 1,Control,84,-3,249.6123656056321
9.0.A,9.0,Fürth 1,Control,112,-3,915.6784446717612
9.0.A,9.0,Fürth 1,Control,147,-3,0.0
9.0.A,9.0,Fürth 1,Control,175,-3,638.491780732896
9.0.A,9.0,Fürth 1,Control,196,-3,468.1855996148986
9.0.A,9.0,Fürth 1,Control,226,-3,275.9835968469824
9.0.A,9.0,Fürth 1,Control,253,-3,265.63721956796434
9.0.A,9.0,Fürth 1,Control,280,-3,230.26704634454535
9.0.A,9.0,Fürth 1,Control,308,-3,389.5050101690835
9.0.A,9.0,Fürth 1,Control,344,-3,390.6599545098983
9.0.A,9.0,Fürth 1,Control,385,-3,572.6599480113124
9.0.A,9.0,Fürth 1,Control,415,-3,225.5510232625308
9.0.A,9.0,Fürth 1,Control,448,-3,291.76783753535113
9.0.A,9.0,Fürth 1,Control,469,-3,159.38233176484746
9.0.A,9.0,Fürth 1,Control,498,-3,381.9016258499308
9.0.A,9.0,Fürth 1,Control,529,-3,187.43785695890244
9.0.A,9.0,Fürth 1,Control,561,-3,421.9878223065104
9.0.A,9.0,Fürth 1,Control,599,-3,392.8254754197003
9.0.A,9.0,Fürth 1,Control,623,-3,189.8776771058644
9.0.A,9.0,Fürth 1,Control,653,-3,258.60168313376255
9.0.A,9.0,Fürth 1,Control,682,-3,260.46884337204403
9.0.A,9.0,Fürth 1,Control,716,-3,281.34446380648654
9.0.A,9.0,Fürth 1,Control,742,-3,209.56947965581563
9.0.A,9.0,Fürth 1,Control,771,-3,102.03452840724472
9.0.A,9.0,Fürth 1,Control,798,-3,236.9897854022504
9.0.A,9.0,Fürth 1,Control,833,-3,351.295598531801
9.0.A,9.0,Fürth 1,Control,868,-3,520.0714781876165
9.0.A,9.0,Fürth 1,Control,899,-3,439.1387469763524
9.0.A,9.0,Fürth 1,Control,926,-3,303.9813748119622
9.0.A,9.0,Fürth 1,Control,955,-3,393.8360517479993
9.0.A,9.0,Fürth 1,Control,987,-3,352.7344665744028
9.0.A,9.0,Fürth 1,Control,1016,-3,309.23637186352966
9.0.A,9.0,Fürth 1,Control,1043,-3,
9.0.A,9.0,Fürth 1,Control,1079,-3,
9.0.A,9.0,Fürth 1,Control,1107,-3,
9.0.A,9.0,Fürth 1,Control,1142,-3,
9.0.B,9.0,Fürth 1,Control,28,-3,422.1321903844996
9.0.B,9.0,Fürth 1,Control,58,-3,472.9497454720501
9.0.B,9.0,Fürth 1,Control,84,-3,201.15282209519225
9.0.B,9.0,Fürth 1,Control,112,-3,751.4838445153138
9.0.B,9.0,Fürth 1,Control,147,-3,0.0
9.0.B,9.0,Fürth 1,Control,175,-3,987.2849995787952
9.0.B,9.0,Fürth 1,Control,196,-3,662.6493685540646
9.0.B,9.0,Fürth 1,Control,226,-3,460.5340927853661
9.0.B,9.0,Fürth 1,Control,253,-3,306.6377469161803
9.0.B,9.0,Fürth 1,Control,280,-3,415.3468918707503
9.0.B,9.0,Fürth 1,Control,308,-3,481.7080738913291
9.0.B,9.0,Fürth 1,Control,344,-3,392.77735266863226
9.0.B,9.0,Fürth 1,Control,385,-3,444.17237980624583
9.0.B,9.0,Fürth 1,Control,415,-3,380.0248412058488
9.0.B,9.0,Fürth 1,Control,448,-3,298.55313604910043
9.0.B,9.0,Fürth 1,Control,469,-3,187.38973428004093
9.0.B,9.0,Fürth 1,Control,498,-3,382.38285263854624
9.0.B,9.0,Fürth 1,Control,529,-3,285.45414116372825
9.0.B,9.0,Fürth 1,Control,561,-3,545.1000865011351
9.0.B,9.0,Fürth 1,Control,599,-3,426.11674854082673
9.0.B,9.0,Fürth 1,Control,623,-3,198.3857677669253
9.0.B,9.0,Fürth 1,Control,653,-3,244.4632384620013
9.0.B,9.0,Fürth 1,Control,682,-3,244.289996750707
9.0.C,9.0,Fürth 1,Control,28,-3,323.3844411817799
9.0.C,9.0,Fürth 1,Control,58,-3,279.496552861183
9.0.C,9.0,Fürth 1,Control,84,-3,166.023262266081
9.0.C,9.0,Fürth 1,Control,112,-3,727.9999742463444
9.0.C,9.0,Fürth 1,Control,147,-3,0.0
9.0.C,9.0,Fürth 1,Control,175,-3,726.1231896022625
9.0.C,9.0,Fürth 1,Control,196,-3,515.3939531861123
9.0.C,9.0,Fürth 1,Control,226,-3,184.2136370900776
9.0.C,9.0,Fürth 1,Control,253,-3,158.80485954630242
9.0.C,9.0,Fürth 1,Control,280,-3,132.818609808051
9.0.C,9.0,Fürth 1,Control,308,-3,341.1417120163668
9.0.C,9.0,Fürth 1,Control,344,-3,399.4182831698658
9.0.C,9.0,Fürth 1,Control,385,-3,284.64568000481376
9.0.C,9.0,Fürth 1,Control,415,-3,317.6097189963295
9.0.C,9.0,Fürth 1,Control,448,-3,287.2443049521632
9.0.C,9.0,Fürth 1,Control,469,-3,174.87783625970275
9.0.C,9.0,Fürth 1,Control,498,-3,374.20199626933027
9.0.C,9.0,Fürth 1,Control,529,-3,425.5970236476322
9.0.C,9.0,Fürth 1,Control,561,-3,894.3264095158812
9.0.C,9.0,Fürth 1,Control,599,-3,730.4734802334676
9.0.C,9.0,Fürth 1,Control,623,-3,329.6885129596565
9.0.C,9.0,Fürth 1,Control,653,-3,376.0499074553222
9.0.C,9.0,Fürth 1,Control,682,-3,312.87444683795655
9.0.D,9.0,Fürth 1,Control,28,-3,293.3077634033335
9.0.D,9.0,Fürth 1,Control,58,-3,314.9629715386004
9.0.D,9.0,Fürth 1,Control,84,-3,252.6440948312173
9.0.D,9.0,Fürth 1,Control,112,-3,757.6916707383116
9.0.D,9.0,Fürth 1,Control,147,-3,0.0
9.0.D,9.0,Fürth 1,Control,175,-3,848.8841583729467
9.0.D,9.0,Fürth 1,Control,196,-3,441.04440531921296
9.0.D,9.0,Fürth 1,Control,226,-3,286.4262193874481
9.0.D,9.0,Fürth 1,Control,253,-3,236.47487268788737
9.0.D,9.0,Fürth 1,Control,280,-3,211.88418080510257
9.0.D,9.0,Fürth 1,Control,308,-3,362.8450426620133
9.0.D,9.0,Fürth 1,Control,344,-3,388.5906791022324
9.0.D,9.0,Fürth 1,Control,385,-3,313.3749229195499
9.0.D,9.0,Fürth 1,Control,415,-3,235.8011550875504
9.0.D,9.0,Fürth 1,Control,448,-3,282.7207726096636
9.0.D,9.0,Fürth 1,Control,469,-3,181.51876673686743
9.0.D,9.0,Fürth 1,Control,498,-3,337.38814248751424
9.0.D,9.0,Fürth 1,Control,529,-3,256.5757181539202
9.0.D,9.0,Fürth 1,Control,561,-3,541.6978127897925
9.0.D,9.0,Fürth 1,Control,599,-3,401.4586848787532
9.0.D,9.0,Fürth 1,Control,623,-3,192.952716584667
9.0.D,9.0,Fürth 1,Control,653,-3,271.07027065407067
9.0.D,9.0,Fürth 1,Control,682,-3,251.5083993020037
9.0.D,9.0,Fürth 1,Control,716,-3,280.53119056501595
9.0.D,9.0,Fürth 1,Control,742,-3,147.6885193814309
9.0.D,9.0,Fürth 1,Control,771,-3,69.72977014260785
9.0.D,9.0,Fürth 1,Control,798,-3,188.97778287502257
9.0.D,9.0,Fürth 1,Control,833,-3,280.1943317889163
9.0.D,9.0,Fürth 1,Control,868,-3,500.5047947529936
9.0.D,9.0,Fürth 1,Control,899,-3,470.6013584451531
9.0.D,9.0,Fürth 1,Control,926,-3,188.7179203802876
9.0.D,9.0,Fürth 1,Control,955,-3,387.0700022865395
9.0.D,9.0,Fürth 1,Control,987,-3,319.0533996028642
9.0.D,9.0,Fürth 1,Control,1016,-3,319.37582164991875
9.0.D,9.0,Fürth 1,Control,1043,-3,
9.0.D,9.0,Fürth 1,Control,1079,-3,
9.0.D,9.0,Fürth 1,Control,1107,-3,
9.0.D,9.0,Fürth 1,Control,1142,-3,
9.0.D,9.0,Fürth 1,Control,1171,-3,
9.1.A,9.1,Fürth 1,Basanite 20,30,-3,
9.1.A,9.1,Fürth 1,Basanite 20,57,-3,
9.1.A,9.1,Fürth 1,Basanite 20,85,-3,
9.1.A,9.1,Fürth 1,Basanite 20,112,-3,
9.1.A,9.1,Fürth 1,Basanite 20,147,-3,
9.1.A,9.1,Fürth 1,Basanite 20,175,-3,
9.1.A,9.1,Fürth 1,Basanite 20,196,-3,572.6599480113124
9.1.A,9.1,Fürth 1,Basanite 20,226,-3,
9.1.A,9.1,Fürth 1,Basanite 20,253,-3,
9.1.A,9.1,Fürth 1,Basanite 20,280,-3,
9.1.A,9.1,Fürth 1,Basanite 20,308,-3,
9.1.A,9.1,Fürth 1,Basanite 20,344,-3,
9.1.A,9.1,Fürth 1,Basanite 20,381,-3,
9.1.A,9.1,Fürth 1,Basanite 20,415,-3,
9.1.A,9.1,Fürth 1,Basanite 20,448,-3,
9.1.A,9.1,Fürth 1,Basanite 20,469,-3,
9.1.A,9.1,Fürth 1,Basanite 20,499,-3,
9.1.A,9.1,Fürth 1,Basanite 20,529,-3,
9.1.A,9.1,Fürth 1,Basanite 20,561,-3,
9.1.A,9.1,Fürth 1,Basanite 20,598,-3,
9.1.A,9.1,Fürth 1,Basanite 20,623,-3,
9.1.A,9.1,Fürth 1,Basanite 20,653,-3,
9.1.B,9.1,Fürth 1,Basanite 20,30,-3,
9.1.B,9.1,Fürth 1,Basanite 20,57,-3,
9.1.B,9.1,Fürth 1,Basanite 20,85,-3,
9.1.B,9.1,Fürth 1,Basanite 20,112,-3,
9.1.B,9.1,Fürth 1,Basanite 20,147,-3,
9.1.B,9.1,Fürth 1,Basanite 20,175,-3,
9.1.B,9.1,Fürth 1,Basanite 20,196,-3,
9.1.B,9.1,Fürth 1,Basanite 20,226,-3,
9.1.B,9.1,Fürth 1,Basanite 20,253,-3,
9.1.B,9.1,Fürth 1,Basanite 20,280,-3,
9.1.B,9.1,Fürth 1,Basanite 20,308,-3,
9.1.B,9.1,Fürth 1,Basanite 20,344,-3,
9.1.B,9.1,Fürth 1,Basanite 20,381,-3,
9.1.B,9.1,Fürth 1,Basanite 20,415,-3,
9.1.B,9.1,Fürth 1,Basanite 20,448,-3,
9.1.B,9.1,Fürth 1,Basanite 20,469,-3,
9.1.B,9.1,Fürth 1,Basanite 20,499,-3,
9.1.B,9.1,Fürth 1,Basanite 20,529,-3,
9.1.B,9.1,Fürth 1,Basanite 20,561,-3,
9.1.B,9.1,Fürth 1,Basanite 20,598,-3,
9.1.B,9.1,Fürth 1,Basanite 20,623,-3,
9.1.B,9.1,Fürth 1,Basanite 20,653,-3,
9.1.C,9.1,Fürth 1,Basanite 20,30,-3,
9.1.C,9.1,Fürth 1,Basanite 20,57,-3,
9.1.C,9.1,Fürth 1,Basanite 20,85,-3,
9.1.C,9.1,Fürth 1,Basanite 20,112,-3,
9.1.C,9.1,Fürth 1,Basanite 20,147,-3,
9.1.C,9.1,Fürth 1,Basanite 20,175,-3,
9.1.C,9.1,Fürth 1,Basanite 20,196,-3,
9.1.C,9.1,Fürth 1,Basanite 20,226,-3,
9.1.C,9.1,Fürth 1,Basanite 20,253,-3,
9.1.C,9.1,Fürth 1,Basanite 20,280,-3,
9.1.C,9.1,Fürth 1,Basanite 20,308,-3,
9.1.C,9.1,Fürth 1,Basanite 20,344,-3,
9.1.C,9.1,Fürth 1,Basanite 20,381,-3,
9.1.C,9.1,Fürth 1,Basanite 20,415,-3,
9.1.C,9.1,Fürth 1,Basanite 20,448,-3,
9.1.C,9.1,Fürth 1,Basanite 20,469,-3,
9.1.C,9.1,Fürth 1,Basanite 20,499,-3,
9.1.C,9.1,Fürth 1,Basanite 20,529,-3,
9.1.C,9.1,Fürth 1,Basanite 20,561,-3,
9.1.C,9.1,Fürth 1,Basanite 20,598,-3,
9.1.C,9.1,Fürth 1,Basanite 20,623,-3,
9.1.C,9.1,Fürth 1,Basanite 20,653,-3,
9.1.D,9.1,Fürth 1,Basanite 20,30,-3,
9.1.D,9.1,Fürth 1,Basanite 20,57,-3,
9.1.D,9.1,Fürth 1,Basanite 20,85,-3,
9.1.D,9.1,Fürth 1,Basanite 20,112,-3,
9.1.D,9.1,Fürth 1,Basanite 20,147,-3,
9.1.D,9.1,Fürth 1,Basanite 20,175,-3,
9.1.D,9.1,Fürth 1,Basanite 20,196,-3,
9.1.D,9.1,Fürth 1,Basanite 20,226,-3,
9.1.D,9.1,Fürth 1,Basanite 20,253,-3,
9.1.D,9.1,Fürth 1,Basanite 20,280,-3,
9.1.D,9.1,Fürth 1,Basanite 20,308,-3,
9.1.D,9.1,Fürth 1,Basanite 20,344,-3,
9.1.D,9.1,Fürth 1,Basanite 20,381,-3,
9.1.D,9.1,Fürth 1,Basanite 20,415,-3,
9.1.D,9.1,Fürth 1,Basanite 20,448,-3,
9.1.D,9.1,Fürth 1,Basanite 20,469,-3,
9.1.D,9.1,Fürth 1,Basanite 20,499,-3,
9.1.D,9.1,Fürth 1,Basanite 20,529,-3,
9.1.D,9.1,Fürth 1,Basanite 20,561,-3,
9.1.D,9.1,Fürth 1,Basanite 20,598,-3,
9.1.D,9.1,Fürth 1,Basanite 20,623,-3,
9.1.D,9.1,Fürth 1,Basanite 20,653,-3,
9.2.A,9.2,Fürth 1,Basanite,30,-3,412.8926349359168
9.2.A,9.2,Fürth 1,Basanite,58,-3,296.5319831518142
9.2.A,9.2,Fürth 1,Basanite,84,-3,194.41564623623563
9.2.A,9.2,Fürth 1,Basanite,112,-3,690.5605256633974
9.2.A,9.2,Fürth 1,Basanite,147,-3,0.0
9.2.A,9.2,Fürth 1,Basanite,175,-3,1145.319896263313
9.2.A,9.2,Fürth 1,Basanite,196,-3,540.9952216138155
9.2.A,9.2,Fürth 1,Basanite,226,-3,249.27550682953247
9.2.A,9.2,Fürth 1,Basanite,253,-3,246.24377760394728
9.2.A,9.2,Fürth 1,Basanite,280,-3,240.85403694566455
9.2.A,9.2,Fürth 1,Basanite,308,-3,210.29613221012093
9.2.A,9.2,Fürth 1,Basanite,344,-3,347.6863970154642
9.2.A,9.2,Fürth 1,Basanite,385,-3,326.7049064323966
9.2.A,9.2,Fürth 1,Basanite,415,-3,255.2427197785667
9.2.A,9.2,Fürth 1,Basanite,448,-3,291.0459972320837
9.2.A,9.2,Fürth 1,Basanite,469,-3,197.49549806847585
9.2.A,9.2,Fürth 1,Basanite,498,-3,299.8043256513629
9.2.A,9.2,Fürth 1,Basanite,529,-3,222.2498070882725
9.2.A,9.2,Fürth 1,Basanite,561,-3,327.426746762415
9.2.A,9.2,Fürth 1,Basanite,599,-3,274.87677501654736
9.2.A,9.2,Fürth 1,Basanite,623,-3,148.98783185953002
9.2.A,9.2,Fürth 1,Basanite,653,-3,178.05393344966603
9.2.A,9.2,Fürth 1,Basanite,682,-3,168.79512890065587
9.2.A,9.2,Fürth 1,Basanite,716,-3,199.19422884650095
9.2.A,9.2,Fürth 1,Basanite,742,-3,116.73600856850592
9.2.A,9.2,Fürth 1,Basanite,771,-3,35.66853391900836
9.2.A,9.2,Fürth 1,Basanite,798,-3,127.21231703471928
9.2.A,9.2,Fürth 1,Basanite,833,-3,176.91823808893434
9.2.A,9.2,Fürth 1,Basanite,868,-3,393.64356098441544
9.2.A,9.2,Fürth 1,Basanite,899,-3,322.73478500511465
9.2.A,9.2,Fürth 1,Basanite,926,-3,163.30914283651242
9.2.A,9.2,Fürth 1,Basanite,955,-3,233.5634502677658
9.2.A,9.2,Fürth 1,Basanite,987,-3,262.36487706841564
9.2.A,9.2,Fürth 1,Basanite,1016,-3,188.58317684577892
9.2.A,9.2,Fürth 1,Basanite,1043,-3,
9.2.A,9.2,Fürth 1,Basanite,1079,-3,
9.2.A,9.2,Fürth 1,Basanite,1107,-3,
9.2.A,9.2,Fürth 1,Basanite,1142,-3,
9.2.A,9.2,Fürth 1,Basanite,1171,-3,
9.2.B,9.2,Fürth 1,Basanite,30,-3,351.68057981828025
9.2.B,9.2,Fürth 1,Basanite,58,-3,228.87148848907876
9.2.B,9.2,Fürth 1,Basanite,84,-3,140.8069754618208
9.2.B,9.2,Fürth 1,Basanite,112,-3,722.2252523015825
9.2.B,9.2,Fürth 1,Basanite,147,-3,0.0
9.2.B,9.2,Fürth 1,Basanite,175,-3,895.8037759191287
9.2.B,9.2,Fürth 1,Basanite,196,-3,454.61500258739994
9.2.B,9.2,Fürth 1,Basanite,226,-3,194.8968730970576
9.2.B,9.2,Fürth 1,Basanite,253,-3,235.8974004693423
9.2.B,9.2,Fürth 1,Basanite,280,-3,117.80433217401767
9.2.B,9.2,Fürth 1,Basanite,308,-3,335.31886707984836
9.2.B,9.2,Fürth 1,Basanite,344,-3,355.86725338468017
9.2.B,9.2,Fürth 1,Basanite,385,-3,289.6985618869968
9.2.B,9.2,Fürth 1,Basanite,415,-3,220.30565061676396
9.2.B,9.2,Fürth 1,Basanite,448,-3,240.7096689331488
9.2.B,9.2,Fürth 1,Basanite,469,-3,107.40983228834467
9.2.B,9.2,Fürth 1,Basanite,498,-3,251.77788651543412
9.2.B,9.2,Fürth 1,Basanite,529,-3,178.43891492869608
9.2.B,9.2,Fürth 1,Basanite,561,-3,287.4367957735051
9.2.B,9.2,Fürth 1,Basanite,599,-3,254.49200577652087
9.2.B,9.2,Fürth 1,Basanite,623,-3,136.87294598023686
9.2.B,9.2,Fürth 1,Basanite,653,-3,157.85203039894097
9.2.B,9.2,Fürth 1,Basanite,682,-3,163.47275997352426
9.2.C,9.2,Fürth 1,Basanite,28,-3,
9.2.C,9.2,Fürth 1,Basanite,58,-3,243.78952066911367
9.2.C,9.2,Fürth 1,Basanite,84,-3,152.45266516637582
9.2.C,9.2,Fürth 1,Basanite,112,-3,730.5023539322463
9.2.C,9.2,Fürth 1,Basanite,147,-3,0.0
9.2.C,9.2,Fürth 1,Basanite,175,-3,908.0750605932968
9.2.C,9.2,Fürth 1,Basanite,196,-3,424.4420790661292
9.2.C,9.2,Fürth 1,Basanite,226,-3,303.84663120524704
9.2.C,9.2,Fürth 1,Basanite,253,-3,230.21892366568383
9.2.C,9.2,Fürth 1,Basanite,280,-3,195.85932679463264
9.2.C,9.2,Fürth 1,Basanite,308,-3,309.91008965641737
9.2.C,9.2,Fürth 1,Basanite,344,-3,331.0840707623804
9.2.C,9.2,Fürth 1,Basanite,385,-3,299.8043256513629
9.2.C,9.2,Fürth 1,Basanite,415,-3,351.1031077682171
9.2.C,9.2,Fürth 1,Basanite,448,-3,235.60866434803535
9.2.C,9.2,Fürth 1,Basanite,469,-3,132.38550565015944
9.2.C,9.2,Fürth 1,Basanite,498,-3,299.41934436488356
9.2.C,9.2,Fürth 1,Basanite,529,-3,204.728337589506
9.2.C,9.2,Fürth 1,Basanite,561,-3,364.9191305090951
9.2.C,9.2,Fürth 1,Basanite,599,-3,308.74552066911366
9.2.C,9.2,Fürth 1,Basanite,623,-3,160.83804300565924
9.2.C,9.2,Fürth 1,Basanite,653,-3,203.63595263252904
9.2.C,9.2,Fürth 1,Basanite,682,-3,215.28645461219085
9.2.C,9.2,Fürth 1,Basanite,716,-3,220.7098811721523
9.2.C,9.2,Fürth 1,Basanite,742,-3,167.4188201215476
9.2.C,9.2,Fürth 1,Basanite,771,-3,106.0672093868464
9.2.C,9.2,Fürth 1,Basanite,798,-3,184.50237318731573
9.2.C,9.2,Fürth 1,Basanite,833,-3,258.32257151453155
9.2.C,9.2,Fürth 1,Basanite,868,-3,430.1205559901318
9.2.C,9.2,Fürth 1,Basanite,899,-3,331.16106721222695
9.2.C,9.2,Fürth 1,Basanite,926,-3,132.52987371081292
9.2.C,9.2,Fürth 1,Basanite,955,-3,239.16974301702868
9.2.C,9.2,Fürth 1,Basanite,987,-3,233.7030060533125
9.2.C,9.2,Fürth 1,Basanite,1016,-3,240.72891798543836
9.2.C,9.2,Fürth 1,Basanite,1043,-3,
9.2.C,9.2,Fürth 1,Basanite,1079,-3,
9.2.C,9.2,Fürth 1,Basanite,1107,-3,
9.2.C,9.2,Fürth 1,Basanite,1142,-3,
9.2.C,9.2,Fürth 1,Basanite,1171,-3,
9.2.D,9.2,Fürth 1,Basanite,28,-3,313.2786774174138
9.2.D,9.2,Fürth 1,Basanite,58,-3,252.6440948312173
9.2.D,9.2,Fürth 1,Basanite,84,-3,133.54045008724952
9.2.D,9.2,Fürth 1,Basanite,112,-3,628.8672437571454
9.2.D,9.2,Fürth 1,Basanite,147,-3,0.0
9.2.D,9.2,Fürth 1,Basanite,175,-3,872.3680284012275
9.2.D,9.2,Fürth 1,Basanite,196,-3,390.7561997713461
9.2.D,9.2,Fürth 1,Basanite,226,-3,270.2569974126
9.2.D,9.2,Fürth 1,Basanite,253,-3,194.0306647572056
9.2.D,9.2,Fürth 1,Basanite,280,-3,200.19036842168603
9.2.D,9.2,Fürth 1,Basanite,308,-3,274.29930296648416
9.2.D,9.2,Fürth 1,Basanite,344,-3,313.3749229195499
9.2.D,9.2,Fürth 1,Basanite,385,-3,249.27550682953247
9.2.D,9.2,Fürth 1,Basanite,415,-3,207.88999795414887
9.2.D,9.2,Fürth 1,Basanite,448,-3,221.65308579336903
9.2.D,9.2,Fürth 1,Basanite,469,-3,146.10047078644925
9.2.D,9.2,Fürth 1,Basanite,498,-3,359.95768168963235
9.2.D,9.2,Fürth 1,Basanite,529,-3,195.02680433239064
9.2.D,9.2,Fürth 1,Basanite,561,-3,271.3830682196722
9.2.D,9.2,Fürth 1,Basanite,599,-3,241.96567109934412
9.2.D,9.2,Fürth 1,Basanite,623,-3,140.67223198143444
9.2.D,9.2,Fürth 1,Basanite,653,-3,185.39264285456403
9.2.D,9.2,Fürth 1,Basanite,682,-3,195.77751821409228
9.3.A,9.3,Fürth 1,Basanite 100,30,-3,
9.3.A,9.3,Fürth 1,Basanite 100,57,-3,
9.3.A,9.3,Fürth 1,Basanite 100,85,-3,
9.3.A,9.3,Fürth 1,Basanite 100,112,-3,
9.3.A,9.3,Fürth 1,Basanite 100,147,-3,
9.3.A,9.3,Fürth 1,Basanite 100,175,-3,
9.3.A,9.3,Fürth 1,Basanite 100,196,-3,
9.3.A,9.3,Fürth 1,Basanite 100,226,-3,
9.3.A,9.3,Fürth 1,Basanite 100,253,-3,
9.3.A,9.3,Fürth 1,Basanite 100,280,-3,
9.3.A,9.3,Fürth 1,Basanite 100,308,-3,
9.3.A,9.3,Fürth 1,Basanite 100,344,-3,
9.3.A,9.3,Fürth 1,Basanite 100,381,-3,
9.3.A,9.3,Fürth 1,Basanite 100,415,-3,
9.3.A,9.3,Fürth 1,Basanite 100,448,-3,
9.3.A,9.3,Fürth 1,Basanite 100,469,-3,
9.3.A,9.3,Fürth 1,Basanite 100,499,-3,
9.3.A,9.3,Fürth 1,Basanite 100,529,-3,
9.3.A,9.3,Fürth 1,Basanite 100,561,-3,
9.3.A,9.3,Fürth 1,Basanite 100,598,-3,
9.3.A,9.3,Fürth 1,Basanite 100,623,-3,
9.3.A,9.3,Fürth 1,Basanite 100,653,-3,
9.3.B,9.3,Fürth 1,Basanite 100,30,-3,
9.3.B,9.3,Fürth 1,Basanite 100,57,-3,
9.3.B,9.3,Fürth 1,Basanite 100,85,-3,
9.3.B,9.3,Fürth 1,Basanite 100,112,-3,
9.3.B,9.3,Fürth 1,Basanite 100,147,-3,
9.3.B,9.3,Fürth 1,Basanite 100,175,-3,
9.3.B,9.3,Fürth 1,Basanite 100,196,-3,
9.3.B,9.3,Fürth 1,Basanite 100,226,-3,
9.3.B,9.3,Fürth 1,Basanite 100,253,-3,
9.3.B,9.3,Fürth 1,Basanite 100,280,-3,
9.3.B,9.3,Fürth 1,Basanite 100,308,-3,
9.3.B,9.3,Fürth 1,Basanite 100,344,-3,
9.3.B,9.3,Fürth 1,Basanite 100,381,-3,
9.3.B,9.3,Fürth 1,Basanite 100,415,-3,
9.3.B,9.3,Fürth 1,Basanite 100,448,-3,
9.3.B,9.3,Fürth 1,Basanite 100,469,-3,
9.3.B,9.3,Fürth 1,Basanite 100,499,-3,
9.3.B,9.3,Fürth 1,Basanite 100,529,-3,
9.3.B,9.3,Fürth 1,Basanite 100,561,-3,
9.3.B,9.3,Fürth 1,Basanite 100,598,-3,
9.3.B,9.3,Fürth 1,Basanite 100,623,-3,
9.3.B,9.3,Fürth 1,Basanite 100,653,-3,
9.3.C,9.3,Fürth 1,Basanite 100,30,-3,
9.3.C,9.3,Fürth 1,Basanite 100,57,-3,
9.3.C,9.3,Fürth 1,Basanite 100,85,-3,
9.3.C,9.3,Fürth 1,Basanite 100,112,-3,
9.3.C,9.3,Fürth 1,Basanite 100,147,-3,
9.3.C,9.3,Fürth 1,Basanite 100,175,-3,
9.3.C,9.3,Fürth 1,Basanite 100,196,-3,
9.3.C,9.3,Fürth 1,Basanite 100,226,-3,
9.3.C,9.3,Fürth 1,Basanite 100,253,-3,
9.3.C,9.3,Fürth 1,Basanite 100,280,-3,
9.3.C,9.3,Fürth 1,Basanite 100,308,-3,
9.3.C,9.3,Fürth 1,Basanite 100,344,-3,
9.3.C,9.3,Fürth 1,Basanite 100,381,-3,
9.3.C,9.3,Fürth 1,Basanite 100,415,-3,
9.3.C,9.3,Fürth 1,Basanite 100,448,-3,
9.3.C,9.3,Fürth 1,Basanite 100,469,-3,
9.3.C,9.3,Fürth 1,Basanite 100,499,-3,
9.3.C,9.3,Fürth 1,Basanite 100,529,-3,
9.3.C,9.3,Fürth 1,Basanite 100,561,-3,
9.3.C,9.3,Fürth 1,Basanite 100,598,-3,
9.3.C,9.3,Fürth 1,Basanite 100,623,-3,
9.3.C,9.3,Fürth 1,Basanite 100,653,-3,
9.3.D,9.3,Fürth 1,Basanite 100,28,-3,286.4262193874481
9.3.D,9.3,Fürth 1,Basanite 100,57,-3,204.81014614597748
9.3.D,9.3,Fürth 1,Basanite 100,84,-3,151.5864568505927
9.3.D,9.3,Fürth 1,Basanite 100,112,-3,
9.3.D,9.3,Fürth 1,Basanite 100,147,-3,
9.3.D,9.3,Fürth 1,Basanite 100,175,-3,
9.3.D,9.3,Fürth 1,Basanite 100,196,-3,
9.3.D,9.3,Fürth 1,Basanite 100,226,-3,
9.3.D,9.3,Fürth 1,Basanite 100,253,-3,
9.3.D,9.3,Fürth 1,Basanite 100,280,-3,
9.3.D,9.3,Fürth 1,Basanite 100,308,-3,
9.3.D,9.3,Fürth 1,Basanite 100,344,-3,
9.3.D,9.3,Fürth 1,Basanite 100,381,-3,
9.3.D,9.3,Fürth 1,Basanite 100,415,-3,
9.3.D,9.3,Fürth 1,Basanite 100,448,-3,
9.3.D,9.3,Fürth 1,Basanite 100,469,-3,
9.3.D,9.3,Fürth 1,Basanite 100,499,-3,
9.3.D,9.3,Fürth 1,Basanite 100,529,-3,
9.3.D,9.3,Fürth 1,Basanite 100,561,-3,
9.3.D,9.3,Fürth 1,Basanite 100,598,-3,
9.3.D,9.3,Fürth 1,Basanite 100,623,-3,
9.3.D,9.3,Fürth 1,Basanite 100,653,-3,
9.4.A,9.4,Fürth 1,Basanite 200,30,-3,363.8074964799326
9.4.A,9.4,Fürth 1,Basanite 200,58,-3,239.16974301702868
9.4.A,9.4,Fürth 1,Basanite 200,84,-3,170.0655677718274
9.4.A,9.4,Fürth 1,Basanite 200,112,-3,700.185062639148
9.4.A,9.4,Fürth 1,Basanite 200,147,-3,0.0
9.4.A,9.4,Fürth 1,Basanite 200,175,-3,943.2046204946146
9.4.A,9.4,Fürth 1,Basanite 200,196,-3,457.83922233588066
9.4.A,9.4,Fürth 1,Basanite 200,226,-3,235.8011550875504
9.4.A,9.4,Fürth 1,Basanite 200,253,-3,175.5996765148324
9.4.A,9.4,Fürth 1,Basanite 200,280,-3,164.09835487093088
9.4.A,9.4,Fürth 1,Basanite 200,308,-3,364.7218275467838
9.4.A,9.4,Fürth 1,Basanite 200,344,-3,377.2818482459835
9.4.A,9.4,Fürth 1,Basanite 200,381,-3,272.1337820566821
9.4.A,9.4,Fürth 1,Basanite 200,415,-3,250.33420590889943
9.4.A,9.4,Fürth 1,Basanite 200,448,-3,229.54520608941573
9.4.A,9.4,Fürth 1,Basanite 200,469,-3,173.04917422227572
9.4.A,9.4,Fürth 1,Basanite 200,499,-3,292.2009415728985
9.4.A,9.4,Fürth 1,Basanite 200,529,-3,170.43130017449906
9.4.A,9.4,Fürth 1,Basanite 200,561,-3,337.5902577476004
9.4.A,9.4,Fürth 1,Basanite 200,599,-3,321.88301341837655
9.4.A,9.4,Fürth 1,Basanite 200,623,-3,159.38233174418548
9.4.A,9.4,Fürth 1,Basanite 200,653,-3,181.67275932366567
9.4.A,9.4,Fürth 1,Basanite 200,682,-3,184.43018915698897
9.4.B,9.4,Fürth 1,Basanite 200,30,-3,346.3870846621337
9.4.B,9.4,Fürth 1,Basanite 200,58,-3,234.0687384559841
9.4.B,9.4,Fürth 1,Basanite 200,84,-3,137.1496514351044
9.4.B,9.4,Fürth 1,Basanite 200,112,-3,670.8302249232805
9.4.B,9.4,Fürth 1,Basanite 200,147,-3,0.0
9.4.B,9.4,Fürth 1,Basanite 200,175,-3,950.9042498345268
9.4.B,9.4,Fürth 1,Basanite 200,196,-3,478.3394861303327
9.4.B,9.4,Fürth 1,Basanite 200,226,-3,171.50924833022444
9.4.B,9.4,Fürth 1,Basanite 200,253,-3,173.24166496179072
9.4.B,9.4,Fürth 1,Basanite 200,280,-3,79.40242977315121
9.4.B,9.4,Fürth 1,Basanite 200,308,-3,323.9619134725314
9.4.B,9.4,Fürth 1,Basanite 200,344,-3,392.68110716649613
9.4.B,9.4,Fürth 1,Basanite 200,381,-3,309.6694761417655
9.4.B,9.4,Fürth 1,Basanite 200,415,-3,275.4542473072989
9.4.B,9.4,Fürth 1,Basanite 200,448,-3,264.1935389614297
9.4.B,9.4,Fürth 1,Basanite 200,469,-3,123.29031823816112
9.4.B,9.4,Fürth 1,Basanite 200,499,-3,288.495494795114
9.4.B,9.4,Fürth 1,Basanite 200,529,-3,131.46155010530114
9.4.B,9.4,Fürth 1,Basanite 200,561,-3,312.29697475125465
9.4.B,9.4,Fürth 1,Basanite 200,599,-3,269.51590805704313
9.4.B,9.4,Fürth 1,Basanite 200,623,-3,134.132359114927
9.4.B,9.4,Fürth 1,Basanite 200,653,-3,191.01818470425417
9.4.B,9.4,Fürth 1,Basanite 200,682,-3,216.46546037667727
9.4.C,9.4,Fürth 1,Basanite 200,30,-3,342.87412864793305
9.4.C,9.4,Fürth 1,Basanite 200,58,-3,242.97143510439855
9.4.C,9.4,Fürth 1,Basanite 200,84,-3,151.34584343221613
9.4.C,9.4,Fürth 1,Basanite 200,112,-3,644.3627484204826
9.4.C,9.4,Fürth 1,Basanite 200,147,-3,0.0
9.4.C,9.4,Fürth 1,Basanite 200,175,-3,1175.1559607677957
9.4.C,9.4,Fürth 1,Basanite 200,196,-3,457.83922233588066
9.4.C,9.4,Fürth 1,Basanite 200,226,-3,240.85403694566455
9.4.C,9.4,Fürth 1,Basanite 200,253,-3,145.61924394969614
9.4.C,9.4,Fürth 1,Basanite 200,280,-3,146.7741883867862
9.4.C,9.4,Fürth 1,Basanite 200,308,-3,298.40876779589627
9.4.C,9.4,Fürth 1,Basanite 200,344,-3,365.7324038750827
9.4.C,9.4,Fürth 1,Basanite 200,381,-3,272.9999903724653
9.4.C,9.4,Fürth 1,Basanite 200,415,-3,311.8349970515675
9.4.C,9.4,Fürth 1,Basanite 200,448,-3,242.8751898429508
9.4.C,9.4,Fürth 1,Basanite 200,469,-3,113.37704518924124
9.4.C,9.4,Fürth 1,Basanite 200,499,-3,310.29507094289664
9.4.C,9.4,Fürth 1,Basanite 200,529,-3,205.81109799626932
9.4.C,9.4,Fürth 1,Basanite 200,561,-3,370.11638037071555
9.4.C,9.4,Fürth 1,Basanite 200,599,-3,303.9525011131837
9.4.C,9.4,Fürth 1,Basanite 200,623,-3,147.42865686780726
9.4.C,9.4,Fürth 1,Basanite 200,653,-3,201.18650797280225
9.4.C,9.4,Fürth 1,Basanite 200,682,-3,199.45890359227388
9.4.D,9.4,Fürth 1,Basanite 200,28,-3,275.4061245562308
9.4.D,9.4,Fürth 1,Basanite 200,58,-3,205.86884520127563
9.4.D,9.4,Fürth 1,Basanite 200,84,-3,174.63722281725734
9.4.D,9.4,Fürth 1,Basanite 200,112,-3,582.0919942234792
9.4.D,9.4,Fürth 1,Basanite 200,147,-3,0.0
9.4.D,9.4,Fürth 1,Basanite 200,175,-3,842.6282093988808
9.4.D,9.4,Fürth 1,Basanite 200,196,-3,387.86883879896504
9.4.D,9.4,Fürth 1,Basanite 200,226,-3,179.0163871472411
9.4.D,9.4,Fürth 1,Basanite 200,253,-3,138.160227811541
9.4.D,9.4,Fürth 1,Basanite 200,280,-3,80.17239273121126
9.4.D,9.4,Fürth 1,Basanite 200,308,-3,242.5383310668512
9.4.D,9.4,Fürth 1,Basanite 200,344,-3,357.3590567422829
9.4.D,9.4,Fürth 1,Basanite 200,381,-3,275.8873515855346
9.4.D,9.4,Fürth 1,Basanite 200,415,-3,282.96138612431554
9.4.D,9.4,Fürth 1,Basanite 200,448,-3,235.8011550875504
9.4.D,9.4,Fürth 1,Basanite 200,469,-3,106.54362394849268
9.4.D,9.4,Fürth 1,Basanite 200,499,-3,265.0597475179012
9.4.D,9.4,Fürth 1,Basanite 200,529,-3,154.1465836692942
9.4.D,9.4,Fürth 1,Basanite 200,561,-3,278.13949313149243
9.4.D,9.4,Fürth 1,Basanite 200,599,-3,247.73558096155003
9.4.D,9.4,Fürth 1,Basanite 200,623,-3,144.1900002188421
9.4.D,9.4,Fürth 1,Basanite 200,653,-3,188.02495370359227
9.4.D,9.4,Fürth 1,Basanite 200,682,-3,163.8481169023407
9.5.A,9.5,Fürth 1,Basanite 400,30,-3,320.9783069980143
9.5.A,9.5,Fürth 1,Basanite 400,58,-3,271.17132847945123
9.5.A,9.5,Fürth 1,Basanite 400,84,-3,221.31622699320056
9.5.A,9.5,Fürth 1,Basanite 400,112,-3,675.6424932908117
9.5.A,9.5,Fürth 1,Basanite 400,147,-3,0.0
9.5.A,9.5,Fürth 1,Basanite 400,175,-3,895.9481439316444
9.5.A,9.5,Fürth 1,Basanite 400,196,-3,436.71336374029727
9.5.A,9.5,Fürth 1,Basanite 400,226,-3,312.1237330765991
9.5.A,9.5,Fürth 1,Basanite 400,253,-3,236.47487268788737
9.5.A,9.5,Fürth 1,Basanite 400,280,-3,296.67635116433
9.5.A,9.5,Fürth 1,Basanite 400,308,-3,265.15599277934894
9.5.A,9.5,Fürth 1,Basanite 400,344,-3,347.63827426439616
9.5.A,9.5,Fürth 1,Basanite 400,385,-3,283.442612912931
9.5.A,9.5,Fürth 1,Basanite 400,415,-3,252.3072360551176
9.5.A,9.5,Fürth 1,Basanite 400,448,-3,283.9238397015464
9.5.A,9.5,Fürth 1,Basanite 400,469,-3,166.408243745111
9.5.A,9.5,Fürth 1,Basanite 400,498,-3,301.729233046513
9.5.A,9.5,Fürth 1,Basanite 400,529,-3,196.1865610445875
9.5.A,9.5,Fürth 1,Basanite 400,561,-3,432.0887738491418
9.5.A,9.5,Fürth 1,Basanite 400,599,-3,307.0131040375474
9.5.A,9.5,Fürth 1,Basanite 400,623,-3,164.95493868904472
9.5.A,9.5,Fürth 1,Basanite 400,653,-3,211.5280729285757
9.5.A,9.5,Fürth 1,Basanite 400,682,-3,204.3818542391239
9.5.B,9.5,Fürth 1,Basanite 400,30,-3,312.3162238401829
9.5.B,9.5,Fürth 1,Basanite 400,58,-3,266.6959186473313
9.5.B,9.5,Fürth 1,Basanite 400,84,-3,186.52352594018893
9.5.B,9.5,Fürth 1,Basanite 400,112,-3,666.258569829713
9.5.B,9.5,Fürth 1,Basanite 400,147,-3,0.0
9.5.B,9.5,Fürth 1,Basanite 400,175,-3,745.9016130934473
9.5.B,9.5,Fürth 1,Basanite 400,196,-3,488.9264766833142
9.5.B,9.5,Fürth 1,Basanite 400,226,-3,359.7170681749804
9.5.B,9.5,Fürth 1,Basanite 400,253,-3,210.1036414706059
9.5.B,9.5,Fürth 1,Basanite 400,280,-3,185.3685815271677
9.5.B,9.5,Fürth 1,Basanite 400,308,-3,286.4262193874481
9.5.B,9.5,Fürth 1,Basanite 400,344,-3,368.1385380588483
9.5.B,9.5,Fürth 1,Basanite 400,385,-3,261.7874047776641
9.5.B,9.5,Fürth 1,Basanite 400,415,-3,212.22103958120223
9.5.B,9.5,Fürth 1,Basanite 400,448,-3,294.51083049521634
9.5.B,9.5,Fürth 1,Basanite 400,469,-3,173.04917422227572
9.5.B,9.5,Fürth 1,Basanite 400,498,-3,132.04864684999097
9.5.B,9.5,Fürth 1,Basanite 400,529,-3,210.48862294963595
9.5.B,9.5,Fürth 1,Basanite 400,561,-3,267.9037980598796
9.5.B,9.5,Fürth 1,Basanite 400,599,-3,292.9805290330345
9.5.B,9.5,Fürth 1,Basanite 400,623,-3,175.63817465319184
9.5.B,9.5,Fürth 1,Basanite 400,653,-3,234.2660414465371
9.5.B,9.5,Fürth 1,Basanite 400,682,-3,227.29306444431072
9.5.C,9.5,Fürth 1,Basanite 400,30,-3,285.3193975570131
9.5.C,9.5,Fürth 1,Basanite 400,58,-3,252.5959720801492
9.5.C,9.5,Fürth 1,Basanite 400,84,-3,156.9761975329442
9.5.C,9.5,Fürth 1,Basanite 400,112,-3,794.5055245201276
9.5.C,9.5,Fürth 1,Basanite 400,147,-3,0.0
9.5.C,9.5,Fürth 1,Basanite 400,175,-3,961.4912406281968
9.5.C,9.5,Fürth 1,Basanite 400,196,-3,460.0047432456826
9.5.C,9.5,Fürth 1,Basanite 400,226,-3,365.347422347915
9.5.C,9.5,Fürth 1,Basanite 400,253,-3,276.320455623082
9.5.C,9.5,Fürth 1,Basanite 400,280,-3,286.4262193874481
9.5.C,9.5,Fürth 1,Basanite 400,308,-3,278.9190805704314
9.5.C,9.5,Fürth 1,Basanite 400,344,-3,327.426746735664
9.5.C,9.5,Fürth 1,Basanite 400,385,-3,284.64568000481376
9.5.C,9.5,Fürth 1,Basanite 400,415,-3,283.73134917865093
9.5.C,9.5,Fürth 1,Basanite 400,448,-3,280.55525194054997
9.5.C,9.5,Fürth 1,Basanite 400,469,-3,167.46694280040916
9.5.C,9.5,Fürth 1,Basanite 400,498,-3,359.6208229135327
9.5.C,9.5,Fürth 1,Basanite 400,529,-3,242.97143510439855
9.5.C,9.5,Fürth 1,Basanite 400,561,-3,439.600724825858
9.5.C,9.5,Fürth 1,Basanite 400,599,-3,371.0114623021842
9.5.C,9.5,Fürth 1,Basanite 400,623,-3,187.96480030398655
9.5.C,9.5,Fürth 1,Basanite 400,653,-3,250.8924289066731
9.5.C,9.5,Fürth 1,Basanite 400,682,-3,241.7972417112943
9.5.D,9.5,Fürth 1,Basanite 400,28,-3,275.8873515855346
9.5.D,9.5,Fürth 1,Basanite 400,58,-3,243.9820114326975
9.5.D,9.5,Fürth 1,Basanite 400,84,-3,162.41406091822614
9.5.D,9.5,Fürth 1,Basanite 400,112,-3,706.9222384018292
9.5.D,9.5,Fürth 1,Basanite 400,147,-3,0.0
9.5.D,9.5,Fürth 1,Basanite 400,175,-3,812.0703045911306
9.5.D,9.5,Fürth 1,Basanite 400,196,-3,370.54467224261384
9.5.D,9.5,Fürth 1,Basanite 400,226,-3,301.34425176003367
9.5.D,9.5,Fürth 1,Basanite 400,253,-3,194.7525050364041
9.5.D,9.5,Fürth 1,Basanite 400,280,-3,241.38338648534807
9.5.D,9.5,Fürth 1,Basanite 400,308,-3,334.16392273903364
9.5.D,9.5,Fürth 1,Basanite 400,344,-3,334.93388555268064
9.5.D,9.5,Fürth 1,Basanite 400,385,-3,281.5177055177809
9.5.D,9.5,Fürth 1,Basanite 400,415,-3,320.6895709729827
9.5.D,9.5,Fürth 1,Basanite 400,448,-3,298.9862400866478
9.5.D,9.5,Fürth 1,Basanite 400,469,-3,189.45900971177568
9.5.D,9.5,Fürth 1,Basanite 400,498,-3,276.320455623082
9.5.D,9.5,Fürth 1,Basanite 400,529,-3,196.74959646188097
9.5.D,9.5,Fürth 1,Basanite 400,561,-3,352.0751859272878
9.5.D,9.5,Fürth 1,Basanite 400,599,-3,287.2346805463626
9.5.D,9.5,Fürth 1,Basanite 400,623,-3,165.590158112237
9.5.D,9.5,Fürth 1,Basanite 400,653,-3,187.62072317227268
9.5.D,9.5,Fürth 1,Basanite 400,682,-3,184.43018915698897
9.6.A,9.6,Fürth 1,Metabasalt,30,-3,322.90321439316443
9.6.A,9.6,Fürth 1,Metabasalt,58,-3,245.90691882784765
9.6.A,9.6,Fürth 1,Metabasalt,84,-3,157.64991513328118
9.6.A,9.6,Fürth 1,Metabasalt,112,-3,619.338952283531
9.6.A,9.6,Fürth 1,Metabasalt,147,-3,0.0
9.6.A,9.6,Fürth 1,Metabasalt,175,-3,746.3828398820626
9.6.A,9.6,Fürth 1,Metabasalt,196,-3,403.17185245802995
9.6.A,9.6,Fürth 1,Metabasalt,226,-3,160.92225768096756
9.6.A,9.6,Fürth 1,Metabasalt,253,-3,150.47963509236416
9.6.A,9.6,Fürth 1,Metabasalt,280,-3,168.33315114026115
9.6.A,9.6,Fürth 1,Metabasalt,308,-3,128.92067233888923
9.6.A,9.6,Fürth 1,Metabasalt,344,-3,311.73875154943136
9.6.A,9.6,Fürth 1,Metabasalt,381,-3,213.66472013959924
9.6.A,9.6,Fürth 1,Metabasalt,415,-3,228.87148848907876
9.6.A,9.6,Fürth 1,Metabasalt,448,-3,247.8318262229978
9.6.A,9.6,Fürth 1,Metabasalt,469,-3,116.21628357903604
9.6.A,9.6,Fürth 1,Metabasalt,499,-3,73.62770761176967
9.6.A,9.6,Fürth 1,Metabasalt,529,-3,148.1601216920392
9.6.A,9.6,Fürth 1,Metabasalt,561,-3,228.9821706695246
9.6.A,9.6,Fürth 1,Metabasalt,599,-3,232.78867503459895
9.6.A,9.6,Fürth 1,Metabasalt,623,-3,130.2392339060925
9.6.A,9.6,Fürth 1,Metabasalt,653,-3,173.32828579336905
9.6.A,9.6,Fürth 1,Metabasalt,682,-3,199.805386942656
9.6.A,9.6,Fürth 1,Metabasalt,716,-3,246.24377760394728
9.6.A,9.6,Fürth 1,Metabasalt,742,-3,155.89824940128767
9.6.A,9.6,Fürth 1,Metabasalt,771,-3,40.06213502617486
9.6.A,9.6,Fürth 1,Metabasalt,798,-3,176.91823808893434
9.6.A,9.6,Fürth 1,Metabasalt,833,-3,203.19322392442388
9.6.A,9.6,Fürth 1,Metabasalt,868,-3,423.59030772007947
9.6.A,9.6,Fürth 1,Metabasalt,899,-3,371.6226203742704
9.6.A,9.6,Fürth 1,Metabasalt,926,-3,231.50861161321376
9.6.A,9.6,Fürth 1,Metabasalt,955,-3,330.5450968168963
9.6.A,9.6,Fürth 1,Metabasalt,987,-3,284.270323124135
9.6.A,9.6,Fürth 1,Metabasalt,1016,-3,246.23415319814669
9.6.A,9.6,Fürth 1,Metabasalt,1043,-3,
9.6.A,9.6,Fürth 1,Metabasalt,1079,-3,
9.6.A,9.6,Fürth 1,Metabasalt,1107,-3,
9.6.A,9.6,Fürth 1,Metabasalt,1142,-3,
9.6.A,9.6,Fürth 1,Metabasalt,1171,-3,
9.6.B,9.6,Fürth 1,Metabasalt,30,-3,307.02272844334794
9.6.B,9.6,Fürth 1,Metabasalt,58,-3,225.2622871412239
9.6.B,9.6,Fürth 1,Metabasalt,84,-3,148.89158649738252
9.6.B,9.6,Fürth 1,Metabasalt,112,-3,599.7048968048618
9.6.B,9.6,Fürth 1,Metabasalt,147,-3,0.0
9.6.B,9.6,Fürth 1,Metabasalt,175,-3,798.210971297912
9.6.B,9.6,Fürth 1,Metabasalt,196,-3,407.502894277634
9.6.B,9.6,Fürth 1,Metabasalt,226,-3,213.85721087911423
9.6.B,9.6,Fürth 1,Metabasalt,253,-3,173.43415570130574
9.6.B,9.6,Fürth 1,Metabasalt,280,-3,181.90374821589745
9.6.B,9.6,Fürth 1,Metabasalt,308,-3,117.32310533726458
9.6.B,9.6,Fürth 1,Metabasalt,344,-3,289.02484433479754
9.6.B,9.6,Fürth 1,Metabasalt,381,-3,200.23849110054755
9.6.B,9.6,Fürth 1,Metabasalt,415,-3,202.1152757927673
9.6.B,9.6,Fürth 1,Metabasalt,448,-3,204.52141002467056
9.6.B,9.6,Fürth 1,Metabasalt,469,-3,110.87466557554606
9.6.B,9.6,Fürth 1,Metabasalt,499,-3,357.8884062819664
9.6.B,9.6,Fürth 1,Metabasalt,529,-3,181.67275932366567
9.6.B,9.6,Fürth 1,Metabasalt,561,-3,310.39131640526193
9.6.B,9.6,Fürth 1,Metabasalt,599,-3,167.00496504001444
9.6.B,9.6,Fürth 1,Metabasalt,623,-3,232.83679772332167
9.6.B,9.6,Fürth 1,Metabasalt,653,-3,191.2587981226307
9.6.B,9.6,Fürth 1,Metabasalt,682,-3,115.220144003851
9.6.C,9.6,Fürth 1,Metabasalt,30,-3,275.4542473072989
9.6.C,9.6,Fürth 1,Metabasalt,58,-3,213.66472013959924
9.6.C,9.6,Fürth 1,Metabasalt,84,-3,113.85827202599434
9.6.C,9.6,Fürth 1,Metabasalt,112,-3,574.8735916721824
9.6.C,9.6,Fürth 1,Metabasalt,147,-3,0.0
9.6.C,9.6,Fürth 1,Metabasalt,175,-3,926.3616807268788
9.6.C,9.6,Fürth 1,Metabasalt,196,-3,381.85350309886275
9.6.C,9.6,Fürth 1,Metabasalt,226,-3,212.22103958120223
9.6.C,9.6,Fürth 1,Metabasalt,253,-3,158.80485954630242
9.6.C,9.6,Fürth 1,Metabasalt,280,-3,157.07244291473614
9.6.C,9.6,Fürth 1,Metabasalt,308,-3,239.74721523557375
9.6.C,9.6,Fürth 1,Metabasalt,344,-3,295.954511101751
9.6.C,9.6,Fürth 1,Metabasalt,381,-3,222.0861899512606
9.6.C,9.6,Fürth 1,Metabasalt,415,-3,320.9301842469463
9.6.C,9.6,Fürth 1,Metabasalt,448,-3,188.6409240748541
9.6.C,9.6,Fürth 1,Metabasalt,469,-3,138.7858227089476
9.6.C,9.6,Fürth 1,Metabasalt,499,-3,303.4135271676996
9.6.C,9.6,Fürth 1,Metabasalt,529,-3,189.80549303808897
9.6.C,9.6,Fürth 1,Metabasalt,561,-3,360.24641781830445
9.6.C,9.6,Fürth 1,Metabasalt,599,-3,304.85720777423427
9.6.C,9.6,Fürth 1,Metabasalt,623,-3,160.42178179272284
9.6.C,9.6,Fürth 1,Metabasalt,653,-3,207.2547785305975
9.6.C,9.6,Fürth 1,Metabasalt,682,-3,197.89972862386423
9.6.C,9.6,Fürth 1,Metabasalt,716,-3,247.23510487995668
9.6.C,9.6,Fürth 1,Metabasalt,742,-3,115.46556969733436
9.6.C,9.6,Fürth 1,Metabasalt,771,-3,64.81163177086468
9.6.C,9.6,Fürth 1,Metabasalt,798,-3,186.2347898429508
9.6.C,9.6,Fürth 1,Metabasalt,833,-3,257.46598784523735
9.6.C,9.6,Fürth 1,Metabasalt,868,-3,564.291413201757
9.6.C,9.6,Fürth 1,Metabasalt,899,-3,396.30474541187795
9.6.C,9.6,Fürth 1,Metabasalt,926,-3,181.73531882784764
9.6.C,9.6,Fürth 1,Metabasalt,955,-3,284.57830844214453
9.6.C,9.6,Fürth 1,Metabasalt,987,-3,286.49840351405015
9.6.C,9.6,Fürth 1,Metabasalt,1016,-3,264.6747659907335
9.6.C,9.6,Fürth 1,Metabasalt,1043,-3,
9.6.C,9.6,Fürth 1,Metabasalt,1079,-3,
9.6.C,9.6,Fürth 1,Metabasalt,1107,-3,
9.6.C,9.6,Fürth 1,Metabasalt,1142,-3,
9.6.C,9.6,Fürth 1,Metabasalt,1171,-3,
9.6.D,9.6,Fürth 1,Metabasalt,28,-3,243.50078464408207
9.6.D,9.6,Fürth 1,Metabasalt,58,-3,205.86884520127563
9.6.D,9.6,Fürth 1,Metabasalt,84,-3,133.54045008724952
9.6.D,9.6,Fürth 1,Metabasalt,112,-3,612.9867578073289
9.6.D,9.6,Fürth 1,Metabasalt,147,-3,0.0
9.6.D,9.6,Fürth 1,Metabasalt,175,-3,697.2977014260786
9.6.D,9.6,Fürth 1,Metabasalt,196,-3,371.8921075877008
9.6.D,9.6,Fürth 1,Metabasalt,226,-3,275.4542473072989
9.6.D,9.6,Fürth 1,Metabasalt,253,-3,150.86461657139418
9.6.D,9.6,Fürth 1,Metabasalt,280,-3,196.34055363138577
9.6.D,9.6,Fürth 1,Metabasalt,308,-3,230.41141440519883
9.6.D,9.6,Fürth 1,Metabasalt,344,-3,295.5695295745833
9.6.D,9.6,Fürth 1,Metabasalt,381,-3,205.4838637222456
9.6.D,9.6,Fürth 1,Metabasalt,415,-3,267.5140042120464
9.6.D,9.6,Fürth 1,Metabasalt,448,-3,267.75461772669837
9.6.D,9.6,Fürth 1,Metabasalt,469,-3,89.17133478548648
9.6.D,9.6,Fürth 1,Metabasalt,499,-3,262.74985859558336
9.6.D,9.6,Fürth 1,Metabasalt,529,-3,128.30951424273422
9.6.D,9.6,Fürth 1,Metabasalt,561,-3,174.13674689860537
9.6.D,9.6,Fürth 1,Metabasalt,599,-3,181.5572648895842
9.6.D,9.6,Fürth 1,Metabasalt,623,-3,113.50457030186452
9.6.D,9.6,Fürth 1,Metabasalt,653,-3,186.30697387327757
9.6.D,9.6,Fürth 1,Metabasalt,682,-3,161.02812758890428
9.7.A,9.7,Fürth 1,Peridotite,30,-3,262.028018292316
9.7.A,9.7,Fürth 1,Peridotite,58,-3,198.79481054215051
9.7.A,9.7,Fürth 1,Peridotite,84,-3,113.85827202599434
9.7.A,9.7,Fürth 1,Peridotite,112,-3,631.1290099283952
9.7.A,9.7,Fürth 1,Peridotite,147,-3,0.0
9.7.A,9.7,Fürth 1,Peridotite,175,-3,724.2464049581804
9.7.A,9.7,Fürth 1,Peridotite,196,-3,464.86513436428186
9.7.A,9.7,Fürth 1,Peridotite,226,-3,224.34795614657924
9.7.A,9.7,Fürth 1,Peridotite,254,-3,267.5140042120464
9.7.A,9.7,Fürth 1,Peridotite,280,-3,201.15282209519225
9.7.A,9.7,Fürth 1,Peridotite,308,-3,444.6536068355497
9.7.A,9.7,Fürth 1,Peridotite,344,-3,311.5462607858475
9.7.A,9.7,Fürth 1,Peridotite,381,-3,205.0026368854925
9.7.A,9.7,Fürth 1,Peridotite,415,-3,262.36487706841564
9.7.A,9.7,Fürth 1,Peridotite,448,-3,221.36434969613092
9.7.A,9.7,Fürth 1,Peridotite,469,-3,108.75726746494976
9.7.A,9.7,Fürth 1,Peridotite,499,-3,275.261756543715
9.7.A,9.7,Fürth 1,Peridotite,529,-3,162.46218359708766
9.7.A,9.7,Fürth 1,Peridotite,561,-3,267.34798106109463
9.7.A,9.7,Fürth 1,Peridotite,599,-3,212.22103958120223
9.7.A,9.7,Fürth 1,Peridotite,623,-3,117.22685996554452
9.7.A,9.7,Fürth 1,Peridotite,653,-3,153.89634572477286
9.7.A,9.7,Fürth 1,Peridotite,682,-3,140.78772640953125
9.7.A,9.7,Fürth 1,Peridotite,716,-3,196.10956473915397
9.7.A,9.7,Fürth 1,Peridotite,742,-3,81.83262535651964
9.7.A,9.7,Fürth 1,Peridotite,771,-3,56.611526301221495
9.7.A,9.7,Fürth 1,Peridotite,798,-3,100.24917679764124
9.7.A,9.7,Fürth 1,Peridotite,833,-3,172.49095108008905
9.7.A,9.7,Fürth 1,Peridotite,868,-3,415.01003285396234
9.7.A,9.7,Fürth 1,Peridotite,899,-3,347.7826425176003
9.7.A,9.7,Fürth 1,Peridotite,926,-3,161.76921694446116
9.7.A,9.7,Fürth 1,Peridotite,955,-3,254.4246339731632
9.7.A,9.7,Fürth 1,Peridotite,987,-3,228.59237691798543
9.7.A,9.7,Fürth 1,Peridotite,1016,-3,219.15070617967385
9.7.A,9.7,Fürth 1,Peridotite,1043,-3,
9.7.A,9.7,Fürth 1,Peridotite,1079,-3,
9.7.A,9.7,Fürth 1,Peridotite,1107,-3,
9.7.A,9.7,Fürth 1,Peridotite,1142,-3,
9.7.A,9.7,Fürth 1,Peridotite,1171,-3,
9.7.B,9.7,Fürth 1,Peridotite,30,-3,259.38127059389853
9.7.B,9.7,Fürth 1,Peridotite,58,-3,198.16921564474396
9.7.B,9.7,Fürth 1,Peridotite,84,-3,86.2839736927613
9.7.B,9.7,Fürth 1,Peridotite,112,-3,549.8497955352308
9.7.B,9.7,Fürth 1,Peridotite,147,-3,0.0
9.7.B,9.7,Fürth 1,Peridotite,175,-3,628.0010354413623
9.7.B,9.7,Fürth 1,Peridotite,196,-3,411.7376903544136
9.7.B,9.7,Fürth 1,Peridotite,226,-3,178.53516028641914
9.7.B,9.7,Fürth 1,Peridotite,254,-3,177.09147975209098
9.7.B,9.7,Fürth 1,Peridotite,280,-3,106.25488785125458
9.7.B,9.7,Fürth 1,Peridotite,308,-3,317.80220975991335
9.7.B,9.7,Fürth 1,Peridotite,344,-3,287.00369167819963
9.7.B,9.7,Fürth 1,Peridotite,381,-3,183.25118339250255
9.7.B,9.7,Fürth 1,Peridotite,415,-3,209.09306509416933
9.7.B,9.7,Fürth 1,Peridotite,448,-3,230.7001505265058
9.7.B,9.7,Fürth 1,Peridotite,469,-3,99.61395735002104
9.7.B,9.7,Fürth 1,Peridotite,499,-3,251.4410277393345
9.7.B,9.7,Fürth 1,Peridotite,529,-3,161.02812758890428
9.7.B,9.7,Fürth 1,Peridotite,561,-3,316.20694284913117
9.7.B,9.7,Fürth 1,Peridotite,599,-3,237.7260624827005
9.7.B,9.7,Fürth 1,Peridotite,623,-3,120.73740981101233
9.7.B,9.7,Fürth 1,Peridotite,653,-3,156.33135355917923
9.7.B,9.7,Fürth 1,Peridotite,682,-3,167.0771490703412
9.7.C,9.7,Fürth 1,Peridotite,30,-3,259.5737613574824
9.7.C,9.7,Fürth 1,Peridotite,58,-3,219.29507424032732
9.7.C,9.7,Fürth 1,Peridotite,84,-3,117.65996413743306
9.7.C,9.7,Fürth 1,Peridotite,112,-3,579.0121424875142
9.7.C,9.7,Fürth 1,Peridotite,147,-3,0.0
9.7.C,9.7,Fürth 1,Peridotite,175,-3,823.860362235995
9.7.C,9.7,Fürth 1,Peridotite,196,-3,404.23055153739693
9.7.C,9.7,Fürth 1,Peridotite,226,-3,204.52141002467056
9.7.C,9.7,Fürth 1,Peridotite,254,-3,206.92754428064265
9.7.C,9.7,Fürth 1,Peridotite,280,-3,117.32310533726458
9.7.C,9.7,Fürth 1,Peridotite,308,-3,256.7826456465491
9.7.C,9.7,Fürth 1,Peridotite,344,-3,321.84451531379744
9.7.C,9.7,Fürth 1,Peridotite,381,-3,200.23849110054755
9.7.C,9.7,Fürth 1,Peridotite,415,-3,266.5034278837475
9.7.C,9.7,Fürth 1,Peridotite,448,-3,228.58275239184064
9.7.C,9.7,Fürth 1,Peridotite,469,-3,132.818609808051
9.7.C,9.7,Fürth 1,Peridotite,499,-3,268.52458078103376
9.7.C,9.7,Fürth 1,Peridotite,529,-3,156.6874614357061
9.7.C,9.7,Fürth 1,Peridotite,561,-3,293.788990121764
9.7.C,9.7,Fürth 1,Peridotite,599,-3,256.686400144413
9.7.C,9.7,Fürth 1,Peridotite,623,-3,146.56244854029737
9.7.C,9.7,Fürth 1,Peridotite,653,-3,160.26778918105782
9.7.C,9.7,Fürth 1,Peridotite,682,-3,159.56038570311088
9.7.C,9.7,Fürth 1,Peridotite,716,-3,216.9611240387508
9.7.C,9.7,Fürth 1,Peridotite,742,-3,96.77953123533304
9.7.C,9.7,Fürth 1,Peridotite,771,-3,18.10375399001143
9.7.C,9.7,Fürth 1,Peridotite,798,-3,189.2472698959023
9.7.C,9.7,Fürth 1,Peridotite,833,-3,232.77905050845416
9.7.C,9.7,Fürth 1,Peridotite,868,-3,401.22769601059025
9.7.C,9.7,Fürth 1,Peridotite,899,-3,269.4870343582646
9.7.C,9.7,Fürth 1,Peridotite,926,-3,162.90491230519285
9.7.C,9.7,Fürth 1,Peridotite,955,-3,291.3491699861604
9.7.C,9.7,Fürth 1,Peridotite,987,-3,245.17545411877973
9.7.C,9.7,Fürth 1,Peridotite,1016,-3,238.64520577652084
9.7.C,9.7,Fürth 1,Peridotite,1043,-3,
9.7.C,9.7,Fürth 1,Peridotite,1079,-3,
9.7.C,9.7,Fürth 1,Peridotite,1107,-3,
9.7.C,9.7,Fürth 1,Peridotite,1142,-3,
9.7.C,9.7,Fürth 1,Peridotite,1171,-3,
9.7.D,9.7,Fürth 1,Peridotite,28,-3,237.2448356459474
9.7.D,9.7,Fürth 1,Peridotite,58,-3,205.0026368854925
9.7.D,9.7,Fürth 1,Peridotite,84,-3,85.65837879535471
9.7.D,9.7,Fürth 1,Peridotite,112,-3,516.837633792647
9.7.D,9.7,Fürth 1,Peridotite,147,-3,0.0
9.7.D,9.7,Fürth 1,Peridotite,175,-3,723.5245646549131
9.7.D,9.7,Fürth 1,Peridotite,196,-3,368.4272740838799
9.7.D,9.7,Fürth 1,Peridotite,226,-3,140.95134352247428
9.7.D,9.7,Fürth 1,Peridotite,254,-3,186.2347898429508
9.7.D,9.7,Fürth 1,Peridotite,280,-3,125.11898025151932
9.7.D,9.7,Fürth 1,Peridotite,308,-3,242.2977175521993
9.7.D,9.7,Fürth 1,Peridotite,344,-3,277.18666393886514
9.7.D,9.7,Fürth 1,Peridotite,381,-3,203.1258521692039
9.7.D,9.7,Fürth 1,Peridotite,415,-3,237.19671296708583
9.7.D,9.7,Fürth 1,Peridotite,448,-3,210.34425488898248
9.7.D,9.7,Fürth 1,Peridotite,469,-3,130.60496631566278
9.7.D,9.7,Fürth 1,Peridotite,499,-3,267.8027404777664
9.7.D,9.7,Fürth 1,Peridotite,529,-3,142.75113193332933
9.7.D,9.7,Fürth 1,Peridotite,561,-3,237.0812185081683
9.7.D,9.7,Fürth 1,Peridotite,599,-3,247.51902882243215
9.7.D,9.7,Fürth 1,Peridotite,623,-3,137.9677370717254
9.7.D,9.7,Fürth 1,Peridotite,653,-3,230.1659887117155
9.7.D,9.7,Fürth 1,Peridotite,682,-3,214.64642288946388
9.8.A,9.8,Fürth 1,Steel Slag,30,-3,530.4082308201456
9.8.A,9.8,Fürth 1,Steel Slag,58,-3,375.3569408508334
9.8.A,9.8,Fürth 1,Steel Slag,84,-3,24.826493045582065
9.8.A,9.8,Fürth 1,Steel Slag,112,-3,1083.9634731331607
9.8.A,9.8,Fürth 1,Steel Slag,147,-3,0.0
9.8.A,9.8,Fürth 1,Steel Slag,175,-3,1054.36802214333
9.8.A,9.8,Fürth 1,Steel Slag,196,-3,779.0100200974787
9.8.A,9.8,Fürth 1,Steel Slag,226,-3,261.4986687526325
9.8.A,9.8,Fürth 1,Steel Slag,254,-3,410.5827460135989
9.8.A,9.8,Fürth 1,Steel Slag,280,-3,347.4457837415007
9.8.A,9.8,Fürth 1,Steel Slag,308,-3,556.2982352728804
9.8.A,9.8,Fürth 1,Steel Slag,344,-3,524.056036584632
9.8.A,9.8,Fürth 1,Steel Slag,381,-3,461.9777731512125
9.8.A,9.8,Fürth 1,Steel Slag,415,-3,478.3394861303327
9.8.A,9.8,Fürth 1,Steel Slag,448,-3,514.3352541067453
9.8.A,9.8,Fürth 1,Steel Slag,469,-3,225.0216737228473
9.8.A,9.8,Fürth 1,Steel Slag,499,-3,502.4008284493652
9.8.A,9.8,Fürth 1,Steel Slag,529,-3,332.0946473313677
9.8.A,9.8,Fürth 1,Steel Slag,561,-3,481.6479207098035
9.8.A,9.8,Fürth 1,Steel Slag,599,-3,405.2700015644744
9.8.A,9.8,Fürth 1,Steel Slag,623,-3,232.70686646062467
9.8.A,9.8,Fürth 1,Steel Slag,653,-3,327.5999884469583
9.8.A,9.8,Fürth 1,Steel Slag,682,-3,288.44737228473434
9.8.A,9.8,Fürth 1,Steel Slag,716,-3,379.97671845478067
9.8.A,9.8,Fürth 1,Steel Slag,742,-3,107.63119663036284
9.8.A,9.8,Fürth 1,Steel Slag,771,-3,119.76773772188456
9.8.A,9.8,Fürth 1,Steel Slag,798,-3,327.9560962753475
9.8.A,9.8,Fürth 1,Steel Slag,833,-3,406.13620988025747
9.8.A,9.8,Fürth 1,Steel Slag,868,-3,694.7568236355978
9.8.A,9.8,Fürth 1,Steel Slag,899,-3,672.379775437752
9.8.A,9.8,Fürth 1,Steel Slag,926,-3,417.97439027618987
9.8.A,9.8,Fürth 1,Steel Slag,955,-3,586.1342997773633
9.8.A,9.8,Fürth 1,Steel Slag,987,-3,428.0657173115109
9.8.A,9.8,Fürth 1,Steel Slag,1016,-3,407.7964426259101
9.8.A,9.8,Fürth 1,Steel Slag,1043,-3,
9.8.A,9.8,Fürth 1,Steel Slag,1079,-3,
9.8.A,9.8,Fürth 1,Steel Slag,1107,-3,
9.8.A,9.8,Fürth 1,Steel Slag,1142,-3,
9.8.A,9.8,Fürth 1,Steel Slag,1171,-3,
9.8.B,9.8,Fürth 1,Steel Slag,30,-3,635.2194382333473
9.8.B,9.8,Fürth 1,Steel Slag,58,-3,216.26334511101751
9.8.B,9.8,Fürth 1,Steel Slag,84,-3,13.224835600985962
9.8.B,9.8,Fürth 1,Steel Slag,112,-3,519.7249950057163
9.8.B,9.8,Fürth 1,Steel Slag,147,-3,0.0
9.8.B,9.8,Fürth 1,Steel Slag,175,-3,675.1612665021963
9.8.B,9.8,Fürth 1,Steel Slag,196,-3,680.2141483843793
9.8.B,9.8,Fürth 1,Steel Slag,226,-3,328.7260593296829
9.8.B,9.8,Fürth 1,Steel Slag,254,-3,441.2368960827968
9.8.B,9.8,Fürth 1,Steel Slag,280,-3,397.0121489861002
9.8.B,9.8,Fürth 1,Steel Slag,308,-3,529.3495317407786
9.8.B,9.8,Fürth 1,Steel Slag,344,-3,494.2199720801492
9.8.B,9.8,Fürth 1,Steel Slag,381,-3,334.11579998796554
9.8.B,9.8,Fürth 1,Steel Slag,415,-3,376.8006212166797
9.8.B,9.8,Fürth 1,Steel Slag,448,-3,324.8281217883146
9.8.B,9.8,Fürth 1,Steel Slag,469,-3,154.6181859799025
9.8.B,9.8,Fürth 1,Steel Slag,499,-3,488.1083911185992
9.8.B,9.8,Fürth 1,Steel Slag,529,-3,292.3741832841928
9.8.B,9.8,Fürth 1,Steel Slag,561,-3,471.0007766063198
9.8.B,9.8,Fürth 1,Steel Slag,599,-3,385.13065792165594
9.8.B,9.8,Fürth 1,Steel Slag,623,-3,253.60414222263913
9.8.B,9.8,Fürth 1,Steel Slag,653,-3,303.01892099404296
9.8.B,9.8,Fürth 1,Steel Slag,682,-3,490.1247314519525
9.8.C,9.8,Fürth 1,Steel Slag,30,-3,318.1871912870811
9.8.C,9.8,Fürth 1,Steel Slag,58,-3,205.58010910403752
9.8.C,9.8,Fürth 1,Steel Slag,84,-3,5.745848554909899
9.8.C,9.8,Fürth 1,Steel Slag,112,-3,666.9804101329803
9.8.C,9.8,Fürth 1,Steel Slag,147,-3,0.0
9.8.C,9.8,Fürth 1,Steel Slag,175,-3,1173.2791761237138
9.8.C,9.8,Fürth 1,Steel Slag,196,-3,684.0158404236115
9.8.C,9.8,Fürth 1,Steel Slag,226,-3,337.82124676575006
9.8.C,9.8,Fürth 1,Steel Slag,254,-3,374.20199626933027
9.8.C,9.8,Fürth 1,Steel Slag,280,-3,407.93599831518145
9.8.C,9.8,Fürth 1,Steel Slag,308,-3,486.6165877609964
9.8.C,9.8,Fürth 1,Steel Slag,344,-3,531.7075434141645
9.8.C,9.8,Fürth 1,Steel Slag,381,-3,403.0274844455142
9.8.C,9.8,Fürth 1,Steel Slag,415,-3,388.97566062940007
9.8.C,9.8,Fürth 1,Steel Slag,448,-3,395.3278548649136
9.8.C,9.8,Fürth 1,Steel Slag,469,-3,113.85827202599434
9.8.C,9.8,Fürth 1,Steel Slag,499,-3,431.51611384559834
9.8.C,9.8,Fürth 1,Steel Slag,529,-3,305.2903118117817
9.8.C,9.8,Fürth 1,Steel Slag,561,-3,491.188242892016
9.8.C,9.8,Fürth 1,Steel Slag,599,-3,401.4490604729526
9.8.C,9.8,Fürth 1,Steel Slag,623,-3,240.2861893062439
9.8.C,9.8,Fürth 1,Steel Slag,653,-3,284.94404067633434
9.8.C,9.8,Fürth 1,Steel Slag,682,-3,301.8014171731151
9.8.C,9.8,Fürth 1,Steel Slag,716,-3,310.4490636019014
9.8.C,9.8,Fürth 1,Steel Slag,742,-3,127.19788023346771
9.8.C,9.8,Fürth 1,Steel Slag,771,-3,178.9201417654492
9.8.C,9.8,Fürth 1,Steel Slag,798,-3,286.7727028100367
9.8.C,9.8,Fürth 1,Steel Slag,833,-3,402.06503086828326
9.8.C,9.8,Fürth 1,Steel Slag,868,-3,669.9447675552078
9.8.C,9.8,Fürth 1,Steel Slag,899,-3,410.2940099885673
9.8.C,9.8,Fürth 1,Steel Slag,926,-3,245.23320127564836
9.8.C,9.8,Fürth 1,Steel Slag,955,-3,282.96138612431554
9.8.C,9.8,Fürth 1,Steel Slag,987,-3,319.13039605271075
9.8.C,9.8,Fürth 1,Steel Slag,1016,-3,213.5684747578073
9.8.C,9.8,Fürth 1,Steel Slag,1043,-3,
9.8.C,9.8,Fürth 1,Steel Slag,1079,-3,
9.8.C,9.8,Fürth 1,Steel Slag,1107,-3,
9.8.D,9.8,Fürth 1,Steel Slag,28,-3,260.58433768578135
9.8.D,9.8,Fürth 1,Steel Slag,58,-3,151.5864568505927
9.8.D,9.8,Fürth 1,Steel Slag,84,-3,0.6015335589111689
9.8.D,9.8,Fürth 1,Steel Slag,112,-3,572.1787212226968
9.8.D,9.8,Fürth 1,Steel Slag,147,-3,0.0
9.8.D,9.8,Fürth 1,Steel Slag,175,-3,973.6181570491608
9.8.D,9.8,Fürth 1,Steel Slag,196,-3,690.5605256633974
9.8.D,9.8,Fürth 1,Steel Slag,226,-3,237.96667590107705
9.8.D,9.8,Fürth 1,Steel Slag,254,-3,257.11950442264873
9.8.D,9.8,Fürth 1,Steel Slag,280,-3,218.28449786389072
9.8.D,9.8,Fürth 1,Steel Slag,308,-3,402.3056441422467
9.8.D,9.8,Fürth 1,Steel Slag,344,-3,475.5483704193995
9.8.D,9.8,Fürth 1,Steel Slag,381,-3,363.8074964799326
9.8.D,9.8,Fürth 1,Steel Slag,415,-3,383.77841049401286
9.8.D,9.8,Fürth 1,Steel Slag,448,-3,293.837112943017
9.8.D,9.8,Fürth 1,Steel Slag,469,-3,122.7128460196161
9.8.D,9.8,Fürth 1,Steel Slag,499,-3,357.3590567422829
9.8.D,9.8,Fürth 1,Steel Slag,529,-3,232.36038315181415
9.8.D,9.8,Fürth 1,Steel Slag,561,-3,398.9466807760631
9.8.D,9.8,Fürth 1,Steel Slag,599,-3,357.23874986461277
9.8.D,9.8,Fürth 1,Steel Slag,623,-3,195.50321886788367
9.8.D,9.8,Fürth 1,Steel Slag,653,-3,272.50913893736083
9.8.D,9.8,Fürth 1,Steel Slag,682,-3,241.99935688067873
'''


In [ ]:
YCOL = "Cumulative HCO3- mmol/m2"          # meq/m²
PAPER_DAYMAX = 660                          # paper data cutoff: last paper sampling ~day 652
                                            # (our next point is ~682); on the GLOBAL timeline this
                                            # matches the paper for every panel incl. the +365-shifted B-soils
ENDWIN = 60                                 # endpoint window (last N days) for stats
# late second-batch soils whose notebook `days` use a 2024-01-30 reference (see prep())

# paper rock aliases -> our rock labels
ALIAS = {"Basalt": "Basanite", "Diabase": "Metabasalt", "Dunite": "Peridotite",
         "Steel Slag": "Steel Slag", "Basanite": "Basanite", "Metabasalt": "Metabasalt",
         "Peridotite": "Peridotite"}

# consistent colours / styles across panels (paper-like)
COL = {"Control": "#1f4fd8", "Basanite": "#e07b39", "Metabasalt": "#2ca02c",
       "Peridotite": "#c0392b", "Steel Slag": "#7d3c98", "Limestone 10": "#17becf",
       "Glacial Dust": "#8c564b", "Cement": "#555555",
       "Basanite 20": "#f0a35e", "Basanite 100": "#e07b39", "Basanite 200": "#b5651d",
       "Basanite 400": "#7a3f10",
       # cross-soil (by soil)
       "LUFA 6S A": "#1f4fd8", "LUFA 2.2 A": "#e07b39", "LUFA 2.1": "#2ca02c", "Fürth 1": "#c0392b"}
STY = {"Control": "-", "Basanite": "--", "Metabasalt": ":", "Peridotite": "-.",
       "Steel Slag": (0, (3, 1, 1, 1)), "Limestone 10": (0, (5, 1)), "Glacial Dust": (0, (1, 1)),
       "Cement": (0, (3, 1, 1, 1, 1, 1)),
       "Basanite 20": ":", "Basanite 100": "-.", "Basanite 200": ":", "Basanite 400": (0, (3, 1, 1, 1)),
       "LUFA 6S A": "-", "LUFA 2.2 A": "--", "LUFA 2.1": ":", "Fürth 1": "-."}

# ── 15-panel spec ─────────────────────────────────────────────────────────────
# kind: control | rate | crosssoil
# For control/rate: soil + list of (rock_in_data, legend_label)
# For crosssoil : rock + list of (soil_in_data, legend_label), baseline first
P = []
def control(png, soil, title, ctrl_label, treatments):
    P.append(dict(kind="control", png=png, soil=soil, title=title, ctrl_label=ctrl_label,
                  treatments=treatments))
def crosssoil(png, rock, title, series):
    P.append(dict(kind="crosssoil", png=png, rock=rock, title=title, series=series))

RCK = [("Basanite", "+Basanite"), ("Metabasalt", "+Metabasalt"),
       ("Peridotite", "+Peridotite"), ("Steel Slag", "+Steel Slag")]

control("01_LUFA_6S_A.png", "LUFA 6S A", "Greenhouse Experiment LUFA 6S A Soil",
        "LUFA 6S A Control", RCK)
control("02_F_rth_1.png", "Fürth 1", "Greenhouse Experiment Fürth 1 soil", "Fürth 1 Control",
        [("Basanite", "+Basanite 40 t/ha"), ("Metabasalt", "+Metabasalt 40 t/ha"),
         ("Peridotite", "+Peridotite 40 t/ha"), ("Steel Slag", "+Steel Slag 40 t/ha")])
control("03_F_rth_1.png", "Fürth 1", "Greenhouse Experiment Fürth 1 soil", "Fürth 1 Control",
        [("Basanite", "+Basalt 40 t/ha"), ("Basanite 200", "+Basalt 200 t/ha"),
         ("Basanite 400", "+Basalt 400 t/ha")])
control("04_F_rth_2.png", "Fürth 2A", "Greenhouse Experiment Fürth 2 soil", "Fürth 2 Control",
        [(f"EW {i}", f"+EW {i} 40 t/ha") for i in range(1, 8)])
control("05_F_rth_2.png", "Fürth 2B", "Greenhouse Experiment Fürth 2 soil", "Fürth 2 Control B",
        [("Basanite", "+Basanite 40 t/ha"), ("Basanite 200", "+Basanite 200 t/ha")])
control("06_LUFA_2_1.png", "LUFA 2.1", "Greenhouse Experiment LUFA 2.1 soil", "LUFA 2.1 Control", RCK)
control("07_LUFA_2_2_A.png", "LUFA 2.2 A", "Greenhouse Experiment LUFA 2.2 A soil",
        "LUFA 2.2 A Control", RCK)
crosssoil("08_LUFA_6S_Basalt.png", "Basanite", "Greenhouse Experiment",
          [("LUFA 6S A", "LUFA 6S + Basalt"), ("LUFA 2.2 A", "LUFA 2.2 + Basalt"),
           ("LUFA 2.1", "LUFA 2.1 + Basalt"), ("Fürth 1", "Fürth + Basalt")])
crosssoil("09_LUFA_6S_Diabase.png", "Metabasalt", "Greenhouse Experiment",
          [("LUFA 6S A", "LUFA 6S + Diabase"), ("LUFA 2.2 A", "LUFA 2.2 + Diabase"),
           ("LUFA 2.1", "LUFA 2.1 + Diabase"), ("Fürth 1", "Fürth + Diabase")])
crosssoil("10_LUFA_6S_Dunite.png", "Peridotite", "Greenhouse Experiment",
          [("LUFA 6S A", "LUFA 6S + Dunite"), ("LUFA 2.2 A", "LUFA 2.2 + Dunite"),
           ("LUFA 2.1", "LUFA 2.1 + Dunite"), ("Fürth 1", "Fürth + Dunite")])
crosssoil("11_LUFA_6S_Steel_Slag.png", "Steel Slag", "Greenhouse Experiment",
          [("LUFA 6S A", "LUFA 6S + Steel Slag"), ("LUFA 2.2 A", "LUFA 2.2 + Steel Slag"),
           ("LUFA 2.1", "LUFA 2.1 + Steel Slag"), ("Fürth 1", "Fürth + Steel Slag")])
control("12_LUFA_2_2_B.png", "LUFA 2.2 B", "Greenhouse Experiment LUFA 2.2 B Soil",
        "LUFA 2.2 B Control",
        [("Basanite", "+Basanite 40 t/ha"), ("Limestone 10", "+Limestone 10 40 t/ha"),
         ("Peridotite", "+Peridotite 40 t/ha"), ("Steel Slag", "+Steel Slag 40 t/ha"),
         ("Glacial Dust", "+Glacial Dust 40 t/ha")])  # paper omits Cement
control("13_LUFA_6S_B.png", "LUFA 6S B", "Greenhouse Experiment LUFA 6S B Soil", "LUFA 6S B Control",
        [("Basanite", "+Basanite"), ("Limestone 10", "+Limestone 10"),
         ("Peridotite", "+Peridotite"), ("Steel Slag", "+Steel Slag")])
control("15_Farmer_2.png", "Farmer 2", "Greenhouse Experiment Farmer 2 soil", "Farmer 2 Control",
        [("Basanite", "+Basanite")])
control("24_Bramstedt.png", "Bramstedt", "Greenhouse Experiment Bramstedt soil", "Bramstedt Control",
        [("Basanite", "+Basanite")])

# fallback palette for treatments lacking an explicit colour (e.g. EW 1..7)
CYCLE_COL = ["#e07b39", "#2ca02c", "#c0392b", "#7d3c98", "#17becf", "#8c564b", "#bcbd22", "#e377c2"]
CYCLE_STY = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1)), (0, (3, 1, 1, 1, 1, 1)), "--"]

# ── stats — parameters ported from the paper's own plot code ──────────────────
ENDPOINT_WINDOW = 45   # paper's endpoint window (days): mean each replicate's last 45 d

def hedges_g(a, b):
    """Hedges' g with the paper's exact small-sample J correction."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    sp = math.sqrt(((na - 1) * np.std(a, ddof=1) ** 2 + (nb - 1) * np.std(b, ddof=1) ** 2)
                   / (na + nb - 2))
    if sp == 0:
        return np.inf
    d = (np.mean(a) - np.mean(b)) / sp
    J = 1 - (3 / (4 * (na + nb - 2) - 1))
    return d * J

def min_achievable_p(na, nb):
    from math import comb
    return 2.0 / comb(na + nb, na)

def endpoint_values(sub, last_day, window=ENDPOINT_WINDOW):
    """One value per replicate: mean of its last `window` days (paper method)."""
    t = sub[sub["days"] >= (last_day - window)]
    return t.groupby("Column/Pot")[YCOL].mean().dropna().values

def interpolate_replicate(rep_df, all_days, creation_day=None):
    """Per-replicate cumulative on a daily grid, defined so the mean is monotone and spans the
    full range (band + endpoint star correct):
      - the line begins at the pot's CREATION day at value 0 (nothing leached yet); before
        creation the pot does not exist -> NaN (excluded from the mean),
      - interpolate linearly between actual measurements (paper method, inside),
      - hold the last value after the pot's last sampling (LOCF) -> no tail dip/truncation.
    Every replicate is then monotone from its creation onward, so their mean is monotone too."""
    s = (rep_df.sort_values("days").groupby("days")[YCOL].mean().reindex(all_days))
    # Anchor value 0 at the pot's creation day, then interpolate: the ramp-up rises linearly
    # from 0 at creation to the first measurement (no vertical step), and stays interpolated
    # between later measurements. Before creation the pot does not exist -> NaN (excluded).
    if creation_day is not None and creation_day in s.index and pd.isna(s.loc[creation_day]):
        s.loc[creation_day] = 0.0
    s = s.interpolate(method="index", limit_area="inside")
    return s.ffill()                      # hold last value after the pot ends (NaN before creation stays NaN)

def stagger_positions(base, min_gap):
    placed, prev = {}, -np.inf
    for k in sorted(base, key=lambda k: base[k]):
        y = base[k]
        if y - prev < min_gap:
            y = prev + min_gap
        placed[k] = y; prev = y
    return placed

def prep(data):
    """Fix the trailing-space pot-ID bug: the SAME physical pot appears as e.g.
    '0.0.A' and '0.0.A ' across its time series, which (a) double-counts replicates
    (n=10 where the paper has 7) and (b) corrupts the per-pot cumulative (cumsum resets
    at the space variant). Trim the ID and recompute the cumulative from per-sampling
    HCO3- mmol/m2. Matches the owner's Sample_ID/Column-Pot cleanup in the lab master."""
    data = data.copy()
    data["Column/Pot"] = data["Column/Pot"].astype(str).str.strip()
    if "HCO3- mmol/m2" in data.columns:
        data = data.sort_values(["Column/Pot", "days"])
        # An exact 0.000 per-sampling alkalinity is a "not measured" placeholder (real leachate
        # carries some alkalinity) -> treat as missing so it neither adds a zero-increment
        # (horizontal cumulative segment) nor a spurious point; the curve interpolates across it.
        inc = data["HCO3- mmol/m2"].mask(data["HCO3- mmol/m2"] == 0)
        data[YCOL] = inc.groupby(data["Column/Pot"]).cumsum()
    # `days` is now global (single 2023-01-30 reference) and `creation_day` (per pot, same
    # reference) marks where each pot's line begins at 0 — see Date_of_Creation_for_each_pot.
    # No per-soil day shift needed (the global reference already places the 2024 B-batch at
    # ~day 365+, matching the paper).
    return data

# ── figure styling ────────────────────────────────────────────────────────────
def paper_figsize(png):
    # The paper's own code uses figsize=(7, 5) for every panel. Matching it is what makes
    # the font-to-figure ratio (and thus line weights) look the same as the originals.
    return (7, 5)

def plot_treatment(ax, sub, color, ls, label, all_days):
    """Per-replicate daily interpolation + mean±SE (paper method). Returns mean endpoint y.
    Cumulative can't fall: each replicate is monotone; the mean only ever drops when a pot
    drops out — the 'inside' interpolation makes that a clean line-end, not an artefact."""
    rep_series = []
    for rep in sub["Column/Pot"].unique():
        rdf = sub[sub["Column/Pot"] == rep].sort_values("days")
        cd = rdf["creation_day"].iloc[0] if "creation_day" in rdf.columns and len(rdf) else None
        s = interpolate_replicate(rdf, all_days, creation_day=cd)
        start = cd if cd is not None else rdf["days"].min()
        last_meas = rdf["days"].max()
        m = (all_days >= start) & (all_days <= last_meas)   # real measured span only; no LOCF hold-flat tail
        ax.plot(all_days[m], s.values[m], color=color, alpha=0.15, lw=0.85, zorder=1)
        ax.scatter(rdf["days"], rdf[YCOL], color=color, s=6, alpha=0.35, linewidths=0, zorder=2)
        rep_series.append(s)
    rep_df = pd.concat(rep_series, axis=1)
    n_valid = rep_df.notna().sum(axis=1)
    mean = rep_df.mean(axis=1); se = rep_df.sem(axis=1)
    # Every replicate now spans the full window (0-filled start, LOCF tail), so the set is
    # constant and the mean is monotone over the whole range. Draw where >=2 replicates exist.
    pm = (n_valid.values >= 2)
    dplot, mplot, splot = all_days[pm], mean.values[pm], se.values[pm]
    ax.fill_between(dplot, mplot - splot, mplot + splot, color=color, alpha=0.20, zorder=2)
    ax.plot(dplot, mplot, color=color, linestyle=ls, lw=2.2, label=label, zorder=3)
    ld = sub["days"].max()
    ep = sub[sub["days"] == ld]
    ax.scatter(ep["days"], ep[YCOL], color=color, s=32, alpha=0.8, edgecolors="white",
               linewidths=0.4, zorder=5)
    return mplot[-1] if len(mplot) else np.nan

def render_groups(png, title, groups, daymax):
    """Shared renderer (paper style). groups[0] = baseline (control / LUFA 6S); the rest are
    compared to it. Each group = dict(label, sub, color, ls)."""
    sns.set_theme(style="ticks")
    # match the paper's font exactly (it sets DejaVu Sans; seaborn's theme otherwise overrides it)
    plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
    fig, ax = plt.subplots(figsize=paper_figsize(png))
    for g in groups:
        g["sub"] = g["sub"][g["sub"]["days"] <= daymax].copy()
    allsub = pd.concat([g["sub"] for g in groups if not g["sub"].empty])
    last_day = allsub["days"].max()
    # start the daily grid at the earliest pot CREATION day (not the first measurement) so each
    # line can begin at value 0 on its creation day, per the owner's request.
    first_day = allsub["days"].min()
    if "creation_day" in allsub.columns and allsub["creation_day"].notna().any():
        first_day = min(first_day, int(allsub["creation_day"].min()))
    all_days = np.arange(int(first_day), int(last_day) + 1)
    base_end = endpoint_values(groups[0]["sub"], last_day)
    nc_disp = groups[0]["sub"]["Column/Pot"].nunique()
    mean_end, stats_res = {}, {}
    for i, g in enumerate(groups):
        if g["sub"].empty:
            mean_end[i] = None; continue
        mean_end[i] = plot_treatment(ax, g["sub"], g["color"], g["ls"], g["label"], all_days)
        if i == 0:
            continue
        vals = endpoint_values(g["sub"], last_day)
        nt_disp = g["sub"]["Column/Pot"].nunique()
        if len(vals) < 2 or len(base_end) < 2:
            stats_res[i] = dict(g=np.nan, sig=False, nt_disp=nt_disp, min_p=np.nan); continue
        gg = hedges_g(vals, base_end)
        try:
            p = mannwhitneyu(vals, base_end, alternative="two-sided").pvalue
        except Exception:
            p = np.nan
        mp = min_achievable_p(len(vals), len(base_end))
        stats_res[i] = dict(g=gg, sig=(p == p) and p <= mp, nt_disp=nt_disp, min_p=mp)
    # x-limits + staggered stars to the right of endpoints
    xr = last_day - int(all_days.min())
    star_x = last_day + xr * 0.08
    ax.set_xlim(0, star_x + xr * 0.04)
    fig.canvas.draw()
    ylo, yhi = ax.get_ylim(); min_gap = (yhi - ylo) * 0.05
    sig_pos = {i: mean_end[i] for i, sr in stats_res.items() if sr["sig"] and mean_end.get(i) is not None}
    for i, sy in stagger_positions(sig_pos, min_gap).items():
        ax.plot(star_x, sy, marker="*", markersize=12, color=groups[i]["color"],
                markeredgecolor="white", markeredgewidth=0.3, linestyle="none", zorder=7)
    # legend (plain line handles; g + star in label text)
    H, L = [], []
    for i, g in enumerate(groups):
        H.append(Line2D([0, 1], [0, 0], color=g["color"], linestyle=g["ls"], lw=2.2))
        if i == 0:
            L.append(f"{g['label']}  (n={nc_disp})")
        else:
            sr = stats_res.get(i, {}); gg = sr.get("g", np.nan)
            nt = sr.get("nt_disp", "?"); sig = sr.get("sig", False)
            gs = ("  g = {:.1f} *".format(gg) if sig else "  g = {:.1f}".format(gg)) if gg == gg else ""
            ns = f"  (n={nt})" if nt != nc_disp else ""
            L.append(g["label"] + gs + ns)
    H += [Patch(color="none"), Patch(color="gray", alpha=0.35),
          Line2D([0], [0], color="gray", marker="*", markerfacecolor="gray", markersize=9, linestyle="none")]
    L += ["", "±1 SE band", "* sig. vs. control (min. achievable p)"]
    ax.legend(H, L, fontsize=8, loc="upper left", framealpha=0.88, borderpad=0.9, handlelength=2.5)
    # footnote (n-note + min-p, listing distinct min-p as the paper does for LUFA 2.2 B)
    ns = sorted(set(sr["nt_disp"] for sr in stats_res.values())) if stats_res else [nc_disp]
    mps = sorted({round(sr["min_p"], 3) for sr in stats_res.values() if sr["min_p"] == sr["min_p"]})
    n_note = (f"n = {nc_disp} replicates per treatment" if (len(ns) == 1 and ns[0] == nc_disp)
              else f"nominal n = {nc_disp} replicates")
    if not mps:
        mps = [min_achievable_p(nc_disp if nc_disp >= 2 else 4, nc_disp if nc_disp >= 2 else 4)]
    p_note = (f"minimum achievable p = {mps[0]:.3f}." if len(mps) == 1
              else "minimum achievable p = " + " / ".join(f"{v:.3f}" for v in mps) + " (depending on n).")
    fig.text(0.02, 0.006, f"Mann-Whitney U, endpoint values only ({n_note}).\n{p_note} Effect size: Hedges' g.",
             fontsize=6.5, color="#555555", ha="left", va="bottom")
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel("Days since start of experiment", fontsize=10)
    ax.set_ylabel("Accumulated Alkalinity meq/m²", fontsize=10)
    ax.tick_params(labelsize=9)
    # match the paper's faint grey background grid (both axes), drawn below the data
    ax.set_axisbelow(True)
    ax.grid(True, which="major", color="0.88", linewidth=0.7, zorder=0)
    sns.despine(ax=ax)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    # actual drawn stats: star count + g per treatment (in group order) — single source of truth
    our_stars = sum(1 for sr in stats_res.values() if sr.get("sig"))
    our_gs = [stats_res[i]["g"] for i in range(1, len(groups)) if i in stats_res]
    return fig, our_stars, our_gs

def _palette(n):
    return sns.color_palette("dark", n_colors=n)

def _styles(n):
    base = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]
    return list(islice(cycle(base), n))

def render_control(spec, data, daymax):
    s = data[data.soil == spec["soil"]]
    rows = [("Control", spec["ctrl_label"])] + list(spec["treatments"])
    pal, lss = _palette(len(rows)), _styles(len(rows))
    groups = [dict(label=lab, sub=s[s.rock == rock], color=pal[i], ls=lss[i])
              for i, (rock, lab) in enumerate(rows)]
    return render_groups(spec["png"], spec["title"], groups, daymax)

def render_crosssoil(spec, data, daymax):
    rock = spec["rock"]
    pal, lss = _palette(len(spec["series"])), _styles(len(spec["series"]))
    groups = [dict(label=lab, sub=data[(data.soil == soil) & (data.rock == rock)],
                   color=pal[i], ls=lss[i]) for i, (soil, lab) in enumerate(spec["series"])]
    return render_groups(spec["png"], spec["title"], groups, daymax)

def render(spec, data, daymax):
    if spec["kind"] == "crosssoil":
        return render_crosssoil(spec, data, daymax)
    return render_control(spec, data, daymax)

# Per-panel explanatory comment shown on the PDF page where our repro still differs from the
# paper. Update these after the corrected-data import; "" = no material difference expected.
NOTES = {
 "01_LUFA_6S_A.png": "Effect sizes run below the paper because the pot-ID space-bug fix recomputed "
    "(raised) the control cumulative on this soil; ranking & the sig. Steel Slag are preserved.",
 "06_LUFA_2_1.png": "Same pot-ID fix as 01 — magnitudes shift, ranking preserved.",
 "07_LUFA_2_2_A.png": "Same pot-ID fix as 01 — magnitudes shift, ranking preserved.",
 "08_LUFA_6S_Basalt.png": "Cross-soil vs LUFA 6S baseline; A-variants + Fürth 1 assumed. |g| slightly "
    "below paper (pot-ID fix on the baseline soil).",
 "09_LUFA_6S_Diabase.png": "Cross-soil vs LUFA 6S baseline; A-variants + Fürth 1 assumed.",
 "10_LUFA_6S_Dunite.png": "Cross-soil vs LUFA 6S baseline; A-variants + Fürth 1 assumed.",
 "11_LUFA_6S_Steel_Slag.png": "Cross-soil vs LUFA 6S baseline; A-variants + Fürth 1 assumed.",
 "12_LUFA_2_2_B.png": "Second-batch soil: x-axis shifted onto the global 2023 timeline (+365 d) to "
    "match the paper.",
 "13_LUFA_6S_B.png": "Second-batch soil: x-axis shifted onto the global 2023 timeline (+365 d) to "
    "match the paper.",
}

# Paper (Hammes-2025) endpoint stats per panel, paper-window: g per treatment (spec order)
# and number of significant treatments (stars). Computed from the paper notebook (fixed).
# OUR stars/g are taken live from the render (single source of truth), so the banner always
# matches the stars actually drawn on OUR paper-window panel (the MIDDLE graphic).
PAPER_STATS = {
  "01_LUFA_6S_A.png": dict(g=[2.42, 1.64, 1.64, 6.67], stars=2),
  "02_F_rth_1.png": dict(g=[-2.26, -3.75, -3.93, 0.53], stars=3),
  "03_F_rth_1.png": dict(g=[-2.26, -2.37, -1.75], stars=2),
  "04_F_rth_2.png": dict(g=[1.93, 0.79, 1.01, 1.49, 1.75, 2.82, 1.9], stars=3),
  "05_F_rth_2.png": dict(g=[-0.6, -0.31], stars=0),
  "06_LUFA_2_1.png": dict(g=[0.91, 5.43, 1.72, 7.79], stars=2),
  "07_LUFA_2_2_A.png": dict(g=[0.26, 3.28, 0.37, 7.36], stars=2),
  "08_LUFA_6S_Basalt.png": dict(g=[-16.12, -21.55, -10.04], stars=2),
  "09_LUFA_6S_Diabase.png": dict(g=[-8.09, -12.19, -8.98], stars=3),
  "10_LUFA_6S_Dunite.png": dict(g=[-5.28, -8.19, -5.44], stars=3),
  "11_LUFA_6S_Steel_Slag.png": dict(g=[-3.27, -4.2, -6.88], stars=3),
  "12_LUFA_2_2_B.png": dict(g=[2.1, 0.07, 0.1, 5.13, 2.3], stars=3),
  "13_LUFA_6S_B.png": dict(g=[0.98, 0.94, 0.14, 3.18], stars=1),
}
G_THRESH = 0.5  # |Δg| above this counts as a significant g change



In [ ]:
# Render every soil panel (our-repro / paper-replacement figures).
# The 4 cross-soil panels (LUFA 6S + Basalt/Diabase/Dunite/Steel Slag) are omitted, as in the paper set.
data = prep(pd.read_csv(io.StringIO(PERPOT_CSV))).dropna(subset=[YCOL,'days','rock','soil'])
os.makedirs('figures', exist_ok=True)
for spec in P:
    if spec['kind'] == 'crosssoil':
        continue
    fig, _stars, _gs = render(spec, data, PAPER_DAYMAX)
    fig.savefig('figures/'+spec['png'], dpi=150); print('saved', spec['png']); plt.show()


## Bonus figure — Leachate Alkalinity Relative to Control (%)
Endpoint alkalinity of each feedstock as a percentage of its soil's control (mean of replicates, ±1 SE via log-ratio; grey dots = individual replicates).

In [ ]:
ROCK_DISPLAY={'Control':'Control','Basanite':'Basanite 40 t/ha','Metabasalt':'Metabasalt 40 t/ha',
 'Peridotite':'Peridotite 40 t/ha','Steel Slag':'Steel Slag 40 t/ha','Limestone 10':'Limestone 10 t/ha',
 'Glacial Dust':'Glacial Rock Dust 40 t/ha','EW 1':'EW1','EW 2':'EW2','EW 3':'EW3','EW 4':'EW4','EW 5':'EW5','EW 6':'EW6','EW 7':'EW7'}
PALETTE={'Control':'#94aad0','Basanite 40 t/ha':'#ff7f00','Metabasalt 40 t/ha':'#33a02c','Peridotite 40 t/ha':'#d62728',
 'Steel Slag 40 t/ha':'#ad94c7','Limestone 10 t/ha':'#a6d854','Glacial Rock Dust 40 t/ha':'#1f78b4'}
EW_COLOR='#b15928'
HUE_ORDER=['Control','Basanite 40 t/ha','Metabasalt 40 t/ha','Peridotite 40 t/ha','Steel Slag 40 t/ha','Limestone 10 t/ha','Glacial Rock Dust 40 t/ha','EW1','EW2','EW3','EW4','EW5','EW6','EW7']
LEGEND=[(k,PALETTE[k]) for k in HUE_ORDER[:7]]+[('EW1-7 40 t/ha',EW_COLOR)]
ROWS=[("Fürth 1",'Fürth 1',None),("LUFA 2.1",'LUFA 2.1',None),("LUFA 2.2 A",'LUFA 2.2 A',None),
 ("LUFA 2.2 A (<250d)",'LUFA 2.2 A',250),("LUFA 2.2 B (<250d)",'LUFA 2.2 B',250),
 ("LUFA 6S A",'LUFA 6S A',None),("LUFA 6S A (<250d)",'LUFA 6S A',250),
 ("LUFA 6S B (<250d)",'LUFA 6S B',250),("Fürth 2 & EW1-7",'Fürth 2A',None)]
EW_ROW="Fürth 2 & EW1-7"
YCOL="HCO3- mmol/m2"
def _prep(base):
    d=base.copy()
    inc=d[YCOL].mask(d[YCOL]==0)                          # exact-0 = not measured (our convention)
    d=d.sort_values(["Column/Pot","days"])
    d["Cum"]=inc.groupby(d["Column/Pot"]).cumsum()
    d["rel_day"]=d["days"]-d["creation_day"]              # days since each pot's creation
    return d
def _endpoints(d,soil,dm):
    s=d[d["soil"]==soil].copy()
    if dm is not None: s=s[s["rel_day"]<=dm]
    s=s.dropna(subset=["Cum"]).sort_values("rel_day")
    return s.groupby("Column/Pot",observed=True).agg(rock=("rock","last"),TA=("Cum","last")).reset_index()
def rel_mean_se(base):
    d=_prep(base); bars,pts=[],[]
    for label,soil,dm in ROWS:
        ep=_endpoints(d,soil,dm); ctrl=ep[ep.rock=="Control"]["TA"].dropna().to_numpy()
        if ctrl.size==0: continue
        Cm=ctrl.mean(); Cse=ctrl.std(ddof=1)/np.sqrt(ctrl.size) if ctrl.size>1 else 0.0
        for rock,sub in ep.groupby("rock"):
            disp=ROCK_DISPLAY.get(rock)
            if disp is None: continue
            if label==EW_ROW and not (rock=="Control" or rock.startswith("EW ")): continue
            if label!=EW_ROW and str(rock).startswith("EW"): continue
            t=sub["TA"].dropna().to_numpy()
            if t.size==0: continue
            rel=t.mean()/Cm*100; Tse=t.std(ddof=1)/np.sqrt(t.size) if t.size>1 else 0.0
            seln=np.sqrt((Tse/t.mean())**2+(Cse/Cm)**2)
            bars.append(dict(row=label,treat=disp,rel=rel,lo=rel*np.exp(-seln),hi=rel*np.exp(seln),n=t.size))
            for v in t: pts.append(dict(row=label,treat=disp,rel=v/Cm*100))
    return pd.DataFrame(bars),pd.DataFrame(pts)

def plot_relative_alkalinity_big(base, width=10, height=5, save=None):
    bars,pts=rel_mean_se(base); horder={h:i for i,h in enumerate(HUE_ORDER)}
    rows=[r[0] for r in ROWS]; col=lambda t: EW_COLOR if str(t).startswith('EW') else PALETTE[t]
    wrap=lambda s: s.replace(' (','\n(').replace(' & ','\n& ')
    sns.set_theme(style='whitegrid'); plt.rcParams['font.family']='DejaVu Sans'; plt.rcParams['font.sans-serif']=['DejaVu Sans']
    fig,ax=plt.subplots(figsize=(width,height)); BAR_W=0.10; YBOT=50
    present={r:sorted(bars[bars.row==r]['treat'].unique(),key=lambda t:horder[t]) for r in rows}
    xmap={}
    for ri,r in enumerate(rows):
        ts=present[r]; span=len(ts)*BAR_W
        for j,t in enumerate(ts): xmap[(r,t)]=ri-span/2+(j+0.5)*BAR_W
    bi=bars.set_index(['row','treat']); stroke=[pe.withStroke(linewidth=1.6,foreground='white')]
    for (r,t),x in xmap.items():
        row=bi.loc[(r,t)]
        ax.bar(x,row['rel'],width=BAR_W*0.9,color=col(t),zorder=2,edgecolor='white',linewidth=0.3)
        ax.errorbar(x,row['rel'],yerr=[[max(row['rel']-row['lo'],0)],[max(row['hi']-row['rel'],0)]],fmt='none',ecolor='black',elinewidth=0.7,capsize=1.5,zorder=4)
    jit=np.random.default_rng(0)
    for _,p in pts.iterrows():
        k=(p['row'],p['treat'])
        if k in xmap: ax.scatter(xmap[k]+jit.uniform(-BAR_W*0.16,BAR_W*0.16),p['rel'],s=5,color='black',alpha=0.5,zorder=3,linewidths=0)
    ax.axhspan(1,100,color='gray',alpha=0.2,zorder=-1); ax.set_yscale('log'); ax.set_ylim(YBOT,1900)
    ax.set_yticks([100,200,500,1000]); ax.set_yticklabels([f'{v}%' for v in [100,200,500,1000]])
    ax.set_xticks(range(len(rows))); ax.set_xticklabels([wrap(r) for r in rows],rotation=0,ha='center',fontsize=8); ax.set_xlim(-0.5,len(rows)-0.5)
    ax.set_title('Leachate Alkalinity Relative to Control (%)',fontsize=11,pad=20)
    ax.text(0.5,1.015,'bars: mean of replicates · vertical lines: ±1 SE (log-ratio) · grey dots: individual replicates',transform=ax.transAxes,ha='center',va='bottom',fontsize=7.5,color='#555555')
    ax.set_ylabel('Relative Alkalinity (% of Control)',fontsize=9); ax.set_xlabel('Soil Type',fontsize=9)
    ax.yaxis.grid(True,linestyle='--',alpha=0.7); ax.xaxis.grid(False)
    ax.legend(handles=[Patch(facecolor=c,label=l) for l,c in LEGEND],title='Feedstock/Treatment',ncol=1,fontsize=7.5,title_fontsize=8,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.tight_layout()
    if save: fig.savefig(save,dpi=150,bbox_inches='tight'); print("saved",save)
    return bars


In [ ]:
# Relative-alkalinity bar chart (endpoint TA / control TA, per soil).
_rel_df = pd.read_csv(io.StringIO(PERPOT_CSV))
os.makedirs('figures', exist_ok=True)
plot_relative_alkalinity_big(_rel_df, save='figures/00_relative_alkalinity.png')
plt.show()


## Batch reproducibility — accumulated alkalinity vs cumulative leachate volume

Companion check to the cumulative-alkalinity panels above. For the two soils that were run in
**both** planting batches, we plot each pot's accumulated alkalinity against its **cumulative
leachate volume** (not time). Putting the signal on a shared water axis makes the 2024 and 2023
batches directly comparable feedstock-by-feedstock:

- **LUFA 6S** — Table 1 (2024, *LUFA 6S B*) vs Table 5 (2023, *LUFA 6S A*)
- **LUFA 2.2** — Table 0 (2024, *LUFA 2.2 B*) vs Table 6 (2023, *LUFA 2.2 A*)

One facet per feedstock (Control / Basanite / Peridotite / Steel Slag). The data is limited to
the paper time window — the first **653 days** since each batch's own start. Per-pot data
embedded below — self-contained, no external files.

In [ ]:
# Batch reproducibility: accumulated alkalinity vs cumulative leachate volume, 2024 vs 2023
# planting batch, one facet per feedstock. Self-contained (per-pot data embedded below).
import io
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DAYMAX = 653   # paper time window: days since each batch's own start
EXP_LABELS = {'0': 'Control', '2': 'Basanite', '7': 'Peridotite', '8': 'Steel Slag'}
COL_ORDER  = ['Control', 'Basanite', 'Peridotite', 'Steel Slag']
TABLE_YEAR = {'0': '2024', '6': '2023', '1': '2024', '5': '2023'}
TABLE_SOIL = {'0': 'LUFA 2.2 B', '6': 'LUFA 2.2 A', '1': 'LUFA 6S B', '5': 'LUFA 6S A'}

def plot_volume_batch(df, tables):
    d = df[df['Table'].astype(str).isin(tables)].copy()
    d['Table'] = d['Table'].astype(str)
    d['Experiment'] = d['Experiment'].astype(str)
    d = d[d['Experiment'].isin(['0', '2', '7', '8'])]
    d = d[d['days'] <= DAYMAX]                          # limit the data's time base
    d = d.sort_values(by=['Column/Pot'])
    d['Experiment'] = d['Experiment'].map(EXP_LABELS).fillna(d['Experiment'])
    d['Start of Experiment'] = (d['Table'].map(TABLE_YEAR).fillna(d['Table'])
                                + ' (' + d['Table'].map(TABLE_SOIL).fillna(d['Table']) + ')')
    g = sns.lmplot(data=d, x='Cumulative Volume in l', y='Cumulative HCO3- mmol',
                   hue='Start of Experiment', col='Experiment', col_order=COL_ORDER,
                   height=3, col_wrap=4, facet_kws=dict(sharex=True, sharey=True),
                   lowess=True, scatter_kws={'s': 10, 'alpha': 0.7})
    g.set_ylabels('Acc. Alkalinity (meq)')             # legend stays outside the axes (seaborn default)
    plt.show()

VOLUME_CSV = '''
Column/Pot,Table,Experiment,days,Cumulative Volume in l,Cumulative HCO3- mmol
0.0.A,0,0,41,5.4,1.6195
0.0.A,0,0,77,11.3,10.4667
0.0.A,0,0,104,16.1,19.6799
0.0.A,0,0,132,21.85,27.2675
0.0.A,0,0,160,24.3,31.5782
0.0.A,0,0,196,31.65,45.9797
0.0.A,0,0,230,37.82,54.4916
0.0.A,0,0,258,41.77,58.6773
0.0.A,0,0,286,46.12,65.8961
0.0.A,0,0,314,51.26,82.8528
0.0.A,0,0,349,57.14,98.2536
0.0.A,0,0,377,58.89,102.3823
0.0.A,0,0,405,61.08,106.9799
0.0.A,0,0,433,65.11,114.3122
0.0.A,0,0,468,68.94,121.4338
0.0.A,0,0,503,76.65,136.6948
0.0.A,0,0,527,83.47,147.7398
0.0.A,0,0,560,88.27,154.0738
0.0.A,0,0,590,93.17,159.1682
0.0.A,0,0,622,97.17,163.087
0.0.A,0,0,651,100.21,166.5515
0.0.A,0,0,678,103.75,171.0813
0.0.B,0,0,41,5.7,1.3676
0.0.B,0,0,77,11.5,11.1085
0.0.B,0,0,104,16.9,22.0131
0.0.B,0,0,132,22.5,29.5148
0.0.B,0,0,160,24.79,37.3442
0.0.B,0,0,196,30.18,53.2936
0.0.B,0,0,230,39.03,79.6584
0.0.B,0,0,258,42.78,87.1561
0.0.B,0,0,286,47.29,98.8784
0.0.B,0,0,314,52.24,109.1712
0.0.B,0,0,349,57.66,116.215
0.0.B,0,0,377,60.07,118.7688
0.0.B,0,0,405,62.12,120.4493
0.0.B,0,0,433,65.41,122.028
0.0.B,0,0,468,69.58,123.9456
0.0.B,0,0,503,77.92,128.1143
0.0.B,0,0,527,85.33,131.9663
0.0.B,0,0,560,89.54,136.9325
0.0.B,0,0,590,94.46,146.2776
0.0.B,0,0,622,99.17,155.9772
0.0.B,0,0,651,102.04,162.3466
0.0.B,0,0,678,105.33,168.6615
0.0.C,0,0,41,6.0,2.6392
0.0.C,0,0,77,12.2,12.5561
0.0.C,0,0,104,17.2,25.8519
0.0.C,0,0,132,23.85,39.0148
0.0.C,0,0,160,25.85,46.1326
0.0.C,0,0,196,32.51,68.1038
0.0.C,0,0,230,37.06,89.5731
0.0.C,0,0,258,42.1,109.3237
0.0.C,0,0,286,47.1,118.221
0.0.C,0,0,314,52.43,125.6807
0.0.C,0,0,349,57.91,130.6111
0.0.C,0,0,377,60.0,132.7005
0.0.C,0,0,405,62.31,135.2407
0.0.C,0,0,433,64.65,137.8607
0.0.C,0,0,468,65.84,139.431
0.0.C,0,0,503,71.57,146.4194
0.0.C,0,0,527,76.59,151.7389
0.0.C,0,0,560,78.38,152.9558
0.0.C,0,0,590,81.0,154.9463
0.0.C,0,0,622,84.56,157.4376
0.0.C,0,0,651,86.82,159.1998
0.0.C,0,0,678,90.98,162.1109
0.0.D,0,0,41,1.5,0.7198
0.0.D,0,0,77,7.7,7.5377
0.0.D,0,0,104,12.7,11.8363
0.0.D,0,0,132,19.4,14.9174
0.0.D,0,0,160,22.72,17.5061
0.0.D,0,0,196,31.25,24.8397
0.0.D,0,0,230,37.42,28.9106
0.0.D,0,0,258,38.87,30.0412
0.0.D,0,0,286,43.78,32.9863
0.0.D,0,0,314,49.3,36.4077
0.0.D,0,0,349,55.2,39.8286
0.0.E,0,0,41,5.3,1.9074
0.0.E,0,0,77,11.3,13.9037
0.0.E,0,0,104,15.8,26.0499
0.0.E,0,0,132,22.75,39.1118
0.0.E,0,0,160,26.54,47.523
0.0.F,0,0,41,5.0,2.3993
0.0.F,0,0,77,11.1,17.0347
0.0.F,0,0,104,15.8,32.6338
0.0.F,0,0,132,22.55,46.6695
0.0.F,0,0,160,24.72,51.442
0.0.F,0,0,196,31.44,69.1773
0.0.F,0,0,230,36.38,85.4742
0.0.F,0,0,258,37.43,92.9269
0.0.F,0,0,286,42.31,104.8303
0.0.F,0,0,314,47.06,113.9475
0.0.G,0,0,41,4.5,1.9794
0.0.G,0,0,77,9.7,13.9357
0.0.G,0,0,104,15.1,31.1023
0.0.G,0,0,132,17.05,36.2487
0.0.G,0,0,160,20.09,57.3397
0.0.G,0,0,196,26.9,73.8148
0.0.G,0,0,230,33.54,94.5252
0.0.G,0,0,258,35.55,102.5627
0.0.G,0,0,286,38.71,117.2205
0.0.G,0,0,314,41.15,131.4169
0.2.A,0,2,41,5.4,2.9151
0.2.A,0,2,77,11.8,15.0713
0.2.A,0,2,104,16.9,26.7977
0.2.A,0,2,132,24.15,35.35
0.2.A,0,2,160,25.78,38.8045
0.2.A,0,2,196,33.08,66.244
0.2.A,0,2,230,38.43,81.0054
0.2.A,0,2,258,41.92,92.1001
0.2.A,0,2,286,46.39,107.204
0.2.A,0,2,314,51.69,126.2781
0.2.A,0,2,349,57.26,134.185
0.2.A,0,2,377,60.29,137.5473
0.2.A,0,2,405,62.12,139.0108
0.2.A,0,2,433,65.15,140.5253
0.2.A,0,2,468,68.24,142.1934
0.2.A,0,2,503,74.76,147.0167
0.2.A,0,2,527,80.9,150.3313
0.2.A,0,2,560,83.71,151.6797
0.2.A,0,2,590,85.0,152.505
0.2.A,0,2,622,87.73,156.8716
0.2.A,0,2,651,89.4,158.1404
0.2.A,0,2,678,91.98,160.5133
0.2.B,0,2,41,5.7,2.7351
0.2.B,0,2,77,12.0,13.8197
0.2.B,0,2,104,16.9,22.1471
0.2.B,0,2,132,23.35,27.4345
0.2.B,0,2,160,26.37,29.7893
0.2.B,0,2,196,34.09,35.1916
0.2.B,0,2,230,39.51,39.3096
0.2.B,0,2,258,43.19,41.2226
0.2.B,0,2,286,48.21,44.6351
0.2.B,0,2,314,53.44,48.295
0.2.B,0,2,349,59.18,52.4265
0.2.B,0,2,377,63.55,55.7029
0.2.B,0,2,405,68.68,59.7031
0.2.B,0,2,433,73.05,62.4117
0.2.B,0,2,468,78.69,65.7946
0.2.B,0,2,503,88.33,74.2752
0.2.B,0,2,527,96.48,79.4895
0.2.B,0,2,560,100.97,82.6316
0.2.B,0,2,590,105.38,85.1886
0.2.B,0,2,622,110.3,88.5331
0.2.B,0,2,651,112.99,90.0928
0.2.B,0,2,678,116.19,91.3085
0.2.C,0,2,41,5.5,2.5292
0.2.C,0,2,77,11.6,12.4081
0.2.C,0,2,104,16.4,24.2125
0.2.C,0,2,132,22.6,35.245
0.2.C,0,2,160,25.59,45.1089
0.2.C,0,2,196,32.39,69.3094
0.2.C,0,2,230,37.37,91.3142
0.2.C,0,2,258,41.3,108.5222
0.2.C,0,2,286,45.37,124.4716
0.2.C,0,2,314,49.9,138.9631
0.2.C,0,2,349,55.1,152.9987
0.2.D,0,2,41,4.7,3.4769
0.2.D,0,2,77,11.1,21.1354
0.2.D,0,2,104,15.9,38.0262
0.2.D,0,2,132,21.9,50.1424
0.2.D,0,2,160,24.13,55.0023
0.2.D,0,2,196,31.7,67.5646
0.2.D,0,2,230,38.81,77.2311
0.2.D,0,2,258,43.25,82.2024
0.2.D,0,2,286,48.81,87.5383
0.2.D,0,2,314,54.35,92.3013
0.2.D,0,2,349,60.24,97.7184
0.2.D,0,2,377,63.51,100.9547
0.2.D,0,2,405,68.32,106.0517
0.2.D,0,2,433,71.59,108.34
0.2.D,0,2,468,75.06,110.6294
0.2.D,0,2,503,81.91,114.8751
0.2.D,0,2,527,87.92,118.1195
0.2.D,0,2,560,91.2,119.8246
0.2.D,0,2,590,95.0,121.8759
0.2.D,0,2,622,99.52,124.9486
0.2.D,0,2,651,101.98,127.0635
0.2.D,0,2,678,104.91,129.6997
0.2.E,0,2,41,3.6,1.7275
0.2.E,0,2,77,7.6,11.7243
0.2.E,0,2,104,11.7,22.0531
0.2.E,0,2,132,17.75,30.3995
0.2.E,0,2,160,21.03,36.2361
0.2.E,0,2,196,28.21,45.1365
0.2.E,0,2,230,33.74,51.881
0.2.E,0,2,258,37.45,55.9608
0.2.E,0,2,286,42.61,64.0079
0.2.E,0,2,314,47.74,74.6749
0.2.E,0,2,349,53.69,83.4782
0.2.F,0,2,41,4.6,2.3913
0.2.F,0,2,77,9.9,16.167
0.2.F,0,2,104,14.5,31.0663
0.2.F,0,2,132,20.35,41.1252
0.2.F,0,2,160,22.02,44.9316
0.2.F,0,2,196,29.1,78.7635
0.2.F,0,2,230,34.15,99.1591
0.2.F,0,2,258,37.43,111.029
0.2.F,0,2,286,42.34,128.2087
0.2.F,0,2,314,48.03,146.0697
0.2.G,0,2,41,4.8,3.263
0.2.G,0,2,77,10.2,16.2189
0.2.G,0,2,104,15.1,28.4651
0.2.G,0,2,132,21.25,35.8428
0.2.G,0,2,160,24.77,43.8659
0.7.A,0,7,41,3.7,1.9234
0.7.A,0,7,77,10.6,12.96
0.7.A,0,7,104,15.4,23.9006
0.7.A,0,7,132,22.1,31.4022
0.7.A,0,7,160,24.67,36.3865
0.7.A,0,7,196,32.07,54.2889
0.7.A,0,7,230,37.38,63.6316
0.7.A,0,7,258,40.85,70.3613
0.7.A,0,7,286,45.27,81.4962
0.7.A,0,7,314,48.7,89.7257
0.7.A,0,7,349,52.44,98.2502
0.7.A,0,7,377,56.87,107.0189
0.7.A,0,7,405,61.94,115.5338
0.7.A,0,7,433,66.37,124.8339
0.7.A,0,7,468,72.22,137.1151
0.7.A,0,7,503,80.87,157.1768
0.7.A,0,7,527,87.8,167.43
0.7.A,0,7,560,91.41,171.3276
0.7.A,0,7,590,93.86,174.0218
0.7.A,0,7,622,96.14,176.6658
0.7.A,0,7,651,97.18,178.3085
0.7.A,0,7,678,97.72,179.4637
0.7.B,0,7,41,5.7,2.7351
0.7.B,0,7,77,12.3,15.931
0.7.B,0,7,104,17.4,29.2889
0.7.B,0,7,132,24.35,40.2665
0.7.B,0,7,160,27.58,48.3389
0.7.B,0,7,196,35.83,70.4421
0.7.B,0,7,230,41.33,106.9507
0.7.B,0,7,258,43.64,128.2423
0.7.B,0,7,286,47.62,164.2103
0.7.C,0,7,41,5.7,3.077
0.7.C,0,7,77,12.1,10.2428
0.7.C,0,7,104,17.2,14.5255
0.7.C,0,7,132,23.5,17.8005
0.7.C,0,7,160,26.01,20.0588
0.7.C,0,7,196,32.83,27.2857
0.7.C,0,7,230,38.21,31.9111
0.7.C,0,7,258,42.15,34.5894
0.7.C,0,7,286,47.47,38.2059
0.7.C,0,7,314,53.12,42.1032
0.7.C,0,7,349,58.01,45.5251
0.7.C,0,7,377,61.08,48.1952
0.7.C,0,7,405,65.73,53.0297
0.7.C,0,7,433,68.8,56.5898
0.7.C,0,7,468,72.55,61.0884
0.7.C,0,7,503,79.43,67.416
0.7.C,0,7,527,85.52,72.1647
0.7.C,0,7,560,89.15,74.342
0.7.C,0,7,590,91.98,76.9448
0.7.C,0,7,622,95.87,82.0003
0.7.C,0,7,651,98.33,92.6241
0.7.C,0,7,678,101.5,116.8354
0.7.D,0,7,41,4.7,1.9734
0.7.D,0,7,77,10.4,11.0905
0.7.D,0,7,104,15.4,23.9865
0.7.D,0,7,132,21.35,37.4293
0.7.D,0,7,160,23.6,46.3366
0.7.D,0,7,196,30.05,64.133
0.7.D,0,7,230,34.91,75.2104
0.7.D,0,7,258,38.66,88.6312
0.7.D,0,7,286,44.03,115.3655
0.7.D,0,7,314,49.15,139.6267
0.7.D,0,7,349,54.36,160.0436
0.7.D,0,7,377,57.86,169.7705
0.7.D,0,7,405,62.7,177.7056
0.7.D,0,7,433,66.2,182.7441
0.7.D,0,7,468,68.56,185.3865
0.7.D,0,7,503,75.6,192.8465
0.7.D,0,7,527,82.7,199.9443
0.7.D,0,7,560,87.63,203.9857
0.7.D,0,7,590,92.75,208.1828
0.7.D,0,7,622,97.36,211.5009
0.7.D,0,7,651,100.29,214.3128
0.7.D,0,7,678,103.91,217.787
0.7.E,0,7,41,4.6,2.1153
0.7.E,0,7,77,9.9,16.4209
0.7.E,0,7,104,14.4,32.4359
0.7.E,0,7,132,20.55,45.3469
0.7.E,0,7,160,22.66,49.8187
0.7.E,0,7,196,29.09,62.4176
0.7.E,0,7,230,33.9,72.6116
0.7.E,0,7,258,37.57,86.4065
0.7.E,0,7,286,42.48,115.7591
0.7.E,0,7,314,47.47,137.3092
0.7.F,0,7,41,5.3,2.7551
0.7.F,0,7,77,11.0,15.2912
0.7.F,0,7,104,16.0,24.6883
0.7.F,0,7,132,23.05,30.3266
0.7.F,0,7,160,25.59,33.3228
0.7.F,0,7,196,34.37,51.5795
0.7.F,0,7,230,41.42,60.8826
0.7.F,0,7,258,46.33,67.0673
0.7.F,0,7,286,51.88,74.0581
0.7.F,0,7,314,57.01,81.3405
0.7.G,0,7,41,4.6,2.0234
0.7.G,0,7,77,10.2,15.907
0.7.G,0,7,104,15.1,32.4639
0.7.G,0,7,132,21.65,45.036
0.7.G,0,7,160,24.26,52.0808
0.8.A,0,8,41,5.5,4.1787
0.8.A,0,8,77,11.4,35.6749
0.8.A,0,8,104,16.4,74.4628
0.8.A,0,8,132,22.6,102.85
0.8.A,0,8,160,25.61,119.6405
0.8.A,0,8,196,34.02,183.5366
0.8.A,0,8,230,40.2,242.599
0.8.A,0,8,258,43.51,269.2031
0.8.A,0,8,286,48.57,301.1724
0.8.A,0,8,314,53.83,327.2539
0.8.B,0,8,41,5.3,3.4969
0.8.B,0,8,77,10.9,28.1292
0.8.B,0,8,104,15.8,100.8226
0.8.B,0,8,132,22.6,164.3148
0.8.B,0,8,160,24.12,174.4349
0.8.B,0,8,196,31.45,215.7632
0.8.B,0,8,230,36.26,241.7291
0.8.B,0,8,258,37.31,249.0138
0.8.B,0,8,286,41.62,295.2027
0.8.B,0,8,314,46.49,335.9032
0.8.B,0,8,349,52.29,372.3158
0.8.B,0,8,377,56.52,396.5885
0.8.B,0,8,405,62.05,425.3355
0.8.B,0,8,433,66.28,443.3497
0.8.B,0,8,468,69.24,451.8719
0.8.B,0,8,503,78.47,486.9349
0.8.B,0,8,527,86.3,518.0886
0.8.B,0,8,560,89.88,530.1137
0.8.B,0,8,590,90.52,532.0203
0.8.B,0,8,622,92.9,538.9677
0.8.B,0,8,651,94.52,544.0206
0.8.B,0,8,678,97.33,552.2232
0.8.C,0,8,41,5.5,2.0893
0.8.C,0,8,77,11.3,21.5713
0.8.C,0,8,104,16.3,66.5573
0.8.C,0,8,132,22.5,104.6134
0.8.C,0,8,160,24.76,121.4226
0.8.C,0,8,196,31.12,154.6114
0.8.C,0,8,230,35.22,174.4493
0.8.C,0,8,258,38.3,191.0761
0.8.C,0,8,286,43.05,219.5672
0.8.D,0,8,41,5.0,2.8991
0.8.D,0,8,77,10.5,30.7204
0.8.D,0,8,104,15.0,90.5518
0.8.D,0,8,132,21.25,141.5359
0.8.D,0,8,160,24.12,159.267
0.8.D,0,8,196,31.62,208.6016
0.8.D,0,8,230,36.53,237.3653
0.8.D,0,8,258,39.47,255.6464
0.8.D,0,8,286,44.29,295.8327
0.8.D,0,8,314,49.09,332.2053
0.8.D,0,8,349,54.18,363.0411
0.8.D,0,8,377,57.33,374.5665
0.8.D,0,8,405,62.28,380.8016
0.8.D,0,8,433,65.43,384.8323
0.8.D,0,8,468,68.28,389.6758
0.8.D,0,8,503,75.06,404.7227
0.8.D,0,8,527,81.11,418.5124
0.8.D,0,8,560,84.07,425.9101
0.8.D,0,8,590,86.35,432.5201
0.8.D,0,8,622,90.71,446.3806
0.8.D,0,8,651,93.31,453.9182
0.8.D,0,8,678,95.99,461.2591
0.8.E,0,8,41,4.9,2.5472
0.8.E,0,8,77,10.4,28.0593
0.8.E,0,8,104,15.4,76.8441
0.8.E,0,8,132,22.45,116.0299
0.8.E,0,8,160,25.14,131.1967
0.8.E,0,8,196,35.92,192.4081
0.8.E,0,8,230,40.17,228.8617
0.8.E,0,8,258,46.32,288.6211
0.8.E,0,8,286,51.67,331.5147
0.8.E,0,8,314,56.92,362.9
0.8.F,0,8,41,4.8,2.5912
0.8.F,0,8,77,9.6,19.386
0.8.F,0,8,104,14.8,114.5163
0.8.F,0,8,132,21.35,173.8409
0.8.F,0,8,160,23.39,182.8548
0.8.F,0,8,196,30.31,219.3811
0.8.F,0,8,230,36.13,249.0538
0.8.F,0,8,258,38.57,267.5921
0.8.F,0,8,286,42.14,301.9961
0.8.F,0,8,314,47.0,345.1395
0.8.F,0,8,349,52.41,377.8057
0.8.F,0,8,377,56.84,399.5502
0.8.F,0,8,405,61.92,418.7467
0.8.F,0,8,433,66.35,447.1784
0.8.F,0,8,468,71.65,483.1012
0.8.F,0,8,503,79.13,547.5587
0.8.F,0,8,527,86.37,605.7502
0.8.F,0,8,560,90.37,630.7824
0.8.F,0,8,590,96.15,667.994
0.8.F,0,8,622,101.56,706.1767
0.8.F,0,8,651,106.85,749.2239
0.8.F,0,8,678,111.62,791.0915
0.8.G,0,8,41,4.5,2.3393
0.8.G,0,8,77,10.2,30.8304
0.8.G,0,8,104,14.9,84.3937
0.8.G,0,8,132,21.4,130.5293
0.8.G,0,8,160,24.07,149.7474
1.0.A,1,0,42,3.0,65.6795
1.0.A,1,0,77,6.0,144.8549
1.0.A,1,0,105,8.5,214.3332
1.0.A,1,0,133,10.8,275.0343
1.0.A,1,0,160,12.29,310.0384
1.0.A,1,0,195,14.76,347.3238
1.0.A,1,0,231,17.71,385.6619
1.0.A,1,0,259,19.07,405.3757
1.0.A,1,0,286,19.89,417.5899
1.0.A,1,0,314,20.46,426.8211
1.0.A,1,0,349,21.99,453.5877
1.0.A,1,0,405,23.6,481.1101
1.0.A,1,0,433,26.46,528.5714
1.0.A,1,0,468,29.22,558.646
1.0.A,1,0,503,32.42,592.2355
1.0.A,1,0,527,34.8,622.69
1.0.A,1,0,560,36.95,646.1178
1.0.A,1,0,590,38.78,660.0214
1.0.A,1,0,622,40.01,668.5058
1.0.A,1,0,651,41.31,678.2527
1.0.A,1,0,678,42.55,686.1863
1.0.B,1,0,42,3.2,73.5771
1.0.B,1,0,77,6.3,169.6472
1.0.B,1,0,105,8.5,231.008
1.0.B,1,0,133,11.3,306.0247
1.0.B,1,0,160,12.49,330.8879
1.0.B,1,0,195,14.65,366.5168
1.0.B,1,0,231,16.78,400.7992
1.0.B,1,0,259,18.04,426.3692
1.0.B,1,0,286,19.0,442.6841
1.0.B,1,0,314,19.92,458.1813
1.0.B,1,0,349,20.82,473.2066
1.0.B,1,0,405,21.91,488.7888
1.0.B,1,0,433,24.38,522.3703
1.0.B,1,0,468,25.56,535.9361
1.0.B,1,0,503,31.38,606.3361
1.0.B,1,0,527,36.7,663.7743
1.0.B,1,0,560,38.65,686.5821
1.0.B,1,0,590,40.6,709.39
1.0.B,1,0,622,43.4,740.7403
1.0.B,1,0,651,45.0,757.8549
1.0.B,1,0,678,47.76,789.0332
1.0.C,1,0,42,4.0,79.1753
1.0.C,1,0,77,8.6,188.1614
1.0.C,1,0,105,11.4,253.1012
1.0.C,1,0,133,15.4,339.8741
1.0.C,1,0,160,19.03,408.8227
1.0.C,1,0,195,22.93,475.8818
1.0.C,1,0,231,26.3,532.1433
1.0.C,1,0,259,27.36,549.5218
1.0.C,1,0,286,28.31,564.907
1.0.C,1,0,314,29.31,581.3519
1.0.C,1,0,349,30.37,599.0484
1.0.C,1,0,405,33.16,616.6199
1.0.C,1,0,433,35.84,635.3741
1.0.C,1,0,468,38.66,651.4431
1.0.C,1,0,503,42.65,677.7689
1.0.C,1,0,527,46.66,703.4249
1.0.C,1,0,560,49.91,721.6192
1.0.C,1,0,590,54.63,748.043
1.0.C,1,0,622,57.22,763.3192
1.0.C,1,0,651,59.03,776.3472
1.0.C,1,0,678,60.13,784.0448
1.0.D,1,0,42,4.0,93.171
1.0.D,1,0,77,9.8,199.2779
1.0.D,1,0,105,14.0,296.2677
1.0.D,1,0,133,17.4,371.0444
1.0.D,1,0,160,19.51,413.0204
1.0.D,1,0,195,22.22,462.0561
1.0.D,1,0,231,24.67,508.3467
1.0.D,1,0,259,26.4,538.4393
1.0.D,1,0,286,27.11,549.086
1.0.E,1,0,42,3.1,72.8893
1.0.E,1,0,77,8.4,192.1022
1.0.E,1,0,105,10.6,243.5661
1.0.E,1,0,133,12.7,298.3591
1.0.E,1,0,160,14.16,329.0095
1.0.E,1,0,195,16.48,364.9583
1.0.E,1,0,231,19.39,408.3038
1.0.E,1,0,259,21.06,431.0088
1.0.E,1,0,286,22.52,450.4207
1.0.F,1,0,42,3.8,82.4343
1.0.F,1,0,77,8.5,194.2595
1.0.F,1,0,105,11.4,264.9975
1.0.F,1,0,133,13.3,311.1531
1.0.F,1,0,160,14.32,334.4018
1.0.F,1,0,195,15.82,365.1423
1.0.F,1,0,231,17.27,397.6122
1.0.F,1,0,259,18.05,415.0787
1.0.F,1,0,286,19.09,438.3675
1.0.G,1,0,42,3.3,74.8867
1.0.G,1,0,77,8.3,196.8487
1.0.G,1,0,105,11.2,258.8894
1.0.G,1,0,133,14.6,331.9666
1.0.G,1,0,160,17.16,380.8474
1.2.A,1,2,42,4.2,91.5315
1.2.A,1,2,77,8.5,197.2786
1.2.A,1,2,105,11.4,270.6257
1.2.A,1,2,133,15.0,349.4412
1.2.A,1,2,160,16.2,374.2734
1.2.A,1,2,195,17.92,403.6763
1.2.A,1,2,231,20.06,438.9753
1.2.A,1,2,259,21.25,460.5076
1.2.A,1,2,286,22.19,476.6705
1.2.A,1,2,314,22.83,488.2189
1.2.A,1,2,349,23.58,502.3895
1.2.A,1,2,405,24.075,505.705
1.2.A,1,2,433,24.705,510.9953
1.2.A,1,2,468,25.555,518.7279
1.2.A,1,2,503,26.635,527.3652
1.2.A,1,2,527,27.475,533.9992
1.2.A,1,2,560,27.955,537.9339
1.2.A,1,2,590,28.385,540.943
1.2.A,1,2,622,28.785,543.5022
1.2.A,1,2,651,28.965,544.8518
1.2.A,1,2,678,29.405,548.2387
1.2.B,1,2,42,4.3,141.8558
1.2.B,1,2,77,8.3,243.4242
1.2.B,1,2,105,12.3,333.796
1.2.B,1,2,133,15.0,404.2441
1.2.B,1,2,160,16.38,439.837
1.2.B,1,2,195,19.08,502.4575
1.2.B,1,2,231,21.96,560.6154
1.2.B,1,2,259,23.48,588.1188
1.2.B,1,2,286,24.91,610.5628
1.2.C,1,2,42,4.5,118.763
1.2.C,1,2,77,9.6,235.5166
1.2.C,1,2,105,12.9,319.3105
1.2.C,1,2,133,16.5,401.0051
1.2.C,1,2,160,18.68,453.9626
1.2.C,1,2,195,22.98,540.3657
1.2.C,1,2,231,26.1,591.8297
1.2.C,1,2,259,27.01,608.2046
1.2.C,1,2,286,27.93,623.2879
1.2.C,1,2,314,28.82,638.4576
1.2.C,1,2,349,29.4,648.7204
1.2.C,1,2,405,31.23,671.5883
1.2.C,1,2,433,33.64,704.1132
1.2.C,1,2,468,37.31,755.4772
1.2.C,1,2,503,41.61,810.0702
1.2.C,1,2,527,45.58,852.536
1.2.C,1,2,560,47.92,880.3733
1.2.C,1,2,590,50.21,906.2422
1.2.C,1,2,622,51.28,921.3245
1.2.C,1,2,651,51.94,930.6936
1.2.C,1,2,678,52.67,939.3049
1.2.D,1,2,42,2.7,63.9701
1.2.D,1,2,77,7.4,172.9761
1.2.D,1,2,105,10.7,253.8009
1.2.D,1,2,133,14.0,332.3165
1.2.D,1,2,160,15.48,366.05
1.2.D,1,2,195,19.46,422.9463
1.2.D,1,2,231,23.04,471.6191
1.2.D,1,2,259,24.48,491.4849
1.2.D,1,2,286,25.36,503.977
1.2.E,1,2,42,3.0,68.9785
1.2.E,1,2,77,6.8,153.3122
1.2.E,1,2,105,10.1,225.5597
1.2.E,1,2,133,14.0,308.6039
1.2.E,1,2,160,16.04,342.4573
1.2.E,1,2,195,20.19,400.9541
1.2.E,1,2,231,23.39,444.7805
1.2.E,1,2,259,24.79,466.6137
1.2.E,1,2,286,25.72,482.0469
1.2.F,1,2,42,3.3,76.5362
1.2.F,1,2,77,7.9,193.7996
1.2.F,1,2,105,11.2,270.3358
1.2.F,1,2,133,14.4,337.195
1.2.F,1,2,160,16.02,364.8883
1.2.G,1,2,42,3.6,94.2906
1.2.G,1,2,77,8.0,198.978
1.2.G,1,2,105,11.5,285.4011
1.2.G,1,2,133,15.3,373.9135
1.2.G,1,2,160,17.94,417.46
1.2.G,1,2,195,22.22,469.2318
1.2.G,1,2,231,25.81,508.3507
1.2.G,1,2,259,27.96,535.4322
1.2.G,1,2,286,29.59,553.5196
1.2.G,1,2,314,31.07,571.2741
1.2.G,1,2,349,32.33,587.523
1.2.G,1,2,405,34.93,619.2331
1.2.G,1,2,433,38.32,666.0006
1.2.G,1,2,468,42.03,715.6991
1.2.G,1,2,503,46.63,766.7432
1.2.G,1,2,527,51.01,816.2218
1.2.G,1,2,560,54.25,852.4985
1.2.G,1,2,590,58.0,894.8603
1.2.G,1,2,622,62.57,942.8303
1.2.G,1,2,651,65.3,970.9406
1.2.G,1,2,678,67.32,996.5866
1.7.A,1,7,42,4.1,99.599
1.7.A,1,7,77,9.0,214.7131
1.7.A,1,7,105,12.6,301.4461
1.7.A,1,7,133,16.2,386.7395
1.7.A,1,7,160,18.05,422.8033
1.7.A,1,7,195,21.32,475.7608
1.7.A,1,7,231,24.27,517.3429
1.7.A,1,7,259,25.62,537.3166
1.7.A,1,7,286,26.66,552.184
1.7.A,1,7,314,27.71,567.5092
1.7.A,1,7,349,28.91,585.3837
1.7.A,1,7,405,31.1,607.4958
1.7.A,1,7,433,34.73,649.2278
1.7.A,1,7,468,37.87,680.9319
1.7.A,1,7,503,40.08,698.8273
1.7.A,1,7,527,42.65,716.5548
1.7.A,1,7,560,44.69,732.6658
1.7.A,1,7,590,46.46,744.698
1.7.A,1,7,622,47.34,752.0877
1.7.A,1,7,651,47.77,755.5697
1.7.A,1,7,678,48.24,759.2345
1.7.B,1,7,42,1.5,36.2887
1.7.B,1,7,77,3.2,84.7236
1.7.B,1,7,105,4.7,126.4106
1.7.B,1,7,133,6.4,171.9564
1.7.B,1,7,160,7.44,197.7404
1.7.B,1,7,195,9.69,241.6017
1.7.B,1,7,231,12.03,284.1765
1.7.B,1,7,259,12.98,303.5505
1.7.B,1,7,286,13.92,324.7878
1.7.C,1,7,42,3.7,92.4712
1.7.C,1,7,77,8.4,188.7912
1.7.C,1,7,105,11.3,255.1805
1.7.C,1,7,133,14.0,318.3408
1.7.C,1,7,160,16.5,373.0738
1.7.C,1,7,195,20.67,440.6068
1.7.C,1,7,231,23.66,484.845
1.7.C,1,7,259,24.53,498.2388
1.7.C,1,7,286,25.53,513.1342
1.7.C,1,7,314,26.28,524.793
1.7.C,1,7,349,27.81,549.5713
1.7.C,1,7,405,29.86,570.8847
1.7.C,1,7,433,31.29,586.8957
1.7.C,1,7,468,33.8,613.7443
1.7.C,1,7,503,36.48,645.6264
1.7.C,1,7,527,38.83,678.986
1.7.C,1,7,560,41.44,719.9503
1.7.C,1,7,590,43.5,746.31
1.7.C,1,7,622,44.92,765.8999
1.7.C,1,7,651,45.33,771.9661
1.7.C,1,7,678,46.49,787.9691
1.7.D,1,7,42,3.4,89.3922
1.7.D,1,7,77,8.2,206.9555
1.7.D,1,7,105,11.6,292.9488
1.7.D,1,7,133,15.3,373.5836
1.7.D,1,7,160,17.33,414.171
1.7.D,1,7,195,21.66,483.4294
1.7.D,1,7,231,24.7,528.1035
1.7.D,1,7,259,25.2,535.6512
1.7.D,1,7,286,26.05,548.5671
1.7.E,1,7,42,2.2,57.4021
1.7.E,1,7,77,6.4,159.8502
1.7.E,1,7,105,9.1,231.3779
1.7.E,1,7,133,11.5,296.1578
1.7.E,1,7,160,12.6,325.4086
1.7.E,1,7,195,14.05,359.9079
1.7.E,1,7,231,16.12,410.1932
1.7.E,1,7,259,16.98,430.053
1.7.E,1,7,286,17.95,450.2228
1.7.F,1,7,42,3.5,83.624
1.7.F,1,7,77,7.6,185.6822
1.7.F,1,7,105,11.0,269.2961
1.7.F,1,7,133,13.8,336.1953
1.7.F,1,7,160,15.08,365.3702
1.7.G,1,7,42,2.7,75.3065
1.7.G,1,7,77,6.5,187.3716
1.7.G,1,7,105,9.4,259.5592
1.7.G,1,7,133,13.6,354.0297
1.7.G,1,7,160,15.31,391.638
1.7.G,1,7,195,19.09,457.3895
1.7.G,1,7,231,21.87,509.0814
1.7.G,1,7,259,23.07,531.3945
1.7.G,1,7,286,24.0,546.8277
1.7.G,1,7,314,24.66,555.6359
1.7.G,1,7,349,26.26,571.7909
1.7.G,1,7,405,28.57,598.3476
1.7.G,1,7,433,29.91,617.2357
1.7.G,1,7,468,30.6,626.2719
1.7.G,1,7,527,36.06,691.2257
1.7.G,1,7,560,39.45,731.893
1.7.G,1,7,590,43.35,771.6606
1.7.G,1,7,622,48.05,820.0556
1.7.G,1,7,651,50.37,840.6971
1.7.G,1,7,678,52.38,856.3703
1.8.A,1,8,42,5.5,91.8214
1.8.A,1,8,77,11.5,232.7775
1.8.A,1,8,105,16.2,344.6027
1.8.A,1,8,133,21.4,459.4869
1.8.A,1,8,160,22.75,490.5272
1.8.A,1,8,195,26.86,568.182
1.8.A,1,8,231,30.65,635.2441
1.8.A,1,8,259,31.72,652.5727
1.8.A,1,8,286,32.69,666.8273
1.8.A,1,8,314,33.32,676.0854
1.8.A,1,8,349,34.64,695.4834
1.8.A,1,8,405,36.94,730.6624
1.8.A,1,8,433,39.8,771.2618
1.8.A,1,8,468,41.61,796.4129
1.8.A,1,8,503,46.56,843.4233
1.8.A,1,8,527,50.45,863.2561
1.8.A,1,8,560,53.04,879.3091
1.8.A,1,8,590,56.63,899.4069
1.8.A,1,8,622,59.84,920.5863
1.8.A,1,8,651,62.01,935.3377
1.8.A,1,8,678,63.18,943.1742
1.8.B,1,8,42,4.2,70.538
1.8.B,1,8,77,9.7,207.9952
1.8.B,1,8,105,13.9,308.7638
1.8.B,1,8,133,18.0,406.7233
1.8.B,1,8,160,20.23,439.94
1.8.B,1,8,195,23.31,506.4473
1.8.B,1,8,231,26.43,574.4421
1.8.B,1,8,259,28.32,612.4192
1.8.B,1,8,286,30.04,641.994
1.8.C,1,8,42,4.9,93.071
1.8.C,1,8,77,10.6,228.1189
1.8.C,1,8,105,14.9,323.1194
1.8.C,1,8,133,19.3,408.0129
1.8.C,1,8,160,23.23,486.9813
1.8.C,1,8,195,28.88,579.0476
1.8.C,1,8,231,33.45,650.3174
1.8.C,1,8,259,35.25,677.309
1.8.C,1,8,286,36.04,687.023
1.8.C,1,8,314,36.78,697.0839
1.8.C,1,8,349,38.03,715.7031
1.8.C,1,8,405,38.85,726.5237
1.8.C,1,8,433,39.61,738.2241
1.8.C,1,8,468,40.6,751.9808
1.8.C,1,8,503,41.95,770.2001
1.8.C,1,8,527,43.19,785.8192
1.8.C,1,8,560,44.61,805.4091
1.8.C,1,8,590,45.76,816.4457
1.8.C,1,8,622,46.69,825.3709
1.8.C,1,8,651,47.69,834.768
1.8.C,1,8,678,49.82,854.9967
1.8.D,1,8,42,5.2,97.2097
1.8.D,1,8,77,10.6,212.1939
1.8.D,1,8,105,15.0,312.4827
1.8.D,1,8,133,20.5,415.8505
1.8.D,1,8,160,23.44,472.2809
1.8.D,1,8,195,27.83,553.4706
1.8.D,1,8,231,31.44,611.2126
1.8.D,1,8,259,32.17,621.9403
1.8.D,1,8,286,32.8,630.6946
1.8.D,1,8,314,33.5,640.1066
1.8.D,1,8,349,34.09,647.7742
1.8.D,1,8,405,38.11,705.2423
1.8.D,1,8,433,41.1,750.6762
1.8.D,1,8,468,43.82,788.4724
1.8.D,1,8,503,50.25,885.5352
1.8.D,1,8,527,55.52,954.0238
1.8.D,1,8,560,57.33,978.2703
1.8.D,1,8,590,59.95,1011.0101
1.8.D,1,8,622,63.99,1064.3215
1.8.D,1,8,651,66.6,1093.5444
1.8.D,1,8,678,69.55,1126.869
1.8.E,1,8,42,4.1,81.9745
1.8.E,1,8,77,9.7,225.8497
1.8.E,1,8,105,14.2,343.2631
1.8.E,1,8,133,18.1,425.1376
1.8.E,1,8,160,19.89,462.5369
1.8.F,1,8,42,4.0,75.5765
1.8.F,1,8,77,8.3,183.9027
1.8.F,1,8,105,11.8,278.0234
1.8.F,1,8,133,15.9,378.0323
1.8.F,1,8,160,17.74,421.6267
1.8.F,1,8,195,21.6,497.645
1.8.F,1,8,231,25.52,563.8724
1.8.F,1,8,259,27.1,590.5661
1.8.F,1,8,286,28.1,604.6617
1.8.G,1,8,42,3.7,74.7167
1.8.G,1,8,77,8.8,213.9034
1.8.G,1,8,105,12.3,301.3761
1.8.G,1,8,133,16.8,407.5431
1.8.G,1,8,160,18.37,440.1889
1.8.G,1,8,195,24.02,539.5979
1.8.G,1,8,231,28.76,610.6758
1.8.G,1,8,259,30.19,630.8325
1.8.G,1,8,286,32.22,660.6642
5.0.A,5,0,35,3.5,26.9416
5.0.A,5,0,59,4.9,38.9219
5.0.A,5,0,85,4.9,38.9219
5.0.A,5,0,114,12.1,84.6996
5.0.A,5,0,149,14.55,102.2851
5.0.A,5,0,178,24.85,150.0623
5.0.A,5,0,198,30.75,178.6094
5.0.A,5,0,227,34.25,195.2642
5.0.A,5,0,255,37.05,206.4607
5.0.A,5,0,281,41.75,227.98
5.0.A,5,0,309,47.55,246.5342
5.0.A,5,0,350,56.25,276.8008
5.0.A,5,0,382,61.65,296.0188
5.0.A,5,0,410,66.45,310.7982
5.0.A,5,0,449,71.65,331.0719
5.0.A,5,0,471,75.45,345.4314
5.0.A,5,0,501,81.25,364.9133
5.0.A,5,0,529,85.12,377.3709
5.0.A,5,0,556,93.47,402.6635
5.0.A,5,0,596,103.35,430.7139
5.0.A,5,0,626,107.87,444.4053
5.0.A,5,0,652,113.35,462.0454
5.0.A,5,0,682,119.87,488.2476
5.0.B,5,0,35,3.4,24.8123
5.0.B,5,0,58,4.7,36.1187
5.0.B,5,0,85,4.7,36.1187
5.0.B,5,0,114,12.1,89.974
5.0.B,5,0,149,15.6,120.9743
5.0.B,5,0,178,24.8,165.4885
5.0.B,5,0,198,30.6,194.7114
5.0.B,5,0,227,34.0,213.2696
5.0.B,5,0,255,36.6,224.1342
5.0.B,5,0,281,41.3,247.6269
5.0.B,5,0,309,47.1,265.6013
5.0.B,5,0,350,55.7,294.6602
5.0.B,5,0,350,62.07,311.3444
5.0.B,5,0,382,67.37,330.7364
5.0.B,5,0,410,72.07,348.8727
5.0.B,5,0,449,78.37,375.3245
5.0.B,5,0,471,82.57,389.684
5.0.B,5,0,501,88.37,405.803
5.0.B,5,0,529,92.03,417.5114
5.0.B,5,0,556,100.84,442.7001
5.0.B,5,0,596,110.73,467.6151
5.0.B,5,0,626,115.16,479.7496
5.0.B,5,0,652,120.51,495.5806
5.0.B,5,0,682,126.9,513.3393
5.0.B,5,0,742,128.09,516.3848
5.0.B,5,0,771,133.03,527.9408
5.0.B,5,0,798,137.39,537.2683
5.0.B,5,0,833,141.4,546.9694
5.0.B,5,0,868,147.72,565.7972
5.0.B,5,0,899,153.79,583.7588
5.0.B,5,0,926,157.42,594.5728
5.0.B,5,0,955,163.92,611.5975
5.0.B,5,0,987,169.09,626.4825
5.0.B,5,0,1016,173.42,638.8624
5.0.B,5,0,1043,177.83,650.413
5.0.B,5,0,1079,181.23,659.998
5.0.B,5,0,1107,185.61,671.9079
5.0.B,5,0,1142,190.4,685.9861
5.0.B,5,0,1171,195.3,701.4653
5.0.C,5,0,32,3.6,25.696
5.0.C,5,0,58,5.1,37.3924
5.0.C,5,0,114,5.1,37.3924
5.0.C,5,0,149,8.0,55.0189
5.0.C,5,0,178,18.2,93.5629
5.0.C,5,0,198,23.5,114.5443
5.0.C,5,0,226,27.0,126.4406
5.0.C,5,0,255,30.1,133.5064
5.0.C,5,0,281,35.0,151.6308
5.0.C,5,0,309,41.0,169.2653
5.0.C,5,0,350,50.1,195.6471
5.0.C,5,0,382,55.6,215.001
5.0.C,5,0,410,59.9,228.5848
5.0.C,5,0,449,65.2,249.7782
5.0.C,5,0,471,68.4,261.4866
5.0.C,5,0,501,75.6,306.8324
5.0.C,5,0,529,78.62,316.9765
5.0.C,5,0,556,86.1,339.9329
5.0.C,5,0,596,95.21,365.2508
5.0.C,5,0,626,100.0,378.9939
5.0.C,5,0,652,105.74,395.979
5.0.C,5,0,682,112.13,413.9932
5.0.D,5,0,32,3.6,25.9119
5.0.D,5,0,58,5.1,36.2587
5.0.D,5,0,114,5.1,36.2587
5.0.D,5,0,149,13.3,85.4434
5.0.D,5,0,178,24.7,134.904
5.0.D,5,0,198,29.8,159.5803
5.0.D,5,0,226,33.8,176.1351
5.0.D,5,0,255,37.6,188.5193
5.0.D,5,0,281,42.4,207.7133
5.0.D,5,0,309,48.3,231.188
5.0.D,5,0,350,53.7,250.1901
5.0.D,5,0,350,59.7,269.3841
5.0.D,5,0,382,65.0,290.7894
5.0.D,5,0,410,69.5,308.244
5.0.D,5,0,449,75.2,331.0369
5.0.D,5,0,471,79.0,345.0925
5.0.D,5,0,501,84.5,363.5668
5.0.D,5,0,529,88.62,375.5934
5.0.D,5,0,556,97.33,401.1058
5.0.D,5,0,596,106.64,428.4686
5.0.D,5,0,626,110.62,441.0017
5.0.D,5,0,652,116.28,460.0134
5.0.D,5,0,682,122.27,479.5347
5.0.D,5,0,742,123.62,483.8534
5.0.D,5,0,771,128.16,498.1046
5.0.D,5,0,798,131.6,509.2467
5.0.D,5,0,833,135.62,522.8301
5.0.D,5,0,868,142.02,542.9198
5.0.D,5,0,899,147.49,558.3404
5.0.D,5,0,926,150.18,565.3322
5.0.D,5,0,955,155.48,578.3662
5.0.D,5,0,987,160.65,592.3208
5.0.D,5,0,1016,165.14,605.8764
5.0.D,5,0,1043,169.51,618.8076
5.0.D,5,0,1079,172.68,628.7583
5.0.D,5,0,1107,176.49,640.4894
5.0.D,5,0,1142,180.9,653.8916
5.0.D,5,0,1171,185.39,668.1654
5.10.A,5,8,59,5.5,33.4296
5.10.A,5,8,85,5.5,33.4296
5.10.A,5,8,114,9.5,81.8945
5.10.A,5,8,149,9.5,81.8945
5.10.A,5,8,178,17.8,163.0432
5.10.A,5,8,198,23.9,215.4869
5.10.A,5,8,227,28.8,268.3904
5.10.A,5,8,255,33.3,315.6257
5.10.A,5,8,281,38.5,363.5548
5.10.A,5,8,309,44.5,407.9409
5.10.A,5,8,350,53.7,466.9865
5.10.A,5,8,350,59.92,495.714
5.10.A,5,8,382,65.02,531.7088
5.10.A,5,8,410,69.22,560.1759
5.10.A,5,8,449,74.22,596.6646
5.10.A,5,8,471,77.12,613.3054
5.10.A,5,8,501,82.52,645.5873
5.10.A,5,8,529,87.02,673.6586
5.10.A,5,8,556,94.87,720.2731
5.10.A,5,8,596,103.8,770.6226
5.10.A,5,8,626,109.56,802.8109
5.10.A,5,8,652,114.99,832.8838
5.10.B,5,8,59,5.9,38.456
5.10.B,5,8,85,5.9,38.456
5.10.B,5,8,114,9.6,92.1633
5.10.B,5,8,149,9.6,92.1633
5.10.B,5,8,178,18.7,181.8614
5.10.B,5,8,198,24.3,232.4696
5.10.B,5,8,227,28.2,272.861
5.10.B,5,8,255,30.9,303.2535
5.10.B,5,8,281,35.5,348.3195
5.10.B,5,8,309,41.3,390.1825
5.10.B,5,8,350,50.4,459.6848
5.10.B,5,8,350,56.59,477.1352
5.10.B,5,8,382,61.09,510.6947
5.10.B,5,8,410,65.29,538.658
5.10.B,5,8,449,70.79,578.2457
5.10.B,5,8,471,74.29,600.4288
5.10.B,5,8,501,80.69,627.8123
5.10.B,5,8,529,83.03,637.684
5.10.B,5,8,556,90.28,664.7905
5.10.B,5,8,596,98.72,692.2964
5.10.B,5,8,626,103.84,710.1596
5.10.B,5,8,652,110.15,733.6255
5.10.C,5,8,59,6.0,44.3862
5.10.C,5,8,85,6.0,44.3862
5.10.C,5,8,114,9.5,84.2038
5.10.C,5,8,149,9.5,84.2038
5.10.C,5,8,178,18.0,158.4706
5.10.C,5,8,198,23.9,215.093
5.10.C,5,8,227,28.5,256.4801
5.10.C,5,8,255,32.1,294.5563
5.10.C,5,8,281,37.8,340.37
5.10.C,5,8,309,43.9,382.4469
5.10.C,5,8,350,53.4,432.7812
5.10.C,5,8,350,58.75,452.784
5.10.C,5,8,382,64.05,481.607
5.10.C,5,8,410,68.45,509.7582
5.10.C,5,8,449,73.95,538.5692
5.10.C,5,8,471,76.85,558.573
5.10.C,5,8,501,82.35,590.0232
5.10.C,5,8,529,85.23,609.5435
5.10.C,5,8,556,92.78,651.8859
5.10.C,5,8,596,101.3,689.7029
5.10.C,5,8,626,105.08,707.5011
5.10.C,5,8,652,110.56,734.783
5.2.A,5,2,35,3.2,31.1583
5.2.A,5,2,59,4.4,45.1219
5.2.A,5,2,85,4.4,45.1219
5.2.A,5,2,114,9.4,92.2073
5.2.A,5,2,149,9.4,92.2073
5.2.A,5,2,178,20.1,163.661
5.2.A,5,2,198,25.6,193.7916
5.2.A,5,2,227,29.9,222.5927
5.2.A,5,2,255,32.8,238.2478
5.2.A,5,2,281,37.6,264.5436
5.2.A,5,2,309,43.5,286.4848
5.2.A,5,2,350,49.5,306.7585
5.2.A,5,2,350,54.92,323.1218
5.2.A,5,2,382,59.32,341.508
5.2.A,5,2,410,63.42,358.6407
5.2.A,5,2,449,68.32,382.6432
5.2.A,5,2,471,72.12,399.358
5.2.A,5,2,501,78.32,423.9024
5.2.A,5,2,529,80.4,432.3861
5.2.A,5,2,556,87.29,457.8712
5.2.A,5,2,596,95.5,485.1199
5.2.A,5,2,626,99.45,498.5457
5.2.A,5,2,652,104.2,515.0706
5.2.A,5,2,682,109.78,533.702
5.2.A,5,2,742,110.78,537.0409
5.2.A,5,2,771,113.72,546.8575
5.2.A,5,2,798,116.32,555.8507
5.2.A,5,2,833,119.9,569.3789
5.2.A,5,2,868,125.13,588.4101
5.2.A,5,2,899,130.51,608.0948
5.2.A,5,2,926,133.87,619.4481
5.2.A,5,2,955,138.54,635.8813
5.2.A,5,2,987,142.61,649.2268
5.2.A,5,2,1016,145.97,660.9159
5.2.A,5,2,1043,148.68,669.3143
5.2.A,5,2,1079,151.04,677.0055
5.2.A,5,2,1107,155.39,690.8342
5.2.A,5,2,1142,160.13,706.2818
5.2.A,5,2,1171,164.9,724.4975
5.2.B,5,2,59,4.4,11.4584
5.2.B,5,2,85,4.4,11.4584
5.2.B,5,2,114,8.6,54.453
5.2.B,5,2,149,8.6,54.453
5.2.B,5,2,178,19.7,139.2306
5.2.B,5,2,198,25.9,172.8242
5.2.B,5,2,227,30.6,201.0154
5.2.B,5,2,255,34.9,216.0607
5.2.B,5,2,281,39.6,236.5463
5.2.B,5,2,309,45.4,262.5222
5.2.B,5,2,350,53.7,290.2356
5.2.B,5,2,350,58.05,305.108
5.2.B,5,2,382,63.35,323.2283
5.2.B,5,2,410,69.05,345.7933
5.2.B,5,2,449,75.95,374.0745
5.2.B,5,2,471,80.15,388.9379
5.2.B,5,2,501,86.95,417.489
5.2.B,5,2,529,89.87,428.6399
5.2.B,5,2,556,97.13,455.1306
5.2.B,5,2,596,106.44,487.5193
5.2.B,5,2,626,110.1,500.0692
5.2.B,5,2,652,117.14,523.857
5.2.B,5,2,682,121.53,538.7784
5.2.B,5,2,771,126.32,557.0705
5.2.B,5,2,798,129.37,570.1814
5.2.B,5,2,833,132.86,585.0442
5.2.B,5,2,868,139.97,615.1812
5.2.B,5,2,899,146.87,641.8069
5.2.B,5,2,926,150.54,657.1427
5.2.B,5,2,955,156.99,677.6473
5.2.B,5,2,987,162.57,693.9359
5.2.B,5,2,1016,167.36,708.7802
5.2.B,5,2,1043,171.76,722.24
5.2.B,5,2,1079,175.12,732.5856
5.2.C,5,2,32,3.7,30.7004
5.2.C,5,2,59,5.2,44.2262
5.2.C,5,2,85,5.2,44.2262
5.2.C,5,2,114,10.4,81.5506
5.2.C,5,2,149,10.4,81.5506
5.2.C,5,2,178,22.1,158.7466
5.2.C,5,2,198,27.9,190.0568
5.2.C,5,2,226,32.4,204.0025
5.2.C,5,2,255,35.2,214.7511
5.2.C,5,2,281,42.2,232.1057
5.2.C,5,2,309,47.8,251.1398
5.2.C,5,2,350,56.6,279.6429
5.2.C,5,2,382,61.4,296.5336
5.2.C,5,2,410,66.6,312.3367
5.2.C,5,2,449,72.5,337.6988
5.2.C,5,2,471,76.3,352.8181
5.2.C,5,2,501,83.0,375.591
5.2.C,5,2,529,86.02,385.1916
5.2.C,5,2,556,93.35,406.5153
5.2.C,5,2,596,101.95,429.2122
5.2.C,5,2,626,106.79,443.1471
5.2.C,5,2,652,112.6,461.2686
5.2.C,5,2,682,118.97,478.5896
5.2.D,5,2,32,3.5,35.6889
5.2.D,5,2,58,4.8,50.8941
5.2.D,5,2,85,4.8,50.8941
5.2.D,5,2,114,8.9,95.8981
5.2.D,5,2,149,8.9,95.8981
5.2.D,5,2,178,19.4,175.6733
5.2.D,5,2,198,24.7,208.9469
5.2.D,5,2,226,27.7,226.4015
5.2.D,5,2,255,29.5,236.9822
5.2.D,5,2,281,33.7,263.5179
5.2.D,5,2,309,39.3,284.4554
5.2.D,5,2,350,47.4,314.74
5.2.D,5,2,382,52.2,333.934
5.2.D,5,2,410,56.5,352.8481
5.2.D,5,2,449,61.5,377.3405
5.2.D,5,2,471,64.6,390.2944
5.2.D,5,2,501,70.4,412.0956
5.2.D,5,2,529,71.94,418.5616
5.2.D,5,2,556,78.88,444.9948
5.2.D,5,2,596,87.45,474.2951
5.2.D,5,2,626,91.14,487.4643
5.2.D,5,2,652,95.58,503.9759
5.2.D,5,2,682,101.16,523.1651
5.7.A,5,7,35,3.0,29.9907
5.7.A,5,7,59,4.3,44.91
5.7.A,5,7,85,4.3,44.91
5.7.A,5,7,114,10.6,106.1269
5.7.A,5,7,149,10.6,106.1269
5.7.A,5,7,178,21.7,183.3589
5.7.A,5,7,198,26.7,213.3495
5.7.A,5,7,227,30.9,242.7404
5.7.A,5,7,255,33.3,256.1282
5.7.A,5,7,281,37.5,277.5416
5.7.A,5,7,309,42.6,300.7903
5.7.A,5,7,350,51.0,331.6927
5.7.A,5,7,350,56.93,349.9514
5.7.A,5,7,382,61.83,367.0961
5.7.A,5,7,410,66.63,387.9216
5.7.A,5,7,449,72.93,418.782
5.7.A,5,7,471,76.53,435.4088
5.7.A,5,7,501,82.23,464.5837
5.7.A,5,7,529,84.79,475.0252
5.7.A,5,7,556,92.4,502.9452
5.7.A,5,7,596,100.9,530.6466
5.7.A,5,7,626,105.44,545.7147
5.7.A,5,7,652,110.48,562.7446
5.7.A,5,7,682,116.36,582.025
5.7.A,5,7,771,121.71,600.1024
5.7.A,5,7,798,126.0,617.8575
5.7.A,5,7,833,131.25,641.0552
5.7.A,5,7,868,138.19,673.108
5.7.A,5,7,899,145.36,703.0693
5.7.A,5,7,926,149.62,721.6371
5.7.A,5,7,955,156.85,747.2234
5.7.A,5,7,987,162.77,768.8838
5.7.A,5,7,1016,168.12,791.7747
5.7.A,5,7,1043,173.36,818.3856
5.7.A,5,7,1079,176.55,833.5015
5.7.A,5,7,1107,181.1,853.5153
5.7.A,5,7,1142,185.6,873.2191
5.7.A,5,7,1171,190.18,894.7384
5.7.B,5,7,32,3.2,32.118
5.7.B,5,7,59,4.6,49.2207
5.7.B,5,7,178,12.7,95.7002
5.7.B,5,7,198,14.9,107.0487
5.7.B,5,7,227,18.2,133.3085
5.7.B,5,7,255,19.7,142.1557
5.7.B,5,7,281,23.5,165.1766
5.7.B,5,7,309,29.3,192.776
5.7.B,5,7,350,36.9,221.0392
5.7.B,5,7,382,41.9,241.0329
5.7.B,5,7,410,46.2,258.4855
5.7.B,5,7,449,50.1,278.7592
5.7.B,5,7,471,53.9,296.0058
5.7.B,5,7,501,60.2,321.198
5.7.B,5,7,529,63.92,337.3377
5.7.B,5,7,556,70.46,363.4242
5.7.B,5,7,596,79.38,395.8829
5.7.B,5,7,626,82.85,408.2322
5.7.B,5,7,652,86.86,422.1827
5.7.B,5,7,682,95.2,450.3631
5.7.C,5,7,32,2.6,25.1602
5.7.C,5,7,59,3.9,39.6636
5.7.C,5,7,85,3.9,39.6636
5.7.C,5,7,114,9.4,91.1276
5.7.C,5,7,149,9.4,91.1276
5.7.C,5,7,178,21.2,178.6564
5.7.C,5,7,198,26.8,211.5741
5.7.C,5,7,226,31.7,239.9853
5.7.C,5,7,255,34.6,255.0606
5.7.C,5,7,281,38.9,272.2552
5.7.C,5,7,309,44.0,292.5469
5.7.C,5,7,350,51.8,318.7467
5.7.C,5,7,350,58.23,338.1593
5.7.C,5,7,382,62.93,356.9534
5.7.C,5,7,410,67.33,377.627
5.7.C,5,7,449,72.73,404.6186
5.7.C,5,7,471,75.83,419.37
5.7.C,5,7,501,81.43,448.257
5.7.C,5,7,529,84.31,460.5795
5.7.C,5,7,556,90.94,485.9645
5.7.C,5,7,596,98.48,511.4418
5.7.C,5,7,626,102.84,526.2612
5.7.C,5,7,652,108.02,543.9713
5.7.C,5,7,682,114.19,564.3259
5.7.C,5,7,771,118.61,578.5539
5.7.C,5,7,798,121.8,589.3327
5.7.C,5,7,833,124.32,598.0996
5.7.C,5,7,868,129.46,616.7006
5.7.C,5,7,899,134.45,630.768
5.7.C,5,7,926,136.63,637.0445
5.7.C,5,7,955,141.05,648.1794
5.7.C,5,7,987,145.09,660.2149
5.7.C,5,7,1016,148.51,671.1555
5.7.C,5,7,1043,150.91,678.3052
5.7.C,5,7,1079,153.37,685.6337
5.7.C,5,7,1107,157.33,699.1727
5.7.C,5,7,1142,161.45,713.094
5.7.C,5,7,1171,165.76,730.5871
5.7.D,5,7,32,2.6,29.6308
5.7.D,5,7,58,3.8,45.5858
5.7.D,5,7,85,3.8,45.5858
5.7.D,5,7,114,10.3,113.6846
5.7.D,5,7,149,10.3,113.6846
5.7.D,5,7,178,21.3,194.6194
5.7.D,5,7,198,27.0,229.7204
5.7.D,5,7,226,30.7,254.5027
5.7.D,5,7,255,33.2,267.0988
5.7.D,5,7,281,37.5,294.0944
5.7.D,5,7,309,43.2,320.3062
5.7.D,5,7,350,48.6,340.6039
5.7.D,5,7,382,53.0,361.1895
5.7.D,5,7,410,57.3,376.6647
5.7.D,5,7,449,62.7,402.5766
5.7.D,5,7,471,65.4,415.2627
5.7.D,5,7,501,71.4,440.4548
5.7.D,5,7,529,73.71,450.6618
5.7.D,5,7,556,81.7,485.8069
5.7.D,5,7,596,89.81,521.3176
5.7.D,5,7,626,93.37,536.7632
5.7.D,5,7,652,94.88,543.2542
5.7.D,5,7,682,95.88,547.8328
5.8.A,5,8,35,4.0,34.7892
5.8.A,5,8,59,5.4,48.113
5.8.A,5,8,85,5.4,48.113
5.8.A,5,8,114,7.0,65.1957
5.8.A,5,8,149,7.0,65.1957
5.8.A,5,8,178,15.7,144.8629
5.8.A,5,8,198,21.6,186.3859
5.8.A,5,8,227,25.4,225.7417
5.8.A,5,8,255,26.9,241.5768
5.8.A,5,8,281,31.7,287.6424
5.8.A,5,8,309,37.9,332.2685
5.8.A,5,8,350,45.6,378.916
5.8.A,5,8,350,52.14,402.7142
5.8.A,5,8,382,57.44,437.8952
5.8.A,5,8,410,61.74,464.0311
5.8.A,5,8,449,67.94,508.0374
5.8.A,5,8,471,71.84,530.9622
5.8.A,5,8,501,78.14,562.2005
5.8.A,5,8,529,81.3,580.8387
5.8.A,5,8,556,89.17,622.6154
5.8.A,5,8,596,98.98,668.9042
5.8.A,5,8,626,103.44,690.1271
5.8.A,5,8,652,108.33,713.5918
5.8.A,5,8,682,114.48,739.7827
5.8.A,5,8,771,119.08,758.9127
5.8.A,5,8,798,122.59,775.8256
5.8.A,5,8,833,126.44,796.5322
5.8.A,5,8,868,132.76,829.8914
5.8.A,5,8,899,139.87,857.6118
5.8.A,5,8,926,144.21,875.6606
5.8.A,5,8,955,151.18,904.368
5.8.A,5,8,987,156.61,925.1042
5.8.A,5,8,1016,161.31,944.0862
5.8.A,5,8,1043,164.41,955.2428
5.8.A,5,8,1079,167.26,965.6135
5.8.A,5,8,1107,170.15,975.4365
5.8.A,5,8,1142,173.94,988.9247
5.8.A,5,8,1171,177.26,1005.254
5.8.B,5,8,35,3.4,28.4151
5.8.B,5,8,58,4.0,35.313
5.8.B,5,8,85,4.0,35.313
5.8.B,5,8,114,5.2,48.9647
5.8.B,5,8,149,5.2,48.9647
5.8.B,5,8,178,13.8,134.4221
5.8.B,5,8,198,19.1,172.1464
5.8.B,5,8,227,22.2,205.9259
5.8.B,5,8,255,23.8,222.4007
5.8.B,5,8,281,28.3,270.1758
5.8.B,5,8,309,34.1,310.7632
5.8.B,5,8,350,43.3,363.1869
5.8.B,5,8,382,48.4,391.3301
5.8.B,5,8,410,52.7,414.9727
5.8.B,5,8,449,58.3,454.1605
5.8.B,5,8,471,62.3,477.2733
5.8.B,5,8,501,68.6,512.9202
5.8.B,5,8,529,70.72,521.2704
5.8.B,5,8,556,78.33,552.3096
5.8.B,5,8,596,87.52,591.0793
5.8.B,5,8,626,92.31,611.3347
5.8.B,5,8,652,97.96,635.2832
5.8.B,5,8,682,104.47,660.6643
5.8.C,5,8,32,4.2,34.3453
5.8.C,5,8,58,6.1,57.5561
5.8.C,5,8,85,6.1,57.5561
5.8.C,5,8,114,6.33,59.722
5.8.C,5,8,149,6.33,59.722
5.8.C,5,8,178,15.13,147.8705
5.8.C,5,8,198,20.33,196.2155
5.8.C,5,8,226,23.63,231.1846
5.8.C,5,8,255,25.43,251.1584
5.8.C,5,8,281,29.73,299.8192
5.8.C,5,8,309,35.03,338.7091
5.8.C,5,8,350,41.33,382.7954
5.8.C,5,8,350,47.48,404.8055
5.8.C,5,8,382,51.88,441.1382
5.8.C,5,8,410,56.68,476.1673
5.8.C,5,8,449,63.28,523.6725
5.8.C,5,8,471,66.68,545.2898
5.8.C,5,8,501,73.78,585.3213
5.8.C,5,8,529,78.58,613.4405
5.8.C,5,8,556,87.03,659.3097
5.8.C,5,8,596,95.64,702.3463
5.8.C,5,8,626,100.77,725.5267
5.8.C,5,8,652,106.92,750.365
5.8.C,5,8,682,112.94,771.1877
5.8.C,5,8,771,117.75,791.2872
5.8.C,5,8,798,121.99,813.4979
5.8.C,5,8,833,125.82,831.9528
5.8.C,5,8,868,133.57,868.3664
5.8.C,5,8,899,140.89,898.9545
5.8.C,5,8,926,146.13,921.8986
5.8.C,5,8,955,151.51,942.7665
5.8.C,5,8,987,155.9,957.9511
5.8.C,5,8,1016,158.03,965.6167
5.8.C,5,8,1043,160.56,974.6713
5.8.C,5,8,1079,165.62,993.2863
5.8.C,5,8,1107,170.63,1012.2182
5.8.C,5,8,1142,176.12,1034.5007
5.8.C,5,8,1171,176.12,1034.5007
5.8.D,5,8,32,4.5,43.8164
5.8.D,5,8,58,6.5,69.2484
5.8.D,5,8,85,6.5,69.2484
5.8.D,5,8,114,6.75,71.5227
5.8.D,5,8,149,6.75,71.5227
5.8.D,5,8,178,15.35,153.7131
5.8.D,5,8,198,20.85,197.0396
5.8.D,5,8,226,24.75,237.0412
5.8.D,5,8,255,26.75,257.3149
5.8.D,5,8,281,31.55,305.2999
5.8.D,5,8,309,37.35,345.6553
5.8.D,5,8,350,46.25,397.4372
5.8.D,5,8,382,51.85,431.9225
5.8.D,5,8,410,57.15,460.0037
5.8.D,5,8,449,64.05,501.3908
5.8.D,5,8,471,67.75,522.9181
5.8.D,5,8,501,75.55,565.3369
5.8.D,5,8,529,80.43,593.0467
5.8.D,5,8,556,88.86,635.6049
5.8.D,5,8,596,98.43,677.8912
5.8.D,5,8,626,103.34,698.7031
5.8.D,5,8,652,109.15,722.2843
5.8.D,5,8,682,115.41,744.6881
6.0.A,6,0,32,5.1,2.6512
6.0.A,6,0,59,9.3,11.2165
6.0.A,6,0,85,13.2,17.6105
6.0.A,6,0,114,20.1,24.2325
6.0.A,6,0,149,24.6,28.2812
6.0.A,6,0,178,33.8,34.1674
6.0.A,6,0,203,41.0,35.175
6.0.A,6,0,227,45.0,35.255
6.0.A,6,0,254,49.3,36.5446
6.0.A,6,0,281,54.0,39.1758
6.0.A,6,0,309,59.5,44.4542
6.0.A,6,0,351,68.8,47.9871
6.0.A,6,0,381,73.0,49.6665
6.0.A,6,0,413,78.7,49.6665
6.0.A,6,0,449,85.9,55.7126
6.0.A,6,0,471,90.6,57.78
6.0.A,6,0,501,97.8,63.3943
6.0.A,6,0,528,101.96,65.3904
6.0.A,6,0,563,111.09,71.7794
6.0.A,6,0,598,118.76,74.9998
6.0.A,6,0,626,124.54,76.8489
6.0.A,6,0,653,129.84,78.8622
6.0.A,6,0,680,135.9,81.4066
6.0.B,6,0,28,5.4,2.5912
6.0.B,6,0,59,9.6,8.6373
6.0.B,6,0,85,12.7,11.6124
6.0.B,6,0,114,19.6,18.9241
6.0.B,6,0,149,24.2,23.2468
6.0.B,6,0,178,33.0,28.1732
6.0.B,6,0,203,39.5,31.1623
6.0.B,6,0,227,43.8,31.6781
6.0.B,6,0,254,47.6,32.8938
6.0.B,6,0,281,52.4,38.1721
6.0.B,6,0,309,58.2,45.014
6.0.B,6,0,351,67.9,57.6201
6.0.B,6,0,381,71.9,59.8594
6.0.B,6,0,413,77.6,59.8594
6.0.B,6,0,449,85.2,67.457
6.0.B,6,0,471,89.3,70.0802
6.0.B,6,0,501,95.6,74.9926
6.0.B,6,0,528,99.28,76.9792
6.0.B,6,0,563,107.86,80.5817
6.0.B,6,0,598,115.85,87.1315
6.0.B,6,0,626,121.57,92.964
6.0.B,6,0,653,127.07,98.3524
6.0.B,6,0,680,132.44,103.7207
6.0.B,6,0,716,137.04,109.3309
6.0.B,6,0,742,139.2,110.3674
6.0.B,6,0,771,144.37,113.4685
6.0.B,6,0,798,148.98,116.6023
6.0.B,6,0,833,154.66,119.8957
6.0.B,6,0,868,162.68,122.9423
6.0.B,6,0,899,170.3,125.5323
6.0.B,6,0,926,174.58,126.9015
6.0.B,6,0,955,181.45,130.8848
6.0.B,6,0,987,186.58,134.0644
6.0.B,6,0,1016,191.19,137.3826
6.0.B,6,0,1043,195.94,140.9915
6.0.B,6,0,1079,201.04,145.3761
6.0.B,6,0,1107,205.86,149.8091
6.0.B,6,0,1142,212.37,156.187
6.0.B,6,0,1171,219.15,160.2537
6.0.C,6,0,28,5.4,2.1593
6.0.C,6,0,59,9.3,5.1224
6.0.C,6,0,85,12.2,8.2534
6.0.C,6,0,114,18.4,12.4681
6.0.C,6,0,149,23.6,15.7951
6.0.C,6,0,178,31.3,18.5662
6.0.C,6,0,203,38.4,21.2634
6.0.C,6,0,227,43.2,21.4553
6.0.C,6,0,254,48.0,23.1828
6.0.C,6,0,281,52.9,25.73
6.0.C,6,0,309,58.2,27.7434
6.0.C,6,0,350,67.1,29.7007
6.0.C,6,0,381,72.0,31.1703
6.0.C,6,0,413,78.3,31.1703
6.0.C,6,0,449,84.1,32.5619
6.0.C,6,0,471,88.7,33.6655
6.0.C,6,0,501,95.2,34.7052
6.0.C,6,0,528,99.27,35.6003
6.0.C,6,0,563,108.77,37.6897
6.0.C,6,0,598,116.46,39.9959
6.0.C,6,0,626,121.6,40.0987
6.0.C,6,0,653,127.08,41.3039
6.0.C,6,0,680,132.4,42.474
6.0.D,6,0,28,5.4,2.8071
6.0.D,6,0,85,8.4,7.4857
6.0.D,6,0,114,15.1,14.3175
6.0.D,6,0,149,18.5,17.6485
6.0.D,6,0,178,28.5,24.4464
6.0.D,6,0,203,35.5,29.9047
6.0.D,6,0,227,40.0,30.8944
6.0.D,6,0,254,43.7,34.7412
6.0.D,6,0,281,49.2,44.0883
6.0.D,6,0,309,55.3,50.4303
6.0.D,6,0,350,64.7,55.1288
6.0.D,6,0,381,69.9,57.936
6.0.D,6,0,413,76.2,70.784
6.0.D,6,0,449,82.6,76.1583
6.0.D,6,0,471,86.3,78.0817
6.0.D,6,0,501,93.3,81.5806
6.0.D,6,0,528,96.77,82.9682
6.0.D,6,0,563,106.13,88.7696
6.0.D,6,0,598,114.59,92.6599
6.0.D,6,0,626,120.0,94.0661
6.0.D,6,0,653,125.43,95.8032
6.0.D,6,0,680,131.38,97.8255
6.0.D,6,0,716,137.44,99.0372
6.0.D,6,0,742,140.12,100.1624
6.0.D,6,0,771,144.84,101.2005
6.0.D,6,0,798,148.91,102.177
6.0.D,6,0,833,153.93,103.3814
6.0.D,6,0,868,161.49,105.7998
6.0.D,6,0,899,168.95,108.0371
6.0.D,6,0,926,172.5,108.676
6.0.D,6,0,955,178.27,109.945
6.0.D,6,0,987,183.65,111.9887
6.0.D,6,0,1016,188.43,113.9957
6.0.D,6,0,1043,193.14,115.5024
6.0.D,6,0,1079,197.9,116.7396
6.0.D,6,0,1107,202.45,117.5584
6.0.D,6,0,1142,208.03,118.6963
6.0.D,6,0,1171,213.92,119.9682
6.2.A,6,2,32,5.3,2.7551
6.2.A,6,2,57,9.3,6.0741
6.2.A,6,2,85,12.4,9.607
6.2.A,6,2,114,18.3,15.7411
6.2.A,6,2,149,21.7,18.9361
6.2.A,6,2,178,29.5,24.8623
6.2.A,6,2,203,36.1,28.4251
6.2.A,6,2,227,41.0,29.6008
6.2.A,6,2,254,45.8,31.6162
6.2.A,6,2,281,50.7,33.3796
6.2.A,6,2,309,56.1,38.0222
6.2.A,6,2,381,59.8,40.2415
6.2.A,6,2,413,65.4,40.2415
6.2.A,6,2,449,71.9,43.2305
6.2.A,6,2,471,76.1,45.3299
6.2.A,6,2,501,82.4,53.2654
6.2.A,6,2,528,86.22,56.1677
6.2.A,6,2,563,93.55,60.2712
6.2.A,6,2,598,99.88,63.8149
6.2.A,6,2,626,104.5,64.831
6.2.A,6,2,653,108.59,66.0576
6.2.A,6,2,680,113.58,67.1551
6.2.A,6,2,716,119.25,68.4021
6.2.A,6,2,742,122.25,70.9213
6.2.A,6,2,771,127.28,71.927
6.2.A,6,2,798,131.57,73.1278
6.2.A,6,2,833,136.46,74.5944
6.2.A,6,2,868,144.17,77.8316
6.2.A,6,2,899,152.19,80.3972
6.2.A,6,2,926,157.06,82.0524
6.2.A,6,2,955,164.1,84.4453
6.2.A,6,2,987,170.14,86.619
6.2.A,6,2,1016,174.43,87.9914
6.2.A,6,2,1043,179.39,89.8756
6.2.A,6,2,1079,184.14,91.3951
6.2.A,6,2,1107,188.64,92.2949
6.2.A,6,2,1142,194.51,93.715
6.2.A,6,2,1171,199.45,95.1372
6.2.B,6,2,32,5.5,3.189
6.2.B,6,2,57,9.5,6.8679
6.2.B,6,2,85,13.0,11.2765
6.2.B,6,2,114,19.4,18.5702
6.2.B,6,2,149,23.0,23.1048
6.2.B,6,2,178,31.4,28.815
6.2.B,6,2,203,38.3,32.8158
6.2.B,6,2,227,43.1,34.5432
6.2.B,6,2,254,47.9,36.6546
6.2.B,6,2,281,53.1,39.9815
6.2.B,6,2,309,58.4,42.7367
6.2.B,6,2,381,63.2,49.6465
6.2.B,6,2,413,69.5,61.6128
6.2.B,6,2,449,76.3,68.4107
6.2.B,6,2,471,80.5,71.2658
6.2.B,6,2,501,86.8,75.0446
6.2.B,6,2,528,89.17,77.4139
6.2.B,6,2,563,96.72,91.9054
6.2.B,6,2,598,102.85,97.5432
6.2.B,6,2,626,105.25,101.382
6.2.B,6,2,653,109.79,106.556
6.2.B,6,2,680,114.73,108.8277
6.2.B,6,2,716,120.27,110.4892
6.2.B,6,2,742,122.36,111.116
6.2.B,6,2,771,127.26,112.7815
6.2.B,6,2,798,130.45,113.9295
6.2.B,6,2,833,133.53,115.5306
6.2.B,6,2,868,140.29,120.3963
6.2.B,6,2,899,146.78,124.9379
6.2.B,6,2,926,149.83,127.2552
6.2.B,6,2,955,155.44,132.6391
6.2.B,6,2,987,158.09,135.1293
6.2.B,6,2,1016,162.89,140.5996
6.2.B,6,2,1043,167.58,145.1006
6.2.B,6,2,1079,172.44,149.8619
6.2.B,6,2,1107,177.09,153.0229
6.2.B,6,2,1142,182.58,158.2917
6.2.B,6,2,1171,188.14,164.4058
6.2.C,6,2,32,5.5,3.079
6.2.C,6,2,57,9.6,8.1615
6.2.C,6,2,85,13.3,15.2632
6.2.C,6,2,114,19.8,23.0608
6.2.C,6,2,149,23.3,27.3995
6.2.C,6,2,178,31.1,33.6375
6.2.C,6,2,203,38.3,38.2441
6.2.C,6,2,227,43.3,40.5434
6.2.C,6,2,254,46.6,42.1929
6.2.C,6,2,281,53.1,45.9617
6.2.C,6,2,309,58.9,49.6725
6.2.C,6,2,351,67.9,53.4514
6.2.C,6,2,381,72.0,55.9106
6.2.C,6,2,413,77.9,58.7417
6.2.C,6,2,449,84.4,65.2397
6.2.C,6,2,471,88.6,67.507
6.2.C,6,2,501,94.6,72.9053
6.2.C,6,2,528,97.51,76.338
6.2.C,6,2,563,104.3,87.3344
6.2.C,6,2,598,110.05,97.4512
6.2.C,6,2,626,113.55,103.3994
6.2.C,6,2,653,115.96,108.4588
6.2.C,6,2,680,117.28,112.2065
6.2.D,6,2,28,5.4,2.4832
6.2.D,6,2,57,9.5,6.7049
6.2.D,6,2,85,12.7,11.8233
6.2.D,6,2,114,19.2,18.7112
6.2.D,6,2,149,23.0,22.8139
6.2.D,6,2,178,32.0,28.0323
6.2.D,6,2,203,38.8,31.5672
6.2.D,6,2,227,42.8,32.047
6.2.D,6,2,254,46.6,33.0347
6.2.D,6,2,281,51.6,36.0338
6.2.D,6,2,309,57.9,38.3011
6.2.D,6,2,351,67.5,40.9882
6.2.D,6,2,381,71.8,43.0516
6.2.D,6,2,413,77.9,44.2712
6.2.D,6,2,449,84.8,48.4099
6.2.D,6,2,471,89.0,50.1734
6.2.D,6,2,501,96.1,52.4447
6.2.D,6,2,528,99.59,54.2589
6.2.D,6,2,563,106.71,58.2449
6.2.D,6,2,598,112.56,61.8707
6.2.D,6,2,626,116.65,63.4244
6.2.D,6,2,653,121.05,65.008
6.2.D,6,2,680,126.27,66.9909
6.7.A,6,7,32,5.7,3.305
6.7.A,6,7,59,10.1,10.3428
6.7.A,6,7,85,12.9,13.0299
6.7.A,6,7,114,18.0,15.987
6.7.A,6,7,149,18.7,16.5329
6.7.A,6,7,178,27.1,19.388
6.7.A,6,7,203,34.0,21.0434
6.7.A,6,7,227,37.9,21.4333
6.7.A,6,7,254,41.3,22.2491
6.7.A,6,7,281,46.2,24.0125
6.7.A,6,7,309,51.7,24.8922
6.7.A,6,7,351,61.2,26.4118
6.7.A,6,7,381,65.1,27.1915
6.7.A,6,7,413,70.6,27.7414
6.7.A,6,7,449,77.2,28.929
6.7.A,6,7,471,81.3,29.6668
6.7.A,6,7,501,88.1,30.6185
6.7.A,6,7,528,90.6,31.1683
6.7.A,6,7,563,98.05,33.4026
6.7.A,6,7,598,103.8,34.3223
6.7.A,6,7,626,108.02,34.9129
6.7.A,6,7,653,112.66,36.0262
6.7.A,6,7,680,118.2,37.0231
6.7.B,6,7,32,5.5,2.7491
6.7.B,6,7,59,9.7,9.551
6.7.B,6,7,85,12.8,12.2782
6.7.B,6,7,114,18.3,15.1373
6.7.B,6,7,149,20.0,16.4629
6.7.B,6,7,178,28.7,18.3763
6.7.B,6,7,203,36.0,20.8575
6.7.B,6,7,227,40.4,22.0011
6.7.B,6,7,254,44.8,23.4967
6.7.B,6,7,281,49.9,25.0262
6.7.B,6,7,309,55.3,26.5377
6.7.B,6,7,351,64.7,28.4171
6.7.B,6,7,381,69.0,29.4488
6.7.B,6,7,413,74.9,30.8644
6.7.B,6,7,449,81.9,32.5439
6.7.B,6,7,471,85.9,33.4236
6.7.B,6,7,501,92.3,34.4473
6.7.B,6,7,528,95.97,35.1077
6.7.B,6,7,563,103.53,37.2238
6.7.B,6,7,598,109.56,38.1883
6.7.B,6,7,626,114.0,38.8097
6.7.B,6,7,653,118.3,39.6694
6.7.B,6,7,680,123.28,40.6651
6.7.B,6,7,716,129.34,41.6344
6.7.B,6,7,742,132.18,41.8048
6.7.B,6,7,771,136.98,42.3806
6.7.B,6,7,798,141.02,42.7845
6.7.B,6,7,833,145.95,43.573
6.7.B,6,7,868,153.74,44.819
6.7.B,6,7,899,161.08,46.4333
6.7.B,6,7,926,165.64,47.2539
6.7.B,6,7,955,171.3,48.3855
6.7.B,6,7,987,177.16,50.2601
6.7.B,6,7,1016,182.64,51.5749
6.7.B,6,7,1043,188.0,52.8609
6.7.B,6,7,1079,193.33,53.7135
6.7.B,6,7,1107,198.52,54.4398
6.7.B,6,7,1142,204.91,55.5258
6.7.B,6,7,1171,211.07,56.548
6.7.C,6,7,32,5.1,2.2433
6.7.C,6,7,59,8.8,6.8299
6.7.C,6,7,85,11.4,9.1691
6.7.C,6,7,114,16.7,13.4078
6.7.C,6,7,149,20.1,15.7191
6.7.C,6,7,178,28.9,20.9975
6.7.C,6,7,203,36.1,25.748
6.7.C,6,7,227,40.3,26.6717
6.7.C,6,7,254,44.6,30.1106
6.7.C,6,7,281,47.8,34.3333
6.7.C,6,7,309,54.0,57.1422
6.7.C,6,7,351,62.9,105.0093
6.7.C,6,7,381,66.7,117.3175
6.7.C,6,7,413,72.4,128.8279
6.7.C,6,7,449,78.7,141.4239
6.7.C,6,7,471,82.5,143.1714
6.7.C,6,7,501,88.7,150.4851
6.7.C,6,7,528,91.78,151.5936
6.7.C,6,7,563,96.9,155.3812
6.7.C,6,7,598,100.22,158.501
6.7.C,6,7,626,103.7,165.4589
6.7.C,6,7,653,108.57,178.8959
6.7.C,6,7,680,113.87,200.1952
6.7.C,6,7,716,119.83,210.5624
6.7.C,6,7,742,121.6,211.9072
6.7.C,6,7,771,126.64,215.5349
6.7.C,6,7,798,130.96,218.0397
6.7.C,6,7,833,134.9,220.1666
6.7.C,6,7,868,143.39,225.4288
6.7.C,6,7,899,152.2,230.7131
6.7.C,6,7,926,156.61,233.2701
6.7.C,6,7,955,162.56,236.3632
6.7.C,6,7,987,167.42,238.5981
6.7.C,6,7,1016,172.06,240.7318
6.7.C,6,7,1043,176.88,242.852
6.7.C,6,7,1079,182.05,246.1597
6.7.C,6,7,1107,187.31,250.2612
6.7.C,6,7,1142,193.06,254.5149
6.7.C,6,7,1171,198.62,258.9615
6.7.D,6,7,32,4.1,1.8034
6.7.D,6,7,59,7.7,7.2737
6.7.D,6,7,85,10.7,10.8126
6.7.D,6,7,114,16.6,16.121
6.7.D,6,7,149,19.5,19.0781
6.7.D,6,7,178,29.3,24.1725
6.7.D,6,7,203,36.9,28.1232
6.7.D,6,7,227,41.4,29.6528
6.7.D,6,7,254,46.3,33.1797
6.7.D,6,7,281,50.5,36.7066
6.7.D,6,7,309,55.0,50.9221
6.7.D,6,7,351,62.2,67.0451
6.7.D,6,7,381,65.7,71.1039
6.7.D,6,7,413,70.6,81.3906
6.7.D,6,7,449,76.5,89.1762
6.7.D,6,7,471,79.9,90.4678
6.7.D,6,7,501,86.2,98.1514
6.7.D,6,7,528,90.52,104.7158
6.7.D,6,7,563,98.31,124.029
6.7.D,6,7,598,104.45,143.057
6.7.D,6,7,626,109.0,147.7876
6.7.D,6,7,653,112.42,150.6595
6.7.D,6,7,680,114.7,152.4829
6.8.A,6,8,32,4.2,4.8705
6.8.A,6,8,59,6.9,30.2426
6.8.A,6,8,85,7.8,37.7462
6.8.A,6,8,114,11.2,69.3564
6.8.A,6,8,149,11.2,69.3564
6.8.A,6,8,178,16.7,96.6279
6.8.A,6,8,203,22.2,128.0781
6.8.A,6,8,227,26.4,145.3767
6.8.A,6,8,254,30.8,164.1149
6.8.A,6,8,281,35.4,188.0274
6.8.A,6,8,309,40.8,208.3251
6.8.A,6,8,351,49.7,248.8965
6.8.A,6,8,381,53.9,271.5694
6.8.A,6,8,413,59.9,296.7616
6.8.A,6,8,449,66.1,319.9423
6.8.A,6,8,471,69.2,329.4874
6.8.A,6,8,501,76.2,349.7811
6.8.A,6,8,528,80.8,362.4731
6.8.A,6,8,563,89.72,386.1929
6.8.A,6,8,598,93.99,397.8891
6.8.A,6,8,626,97.81,407.6652
6.8.A,6,8,653,101.88,417.9998
6.8.A,6,8,680,107.14,430.5147
6.8.A,6,8,716,113.06,443.7714
6.8.A,6,8,742,116.0,449.532
6.8.A,6,8,771,121.59,459.9261
6.8.A,6,8,798,125.95,470.2125
6.8.A,6,8,833,129.99,481.2787
6.8.A,6,8,868,138.24,506.8457
6.8.A,6,8,899,145.82,523.8196
6.8.A,6,8,926,150.1,532.976
6.8.A,6,8,955,156.78,551.0064
6.8.A,6,8,987,161.95,565.3745
6.8.A,6,8,1016,166.71,578.7935
6.8.A,6,8,1043,171.49,590.7398
6.8.A,6,8,1079,176.74,603.5458
6.8.A,6,8,1107,181.66,614.3664
6.8.A,6,8,1142,187.29,629.6753
6.8.A,6,8,1171,192.69,646.1941
6.8.B,6,8,32,4.0,5.3583
6.8.B,6,8,59,6.1,25.1342
6.8.B,6,8,85,6.9,33.7715
6.8.B,6,8,114,11.3,88.4904
6.8.B,6,8,149,11.3,88.4904
6.8.B,6,8,178,17.5,126.9185
6.8.B,6,8,203,24.0,167.3359
6.8.B,6,8,227,27.6,193.0319
6.8.B,6,8,254,30.8,217.9201
6.8.B,6,8,281,35.5,248.9305
6.8.B,6,8,309,40.9,281.1044
6.8.B,6,8,351,50.1,326.3543
6.8.B,6,8,381,54.0,344.6007
6.8.B,6,8,413,59.0,366.7938
6.8.B,6,8,449,63.6,383.1647
6.8.B,6,8,471,66.4,391.5061
6.8.B,6,8,501,72.6,414.6868
6.8.B,6,8,528,75.47,422.8351
6.8.B,6,8,563,79.82,435.2722
6.8.B,6,8,598,85.28,451.8655
6.8.B,6,8,626,88.86,461.4569
6.8.B,6,8,653,92.56,472.4794
6.8.B,6,8,680,96.84,485.4866
6.8.C,6,8,32,4.5,8.7273
6.8.C,6,8,59,8.2,62.7305
6.8.C,6,8,85,8.5,66.0894
6.8.C,6,8,114,12.0,104.2275
6.8.C,6,8,149,12.0,104.2275
6.8.C,6,8,178,17.2,129.0758
6.8.C,6,8,203,22.8,158.9705
6.8.C,6,8,227,27.6,189.6809
6.8.C,6,8,254,33.0,213.4335
6.8.C,6,8,281,37.8,237.3301
6.8.C,6,8,309,43.5,266.619
6.8.C,6,8,351,53.2,321.116
6.8.C,6,8,381,57.5,341.3197
6.8.C,6,8,413,62.9,368.8511
6.8.C,6,8,449,66.8,384.3683
6.8.C,6,8,471,69.1,394.3012
6.8.C,6,8,501,75.9,426.3872
6.8.C,6,8,528,78.71,437.6799
6.8.C,6,8,563,83.55,461.0982
6.8.C,6,8,598,89.22,479.3499
6.8.C,6,8,626,94.2,492.6921
6.8.C,6,8,653,98.09,504.5141
6.8.C,6,8,680,103.29,518.2378
6.8.C,6,8,716,109.28,537.5196
6.8.C,6,8,742,112.04,545.2452
6.8.C,6,8,771,117.12,558.8553
6.8.C,6,8,798,120.58,568.2636
6.8.C,6,8,833,125.64,583.3377
6.8.C,6,8,868,133.24,606.5865
6.8.C,6,8,899,140.22,622.6355
6.8.C,6,8,926,143.83,629.8532
6.8.C,6,8,955,150.63,644.1288
6.8.C,6,8,987,156.38,657.0048
6.8.C,6,8,1016,160.91,668.5074
6.8.C,6,8,1043,165.25,678.1392
6.8.C,6,8,1079,168.64,685.3237
6.8.C,6,8,1107,173.0,694.1282
6.8.C,6,8,1142,177.86,706.8574
6.8.C,6,8,1171,182.93,722.2654
6.8.D,6,8,32,4.0,6.6379
6.8.D,6,8,59,7.8,65.9755
6.8.D,6,8,85,8.3,74.1729
6.8.D,6,8,114,12.4,107.3726
6.8.D,6,8,149,12.4,107.3726
6.8.D,6,8,178,17.5,135.7197
6.8.D,6,8,203,23.0,170.5789
6.8.D,6,8,227,27.7,202.341
6.8.D,6,8,254,31.1,225.4538
6.8.D,6,8,281,35.6,257.7537
6.8.D,6,8,309,41.8,307.3383
6.8.D,6,8,351,51.2,371.2384
6.8.D,6,8,381,55.5,392.7317
6.8.D,6,8,413,60.6,420.467
6.8.D,6,8,449,65.9,453.7407
6.8.D,6,8,471,69.0,472.2729
6.8.D,6,8,501,74.8,504.2789
6.8.D,6,8,528,77.11,512.6385
6.8.D,6,8,563,80.95,525.9976
6.8.D,6,8,598,86.29,543.7208
6.8.D,6,8,626,90.98,558.3491
6.8.D,6,8,653,94.96,570.8424
6.8.D,6,8,680,100.07,586.4741
'''

_vb = pd.read_csv(io.StringIO(VOLUME_CSV))
plot_volume_batch(_vb, ['1', '5'])   # LUFA 6S : 2024 (B) vs 2023 (A)
plot_volume_batch(_vb, ['0', '6'])   # LUFA 2.2: 2024 (B) vs 2023 (A)
